In [ ]:
%load_ext autoreload
%autoreload 2

# IMPORT

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import scipy
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import os
import string

from pycns import CnsStream, CnsReader
import physio
from configuration import *

from tools import *
from multi_projects_jobs import *
from icp_slow_rises_jobs import *


In [ ]:
def get_plot_letters(kind = 'upper'):
    if kind == 'upper':
        alphabet = list(string.ascii_uppercase)
    else:
        alphabet = list(string.ascii_lowercase)
    return [f'{letter})' for letter in alphabet]

# Results

In [ ]:
compliance_all = concat_slow_icp_features('compliance')

In [ ]:
compliance_all['patient'].unique().size

In [ ]:
(compliance_all['patient'].value_counts() / 12).quantile([0.25,0.5,0.75])

In [ ]:
mapper_clean_name = {
    "median_icp_mmHg" : "ICP (mmHg)",
    "heart_in_icp_spectrum" : "Heart in ICP\nSpectrum\n(mmHg)",
    "icp_pulse_amplitude_mmHg" : "ICP\nPulse Amplitude\n(mmHg)",
    "resp_in_icp_spectrum" : "Resp in ICP\nSpectrum\n(mmHg)",
    "icp_resp_modulated" : "Resp Modulation\nof ICP\n(mmHg)",
    "ratio_heart_resp_in_icp_spectrum" : "Heart / Resp\nSpectral Ratio",
    "icp_pulse_resp_modulated" : "Resp Modulation of\nICP Pulse Amplitude\n(mmHg)",
    "median_abp_mmHg" : "ABP (mmHg)",
    "median_cpp_mmHg" : "CPP (mmHg)",
    "abp_pulse_amplitude_mmHg" : "ABP\nPulse Amplitude\n(mmHg)",
    "abp_pulse_resp_modulated" : "Resp Modulation of\nABP Pulse Amplitude\n(mmHg)",
    "abp_resp_modulated" : "Resp Modulation\nof ABP\n(mmHg)",
    "RAQ_2" : "RAQ",
    "RAQ_ABP" : "RAQ_ABP",
    "P2P1_ratio" : "P2P1 Ratio",
    "PSI" : "PSI",
    "PRx" : "PRx",
    "heart_rate_bpm" : "Heart Rate (bpm)",
    "resp_rate_cpm" : "Resp Rate (cpm)"
}


In [ ]:
def get_relative_variation_to_baseline_res(compliance_all, metrics):
    rows = []
    for p in compliance_all['patient'].unique():
        for e in compliance_all['n_event'].unique():
            df_p_e = compliance_all[(compliance_all['patient'] == p) & (compliance_all['n_event'] == e)].reset_index(drop = True)
            if df_p_e.shape[0] > 0:
                baseline_values = df_p_e.set_index('win_label').loc['baseline_before',metrics].values
                for win in compliance_all['win_label'].unique():
                    win_values = df_p_e.set_index('win_label').loc[win,metrics].values
                    norm_values = ((win_values - baseline_values) / baseline_values) * 100
                    row = {'patient':p, 'n_event':e, 'win_label':win} | {k:v for k,v in zip(metrics,norm_values)}
                    rows.append(row)
    res_norm = pd.DataFrame(rows)
    return res_norm

In [ ]:
metrics = ['median_icp_mmHg','median_abp_mmHg','median_cpp_mmHg',
           'heart_in_icp_spectrum','resp_in_icp_spectrum','ratio_heart_resp_in_icp_spectrum',
           'icp_pulse_amplitude_mmHg','icp_resp_modulated','icp_pulse_resp_modulated',
           'abp_pulse_amplitude_mmHg','abp_resp_modulated','abp_pulse_resp_modulated',
           'P2P1_ratio','PSI','PRx',
           'RAQ_2','RAQ_ABP','heart_rate_bpm',
           'resp_rate_cpm'
           ]
res_norm = get_relative_variation_to_baseline_res(compliance_all, metrics)

In [ ]:
# lmm_res = pd.read_excel(base_folder / 'figures' / 'slow_icp_rises_figs' / 'res_matrix' / 'results_compliance_lmm.xlsx')

In [ ]:
# lmm_win_mapper = {v:k for k,v in zip(lmm_res.columns[1:],compliance_all['win_label'].unique())} 
# lmm_metric_mapper = {v:k for k,v in zip(lmm_res.iloc[:,0].unique(),metrics)} 

# def get_pval_lmm(metric_name, win_label):
#     return float(lmm_res.set_index('Metric / Window Label').loc[lmm_metric_mapper[metric_name],lmm_win_mapper[win_label]].split(',')[2][3:])

# def get_signifiance_lmm_dict(metric_name):
#     signif = {win_label:lmm_res.set_index('Metric / Window Label').loc[lmm_metric_mapper[metric_name],lmm_win_mapper[win_label]].split(',')[3][1:] for win_label in compliance_all['win_label'].unique() if not win_label == 'baseline_before'}
#     return signif

# def get_inds_signifs(metric_name):
#     bool_signif = (pd.Series(get_signifiance_lmm_dict(metric_name)) != 'ns').values
#     bool_signif = np.insert(bool_signif, 0, False)
#     inds_signif = np.ones(bool_signif.shape)
#     inds_signif[~bool_signif] = np.nan
#     return inds_signif

In [ ]:
dpi = 500
img_extension = '.png'

x_letter = - 0.07
y_letter = 1.1

color_ax2 = 'b'

p_titles = dict(
    fontsize = 25,
    weight = 'bold',
    loc = 'center',
)

p_letter = dict(
    fontsize = 25,
    weight = 'bold',
    ha = 'right',
    va = 'top',
)

p_labels = dict(
    fontsize = 16,
    weight = 'bold',
)

p_ticks = dict(
    fontsize = 15,
    fontweight = 'bold',
)

p_pval_line = dict(
    label = 'p<0.05',
    color = 'darkorange',
    lw = 4,
)


# save_folder = base_folder / 'figures' / 'slow_icp_rises_figs' / 'manuscript_compliance' / 'fig2'
save = True

In [ ]:
estimator = 'median'
# estimator = 'mean'

# metrics = ['median_icp_mmHg','heart_in_icp_spectrum','icp_pulse_amplitude_mmHg',
#            'resp_in_icp_spectrum','icp_resp_modulated','ratio_heart_resp_in_icp_spectrum', 
#            'icp_pulse_resp_modulated','median_abp_mmHg','median_cpp_mmHg',
#            'abp_pulse_amplitude_mmHg','abp_pulse_resp_modulated','abp_resp_modulated',
#            'RAQ_2','RAQ_ABP','P2P1_ratio', 
#            'PSI','PRx','heart_rate_bpm',
#            'resp_rate_cpm'
#            ]

metrics = ['median_icp_mmHg','median_abp_mmHg','median_cpp_mmHg',
           'heart_in_icp_spectrum','resp_in_icp_spectrum','ratio_heart_resp_in_icp_spectrum',
           'icp_pulse_amplitude_mmHg','icp_resp_modulated','icp_pulse_resp_modulated',
           'abp_pulse_amplitude_mmHg','abp_resp_modulated','abp_pulse_resp_modulated',
           'P2P1_ratio','PSI','PRx',
           'RAQ_2','RAQ_ABP','heart_rate_bpm',
           'resp_rate_cpm'
           ]

letters = get_plot_letters()[:len(metrics)]
d_letters = {m:l for m,l in zip(metrics, letters)}

nrows = 7
ncols = 3
poss = attribute_subplots(metrics, nrows, ncols)

fig, axs = plt.subplots(nrows, ncols, figsize = (ncols * 5,nrows * 3.), constrained_layout = True, sharex = True)
# fig.suptitle(f'Estimator : {estimator}')

for m, (r,c) in poss.items():
        ax = axs[r,c]  

        if m != 'PRx':
                ax2 = ax.twinx()
                sns.pointplot(data = res_norm,
                                x = 'win_label',
                                y = m,
                                ax=ax2,
                                errorbar=('ci',95),
                                estimator=estimator,
                                color = 'b',
                                linestyles="--",
                                )
                ax2.set_ylim(-20, 70)
                ax2.set_xticks(ax2.get_xticks(), ax2.get_xticklabels(), rotation = 90, **p_ticks)
                ax2.set_yticks(ax2.get_yticks()[1:-1], ax2.get_yticklabels()[1:-1], rotation = 0, **p_ticks | {'color':color_ax2})
                ax2.set_xlabel('')
                ax2.set_xticks(ax2.get_xticks(), [n.replace('_',' ') for n in compliance_all['win_label'].unique()], **p_ticks)
                ax2.set_ylabel('% vs baseline', **p_labels | {'color':color_ax2})


        sns.pointplot(data = compliance_all,
                        x = 'win_label',
                        y = m,
                        ax=ax,
                        errorbar=('ci',95),
                        estimator=estimator,
                        color = 'k',
                        )
        
        ax.set_yticks(ax.get_yticks()[1:-1], ax.get_yticklabels()[1:-1], rotation = 0, **p_ticks)
        ax.set_xlabel('')
        ax.set_xticks(ax.get_xticks(), [n.replace('_',' ') for n in compliance_all['win_label'].unique()], rotation = 90, **p_ticks)
        ax.set_ylabel(mapper_clean_name[m], **p_labels)
        letter = d_letters[m]
        ax.axvspan(1, 5.5, color = 'r', alpha = 0.2)
        ax.axvspan(5.5, 10, color = 'g', alpha = 0.2)

        # ylim = ax.get_ylim()
        # y_signif_line = min(ylim) + 0.05 * (max(ylim) - min(ylim))
        # ax.plot(np.arange(compliance_all['win_label'].unique().size), get_inds_signifs(m) * y_signif_line, **p_pval_line)

        # ax.text(x_letter, y_letter, letter, transform=ax.transAxes, **p_letter)
        # ax.set_title('Frequency domain : Heart and Respiration in ICP', **p_titles)
        # ax.legend(loc = 'upper right', fontsize = 11, framealpha = 0.5)
fig.delaxes(axs[nrows-1, ncols-2])
fig.delaxes(axs[nrows-1, ncols-1])
# fig.savefig(save_folder / f'fig2_{estimator}{img_extension}', dpi = dpi, bbox_inches = 'tight')

if save:
        hash_ = joblib.hash(slow_icp_rise_detection_params)
        save_path = Path.cwd() / 'figs_sensitivy_analysis' / hash_
        if not os.path.exists(save_path):
                os.makedirs(save_path)
                with open(save_path / '__params__.json', mode='w') as f:
                        json.dump(slow_icp_rise_detection_params, f, indent=4)
        fig.savefig(save_path / 'fig2.png', dpi = dpi, bbox_inches = 'tight')

plt.show()

# P2P1 distrib

In [ ]:
concat_subs = new_names_subs_compliance
concat = [ratio_P1P2_job.get(s)['ratio_P1P2'].to_series().quantile([0.5,0.6,0.7,0.75]).values for s in concat_subs]
concat = np.concatenate(concat)



In [ ]:
fig, ax = plt.subplots()
ax.hist(concat, bins = 30, density = True, color = 'k')
ax.set_xlabel('P2/P1 ratio')
ax.set_ylabel('Density')
plt.show()

# sensitivity analysis slow ICP rises detection

In [ ]:
folder = Path("/mnt/data/NEURO_REA_MONITORAGE/precompute/slow_icp_detection_compliance_features/07f1d5c0e7d4b41150d168bb8f269500")
subs = [str(f).split('/')[-1].split('.')[0] for f in folder.iterdir()]
len(subs)

In [ ]:
meta = get_metadata().set_index('ID_pseudo')
meta.columns

In [ ]:
[meta.at[s,'ID_pseudo_old'] for s in subs]

In [ ]:
np.array([meta.at[s,'ID_pseudo_old'] for s in subs])

In [ ]:
np.array(subs)

In [ ]:
def detect_slow_icp_rises_sensitivity(raw_icp,
                            icp_filtered, 
                            icp_smoothed_for_trough,
                            times,
                          datetimes,
                          datetimes_raw,
                          datetimes_smoothed,
                          params_detection = {'min_rise_duration_min':15,'min_rise_amplitude_mmHg':5, 'min_peak_amplitude_mmHg':20,'max_peak_amplitude_raw_mmHg':80,'min_decay_amplitude_mmHg':2, 'max_trough_amplitude_smoothed_mmHg':20},
                          ):
    """
    Inputs:
    - raw_icp : np.array = the raw ICP signal
    - icp_filtered : np.array = the ICP filtered signal
    - icp_smoothed_for_trough : np.array = an ICP filtered signal that fit better to raw trace to filter for trough amplitude
    - srate : float or int = sampling rate of the filtered ICP signal
    - datetimes : np.datetime64 = datetime vector of the filtered ICP signal
    - datetimes_raw = : np.datetime64 = datetime vector of the raw ICP signal
    - datetimes_smoothed = : np.datetime64 = datetime vector of the smoothed ICP signal
    - params_detection : dict = dictionnary of parameters aiming to post-process the detection by applying a mask
    - verbose : bool = print the highcut frequency of the filter

    Outputs :
    - detection : pd.DataFrame = detections with events in rows and features in columns
    """
    firt_derivarive_sig = np.gradient(icp_filtered)
    detection = detect_cross(firt_derivarive_sig, thresh = 0)
    detection.columns = ['trough_ind','peak_ind']
    detection['next_trough_ind'] = np.append(detection.loc[1:,'trough_ind'].values, np.nan)
    detection = detection.dropna()
    detection = detection.astype(int)
    detection['trough_time_s'] = times[detection['trough_ind']]
    detection['peak_time_s'] = times[detection['peak_ind']]
    detection['next_trough_time_s'] = times[detection['next_trough_ind']]
    detection['trough_date'] = datetimes[detection['trough_ind']]
    detection['peak_date'] = datetimes[detection['peak_ind']]
    detection['next_trough_date'] = datetimes[detection['next_trough_ind']]
    detection['rise_duration_s'] = detection['peak_time_s'] - detection['trough_time_s']
    detection['rise_duration_min'] = detection['rise_duration_s'] / 60
    detection['decay_duration_s'] = detection['next_trough_time_s'] - detection['peak_time_s']
    detection['decay_duration_min'] = detection['decay_duration_s'] / 60
    detection['trough_amplitude_mmHg'] = icp_filtered[detection['trough_ind']]
    detection['peak_amplitude_mmHg'] = icp_filtered[detection['peak_ind']]
    detection['next_trough_amplitude_mmHg'] = icp_filtered[detection['next_trough_ind']]
    detection['rise_amplitude_mmHg'] = detection['peak_amplitude_mmHg'] - detection['trough_amplitude_mmHg']
    raw_peak_inds = np.searchsorted(datetimes_raw, detection['peak_date'].values)
    detection['peak_amplitude_raw_mmHg'] = raw_icp[raw_peak_inds]
    smoothed_trough_inds = np.searchsorted(datetimes_smoothed, detection['trough_date'].values)
    detection['trough_amplitude_smoothed_mmHg'] = icp_smoothed_for_trough[smoothed_trough_inds]
    detection['decay_amplitude_mmHg'] = detection['peak_amplitude_mmHg'] - detection['next_trough_amplitude_mmHg']
    params_detection = {k:v for k,v in params_detection.items() if not v is None}
    if len(params_detection) != 0:
        masking = pd.DataFrame(index = detection.index, columns = params_detection.keys(), dtype = bool)
        masking[:] = True
        for param_name, threshold in params_detection.items():
            metric_name = param_name[4:]
            if 'min' in param_name:
                masking.loc[:,param_name] = detection[metric_name] > threshold
            elif 'max' in param_name:
                masking.loc[:,param_name] = detection[metric_name] < threshold
        mask = masking.all(axis = 1)
        detection = detection[mask].reset_index(drop = True)
    return detection

def slow_icp_rise_detection_sensivity(sub, **p):
    cns_reader = CnsReader(data_path / sub)
    stream = cns_reader.streams['ICP']
    raw_icp, datetimes_raw = stream.get_data(apply_gain = True, with_times = True)
    icp_filtered = icp_filter_for_detection_job.get(sub)['icp_filter_for_detection']
    icp_smoothed_for_trough = icp_filter_for_trough_filtering_job.get(sub)['icp_filter_for_trough_filtering']
    datetimes = icp_filtered['datetime'].values
    times = icp_filtered.attrs['time']
    datetimes_smoothed = icp_smoothed_for_trough['datetime'].values
    icp_filtered = icp_filtered.values
    params_detection = p
    detections = detect_slow_icp_rises_sensitivity(raw_icp, icp_filtered, icp_smoothed_for_trough, times, datetimes, datetimes_raw, datetimes_smoothed, params_detection=params_detection)
    # for col in detections.columns:
    #     if 'date' in col:
    #         detections[col] = detections[col].astype('datetime64[ns]')
    # if detections.shape[0] == 0:
    #     print(f'Warning : No event detected in sub {sub}')
    return detections.shape[0]

In [ ]:
sub = subs[0]

initial_params = {'min_rise_duration_min':15,
                'min_rise_amplitude_mmHg':5, 
                'min_peak_amplitude_mmHg':20,
                'max_peak_amplitude_raw_mmHg':80,
                'min_decay_amplitude_mmHg':2,
                'max_trough_amplitude_smoothed_mmHg':20
}


# test = slow_icp_rise_detection(sub, **slow_icp_rise_detection_params)

In [ ]:
# min_rise_duration_mins = [5,10,15,20,25]
# min_rise_amplitude_mmHgs = [3,5,7,10]
# min_peak_amplitude_mmHgs = [10,15,20,25,30]


compute_subs = subs[:10]
min_rise_duration_mins = [10,20,30,60,90]
min_rise_amplitude_mmHgs = [3,5,10,20]
min_peak_amplitude_mmHgs = [15,20,25,30]

total_iterations = len(compute_subs) * len(min_rise_duration_mins) * len(min_rise_amplitude_mmHgs) * len(min_peak_amplitude_mmHgs)

counter = 0

rows = []

for s in compute_subs:
    print(s)
    for min_rise_duration_min in min_rise_duration_mins:
        for min_rise_amplitude_mmHg in min_rise_amplitude_mmHgs:
            for min_peak_amplitude_mmHg in min_peak_amplitude_mmHgs:
                new_params = initial_params.copy()
                new_params['min_rise_duration_min'] = min_rise_duration_min
                new_params['min_rise_amplitude_mmHg'] = min_rise_amplitude_mmHg
                new_params['min_peak_amplitude_mmHg'] = min_peak_amplitude_mmHg
                N = slow_icp_rise_detection_sensivity(s, **new_params)
                row = dict(sub=s, min_rise_duration_min=min_rise_duration_min, min_rise_amplitude_mmHg=min_rise_amplitude_mmHg, min_peak_amplitude_mmHg=min_peak_amplitude_mmHg, N=N)
                rows.append(row)

                counter += 1
                if counter % 10 == 0:
                    print(counter, '/', total_iterations)

sensitivity = pd.DataFrame(rows)

In [ ]:
sensitivity

In [ ]:
# sensitivity.to_excel('sensitivity_slow_icp_rises.xlsx')
sensitivity = pd.read_excel('sensitivity_slow_icp_rises.xlsx')
sensitivity

In [ ]:
coords = dict(sub = sensitivity['sub'].unique(),
              min_rise_duration_min = sensitivity['min_rise_duration_min'].unique(),
              min_rise_amplitude_mmHg = sensitivity['min_rise_amplitude_mmHg'].unique(),
              min_peak_amplitude_mmHg = sensitivity['min_peak_amplitude_mmHg'].unique(),
             )
da = xr.DataArray(dims = coords.keys(), coords=coords)
for s in sensitivity['sub'].unique():
    for rise_dur in sensitivity['min_rise_duration_min'].unique():
        for rise_amp in sensitivity['min_rise_amplitude_mmHg'].unique():
            for peak_amp in sensitivity['min_peak_amplitude_mmHg'].unique():
                da.loc[s,rise_dur,rise_amp,peak_amp] = sensitivity.set_index(['sub','min_rise_duration_min','min_rise_amplitude_mmHg','min_peak_amplitude_mmHg']).loc[(s,rise_dur,rise_amp,peak_amp),'N']

In [ ]:
n_fig = 0

for s in da['sub'].values:
    print(s)
    n_fig += 1
    ncols = len(sensitivity['min_rise_duration_min'].unique())
    fig, axs = plt.subplots(ncols = ncols, figsize = (ncols * 4,4))
    fig.suptitle(f'S{n_fig}')
    for rise_dur, ax in zip(sensitivity['min_rise_duration_min'].unique(),axs):
        ax.set_title(f'min_rise_duration_minutes : {rise_dur}')
        da_plot = da.loc[s, rise_dur]
        ax.pcolormesh(da_plot['min_rise_amplitude_mmHg'].astype(str), da_plot['min_peak_amplitude_mmHg'].astype(str), da_plot)
        ax.set_xlabel('min_rise_amplitude_mmHg')
        ax.set_ylabel('min_peak_amplitude_mmHg')
        ax.set_xticks(range(da_plot['min_rise_amplitude_mmHg'].size), da_plot['min_rise_amplitude_mmHg'].values)
        # ax.set_xticks(da_plot['min_rise_amplitude_mmHg'].values)
        ax.set_yticks(range(da_plot['min_peak_amplitude_mmHg'].size), da_plot['min_peak_amplitude_mmHg'].values)

        data = da_plot.values
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                val = data[i, j]
                color = 'white' if val < data.mean() else 'black'
                
                ax.text(j, i, f"{val:.0f}",
                        ha='center', va='center', color=color)
    plt.show()

In [ ]:
n_fig = 0

# for s in da['sub'].values:
print(s)
n_fig += 1
ncols = len(sensitivity['min_rise_duration_min'].unique())
fig, axs = plt.subplots(ncols = ncols, figsize = (ncols * 4,4))
# fig.suptitle(f'S{n_fig}')
for rise_dur, ax in zip(sensitivity['min_rise_duration_min'].unique(),axs):
    ax.set_title(f'min_rise_duration_minutes : {rise_dur}')
    da_plot = da.sum('sub').loc[rise_dur]
    ax.pcolormesh(da_plot['min_rise_amplitude_mmHg'].astype(str), da_plot['min_peak_amplitude_mmHg'].astype(str), da_plot)
    ax.set_xlabel('min_rise_amplitude_mmHg')
    ax.set_ylabel('min_peak_amplitude_mmHg')
    ax.set_xticks(range(da_plot['min_rise_amplitude_mmHg'].size), da_plot['min_rise_amplitude_mmHg'].values)
    # ax.set_xticks(da_plot['min_rise_amplitude_mmHg'].values)
    ax.set_yticks(range(da_plot['min_peak_amplitude_mmHg'].size), da_plot['min_peak_amplitude_mmHg'].values)

    data = da_plot.values
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data[i, j]
            color = 'white' if val < data.mean() else 'black'
            
            ax.text(j, i, f"{val:.0f}",
                    ha='center', va='center', color=color)
plt.show()

# update new patients

In [ ]:
get_patient_list(['ECG_II'], verbose = True)

In [ ]:
ecg_peaks = detect_ecg_job.get('P135').to_dataframe()
ecg_peaks

# fourier fig

In [ ]:
save_folder = base_folder / 'documents_valentin' / 'pour_fourier_fig' 

In [ ]:
srate = 100
time = np.arange(0,5, 1 / srate)


dict_sigs = dict(
    sig1 = {'f':1, 'a':10},
    sig2 = {'f':3, 'a':5},
    # sig3 = {'f':7, 'a':2}, 
)
sigs = np.zeros((len(dict_sigs), time.size))
for i, sig_name in enumerate(dict_sigs):
    dict_sig = dict_sigs[sig_name]
    sigs[i,:] = dict_sig['a'] * np.sin(2 * np.pi * time * dict_sig['f'])
sum_sig = np.sum(sigs, axis = 0)

figsize = 4
figsize = (figsize,figsize)

fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=figsize)
ax.plot(time, [0] * time.size, sum_sig, color = 'k')
ax.set_xlabel('Time (s)')
ax.set_zlabel('Amplitude (AU)')
ax.set_yticks([0])
# ax.grid(False)
# ax.xaxis.pane.set_visible(False)
# ax.yaxis.pane.set_visible(False)
# ax.zaxis.pane.set_visible(False)
# ax.xaxis.line.set_visible(False)
# ax.yaxis.line.set_visible(False)
# ax.zaxis.line.set_visible(False)

fig.savefig(save_folder / 'signal.svg', dpi = 300, bbox_inches = 'tight')

fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=figsize)
for i in range(sigs.shape[0]):
    ax.plot(time, [dict_sigs[f'sig{i+1}']['f']] * time.size, sigs[i,:])

ax.set_xlabel('Time (s)')
ax.set_zlabel('Amplitude (AU)')
ax.set_ylabel('Frequency (Hz)')

ax.zaxis.pane.set_visible(False)



fig.savefig(save_folder / 'signal_decomposed.svg', dpi = 300, bbox_inches = 'tight')

fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=figsize)
for i in range(sigs.shape[0]):
    f, Pxx = scipy.signal.periodogram(sigs[i,:], fs=srate, scaling = 'spectrum', detrend = False)
    mask_f = f < 4
    f = f[mask_f]
    Pxx = Pxx[mask_f]
    ax.plot([0] * f.size, f, np.sqrt(Pxx * 2))

# ax.set_xlabel('Time (s)')
ax.set_zlabel('Amplitude (AU)')
ax.set_ylabel('Frequency (Hz)')

ax.set_xticks([0])


ax.set_ylim(0,4)

fig.savefig(save_folder / 'fourier.svg', dpi = 300, bbox_inches = 'tight')

ax.view_init(elev=30, azim=-60)

In [ ]:
srate = 100
time = np.arange(0,5, 1 / srate)


dict_sigs = dict(
    sig1 = {'f':1, 'a':10},
    sig2 = {'f':3, 'a':5},
    # sig3 = {'f':7, 'a':2}, 
)
sigs = np.zeros((len(dict_sigs), time.size))
for i, sig_name in enumerate(dict_sigs):
    dict_sig = dict_sigs[sig_name]
    sigs[i,:] = dict_sig['a'] * np.sin(2 * np.pi * time * dict_sig['f'])
sum_sig = np.sum(sigs, axis = 0)


fig, ax = plt.subplots(figsize=(6,3))
ax.plot(time, sum_sig, color = 'k', lw = 2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (AU)')

# ax.zaxis.line.set_visible(False)

fig.savefig(save_folder / 'signal.png', dpi = 300, bbox_inches = 'tight')

fig, ax = plt.subplots(subplot_kw={"projection": "3d"}, figsize=(7,7))
for i in range(sigs.shape[0]):
    ax.plot(time, [dict_sigs[f'sig{i+1}']['f']] * time.size, sigs[i,:], lw = 2)
ax.set_yticks([1,3])
ax.set_ylim(0,4)
ax.set_xlabel('Time (s)')
ax.set_zlabel('Amplitude (AU)')
ax.set_ylabel('Frequency (Hz)')

fig.savefig(save_folder / 'signal_decomposed.png', dpi = 300, bbox_inches = 'tight')

fig, ax = plt.subplots(figsize=(6,3))
for i in range(sigs.shape[0]):
    f, Pxx = scipy.signal.periodogram(sigs[i,:], fs=srate, scaling = 'spectrum', detrend = False)
    mask_f = f < 4
    f = f[mask_f]
    Pxx = Pxx[mask_f]
    ax.plot(f, np.sqrt(Pxx * 2), lw = 2)

ax.set_ylabel('Amplitude (AU)')
ax.set_xlabel('Frequency (Hz)')


# ax.set_ylim(0,4)

fig.savefig(save_folder / 'fourier.png', dpi = 300, bbox_inches = 'tight')


In [ ]:
srate = 50
time = np.arange(0, 10, 1 / srate)


dict_sigs = dict(
    sig1 = {'f':1, 'a':10},
    sig2 = {'f':2, 'a':6},
    sig3 = {'f':4, 'a':4}, 
)
sigs = np.zeros((len(dict_sigs), time.size))
for i, sig_name in enumerate(dict_sigs):
    dict_sig = dict_sigs[sig_name]
    sigs[i,:] = dict_sig['a'] * np.sin(2 * np.pi * time * dict_sig['f'] + np.random.rand(1) * 2 * np.pi)
sum_sig = np.sum(sigs, axis = 0)


fig, axs = plt.subplots(nrows = 3, figsize=(6,8), constrained_layout = True)

ax = axs[0]
ax.plot(time, sum_sig, color = 'k', lw = 2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (AU)')
# ax.set_yticks(np.arange(-10,11,2))
ax.set_xlim(0,2)
ax.set_title('What you have', fontsize = 14)
ax.grid()

ax = axs[1]
for i in range(sigs.shape[0]):
    f, Pxx = scipy.signal.periodogram(sigs[i,:], fs=srate, scaling = 'spectrum', detrend = False)
    ax.plot(f, np.sqrt(Pxx * 2), lw = 2)
ax.set_xlim(0,5)
ax.set_xticks(np.arange(0,5,1))

ax.set_ylabel('Amplitude (AU)')
ax.set_xlabel('Frequency (Hz)')
ax.set_title('What Fourier gives', fontsize = 14)
ax.grid()

ax = axs[2]
for i in range(sigs.shape[0]):
    ax.plot(time, sigs[i,:], lw = 2)
ax.set_ylabel('Amplitude (AU)')
ax.set_xlabel('Time (s)')
ax.set_yticks(np.arange(-10,11,2))
ax.set_xlim(0,2)
ax.set_title('What Fourier means', fontsize = 14)
ax.grid()


# ax.set_ylim(0,4)

fig.savefig(save_folder / 'all_fourier.png', dpi = 300, bbox_inches = 'tight')


In [ ]:
srate = 60
time = np.arange(0, 10, 1 / srate)

freqs = np.arange(0.1,30,0.1)
amps = 1 / freqs

sigs = np.zeros((freqs.size, time.size))

for i, fi in enumerate(freqs):
    sigs[i,:] = amps[i] * np.sin(2 * np.pi * time * fi + np.random.rand(1) * 2 * np.pi)
sum_sig = np.sum(sigs, axis = 0)
fig, axs = plt.subplots(nrows = 3, figsize=(6,8), constrained_layout = True)

ax = axs[0]
ax.plot(time, sum_sig, color = 'k', lw = 2)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude (AU)')
# ax.set_yticks(np.arange(-10,11,2))
ax.set_title('What you have', fontsize = 14)
ax.grid()

ax = axs[1]
for i in range(sigs.shape[0]):
    f, Pxx = scipy.signal.periodogram(sigs[i,:], fs=srate, scaling = 'spectrum', detrend = False)
    ax.plot(f, np.sqrt(Pxx * 2), lw = 2, alpha = 1)
f, Pxx = scipy.signal.periodogram(sum_sig, fs=srate, scaling = 'spectrum', detrend = False)
# ax.plot(f, np.sqrt(Pxx * 2), lw = 1, color = 'k', alpha = 1)
# ax.set_xticks(np.arange(0,5,1))
ax.set_xlim(-1,20)
ax.set_ylabel('Amplitude (AU)')
ax.set_xlabel('Frequency (Hz)')
ax.set_title('What Fourier gives', fontsize = 14)
ax.grid()

ax = axs[2]
for i in range(sigs.shape[0]):
    ax.plot(time, sigs[i,:], lw = 2)
ax.set_ylabel('Amplitude (AU)')
ax.set_xlabel('Time (s)')
# ax.set_yticks(np.arange(-10,11,2))
ax.set_title('What Fourier means', fontsize = 14)
ax.grid()


# ax.set_ylim(0,4)

fig.savefig(save_folder / 'all_fourier_lot.png', dpi = 300, bbox_inches = 'tight')


# anais histogramme phase fig

In [ ]:
sub = 'P60'

p = {
    'rolling_P2P1':60,
    'rolling_icp_abp':600,
    'resp_wsize_in_mins':3, 
    'ratio_sat':4, 
    'rate_bins_resp':np.arange(5, 30, 0.5), 
    'plot_type': '2d',
    'quantile_saturation_2d':0.025,
}

icp_resp = icp_by_resp_cycle_job.get(sub)['icp_by_resp_cycle']
resp_cycles = detect_resp_job.get(sub).to_dataframe()
p2p1 = ratio_P1P2_job.get(sub)['ratio_P1P2'].rolling(date = p['rolling_P2P1']).median('date').bfill('date').ffill('date')

In [ ]:
max_icp_phase = icp_resp.idxmax('phase')
max_icp_phase

In [ ]:
start_dates = resp_cycles['inspi_date'].values
stop_dates = resp_cycles['next_inspi_date'].values

p2p1_values = p2p1.values
p2p1_datetimes = p2p1['date'].values


df = pd.DataFrame(index = start_dates)
df['max_icp_phase'] = max_icp_phase.values
df['P2P1'] = np.nan

for start_date, stop_date in zip(start_dates, stop_dates):
    local_mask = (p2p1_datetimes >= start_date) & (p2p1_datetimes < stop_date)
    if np.any(local_mask):
        local_p2p1 = p2p1_values[local_mask]
        p2p1_average = np.nanmedian(local_p2p1)
        df.loc[start_date,'P2P1'] = p2p1_average



In [ ]:
df

# anais histogramme phase fig

In [ ]:
subs = get_patient_list(['ICP','CO2'])

bins = np.arange(0,1,0.025) 

nrows = 10 
ncols = 10

fig, axs = plt.subplots(nrows, ncols, figsize = (nrows * 2, ncols * 2), constrained_layout = True)
for i, ax in enumerate(axs.ravel()):
    sub = subs[i]
    try:
        icp_resp = icp_by_resp_cycle_job.get(sub)['icp_by_resp_cycle']
        ax.hist(icp_resp.idxmax('phase'), bins=bins, edgecolor = 'k')
        ax.set_title(sub)
    except:
        continue
fig.savefig(base_folder / 'figures' / 'figures_anais' / 'histogramme_phases.png', dpi = 300, bbox_inches = 'tight')


# Figure Anais exploration

- Carte de la france PIC dans CO2 (t0 à t1)
- P2P1
- ICP
- ABP
- PPC
- Variabilité respi
- Assisté / controlé
- PaCO2
- Sédation
- Glasgow
- CSF


In [ ]:
sub = 'P14'

p = {
    'rolling_P2P1':300,
    'rolling_icp_abp':600,
    'resp_wsize_in_mins':3, 
    'ratio_sat':4, 
    'rate_bins_resp':np.arange(5, 30, 0.5), 
    'plot_type': '2d',
    'quantile_saturation_2d':0.025,
}

reader = pycns.CnsReader(data_path / sub)


icp_by_resp_cycle = icp_by_resp_cycle_job.get(sub)['icp_by_resp_cycle']
p2_p1_da = ratio_P1P2_job.get(sub)['ratio_P1P2'].rolling(date = p['rolling_P2P1']).median('date').bfill('date').ffill('date')
all_streams = reader.streams.keys()
if 'ABP' in all_streams:
    abp_name = 'ABP'
elif 'ART' in all_streams:
    abp_name = 'ART'
else:
    raise NotImplementedError('No blood pressure stream in data')
assert 'ICP' in all_streams, 'No ICP stream in data'
stream_names = ['ICP_Mean',f'{abp_name}_Mean']
srate = max([reader.streams[stream_name].sample_rate for stream_name in stream_names])
ds = reader.export_to_xarray(stream_names, start=None, stop=None, resample=True, sample_rate=srate)
icp = ds['ICP_Mean'].rolling(times = p['rolling_icp_abp']).median('times').bfill('times').ffill('times')
abp = ds[f'{abp_name}_Mean'].rolling(times = p['rolling_icp_abp']).median('times').bfill('times').ffill('times')
ppc = abp - icp
resp_cycles = detect_resp_job.get(sub).to_dataframe()
pco2_art = icca_bio_job.get(sub)['pCO2 artériel'] * 7.50062
sedation = qualitative_sedation_level_job.get(sub)['qualitative_sedation_level']
glasgow = icca_clinical_job.get(sub)['Glasgow, total']
csf = icca_csf_job.get(sub)['Vol vidé (E_S)']

meta = get_metadata(sub=sub)
gcs_sortie = meta['GCS_sortie']                                         
mrs_sortie = meta['mRS_sortie']                                                                      
mrs_6 = meta['mRS_6mois']
motif = meta['motif']
duree = meta['durée']
age = (meta['entree_rea'] - meta['ddn']).days / 365.25

d0 = icp_by_resp_cycle['cycle_date'].values[0]
d1 = icp_by_resp_cycle['cycle_date'].values[-1]

dict_das = {
    'icp_in_resp':icp_by_resp_cycle,
    'P2P1':p2_p1_da,
    'icp':icp,
    'abp':abp,
    'ppc':ppc,
    'co2':resp_cycles,
    'ventilation':resp_cycles,
    'pco2_art':pco2_art,
    'sedation':sedation,
    'glasgow':glasgow,
    'csf':csf,
}

ylims = {
    'icp_in_resp':None,
    'P2P1':(0.5,2),
    'icp':(0,40),
    'abp':(50, 200),
    'ppc':(30,130),
    'co2':None,
    'ventilation':None,
    'pco2_art':(20,50),
    'sedation':None,
    'glasgow':(2.5,15.5),
    'csf':(0,50),
}

ylabels = {
    'icp_in_resp':'Phase',
    'P2P1':'Score',
    'icp':'mmHg',
    'abp':'mmHg',
    'ppc':'mmHg',
    'co2':'cpm',
    'ventilation':'mode',
    'pco2_art':'mmHg',
    'sedation':'Level',
    'glasgow':'Score',
    'csf':'mL',
}


nrows = len(dict_das)
figsize = (12,nrows * 3)

fig, axs = plt.subplots(nrows = nrows, figsize=figsize , constrained_layout = True, sharex = False)
fig.suptitle(f'{sub}\nage : {int(round(age,0))} - pathologie : {motif} - duree : {duree} jours\ngcs sortie : {gcs_sortie} - mrs sortie : {mrs_sortie} - mrs 6 mois : {mrs_6}')

for i, name in enumerate(dict_das):
    ax = axs[i]
    ax.set_title(name)
    ax.set_xlim(d0, d1)
    ax.set_ylabel(ylabels[name])
    if not ylims[name] is None:
        ax.set_ylim(ylims[name])

    da = dict_das[name]
    if da.ndim == 1:
        ax.plot(da[da.dims[0]], da.values, color = 'k')
        if name in ['pco2_art','csf','glasgow']:
            ax.scatter(da[da.dims[0]], da.values, color = 'r')
    if name == 'icp_in_resp':
        da = da - da.median('phase')
        da = da[::10,::2]
        vmin = da.quantile(p['quantile_saturation_2d'])
        vmax = da.quantile(1 - p['quantile_saturation_2d'])
        vmin = vmin if abs(vmin) > abs(vmax) else -vmax
        vmax = vmax if abs(vmax) > abs(vmin) else -vmin
        ax.pcolormesh(da[da.dims[0]].values, da['phase'].values, da.values.T, vmin = vmin, vmax=vmax, cmap = 'seismic')
        ax.axhline(icp_by_resp_cycle.attrs['cycle_ratio'], color = 'g', lw = 2)
    if name == 'icp':
        ax.axhline(20, color = 'r', lw = 2)
    if name == 'co2':
        view = Respi_Rate(resp_cycles, resp_wsize_in_mins = p['resp_wsize_in_mins'], ratio_sat = p['ratio_sat'], rate_bins_resp = p['rate_bins_resp'], plot_type = p['plot_type'])
        view.plot(ax, d0, d1)
    if name == 'ventilation':
        local_resp_features = resp_cycles[(resp_cycles['inspi_date'] > d0) & (resp_cycles['inspi_date'] < d1)]
        local_mode = local_resp_features['is_ventilation_controlled'].values
        datetimes = local_resp_features['inspi_date'].values
        ax.plot(datetimes, local_mode, color = 'k', lw = 2)
        ax.set_ylim(-0.1, 1.1)
        ax.set_yticks([0,1], ['assisted','controlled'])


In [ ]:
meta

In [ ]:
(meta['entree_rea'] - meta['ddn']).days / 365.25

In [ ]:
icp_by_resp_cycle

# CSF DVE

In [ ]:
sub = 'P14'

In [ ]:
ds = icca_csf_job.get(sub)['Vol vidé (E_S)']
ds

In [ ]:
icp_by_resp_cycle = icp_by_resp_cycle_job.get(sub)['icp_by_resp_cycle']

In [ ]:
icp_by_resp_cycle

In [ ]:
%matplotlib widget

In [ ]:
resp_cycles = detect_resp_job.get(sub).to_dataframe()
icp_by_resp_cycle = icp_by_resp_cycle_job.get(sub)['icp_by_resp_cycle']
p2_p1_da = ratio_P1P2_job.get(sub)['ratio_P1P2'].rolling(date = 25).median('date').bfill('date').ffill('date')

fig, axs = plt.subplots(nrows = 3, figsize = (10,6), sharex = True)

ax = axs[0]
ax.plot(resp_cycles['inspi_date'], resp_cycles['is_ventilation_controlled'])
ax.plot(resp_cycles['inspi_date'], resp_cycles['is_ventilation_controlled'])

ax2 = ax.twinx()
ax2.plot(ds['datetime_Vol vidé (E_S)'], ds.values)
ax2.scatter(ds['datetime_Vol vidé (E_S)'], ds.values, color = 'k')

ax = axs[1]
icp_by_resp_cycle = icp_by_resp_cycle - icp_by_resp_cycle.median('phase')
data = icp_by_resp_cycle.values.T
q_factor_phase = 3
q_factor_cycle = 100
data = data[::q_factor_phase,::q_factor_cycle]
q = 0.025
vmin = np.quantile(data, q)
vmax = np.quantile(data, 1-q)
ax.pcolormesh(icp_by_resp_cycle['cycle_date'].values[::q_factor_cycle], icp_by_resp_cycle['phase'].values[::q_factor_phase], data, vmin = vmin, vmax=vmax , cmap = 'seismic')
ax.axhline(0.4, color = 'g', lw = 2)

ax = axs[2]
ax.plot(p2_p1_da['date'], p2_p1_da.values, color = 'k')


In [ ]:
plt.figure()
icp_by_resp_cycle.mean('cycle_date').plot.line()
plt.show()

In [ ]:
plt.figure()
icp_by_resp_cycle[50000:50010,:].plot.line(y = 'phase', col = 'cycle_date', col_wrap = 4)
plt.show()

# resp in abp vs resp in heart

In [ ]:
sub = 'P100'

In [ ]:
resp_in_abp = abp_by_resp_cycle_job.get(sub)['abp_by_resp_cycle']
resp_in_heart = heart_rate_by_resp_cycle_job.get(sub)['heart_rate_by_resp_cycle']
if resp_in_abp.shape[0] > resp_in_heart.shape[0]:
    resp_in_abp = resp_in_abp.loc[resp_in_heart['cycle_date']]
elif resp_in_heart.shape[0] > resp_in_abp.shape[0]:
    resp_in_heart = resp_in_heart.loc[resp_in_abp['cycle_date']]

In [ ]:
resp_in_heart[100:105,:].plot.line(x = 'phase', col = 'cycle_date')

In [ ]:
resp_in_abp[100:105,:].plot.line(x = 'phase', col = 'cycle_date')

In [ ]:
ptp_heart = resp_in_heart.max('phase') - resp_in_heart.min('phase')
ptp_abp = resp_in_abp.max('phase') - resp_in_abp.min('phase')

In [ ]:
fig, ax = plt.subplots()
ax.scatter(np.log(ptp_heart), np.log(ptp_abp), alpha = 0.005)
ax.set_xlabel('log RespHRV Amplitude')
ax.set_ylabel('log Traube Herring')
# ax.set_xticks(ax.get_xticks()[:3], np.exp(ax.get_xticks()[:3]))
ax.set_ylim(-0.5,4)
ax.set_xlim(-6,6)

# see artifact

In [ ]:
ds

In [ ]:
sub = 'JL5'
da = eeg_quality_averaged_job.get(sub)['eeg_quality_averaged']
da.loc['ecog'].plot(figsize = (12,4))

# rmplot resphrv design matrix sufentanil

In [ ]:
from resphrv_design_matrix import concat_and_save
concat = concat_and_save(down_sample_factor = 10, minimal_n_points_by_sub = 10)

In [ ]:
concat

In [ ]:
concat['noradrenaline'].plot.hist(bins = 50)

In [ ]:
alpha = 0.1
sns.lmplot(data = concat, x='noradrenaline', y = 'resphrv', hue = 'sub', ci = None, legend = False, line_kws={'alpha':1}, scatter_kws={'alpha':alpha})

In [ ]:
sns.lmplot(data = concat, x='sufentanil', y = 'resphrv', hue = 'sub', line_kws={'alpha':1}, scatter_kws={'alpha':alpha}, legend = False, ci = False)

In [ ]:
sns.lmplot(data = concat, x='time', y = 'resphrv', hue = 'sub', line_kws={'alpha':1}, scatter_kws={'alpha':alpha}, legend = False, ci = False)

In [ ]:
concat

In [ ]:
concat2 = concat.copy()
concat2['resphrv_log'] = np.log(concat2['resphrv'])
sns.boxplot(data = concat2, x = 'sedation', y = 'resphrv_log')
sns.lineplot(data = concat2, x = 'sedation', y = 'resphrv_log', hue = 'sub', legend = False)

# Figure ANR

In [ ]:
from custom_view import *

In [ ]:
sub = 'MF12'
cns_reader = pycns.CnsReader(data_path / sub)
r_peaks = detect_ecg_job.get(sub).to_dataframe()
resp_cycles = detect_resp_job.get(sub).to_dataframe()
rsa_cycles = rsa_job.get(sub).to_dataframe()
abp_features = detect_abp_job.get(sub).to_dataframe()
icp_features = detect_icp_job.get(sub).to_dataframe()

In [ ]:
rsa_cycles['peak_date'] = rsa_cycles['cycle_date'] + pd.to_timedelta(rsa_cycles['peak_time'] - rsa_cycles['cycle_time'], 's')
rsa_cycles['trough_date'] = rsa_cycles['cycle_date'] + pd.to_timedelta(rsa_cycles['trough_time'] - rsa_cycles['cycle_time'], 's')

In [ ]:
duration_hours = 24
d0 = "2022-09-10T23:00:00"
d0 = np.datetime64(d0)
d1 = d0 + np.timedelta64(duration_hours, 'h')

yfontsize = 12

rate_bins_resp = np.arange(5, 30, 0.5)
step_bins_ecg = 1

views = {
        'Heart\nRate\n(bpm)':Heart_Rate(r_peaks, step_bins_ecg = step_bins_ecg, hrv_wsize_in_mins = 3, ratio_sat = 3, plot_type = '2d'),
        'Resp\nRate\n(cpm)':Respi_Rate(resp_cycles, resp_wsize_in_mins = 3, ratio_sat = 4, rate_bins_resp=rate_bins_resp, plot_type = '2d'),
        'RespHRV\n(bpm)':RSA(rsa_cycles),
        'Sedative\nlevel':Sedative_level_view(qualitative_sedation_level_job.get(sub)),
}

fig, axs = plt.subplots(nrows = len(views) + 1, figsize = (6,5), sharex = True)
fig.subplots_adjust(hspace = 0.05)

ax = axs[0]
stream_name = 'ICP_Mean'
stream = cns_reader.streams[stream_name]
sig, datetimes = stream.get_data(sel = slice(d0, d1), with_times = True, apply_gain = True)
sig = pd.Series(sig).rolling(300, center = True).median().bfill().ffill().values
ax.plot(datetimes, sig, color = 'k') 
ax.set_ylabel('ICP\n(mmHg)', fontsize = yfontsize)

for r, key in enumerate(views):
    r += 1
    ax = axs[r]
    view = views[key]
    view.plot(ax, d0, d1)
    ax.set_ylabel(key, fontsize = yfontsize)
ax.text(x = 0.7, y = 0.4, s = 'OFF Sedation', color = 'g', weight = 'bold', transform = ax.transAxes, ha=  'center', fontsize = 15)

for ax in axs:
    ax.set_xlim(d0, d1)
# xticks = np.arange(25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
# Rotates and right-aligns the x labels so they don't crowd each other.
for label in ax.get_xticklabels(which='major'):
    label.set(fontsize = yfontsize)



In [ ]:
duration_s = 30
d0 = "2022-09-10T23:00:00"
d0 = np.datetime64(d0)
d1 = d0 + np.timedelta64(duration_s, 's')
print(d0, d1)

In [ ]:
ihr_da = ihr_job.get(sub)['ihr']

In [ ]:
rsa_cycles

In [ ]:
duration_s = 15
d0 = "2022-09-09T23:01:00"
d0 = np.datetime64(d0)
d1 = d0 + np.timedelta64(duration_s, 's')

views = {
        'ECG':dict(stream_name = 'ECG_II', detections = r_peaks),
        'ABP':dict(stream_name = 'ABP', detections = abp_features),
        'ICP':dict(stream_name = 'ICP', detections = icp_features),
        'CO2':dict(stream_name = 'CO2', detections = resp_cycles),
        'RespHRV':dict(stream_name = None, detections = rsa_cycles),
}

fig, axs = plt.subplots(nrows = len(views), figsize = (6,5), sharex = True)
fig.subplots_adjust(hspace = 0.05)

for r, key in enumerate(views):
    ax = axs[r]
    d = views[key]
    if not key in ['RespHRV']:
        stream = cns_reader.streams[d['stream_name']]
        detections = d['detections']
        sig, datetimes = stream.get_data(sel = slice(d0, d1), with_times = True, apply_gain = True)
        colname = 'inspi_date' if key == 'CO2' else 'peak_date'
        local_detections = detections[(detections[colname] > d0) & (detections[colname] <= d1)]
        ax.plot(datetimes, sig, color = 'k')
        if key == 'ECG':
            inds = np.searchsorted(datetimes, local_detections['peak_date'].values)
            while inds[-1] >= sig.size:
                inds = inds[:-1]
            ax.scatter(datetimes[inds], sig[inds], color = 'm')
        elif key in ['ABP','ICP']:
            ind_troughs = np.searchsorted(datetimes, local_detections['trough_date'].values)
            ind_peaks = np.searchsorted(datetimes, local_detections['peak_date'].values)
            while ind_troughs[-1] >= sig.size:
                ind_troughs = ind_troughs[:-1]
            while ind_peaks[-1] >= sig.size:
                ind_peaks = ind_peaks[:-1]
            ax.scatter(datetimes[ind_troughs], sig[ind_troughs], color = 'y')  
            ax.scatter(datetimes[ind_peaks], sig[ind_peaks], color = 'c')
        elif key == 'CO2':
            ind_inspi = np.searchsorted(datetimes, local_detections['inspi_date'].values)
            ind_expi= np.searchsorted(datetimes, local_detections['expi_date'].values)
            while ind_inspi[-1] >= sig.size:
                ind_inspi = ind_inspi[:-1]
            while ind_expi[-1] >= sig.size:
                ind_expi = ind_expi[:-1]
            ax.scatter(datetimes[ind_inspi], sig[ind_inspi], color = 'g')  
            ax.scatter(datetimes[ind_expi], sig[ind_expi], color = 'r')
    else:
        local_da = ihr_da.loc[d0:d1]
        datetimes = local_da['datetime'].values
        sig = local_da.values
        detections = d['detections']
        colname = 'cycle_date'
        ax.plot(datetimes, sig, color = 'k', label = 'Heart Rate')
        ind_peaks, _ = scipy.signal.find_peaks(sig)
        ind_troughs, _ = scipy.signal.find_peaks(-sig)
        resphrvs = sig[ind_peaks] - sig[ind_troughs]

        ax.scatter(datetimes[ind_troughs], sig[ind_troughs], color = 'purple')  
        ax.scatter(datetimes[ind_peaks], sig[ind_peaks], color = 'blue')

        ax.quiver(datetimes[ind_peaks], sig[ind_peaks], 
                np.zeros_like(resphrvs), -resphrvs, 
                angles='xy', scale_units='xy', scale=0.9, 
                color='r', width=0.01, headwidth=2.5, headlength=5, label = 'RespHRV')
        ax.legend(loc = 'upper center', ncols = 2, fontsize = 8, framealpha = 1)


    ax.set_ylabel(key, fontsize = yfontsize)

for ax in axs:
    ax.set_xlim(d0, d1)
# xticks = np.arange(25)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%S'))
# Rotates and right-aligns the x labels so they don't crowd each other.
for label in ax.get_xticklabels(which='major'):
    label.set(fontsize = yfontsize)
ax.set_xlabel('Time (s)', fontsize = yfontsize)


# Onde dicrote detection

In [ ]:
def compute_abp(raw_abp, srate, date_vector = None, lowcut = 0.5, highcut = 10, order = 1, ftype = 'bessel', peak_prominence = 15, h_distance_s = 0.3, rise_amplitude_limits = (15,200), amplitude_at_trough_low_limit = 20, range_first_derivative = 3, compute_cardiac_output = True, show = False, verbose = False):
    assert not np.any(np.isnan(raw_abp)), 'Nans in ABP sig'
    abp_filt = iirfilt(raw_abp, srate, lowcut = lowcut, highcut = highcut, order = order, ftype = ftype)
    maximums,_ = scipy.signal.find_peaks(abp_filt, distance = int(srate * h_distance_s), prominence = peak_prominence)
    minimums,_ = scipy.signal.find_peaks(-abp_filt, distance = int(srate * h_distance_s), prominence = peak_prominence)
    if minimums[0] > maximums[0]: # first point detected has to be a minimum
        maximums = maximums[1:] # so remove the first maximum if is before first minimum
    if maximums[-1] > minimums[-1]: # last point detected has to be a minimum
        maximums = maximums[:-1] # so remove the last maximum if is after last minimum

    peak_index = maximums
    trough_index = minimums[np.searchsorted(minimums, peak_index) - 1]

    detection = pd.DataFrame()
    detection['trough_ind'] = trough_index
    detection['trough_time'] =  detection['trough_ind'] / srate
    next_trough_inds = trough_index[1:]
    next_trough_inds = np.append(next_trough_inds, np.nan)
    detection['next_trough_ind'] = next_trough_inds
    detection['next_trough_time'] =  detection['next_trough_ind'] / srate
    detection['peak_ind'] = peak_index
    detection['peak_time'] =  detection['peak_ind'] / srate
    detection = detection.iloc[:-1,:]
    detection['next_trough_ind'] = detection['next_trough_ind'].astype(int)
    detection['rise_duration'] = detection['peak_time'] - detection['trough_time']
    detection['decay_duration'] = detection['next_trough_time'] - detection['peak_time']
    detection['total_duration'] = detection['rise_duration'] + detection['decay_duration']

    detection['amplitude_at_trough'] = raw_abp[detection['trough_ind']]
    detection['amplitude_at_peak'] = raw_abp[detection['peak_ind']]
    detection['amplitude_at_next_trough'] = raw_abp[detection['next_trough_ind']]

    detection['rise_amplitude'] = detection['amplitude_at_peak'] - detection['amplitude_at_trough']
    detection['decay_amplitude'] = detection['amplitude_at_peak'] - detection['amplitude_at_next_trough']

    if not date_vector is None:
        detection['trough_date'] =  date_vector[detection['trough_ind']].astype('datetime64[ns]')
        detection['peak_date'] =  date_vector[detection['peak_ind']].astype('datetime64[ns]')
        detection['next_trough_date'] =  date_vector[detection['next_trough_ind']].astype('datetime64[ns]')
    
    #cleaning
    detection_clean = detection.copy()
    detection_clean = detection_clean[(detection_clean['amplitude_at_trough'] >= amplitude_at_trough_low_limit)]
    detection_clean = detection_clean[(detection_clean['rise_amplitude'] >= rise_amplitude_limits[0]) & (detection_clean['rise_amplitude'] <= rise_amplitude_limits[1]) ]
    detection_clean = detection_clean.reset_index(drop = True)


    if compute_cardiac_output:
        # dicrotic

        first_derivative = np.gradient(abp_filt)
        second_derivative = np.gradient(first_derivative)

        mask_amp = (abp_filt > np.quantile(abp_filt, 0.25)) & (abp_filt < np.quantile(abp_filt, 0.75))
        mask_1st_derivative = (first_derivative > -range_first_derivative) & (first_derivative < range_first_derivative) # search where first derivative is approachnig 0
        mask_2nd_derivative = second_derivative > 0 # search positive inflexion point

        mask = mask_amp & mask_1st_derivative & mask_2nd_derivative
        crossings = detect_cross(mask, 0.5)
        crossings['rises'] = crossings['rises'] + 1

        detection_clean['dicrotic_notch_ind'] = np.nan
        detection_clean['auc_cardiac_output'] = np.nan

        for i, row in detection_clean.iterrows():
            peak_ind = int(row['peak_ind'])
            next_trough_ind = int(row['next_trough_ind'])
            local_sig = abp_filt[peak_ind:next_trough_ind]
            local_mask = mask[peak_ind:next_trough_ind]

            local_crossings = crossings[(crossings['rises'] > peak_ind) & (crossings['decays'] < next_trough_ind)]

            if local_crossings.shape[0] > 0:
                local_crossings = local_crossings.iloc[0,:]

                first_derivation_dicrotic_zone = first_derivative[local_crossings['rises']:local_crossings['decays']]
                change_derivative_sign_in_dicrotic_zone = (first_derivation_dicrotic_zone[:-1] <= 0) & (first_derivation_dicrotic_zone[1:] > 0)
                if np.any(change_derivative_sign_in_dicrotic_zone):
                    dicrotic_notch_ind = np.where(change_derivative_sign_in_dicrotic_zone)[0][0] + local_crossings['rises']
                else:
                    dicrotic_notch_ind = local_crossings['rises']

                local_sig_cardiac_output = raw_abp[peak_ind:dicrotic_notch_ind] - row['amplitude_at_trough']
                detection_clean.loc[i, 'dicrotic_notch_ind'] = dicrotic_notch_ind
                detection_clean.loc[i,'auc_cardiac_output'] = np.trapz(local_sig_cardiac_output)
        detection_clean['dicrotic_notch_time'] = detection_clean['dicrotic_notch_ind'] / srate
        detection_clean['dicrotic_notch_ind'] = detection_clean['dicrotic_notch_ind'].astype('Int64')
        detection_clean['dicrotic_notch_amplitude'] = raw_abp[detection_clean['dicrotic_notch_ind']]
        
    if verbose:
        print("{n_removed} abp cycles were removed by cleaning".format(n_removed = detection.shape[0] - detection_clean.shape[0]))

    if show:
        t = np.arange(raw_abp.size) / srate
        fig, ax = plt.subplots()
        ax.plot(t, raw_abp)
        ax.scatter(t[detection_clean['trough_ind']], raw_abp[detection_clean['trough_ind']], color = 'r')
        ax.scatter(t[detection_clean['peak_ind']], raw_abp[detection_clean['peak_ind']], color = 'g')
        plt.show()

    # return detection_clean.reset_index(drop = True), mask, crossings
    return detection_clean.reset_index(drop = True)

In [ ]:
sub = 'MF12'
reader = pycns.CnsReader(data_path / sub)
stream_name = 'ABP'
stream = reader.streams['ABP']
srate = stream.sample_rate

duration_s = 300
t0 = 3600
t1 = t0 + duration_s 
i0 = int(t0 * srate)
i1 = int(t1 * srate)

raw_abp, times = stream.get_data(isel = slice(i0,i1), with_times = True, time_as_second=True, apply_gain = True)
times = times - times[0]


# detections, mask, crossings = compute_abp(raw_abp, srate)
detections = compute_abp(raw_abp, srate, compute_cardiac_output = True)

In [ ]:
detections

In [ ]:
crossings

In [ ]:
%matplotlib widget

In [ ]:
inds = detections['dicrotic_notch_ind'].dropna().values

fig, ax = plt.subplots(figsize=  (12,4))
ax.plot(times, raw_abp)
ax.scatter(times[inds], raw_abp[inds], color = 'r')
# ax.scatter(times[crossings['rises']], raw_abp[crossings['rises']], color = 'm')
# ax.scatter(times[crossings['decays']], raw_abp[crossings['decays']], color = 'c')
# ax.fill_between(times, raw_abp, where = mask, alpha = 0.3, color = 'r')

In [ ]:
crossings / srate

In [ ]:
fig, ax = plt.subplots()
ax.hist(raw_abp, bins = 100)
ax.axvline(np.quantile(raw_abp, 0.25), color = 'r')
ax.axvline(np.quantile(raw_abp, 0.75), color = 'r')

fig, ax = plt.subplots()
ax.hist(np.gradient(raw_abp), bins = 100)
plt.show()

In [ ]:
detections, mask = compute_abp(raw_abp, srate)
# detections = detections.dropna(subset = 'dicrotic_notch_ind')

In [ ]:
%matplotlib widget

In [ ]:
alpha = 0.5

fig, ax = plt.subplots(figsize = (12,4))
ax.plot(times, raw_abp)
# ax.twinx().plot(times, np.gradient(raw_abp), color = 'r', alpha = alpha)
# ax.twinx().plot(times, np.gradient(np.gradient(raw_abp)), color = 'g', alpha = alpha)
ax.scatter(times[detections['trough_ind']], raw_abp[detections['trough_ind']], color = 'r')
ax.scatter(times[detections['peak_ind']], raw_abp[detections['peak_ind']], color = 'g')
ax.fill_between(times, raw_abp, where = mask, alpha = 0.2, color = 'g')
# ax.scatter(times[detections['dicrotic_notch_ind'].astype(int)], raw_abp[detections['dicrotic_notch_ind'].astype(int)], color = 'c')


# for t in times[detections['peak_ind']]:
#     ax.axvspan(t + 0.15, t + 0.25, color = 'k' , alpha = 0.1)

# ax2 = ax.twinx()

In [ ]:
abp_highpass = iirfilt(raw_abp, srate, lowcut = 0.3)

range_first_derivative = 3

first_derivative_abp = np.gradient(abp_highpass)
second_derivative_abp = np.gradient(first_derivative_abp)

mask_amp = (abp_highpass > np.quantile(abp_highpass, 0.25)) & (abp_highpass < np.quantile(abp_highpass, 0.75))
mask_1st_derivative = (first_derivative_abp > -range_first_derivative) & (first_derivative_abp < range_first_derivative)
mask_2nd_derivative = second_derivative_abp > 0

mask = mask_amp & mask_1st_derivative & mask_2nd_derivative

In [ ]:
fig, ax = plt.subplots()
ax.hist(abp_highpass, bins = 100)
ax.axvline(np.quantile(abp_highpass, 0.25), color = 'r')
ax.axvline(np.quantile(abp_highpass, 0.75), color = 'r')

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))
ax.plot(times, abp_highpass)
ax.fill_between(times, np.min(abp_highpass), abp_highpass, where = mask, color = 'm')

In [ ]:
alpha = 0.5

fig, ax = plt.subplots(figsize = (12,4))
ax.plot(times, raw_abp)
ax.twinx().plot(times, np.gradient(raw_abp), color = 'r', alpha = alpha)
ax.twinx().plot(times, np.gradient(np.gradient(raw_abp)), color = 'g', alpha = alpha)
ax.scatter(times[detections['trough_ind']], raw_abp[detections['trough_ind']], color = 'r')
ax.scatter(times[detections['peak_ind']], raw_abp[detections['peak_ind']], color = 'g')
# ax.scatter(times[dicrotic_notch_inds], raw_abp[dicrotic_notch_inds], color = 'm')

for t in times[detections['peak_ind']]:
    ax.axvspan(t + 0.15, t + 0.25, color = 'k' , alpha = 0.1)

plt.show()
# ax2 = ax.twinx()

# RespHRV vs ICCA

In [ ]:
sub = 'MF12'
resp_hrv = rsa_job.get(sub).to_dataframe()
resp_hrv

In [ ]:
resp_hrv['decay_amplitude'].plot()

In [ ]:
d0 = resp_hrv['cycle_date'].values[0]
d1 = resp_hrv['cycle_date'].values[-1]

In [ ]:
d0.astype('datetime64[m]')

In [ ]:
def get_rsa_by_minute(sub, metric = 'decay_amplitude', remove_rsa_outliers_threshold = 30):
    resp_hrv = rsa_job.get(sub).to_dataframe()
    d0 = resp_hrv['cycle_date'].values[0]
    d1 = resp_hrv['cycle_date'].values[-1]
    starts = np.arange(d0.astype('datetime64[m]'), d1.astype('datetime64[m]'), np.timedelta64(1, 'm')).astype('datetime64[ns]')
    stops = starts + np.timedelta64(1, 'm')
    inds = range(starts.size)
    resp_hrvs = np.zeros(starts.shape)
    for i, start, stop in zip(inds, starts,stops):
        local_resp_hrv = resp_hrv[(resp_hrv['cycle_date'] >= start) & (resp_hrv['cycle_date'] < stop)][metric].values
        if local_resp_hrv.shape[0] == 0:
            local_value = np.nan
        else:
            local_value = np.nanmedian(local_resp_hrv)
        resp_hrvs[i] = local_value
    resp_hrvs[resp_hrvs > remove_rsa_outliers_threshold] = np.nan
    resp_hrvs = pd.Series(resp_hrvs).ffill().bfill().values
    return xr.DataArray(data = resp_hrvs, coords = {'datetime_resphrv':starts})


def get_ventilation_by_minute(sub):
    resp_cycles = detect_resp_job.get(sub).to_dataframe()
    d0 = resp_cycles['inspi_date'].values[0]
    d1 = resp_cycles['expi_date'].values[-1]
    starts = np.arange(d0.astype('datetime64[m]'), d1.astype('datetime64[m]'), np.timedelta64(1, 'm')).astype('datetime64[ns]')
    stops = starts + np.timedelta64(1, 'm')
    inds = range(starts.size)
    ventilation_modes = np.zeros(starts.shape)
    for i, start, stop in zip(inds, starts,stops):
        local_resp = resp_cycles[(resp_cycles['inspi_date'] >= start) & (resp_cycles['expi_date'] < stop)]['is_ventilation_controlled'].values
        if local_resp.shape[0] == 0:
            local_value = np.nan
        else:
            local_value = np.nanmedian(local_resp)
        ventilation_modes[i] = local_value
    ventilation_modes = pd.Series(ventilation_modes).ffill().bfill().values
    return xr.DataArray(data = ventilation_modes, coords = {'datetime_ventilation_mode':starts})



In [ ]:
def load_reshrv_to_icca(sub):
    da_resphrv = get_rsa_by_minute(sub)
    da_ventilation_mode = get_ventilation_by_minute(sub)
    da_qualitative_sedation = qualitative_sedation_level_job.get(sub)['qualitative_sedation_level']
    ds_clinical = resample_clinical_job.get(sub)
    ds_pse = resample_pse_tt_job.get(sub)
    da_glasgow = ds_clinical['Glasgow, total']
    da_na = ds_pse['Norépinéphrine']
    da_sufentanil = ds_pse['Sufentanil']
    da_list = [da_resphrv, da_ventilation_mode, da_qualitative_sedation, da_glasgow, da_na, da_sufentanil]
    min_date = max([da[da.dims[0]].values[0] for da in da_list])
    max_date = min([da[da.dims[0]].values[-1] for da in da_list])
    metric_da = {'resphrv':da_resphrv,
             'ventilation':da_ventilation_mode,
             'sedation':da_qualitative_sedation,
             'glasgow':da_glasgow,
             'noradrenaline':da_na,
             'sufentanil':da_sufentanil
             }
    da_all = None
    for m, da in metric_da.items():
        local_data = da.loc[min_date:max_date]
        if da_all is None:
            local_datetimes = local_data[local_data.dims[0]].values
            da_all = xr.DataArray(data = np.nan, dims = ['dtype','datetime'], coords = {'dtype':list(metric_da.keys()), 'datetime':local_datetimes})
        da_all.loc[m, :] = local_data.values
    df = pd.DataFrame(da_all.T.values, columns = da_all['dtype'].values)
    df['time'] = df.index + 1
    return df


In [ ]:
subs_icca = get_icca_subs()
subs_resphrv = get_patient_list(['CO2','ECG_II'])
subs = list(set(subs_icca + subs_resphrv))

In [ ]:
concat_df = []
for sub in subs[:5]:
    df = load_reshrv_to_icca(sub)
    df['sub'] = sub
    concat_df.append(df)
concat_df = pd.concat(concat_df)

In [ ]:
len(subs)

In [ ]:
df = load_reshrv_to_icca('MF12')

In [ ]:
df

In [ ]:
import statsmodels.api as sm

In [ ]:
Y = df['resphrv']
X = df[[c for c in df.columns if not c == 'resphrv']]
X = sm.add_constant(X)

In [ ]:
model = sm.OLS(Y,X)

In [ ]:
results = model.fit()

In [ ]:
print(results.summary())

In [ ]:
da_all.plot.line(x = 'datetime')

# Resample clinical

In [ ]:
sub = 'MF12'

In [ ]:
def resample_clinical_or_bio(ds, **p): 
    # print(ds)
    new_ds = xr.Dataset()
    names = list(ds.keys())
    for name in names:
        da = ds[name]
        data = da.values
        if not np.issubdtype(data.dtype, np.number):
            print(f'{name} is of type {data.dtype} so not resampled')
            new_da = da.copy()
        else:
            initial_datetimes = da[da.dims[0]].values
            d_start, d_stop = initial_datetimes[0], initial_datetimes[-1]
            initial_times = (initial_datetimes - d_start).astype(float) / 1e9 / 60 

            new_times = np.arange(initial_times[0], initial_times[-1] + 1, 1)
            new_datetimes = np.arange(initial_datetimes[0], initial_datetimes[-1] + np.timedelta64(1, 'm'), np.timedelta64(1, 'm'))

            f = scipy.interpolate.interp1d(initial_times, data, fill_value="extrapolate", kind = p['interpolation_kind'])
            new_data = f(new_times)

            new_da = xr.DataArray(data = new_data, coords = {f'datetime_{name}':new_datetimes}, attrs = da.attrs)
        new_ds[name] = new_da
    return new_ds

In [ ]:
p = {'interpolation_kind':'previous'}

ds = icca_clinical_job.get(sub)
new_ds = resample_clinical_or_bio(ds, **p)

In [ ]:
ds

In [ ]:
new_ds

In [ ]:
ds['Glasgow, total'].plot.scatter()

In [ ]:
new_ds['Gauche'].plot()

In [ ]:
p = {'interpolation_kind':'linear'}

ds = icca_bio_job.get(sub)
new_ds = xr.Dataset()
names = list(ds.keys())
for name in names:
    da = ds[name]

    data = da.values
    if not data.dtype in [float,int]:
        print(f'{name} is of type {data.dtype} so not resampled')
        new_da = xr.DataArray(data = data, coords = {f'datetime_{name}':da[f'datetime_{name}']}, attrs = da.attrs)
    else:
        initial_datetimes = da[da.dims[0]].values
        d_start, d_stop = initial_datetimes[0], initial_datetimes[-1]
        initial_times = (initial_datetimes - d_start).astype(float) / 1e9 / 60 

        new_times = np.arange(initial_times[0], initial_times[-1] + 1, 1)
        new_datetimes = np.arange(initial_datetimes[0], initial_datetimes[-1] + np.timedelta64(1, 'm'), np.timedelta64(1, 'm'))

        f = scipy.interpolate.interp1d(initial_times, data, fill_value="extrapolate", kind = p['interpolation_kind'])
        new_data = f(new_times)

        new_da = xr.DataArray(data = new_data, coords = {f'datetime_{name}':new_datetimes}, attrs = da.attrs)

    new_ds[name] = new_da

In [ ]:
ds

In [ ]:
ds['Hémoglobine'].plot()
ds['Hémoglobine'].plot.scatter()

In [ ]:
new_ds['Hémoglobine'].plot()

# See effect of derivative on PSD

In [ ]:
%matplotlib widget

In [ ]:
sub = 'MF12'
ihr_da = ihr_job.get(sub)['ihr']
srate = ihr_da.attrs['srate']
times = ihr_da.attrs['time']
sig = ihr_da.values
sig_p = np.gradient(sig)
sig_pp = np.gradient(sig_p)

win_size_s = 60
nperseg = int(win_size_s * srate)
f1, Pxx = scipy.signal.welch(sig, srate, nperseg = nperseg)
f2, Pxx_p = scipy.signal.welch(sig_p, srate, nperseg = nperseg)
f3, Pxx_pp = scipy.signal.welch(sig_pp, srate, nperseg = nperseg)

figsize = (9,6)

fig, axs = plt.subplots(nrows = 3, figsize=figsize, sharex = True)
ax = axs[0]
ax.plot(times, sig)
ax = axs[1]
ax.plot(times, sig_p)
ax = axs[2]
ax.plot(times, sig_pp)

fig, ax = plt.subplots(figsize=figsize, sharex = True)
ax.plot(f1, Pxx)
ax.plot(f2, Pxx_p)
ax.plot(f3, Pxx_pp)

In [ ]:
sub = 'MF12'

reader = pycns.CnsReader(data_path / sub)
stream = reader.streams['EEG']
srate = 256
duration_s = 60
t0 = 100000
t1 = t0 + duration_s
i0 = int(srate * t0)
i1 = int(srate * t1)
sigs = stream.get_data(isel = slice(i0,i1), time_as_second = True, apply_gain = True, with_times = False)
times = np.arange(sigs.shape[0]) / srate
chans = stream.channel_names
sig = sigs[:,chans.index('C4')]
sig_p = np.gradient(sig)
sig_pp = np.gradient(sig_p)

win_size_s = 5
nperseg = int(win_size_s * srate)
f1, Pxx = scipy.signal.welch(sig, srate, nperseg = nperseg)
f2, Pxx_p = scipy.signal.welch(sig_p, srate, nperseg = nperseg)
f3, Pxx_pp = scipy.signal.welch(sig_pp, srate, nperseg = nperseg)

figsize = (9,6)

fig, axs = plt.subplots(nrows = 3, figsize=figsize, sharex = True)
ax = axs[0]
ax.plot(times, sig)
ax = axs[1]
ax.plot(times, sig_p)
ax = axs[2]
ax.plot(times, sig_pp)

fig, ax = plt.subplots(figsize=figsize, sharex = True)
ax.loglog(f1, Pxx)
ax.loglog(f2, Pxx_p)
ax.loglog(f3, Pxx_pp)

In [ ]:
morlet_power?

In [ ]:
f_start = 1
f_stop = 20
n_steps = 40
n_cycles = 5
amplitude_exponent=1,
duration_s='auto'
powers = None
t0_compute = 10
t1_compute = 40
mask_t_compute = (times > t0_compute) & (times < t1_compute)
for i, sig_to_study in enumerate([sig, sig_p, sig_pp]):
    sig_crop = sig_to_study[mask_t_compute]
    t_crop = times[mask_t_compute]
    freqs, power = morlet_power(sig_crop, srate, f_start, f_stop, n_steps, n_cycles, amplitude_exponent)
    power = power[:,int(srate * 10):int(srate * 20)]
    t_crop = np.arange(power.shape[1]) / srate
    # fig, ax = plt.subplots()
    # ax.pcolormesh(t_crop, freqs, power)
    if powers is None:
        coords = {'p':[0,1,2], 'freq':freqs, 'time':t_crop}
        powers = xr.DataArray(data = np.nan, dims = coords.keys(), coords = coords)
    powers.loc[i,:,:] = power

In [ ]:
powers.plot.pcolormesh(x = 'time', y = 'freq', col = 'p', robust  = True)
plt.show()
for p in powers['p']:
    plt.figure()
    powers.loc[p].plot()
    plt.show()

# Search "normal" phase of RespHRV patients

In [ ]:
sub = 'MF12'
da = heart_rate_by_resp_cycle_job.get(sub)['heart_rate_by_resp_cycle']
da

In [ ]:
da.mean('cycle_date').plot()

In [ ]:
def get_cycle_ratios(sub):
    times_ihr = ihr_job.get(sub)['ihr'].attrs['time']
    resp_cycles = detect_resp_job.get(sub).to_dataframe()
    resp_cycles = resp_cycles[(resp_cycles['inspi_time'] > times_ihr[0]) & (resp_cycles['next_inspi_time'] < times_ihr[-1])]
    return resp_cycles['cycle_ratio'].values

In [ ]:
cycle_ratios = get_cycle_ratios(sub)
cycle_ratios.shape

In [ ]:
da.idxmax('phase').shape

In [ ]:
(da.idxmax('phase') < da.attrs['med_cycle_ratio'].sum()).to_series().value_counts(normalize = True)

In [ ]:
(da.idxmax('phase') < cycle_ratios).to_series().value_counts(normalize = True)

In [ ]:
subs = get_patient_list(['CO2','ECG_II'])
rows = []
for sub in tqdm(subs):
    try:
        da = heart_rate_by_resp_cycle_job.get(sub)['heart_rate_by_resp_cycle']
        cycle_ratios = get_cycle_ratios(sub)
        props = (da.idxmax('phase') < cycle_ratios).to_series().value_counts(normalize = True)
        if True in props.index:
            prop_cycles_max_ihr_inspi = props[True]
        else:
            prop_cycles_max_ihr_inspi = 0
        d = {'sub':sub, 'prop_max_inspi':prop_cycles_max_ihr_inspi}
        rows.append(d)
    except:
        continue
df = pd.DataFrame(rows)
df

In [ ]:
df.sort_values(by = 'prop_max_inspi')

In [ ]:
df['prop_max_inspi'].plot.hist(bins = np.arange(0, 1, 0.025))

# 0.1 Hz neuro

In [ ]:
relative_power = True
wsize_s = 100
scaling = 'density'
duration_s = 7200
sr = 256

nperseg = int(wsize_s * sr)
nfft=  nperseg * 2

subs = get_patient_list(['EEG'])
pxx_da_subs = None
# for sub in tqdm(subs[:5]):
for sub in tqdm(subs):
    reader = pycns.CnsReader(data_path / sub)
    stream = reader.streams['EEG']

    all_chans = stream.channel_names
    scalp_chans = [ch for ch in all_chans if not 'ECoG' in ch]
    if len(scalp_chans) != 13:
        continue

    sample_inds = stream.index['sample_ind']
    i0 = sample_inds[sample_inds.size // 2]
    i1 = i0 + int(duration_s * sr)
    data = stream.get_data(isel = slice(i0, i1))

    data = data[:,[all_chans.index(ch) for ch in all_chans if ch in scalp_chans]]
    data = data.T
    f, Pxx = scipy.signal.welch(data, sr, nperseg=nperseg, axis = 1, scaling = scaling, nfft=nfft)
    pxx_da = xr.DataArray(data = Pxx, coords = {'chan':scalp_chans, 'freq':f})
    if relative_power:
        pxx_da = pxx_da / pxx_da.sum('freq')

    coords = dict(sub = subs, chan = scalp_chans, freq = f)
    if pxx_da_subs is None:
        pxx_da_subs = xr.DataArray(data = np.nan, dims = coords.keys(), coords = coords)
    if pxx_da.shape == pxx_da_subs.loc[sub,:,:].shape:
        pxx_da_subs.loc[sub,:,:] = pxx_da.values
    else:
        continue
pxx_da_subs = pxx_da_subs.dropna(dim = 'sub')
    

In [ ]:
pxx_da_subs.loc[:,:,0.03:0.5].median('sub').plot.pcolormesh(x = 'freq', y = 'chan', vmin = 0, vmax = 0.05)

In [ ]:
f0 = 0.03
f1 = 0.5
data = pxx_da_subs.loc[:,:,f0:f1].mean(['sub','chan'])
fig, ax = plt.subplots(figsize = (4,3))
ax.plot(data['freq'], data.values, color = 'k', lw = 2)
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Relative power')
ax.set_ylim(-0.01,0.09)
ax.set_xlim(f0,f1)


In [ ]:
pxx_da_subs.loc[:,:,0.03:0.5].plot.pcolormesh(x = 'freq', y = 'chan', col = 'sub', col_wrap = 5, robust = True)

In [ ]:
stream.index['sample_ind'][0]

In [ ]:
reader = pycns.CnsReader(data_path / sub)
stream = reader.streams['EEG']
data, times = stream.get_data(isel = slice(0, int(256 * 7200)), with_times = True, time_as_second = True)
sr = 1 / (np.median(np.diff(times)))
print(times[-1] - times[0])

In [ ]:
n_scalp_chans = pd.Series()
subs = get_patient_list(['EEG'])
for sub in subs:
    reader = pycns.CnsReader(data_path / sub)
    stream = reader.streams['EEG']
    chans = stream.channel_names
    scalp_chans = [ch for ch in chans if not 'ECoG' in ch]
    n_scalp_chans[sub] = len(scalp_chans)

In [ ]:
n_scalp_chans.value_counts()

In [ ]:
relative_power = True

In [ ]:
wsize_s = 300
scaling = 'density'
nperseg = int(wsize_s * sr)
f, Pxx = scipy.signal.welch(data, sr, nperseg=nperseg, axis = 0, scaling = scaling)
pxx_da = xr.DataArray(data = Pxx, coords = {'f':f, 'chan':stream.channel_names})
if relative_power:
    pxx_da = pxx_da / pxx_da.sum('f')
pxx_da_scalp = pxx_da.loc[:,[ch for ch in pxx_da['chan'].values if not 'ECoG' in ch]]

In [ ]:
pxx_da_scalp.loc[0.03:0.5,:].plot.line(x = 'f', hue = 'chan', xscale=  'linear', yscale = 'linear')

In [ ]:
pxx_da_scalp.loc[0.03:0.5,:].plot.line(x = 'f', hue = 'chan', xscale=  'linear', yscale = 'linear')

In [ ]:
pxx_da_scalp.loc[0.03:0.5,:].plot.pcolormesh(x = 'f', y = 'chan')

In [ ]:
pxx_da_scalp.loc[0.01:0.5,:].plot.line(x = 'f', hue = 'chan', xscale=  'linear', yscale = 'log')

# RSA vs Resp cycle freq

In [ ]:
subs = get_patient_list(['CO2'])
s = pd.Series()
for sub in subs:
    s[sub] = detect_resp_job.get(sub).to_dataframe()['cycle_freq'].std()


In [ ]:
s.plot.hist()

In [ ]:
s.sort_values(ascending = False)

In [ ]:
%matplotlib widget

In [ ]:
sub = 'P89' #cool
# sub = 'P17' #cool

figsize = (10,7)
resp_cycles = detect_resp_job.get(sub).to_dataframe()
rsa_cycles = rsa_job.get(sub).to_dataframe()

fig, ax =plt.subplots(figsize=figsize)
ax.plot(resp_cycles['inspi_date'], resp_cycles['cycle_freq_cpm'])

fig, ax =plt.subplots(figsize=figsize)
log_cycle_freq = np.log(resp_cycles['cycle_freq_cpm'].values)
ax.scatter(log_cycle_freq, rsa_cycles['decay_amplitude'], alpha = 0.2)
xticks = np.arange(np.min(log_cycle_freq), np.max(log_cycle_freq), 0.5)
labels = np.round(np.exp(xticks),2)
ax.set_xticks(xticks, labels = labels)
# ax.scatter(resp_cycles['cycle_freq_cpm'], rsa_cycles['decay_amplitude'], alpha = 0.2)
# ax.set_xticks(np.exp(ax.get_xticks()))
# ax.scatter(resp_cycles['cycle_freq_cpm'], rsa_cycles['rsa_amplitude_clean'], alpha = 0.2)
# ax.scatter(resp_cycles['cycle_freq_cpm'], rsa_cycles['rising_amplitude'], alpha = 0.2)
ax.set_xlabel('Cycle freq (cpm)')
ax.set_ylabel('RSA (bpm)')
# ax.axvline(7.2, color ='r')
# ax.set_xlim(-1,30)
# ax.set_xlim(xticks[0],xticks[-1])
ax.set_xlim(1,xticks[-1])


In [ ]:
%matplotlib inline
plt.show()

In [ ]:
import matplotlib.ticker as ticker

In [ ]:
yscale = 'log' 
# yscale = 'linear' 

meta = get_metadata().set_index('ID_pseudo')

xmin = 0.5
xmax = 60
ymin = 0.1
ymax = 60

if yscale == 'log':
    bins_rsa = np.logspace(np.log10(ymin), np.log10(ymax), 50)
else:
    bins_rsa = np.arange(ymin, ymax, 1)
# ticks = np.arange(xmin,xmax,2)
xticks = [1,2,3,4,5,10,20,30,40,50,60]
yticks = [1,5,10,15,20,30,40]

figsize = (10,4)

subs = get_patient_list(['CO2','ECG_II'])
n_subs = 0
for sub in subs:
# for sub in subs[:2]:
    try:
        resp_cycles = detect_resp_job.get(sub).to_dataframe()
        rsa_cycles = rsa_job.get(sub).to_dataframe()

        fig, axs =plt.subplots(ncols = 2, figsize=figsize)
        gg = meta.loc[sub,'GCS_sortie']
        fig.suptitle(f'{sub} - GCS sortie : {int(gg)}')

        ax = axs[0]
        
        # ax.scatter(resp_cycles['cycle_freq_cpm'], rsa_cycles['decay_amplitude'], alpha = 0.05, color = 'k')
        ax.scatter(resp_cycles['cycle_freq_cpm'], rsa_cycles['decay_amplitude'], alpha = 0.1, c = range(rsa_cycles.shape[0]), cmap = 'nipy_spectral')

        ax.set_xlim(xmin, xmax)  
        
        ax.set_xlabel('Resp cycle frequency (cpm)')
        ax.set_ylabel('RespHRV (bpm)')
        
        ax.set_yticks(yticks)
        ax.set_xticks(xticks)

        if yscale == 'log':
            ax.set_xscale('log')
            ax.set_yscale('log')
            ax.get_yaxis().set_major_formatter(ticker.ScalarFormatter())
            ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
            ax.set_yticks(yticks)
            ax.set_xticks(xticks)

        ax.axvline(7.2, color = 'r', label = 'Corner Frequency\n(Hirsch & Bishop)')
        ax.legend(loc = 'lower left', framealpha = 1, fontsize = 9)

        ax.set_ylim(ymin, ymax)

        ax = axs[1]
        ax.hist(rsa_cycles['decay_amplitude'], bins=bins_rsa, orientation='horizontal', color="k", log = False, density = True)
        ax.set_yticks(yticks)

        if yscale == 'log':
            ax.set_yscale('log')
            ax.get_yaxis().set_major_formatter(ticker.ScalarFormatter())
            ax.set_yticks(yticks)

        ax.set_ylabel('RespHRV (bpm)')
        if yscale == 'log':
            ax.set_xlabel('Density')
        else:
            ax.set_xlabel('Count')
        ax.set_ylim(ymin, ymax)
        fig.savefig(base_folder / 'figures' / 'resphrv_amplitude_vs_resp_freq' / f'{sub}.png', dpi = 300, bbox_inches = 'tight')
        plt.show()
        n_subs += 1
    except Exception as e:
        print(sub)
        print(f"Error with {sub}: {e}")
        print(resp_cycles.shape)
        print(rsa_cycles.shape)
        continue

print(f'N patients:', n_subs)

In [ ]:
yscale = 'log' 
# yscale = 'linear' 

meta = get_metadata().set_index('ID_pseudo')

xmin = 0.5
xmax = 60
ymin = 0.1
ymax = 60

if yscale == 'log':
    bins_rsa = np.logspace(np.log10(ymin), np.log10(ymax), 50)
else:
    bins_rsa = np.arange(ymin, ymax, 1)
# ticks = np.arange(xmin,xmax,2)
xticks = [1,2,3,4,5,10,20,30,40,50,60]
yticks = [1,5,10,15,20,30,40]

figsize = (10,4)

subs = get_patient_list(['CO2','ECG_II'])
n_subs = 0
for sub in subs:
# for sub in subs[:2]:
    try:
        resp_cycles = detect_resp_job.get(sub).to_dataframe()
        rsa_cycles = rsa_job.get(sub).to_dataframe()

        fig, axs =plt.subplots(ncols = 2, figsize=figsize)
        gg = meta.loc[sub,'GCS_sortie']
        fig.suptitle(f'{sub} - GCS sortie : {int(gg)}')

        ax = axs[0]
        
        # ax.scatter(resp_cycles['cycle_freq_cpm'], rsa_cycles['decay_amplitude'], alpha = 0.05, color = 'k')
        ax.scatter(resp_cycles['cycle_freq_cpm'], rsa_cycles['decay_amplitude'], alpha = 0.1, c = range(rsa_cycles.shape[0]), cmap = 'nipy_spectral')

        ax.set_xlim(xmin, xmax)  
        
        ax.set_xlabel('Resp cycle frequency (cpm)')
        ax.set_ylabel('RespHRV (bpm)')
        
        ax.set_yticks(yticks)
        ax.set_xticks(xticks)

        if yscale == 'log':
            ax.set_xscale('log')
            ax.set_yscale('log')
            ax.get_yaxis().set_major_formatter(ticker.ScalarFormatter())
            ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
            ax.set_yticks(yticks)
            ax.set_xticks(xticks)

        ax.axvline(7.2, color = 'r', label = 'Corner Frequency\n(Hirsch & Bishop)')
        ax.legend(loc = 'lower left', framealpha = 1, fontsize = 9)

        ax.set_ylim(ymin, ymax)

        ax = axs[1]
        ax.hist(rsa_cycles['decay_amplitude'], bins=bins_rsa, orientation='horizontal', color="k", log = False, density = True)
        ax.set_yticks(yticks)

        if yscale == 'log':
            ax.set_yscale('log')
            ax.get_yaxis().set_major_formatter(ticker.ScalarFormatter())
            ax.set_yticks(yticks)

        ax.set_ylabel('RespHRV (bpm)')
        if yscale == 'log':
            ax.set_xlabel('Density')
        else:
            ax.set_xlabel('Count')
        ax.set_ylim(ymin, ymax)

        plt.show()
        n_subs += 1
    except Exception as e:
        print(sub)
        print(f"Error with {sub}: {e}")
        print(resp_cycles.shape)
        print(rsa_cycles.shape)
        continue

print(f'N patients:', n_subs)

In [ ]:
yscale = 'log' 
# yscale = 'linear' 

meta = get_metadata().set_index('ID_pseudo')

xmin = 0.5
xmax = 60
ymin = 0.1
ymax = 60

if yscale == 'log':
    bins_rsa = np.logspace(np.log10(ymin), np.log10(ymax), 50)
else:
    bins_rsa = np.arange(ymin, ymax, 1)
# ticks = np.arange(xmin,xmax,2)
xticks = [1,2,3,4,5,10,20,30,40,50,60]
yticks = [1,5,10,15,20,30,40]

figsize = (10,4)

subs = get_patient_list(['CO2','ECG_II'])
n_subs = 0
for sub in subs:
# for sub in subs[:2]:
    try:
        resp_cycles = detect_resp_job.get(sub).to_dataframe()
        rsa_cycles = rsa_job.get(sub).to_dataframe()

        fig, axs =plt.subplots(ncols = 2, figsize=figsize)
        gg = meta.loc[sub,'GCS_sortie']
        fig.suptitle(f'{sub} - GCS sortie : {int(gg)}')

        ax = axs[0]
        
        # ax.scatter(resp_cycles['cycle_freq_cpm'], rsa_cycles['decay_amplitude'], alpha = 0.05, color = 'k')
        ax.scatter(resp_cycles['cycle_freq_cpm'], rsa_cycles['decay_amplitude'], alpha = 0.1, c = range(rsa_cycles.shape[0]), cmap = 'nipy_spectral')

        ax.set_xlim(xmin, xmax)  
        
        ax.set_xlabel('Resp cycle frequency (cpm)')
        ax.set_ylabel('RespHRV (bpm)')
        
        ax.set_yticks(yticks)
        ax.set_xticks(xticks)

        if yscale == 'log':
            ax.set_xscale('log')
            ax.set_yscale('log')
            ax.get_yaxis().set_major_formatter(ticker.ScalarFormatter())
            ax.get_xaxis().set_major_formatter(ticker.ScalarFormatter())
            ax.set_yticks(yticks)
            ax.set_xticks(xticks)

        ax.axvline(7.2, color = 'r', label = 'Corner Frequency\n(Hirsch & Bishop)')
        ax.legend(loc = 'lower left', framealpha = 1, fontsize = 9)

        ax.set_ylim(ymin, ymax)

        ax = axs[1]
        ax.hist(rsa_cycles['decay_amplitude'], bins=bins_rsa, orientation='horizontal', color="k", log = False, density = True)
        ax.set_yticks(yticks)

        if yscale == 'log':
            ax.set_yscale('log')
            ax.get_yaxis().set_major_formatter(ticker.ScalarFormatter())
            ax.set_yticks(yticks)

        ax.set_ylabel('RespHRV (bpm)')
        if yscale == 'log':
            ax.set_xlabel('Density')
        else:
            ax.set_xlabel('Count')
        ax.set_ylim(ymin, ymax)

        plt.show()
        n_subs += 1
    except Exception as e:
        print(sub)
        print(f"Error with {sub}: {e}")
        print(resp_cycles.shape)
        print(rsa_cycles.shape)
        continue

print(f'N patients:', n_subs)

In [ ]:
fig, ax = plt.subplots()
ax.hist( rsa_cycles['decay_amplitude'], bins = np.arange(0.1 , 20.1, 0.1))

fig, ax = plt.subplots()
ax.hist( np.log(rsa_cycles['decay_amplitude']), bins = np.arange(0.1 , 3, 0.1))
# ax.set_xlim()

In [ ]:
fig, ax = plt.subplots()
ax.hist( rsa_cycles['decay_amplitude'], bins = np.arange(0.1 , 20.1, 0.1))

fig, ax = plt.subplots()
ax.hist( np.log(rsa_cycles['decay_amplitude']), bins = np.arange(0.1 , 3, 0.1))
# ax.set_xlim()

In [ ]:
xticks

In [ ]:
np.log(15)

In [ ]:
np.exp(-2.7)

In [ ]:
ax.get_xticks()

# See resp mode vs sedation level

In [ ]:
get_patient_list(['CO2'])

In [ ]:
sub = 'DV4'
resp_cycles = detect_resp_job.get(sub).to_dataframe()
sedation = qualitative_sedation_level_job.get(sub)['qualitative_sedation_level']

band = 'lf'

figsize = (12,5)
fig, axs = plt.subplots(nrows = 2, figsize = figsize, sharex = True)
ax = axs[0]
ax.plot(resp_cycles['inspi_date'], resp_cycles['is_ventilation_controlled'])
ax = axs[1]
ax.plot(sedation[sedation.dims[0]], sedation.values)
# ax.set_xlim(resp_cycles['inspi_date'].values[0], resp_cycles['inspi_date'].values[-1])


figsize = (12,5)
fig, axs = plt.subplots(nrows = 2, figsize = figsize, sharex = True)
ax = axs[0]
ax.plot(resp_cycles['inspi_date'], resp_cycles['is_ventilation_controlled'])
ax = axs[1]
ax.plot(sedation[sedation.dims[0]], sedation.values)
ax.set_xlim(resp_cycles['inspi_date'].values[0], resp_cycles['inspi_date'].values[-1])

figsize = (12,5)
fig, axs = plt.subplots(nrows = 2, figsize = figsize, sharex = True)
ax = axs[0]
ax.plot(resp_cycles['inspi_date'], resp_cycles['sliding_variability_cpm'])
ax = axs[1]
ax.plot(sedation[sedation.dims[0]], sedation.values)
ax.set_xlim(resp_cycles['inspi_date'].values[0], resp_cycles['inspi_date'].values[-1])

# fig, ax = plt.subplots(figsize = figsize)
# x = sedation.loc[da[da.dims[1]]].values
# y = da.loc[da[da.dims[1]]].values
# noise =  np.random.randn(y.size) * 1e-2
# # y = y + noise * 10
# x = x + noise
# ax.scatter(x, y, alpha = 0.1)
# sedation.loc[da_gg[da_gg.dims[0]]]

# See hrv vs sedation level

In [ ]:
hrv_freq_job.get(sub)

In [ ]:
sub = 'P70'
# da = hrv_freq_job.get(sub)['hrv_freq']
da = hr_filtered_job.get(sub)['hr_filtered']
sedation = qualitative_sedation_level_job.get(sub)['qualitative_sedation_level']

band = 'lf'

figsize = (12,5)
fig, axs = plt.subplots(nrows = 2, figsize = figsize, sharex = True)
ax = axs[0]
ax.plot(da[da.dims[1]], da.loc[band].values)
ax = axs[1]
ax.plot(sedation[sedation.dims[0]], sedation.values)

# fig, ax = plt.subplots(figsize = figsize)
# x = sedation.loc[da[da.dims[1]]].values
# y = da.loc[da[da.dims[1]]].values
# noise =  np.random.randn(y.size) * 1e-2
# # y = y + noise * 10
# x = x + noise
# ax.scatter(x, y, alpha = 0.1)
# sedation.loc[da_gg[da_gg.dims[0]]]

# See Glasgow vs sedation level

In [ ]:
ds = icca_pse_tt_job.get(sub)
ds['Norépinéphrine'].plot.scatter()

In [ ]:
ds = resample_pse_tt_job.get(sub)
ds['Norépinéphrine'].plot()

In [ ]:
meta = get_metadata()
meta.dropna(subset='GCS_sortie').sort_values(by = 'GCS_sortie', ascending = False)['ID_pseudo'].values

In [ ]:
sedation

In [ ]:
sub = 'LD16'
da_gg = icca_clinical_job.get(sub)['Glasgow, total']
sedation = qualitative_sedation_level_job.get(sub)['qualitative_sedation_level']

figsize = (12,5)
fig, axs = plt.subplots(nrows = 2, figsize = figsize, sharex = True)
ax = axs[0]
ax.plot(da_gg[da_gg.dims[0]], da_gg.values)
ax = axs[1]
ax.plot(sedation[sedation.dims[0]], sedation.values)

fig, ax = plt.subplots(figsize = figsize)
x = sedation.loc[da_gg[da_gg.dims[0]]].values
y = da_gg.loc[da_gg[da_gg.dims[0]]].values
noise =  np.random.randn(y.size) * 1e-2
# y = y + noise * 10
x = x + noise
ax.scatter(x, y, alpha = 0.1)
# sedation.loc[da_gg[da_gg.dims[0]]]

# Dvp 3 stades vector sedation

## 3 stades :

3 drugs : Thiopenthal , Propofol, Midazolam
- 0 : Nothing
- 1 : 1 sedation
- 2 : 2 or more

In [ ]:
sub = 'P12'
sedatives = ['Thiopenthal','Midazolam','Propofol']

ds = resample_pse_tt_job.get(sub)
possible_tts = list(ds.keys())
possible_sedatives = [s for s in possible_tts if s in sedatives]
possible_sedatives

In [ ]:
meta = get_metadata(sub)

In [ ]:
edges_datetime_mode = 'sedative_date' # hospit_date | sedative_date

if edges_datetime_mode == 'hospit_date':
    min_datetime_pse = min([ds[name][ds[name].dims[0]].values[0] for name in possible_sedatives])
    max_datetime_pse = max([ds[name][ds[name].dims[0]].values[-1] for name in possible_sedatives])
    min_datetime_hospit = np.datetime64(meta['entree_rea'])
    max_datetime_hospit = np.datetime64(meta['date_sortie_rea'])
    min_datetime = min([min_datetime_pse, min_datetime_hospit])
    max_datetime = max([max_datetime_pse, max_datetime_hospit])
elif edges_datetime_mode == 'sedative_date':
    min_datetime = min([ds[name][ds[name].dims[0]].values[0] for name in possible_sedatives])
    max_datetime = max([ds[name][ds[name].dims[0]].values[-1] for name in possible_sedatives])

In [ ]:
min_datetime

In [ ]:
max_datetime

In [ ]:
min_dt = np.timedelta64(1 ,'m')
datetimes_global = np.arange(min_datetime, max_datetime + min_dt, min_dt)
datetimes_global.shape

In [ ]:
datetimes_global

In [ ]:
bool_global = np.zeros((len(possible_sedatives), datetimes_global.size), dtype = bool)

for i, name in enumerate(possible_sedatives):
    da = ds[name]
    datetimes_test = da[da.dims[0]].values
    dose_test = da.values
    non_zero_dose = dose_test != 0
    datetimes_non_zero = datetimes_test[non_zero_dose]
    bool_tt = np.isin(datetimes_global, datetimes_non_zero)
    bool_global[i,:] = bool_tt

qualitative_sedation_level = np.sum(bool_global, axis = 0)
qualitative_sedation_level[qualitative_sedation_level >= 2] = 2

In [ ]:
fig, axs = plt.subplots(nrows = len(possible_sedatives) + 1, figsize = (10,6), sharex=  True)

for r, name in enumerate(possible_sedatives):
    ax = axs[r]
    da = ds[name]
    ax.plot(da[da.dims[0]], da)
    ax.set_ylabel(name)

ax = axs[-1]
ax.plot(datetimes_global, qualitative_sedation_level)
ax.set_ylabel('Sedation level')

In [ ]:
qualitative_sedation_level_params = {
    'resample_pse_tt_params':resample_pse_tt_params,
    'edges_datetime_mode':'hospit_date' # hospit_date | sedative_date. I prefer hospit_date but warning to period between start hospit and start sedative where no data available will give 0 level.
}

def qualitative_sedation_level(sub, **p):
    sedatives = ['Thiopenthal','Midazolam','Propofol']
    ds = resample_pse_tt_job.get(sub)
    possible_tts = list(ds.keys())
    possible_sedatives = [s for s in possible_tts if s in sedatives]
    
    assert p['edges_datetime_mode'] in ['hospit_date','sedative_date'], "'edges_datetime_mode' should be one['hospit_date','sedative_date']"
    if p['edges_datetime_mode'] == 'hospit_date':
        meta = get_metadata(sub)
        min_datetime_pse = min([ds[name][ds[name].dims[0]].values[0] for name in possible_sedatives])
        max_datetime_pse = max([ds[name][ds[name].dims[0]].values[-1] for name in possible_sedatives])
        min_datetime_hospit = np.datetime64(meta['entree_rea'])
        max_datetime_hospit = np.datetime64(meta['date_sortie_rea'])
        min_datetime = min([min_datetime_pse, min_datetime_hospit])
        max_datetime = max([max_datetime_pse, max_datetime_hospit])
    elif p['edges_datetime_mode'] == 'sedative_date':
        min_datetime = min([ds[name][ds[name].dims[0]].values[0] for name in possible_sedatives])
        max_datetime = max([ds[name][ds[name].dims[0]].values[-1] for name in possible_sedatives])

    minute_delta_t = np.timedelta64(1 ,'m')
    datetimes_global = np.arange(min_datetime, max_datetime + minute_delta_t, minute_delta_t)

    bool_global = np.zeros((len(possible_sedatives), datetimes_global.size), dtype = bool)

    for i, name in enumerate(possible_sedatives):
        da = ds[name]
        datetimes_test = da[da.dims[0]].values
        dose_test = da.values
        non_zero_dose = dose_test != 0
        datetimes_non_zero = datetimes_test[non_zero_dose]
        bool_tt = np.isin(datetimes_global, datetimes_non_zero)
        bool_global[i,:] = bool_tt

    qualitative_sedation_level = np.sum(bool_global, axis = 0)
    qualitative_sedation_level[qualitative_sedation_level >= 2] = 2
    
    da = xr.DataArray(data = qualitative_sedation_level, coords = {'datetime':datetimes_global})
    ds = xr.Dataset()
    ds['qualitative_sedation_level'] = da
    return ds

def test_qualitative_sedation_level(sub):
    print(sub)
    ds = qualitative_sedation_level(sub, **qualitative_sedation_level_params)
    print(ds['qualitative_sedation_level'])

In [ ]:
test_qualitative_sedation_level('MF12')

# Dvp resample pse tt func

In [ ]:
sub = 'LA19'

sedatives = ['Thiopenthal','Midazolam','Propofol']

ds = icca_pse_tt_job.get(sub)

possible_tts = list(ds.keys())
possible_sedatives = [s for s in possible_tts if s in sedatives]
possible_sedatives

# bins = np.arange(0, 250, 10)
# fig, ax = plt.subplots()
# for tt in possible_sedatives:
#     # ds[tt].plot.line()
#     delta_tt_min = np.diff(ds[tt][ds[tt].dims[0]]).astype(float) / 1e9 / 60
#     count, bins = np.histogram(delta_tt_min, bins=bins, density=True)
#     ax.plot(bins[:1], count,  label = tt, alpha = 0.2)
#     # ax.hist(delta_tt_min, bins=bins, label = tt, alpha = 0.2, density = True)
    
# ax.legend()

In [ ]:
initial_times = 

In [ ]:
%matplotlib widget

In [ ]:
mask_where_zeros_should_be = np.diff(initial_times) > n_mins_zero_dose
mask_where_zeros_should_be = np.append(mask_where_zeros_should_be, False)

In [ ]:
n_mins_zero_dose = 120

mask_where_zeros_should_be = np.diff(initial_times) > n_mins_zero_dose
mask_where_zeros_should_be = np.append(mask_where_zeros_should_be, False)
dates_where_zero_should_be = initial_datetimes[mask_where_zeros_should_be] + np.timedelta64(n_mins_zero_dose, 'm')
dates_where_zero_should_be

In [ ]:
dose_with_zeros = dose.copy()
inds_where_insert_zeros = np.searchsorted(initial_datetimes, dates_where_zero_should_be)
dose_with_zeros = np.insert(dose_with_zeros,inds_where_insert_zeros , 0)
initial_datetimes_with_zeros =  np.insert(initial_datetimes, inds_where_insert_zeros, dates_where_zero_should_be)


In [ ]:
name = 'Thiopenthal'
n_mins_zero_dose = 120

da = ds[name]
dose = da.values
initial_datetimes = da[da.dims[0]].values
d_start, d_stop = initial_datetimes[0], initial_datetimes[-1]
initial_times = (initial_datetimes - d_start).astype(float) / 1e9 / 60 

mask_where_zeros_should_be = np.diff(initial_times) > n_mins_zero_dose
mask_where_zeros_should_be = np.append(mask_where_zeros_should_be, False)
dates_where_zero_should_be = initial_datetimes[mask_where_zeros_should_be] + np.timedelta64(n_mins_zero_dose, 'm')
dose_with_zeros = dose.copy()
inds_where_insert_zeros = np.searchsorted(initial_datetimes, dates_where_zero_should_be)
dose_with_zeros = np.insert(dose_with_zeros,inds_where_insert_zeros , 0)
initial_datetimes_with_zeros =  np.insert(initial_datetimes, inds_where_insert_zeros, dates_where_zero_should_be)
initial_times_with_zeros = (initial_datetimes_with_zeros - d_start).astype(float) / 1e9 / 60 

new_times = np.arange(initial_times_with_zeros[0], initial_times_with_zeros[-1] + 1, 1)
new_datetimes = np.arange(initial_datetimes_with_zeros[0], initial_datetimes_with_zeros[-1] + np.timedelta64(1, 'm'), np.timedelta64(1, 'm'))


# f = scipy.interpolate.interp1d(initial_times, dose, fill_value="extrapolate", kind = kind)
# new_dose = f(new_times)

fig, axs = plt.subplots(nrows = 3, sharex = True, sharey = True, figsize = (12,6))

ax = axs[0]
ax.plot(initial_datetimes, dose)
ax.scatter(initial_datetimes, dose, color = 'k')

ax = axs[1]
ax.plot(initial_datetimes_with_zeros, dose_with_zeros)
ax.scatter(initial_datetimes_with_zeros, dose_with_zeros, color = 'k')

ax = axs[2]
alpha = 0.8
kinds = ["linear", "nearest", "zero",  "previous"]#, "nearest-up", "slinear", "quadratic", "cubic", "next"]
for kind in kinds:
    f = scipy.interpolate.interp1d(initial_times_with_zeros, dose_with_zeros, fill_value="extrapolate", kind = kind)
    new_dose = f(new_times)
    ax.plot(new_datetimes, new_dose, label = kind, alpha = alpha)
    ax.scatter(new_datetimes, new_dose, alpha = alpha)
ax.legend()

In [ ]:
resample_pse_tt_params = {
    'icca_pse_tt_params':icca_pse_tt_params,
    'interpolation_kind':'previous',
    'n_mins_considering_zero_dose':130, # when this amount of minutes has passed since last available dose, a zero dose is initialized until next available dose
}

def resample_pse_tt(sub, **p):
    ds = icca_pse_tt_job.get(sub)
    new_ds = xr.Dataset()
    names = list(ds.keys())
    for name in names:
        da = ds[name]

        dose = da.values
        initial_datetimes = da[da.dims[0]].values
        d_start, d_stop = initial_datetimes[0], initial_datetimes[-1]
        initial_times = (initial_datetimes - d_start).astype(float) / 1e9 / 60 

        mask_where_zeros_should_be = np.diff(initial_times) > p['n_mins_considering_zero_dose']
        mask_where_zeros_should_be = np.append(mask_where_zeros_should_be, False)
        dates_where_zero_should_be = initial_datetimes[mask_where_zeros_should_be] + np.timedelta64(p['n_mins_considering_zero_dose'], 'm')
        dose_with_zeros = dose.copy()
        inds_where_insert_zeros = np.searchsorted(initial_datetimes, dates_where_zero_should_be)
        dose_with_zeros = np.insert(dose_with_zeros,inds_where_insert_zeros , 0)
        initial_datetimes_with_zeros =  np.insert(initial_datetimes, inds_where_insert_zeros, dates_where_zero_should_be)
        initial_times_with_zeros = (initial_datetimes_with_zeros - d_start).astype(float) / 1e9 / 60 

        new_times = np.arange(initial_times_with_zeros[0], initial_times_with_zeros[-1] + 1, 1)
        new_datetimes = np.arange(initial_datetimes_with_zeros[0], initial_datetimes_with_zeros[-1] + np.timedelta64(1, 'm'), np.timedelta64(1, 'm'))

        f = scipy.interpolate.interp1d(initial_times_with_zeros, dose_with_zeros, fill_value="extrapolate", kind = p['interpolation_kind'])
        new_dose = f(new_times)

        new_da = xr.DataArray(data = new_dose, coords = {f'datetime_{name}':new_datetimes}, attrs = da.attrs)

        new_ds[name] = new_da
    return new_ds

def test_resample_pse_tt(sub):
    print(sub)
    ds = resample_pse_tt(sub, **resample_pse_tt_params)
    print(ds)

In [ ]:
test_resample_pse_tt('GA9')

In [ ]:
new_da = xr.DataArray(data = new_dose, coords = {f'datetime_{name}':new_datetimes}, attrs = da.attrs)

In [ ]:
new_da

In [ ]:
ds_test = xr.Dataset()

In [ ]:
ds_test['test'] = new_da

In [ ]:
# name = 'Propofol'
name = 'Norépinéphrine'
kind = 'previous'
n_mins_considering_zero_dose = 130

ds_test = xr.Dataset()

da = ds[name]
dose = da.values
initial_datetimes = da[da.dims[0]].values
d_start, d_stop = initial_datetimes[0], initial_datetimes[-1]
initial_times = (initial_datetimes - d_start).astype(float) / 1e9 / 60 

mask_where_zeros_should_be = np.diff(initial_times) > n_mins_considering_zero_dose
mask_where_zeros_should_be = np.append(mask_where_zeros_should_be, False)
dates_where_zero_should_be = initial_datetimes[mask_where_zeros_should_be] + np.timedelta64(n_mins_considering_zero_dose, 'm')
dose_with_zeros = dose.copy()
inds_where_insert_zeros = np.searchsorted(initial_datetimes, dates_where_zero_should_be)
dose_with_zeros = np.insert(dose_with_zeros,inds_where_insert_zeros , 0)
initial_datetimes_with_zeros =  np.insert(initial_datetimes, inds_where_insert_zeros, dates_where_zero_should_be)
initial_times_with_zeros = (initial_datetimes_with_zeros - d_start).astype(float) / 1e9 / 60 

new_times = np.arange(initial_times_with_zeros[0], initial_times_with_zeros[-1] + 1, 1)
new_datetimes = np.arange(initial_datetimes_with_zeros[0], initial_datetimes_with_zeros[-1] + np.timedelta64(1, 'm'), np.timedelta64(1, 'm'))

f = scipy.interpolate.interp1d(initial_times_with_zeros, dose_with_zeros, fill_value="extrapolate", kind = kind)
new_dose = f(new_times)

new_da = xr.DataArray(data = new_dose, coords = {f'datetime_{name}':new_datetimes}, attrs = da.attrs)
ds_test['test'] = new_da

fig, axs = plt.subplots(nrows = 3, sharex = True, sharey = True, figsize = (12,6))

ax = axs[0]
ax.plot(initial_datetimes, dose)
ax.scatter(initial_datetimes, dose, color = 'k')

ax = axs[1]
ax.plot(initial_datetimes_with_zeros, dose_with_zeros)
ax.scatter(initial_datetimes_with_zeros, dose_with_zeros, color = 'k')

ax = axs[2]
ax.plot(new_datetimes, new_dose, label = kind, alpha = alpha)
ax.scatter(new_datetimes, new_dose, alpha = alpha)
ax.legend()

In [ ]:
pd.Series(initial_datetimes_with_zeros).value_counts().value_counts()

In [ ]:
_, unique_idx = np.unique(initial_datetimes_with_zeros, return_index=True)

# Detect new bugged patients ICCA

In [ ]:
def get_icca_edges_sub(sub):
    datetimes = icca_clinical_job.get(sub)['Glasgow, total']['datetime_Glasgow, total'].values
    return datetimes[0], datetimes[-1]

In [ ]:
meta = get_metadata().set_index('ID_pseudo')
import os.path, time
pb_subs = []
for sub in get_icca_subs():
    if sub in ['P89']: # parce que il est ok il doit juste avoir un arrondi de baptiste à 2h prêt quant à la date de début / fin d'hospit
        continue
    d0_icca, d1_icca = get_icca_edges_sub(sub)
    d0_hospit, d1_hospit = meta.at[sub, 'entree_rea'], meta.at[sub, 'date_sortie_rea']
    d0_hospit, d1_hospit =  np.datetime64(d0_hospit),  np.datetime64(d1_hospit)
    d0_hospit_r, d1_hospit_r, d0_icca_r, d1_icca_r = d0_hospit.astype('datetime64[D]'),d1_hospit.astype('datetime64[D]'), d0_icca.astype('datetime64[D]'), d1_icca.astype('datetime64[D]')
    if d0_icca_r < d0_hospit_r or d1_icca_r > d1_hospit_r:
        path = icca_path / sub / f'{sub}_ICCA_biology_anonymous.xlsx' 
        pb_subs.append(sub)
        print(sub)
        print('première date hospit:', d0_hospit)
        print('première date ICCA:', d0_icca)
        print('dernière date ICCA:', d1_icca)
        print('dernière date hospit:', d1_hospit)
        print('date modification fichier ICCA :', time.ctime(os.path.getmtime(path)))
        print()


In [ ]:
len(pb_subs)

In [ ]:
pb_subs

# ACSOS vs CSD

ACSOS : 
- hémoglobinémie (-)
- PaCO2 (+ ou -)
- PaO2 (-)
- glycémie (+)
- température (+ ou ---)
- ABP (-)

## 1. CSD Frequency

In [ ]:
patients_with_sds = get_patient_list(stream_selection = ['ECoG'], threshold_N_SDs = 0, threshold_duration_mins=0)
np.array(patients_with_sds)
# np.array(patients_with_sds)

In [ ]:
step = 'day'

threshold_N_SDs = 0
dict_frequency_sd = {}
subs = get_patient_list(stream_selection = ['ECoG'], threshold_N_SDs = threshold_N_SDs, threshold_duration_mins=0)
keep_subs = []
pb_subs = []
for sub in subs:
    reader = pycns.CnsReader(data_path / sub, with_events = True, event_time_zone='Europe/Paris')
    events = pd.DataFrame(reader.events)

    if events.shape[0] != 0:
        events_sd = filter_events(events, pattern  = 'SD', mode = 'with')
        sd_datetimes = events_sd['start_time'].values
    else:
        sd_datetimes = None

    d0, d1 = get_stream_index_datetime_edges(sub)
    duration_stay_s = (d1 - d0).astype(float) / 1e6
    duration_stay_hours = duration_stay_s / 3600
    duration_stay_days = duration_stay_hours / 24
    if duration_stay_days < 1 or sub in pb_subs:
        continue

    times = np.arange(0, duration_stay_days, 1)
    datetimes = np.arange(d0, d1, np.timedelta64(1, 'D'))
    denominator = (3600 * 24)

    count = np.zeros(times.shape)

    if not sd_datetimes is None:
        sd_times_s = ((sd_datetimes - d0).astype(float) / 1e6)
        sd_times_unit = sd_times_s / denominator
        count, _ = np.histogram(sd_times_unit, bins = times)
        count = np.append(count, 0)

    dict_frequency_sd[sub] = {'times' : times , 'datetimes': datetimes,'count' : count}

    # fig, ax = plt.subplots(figsize = (8,3))
    # ax.plot(times, count)
    # ax.scatter(frequency_sd.index, frequency_sd.values, color = 'k')
    # np.floor(events_sd['time_after_start_recording']).value_counts().sort_index().plot.scatter(ax=ax, color = 'k')
    # ax.set_title(f'{sub} - N SD = {n_sd}')
keep_subs = list(dict_frequency_sd.keys())

In [ ]:
len(keep_subs)

In [ ]:
pb_subs

In [ ]:
np.array(keep_subs)

In [ ]:
# s = 'HA1'
s = 'P115'
fig, ax = plt.subplots()
ax.plot(dict_frequency_sd[s]['times'], dict_frequency_sd[s]['count'])
ax.scatter(dict_frequency_sd[s]['times'], dict_frequency_sd[s]['count'])

## 2. ACSOS

ACSOS : 
- hémoglobinémie (-)
- PaCO2 (+ ou -)
- PaO2 (-)
- glycémie (+)
- température (+ ou ---)
- ABP (-)

Par jour ou 12h à discuter
-Valeur de l’Hb -> interpolation linéaire pour les jours sans valeur(Hb<8g/dl ou 9 ou 10 ?)
- Valeur de la PAO2 (seuil 80 ou à 100mmHg)
- Pourcentage de temps SpO2<94%
- Pourcentage dextro <6mmol/l
- Pourcentage dextro <8mmol/l
- Pourcentage de temps PPC<60mmHg
- Pourcentage de temps PPC<70mmHg
- Pourcentage de temps PIC>20mmHg
- Pourcentage de temps PIC>22mmHg
- Pourcentage de temps PIC>25mmHg
- Nombre de CSD


In [ ]:
minimal_days = 1
lowpass_hours = 0.5
minimal_prop_duration_day = 0.5
sr = 1
n_roll = int(sr * 3600 * lowpass_hours)

threshs = {'ICP_Mean':20, 'CPP_Mean':70, 'SpO2':94}

rows = []

subs_icp_art = get_patient_list(['ICP_Mean','ART_Mean','SpO2','ECoG'], threshold_duration_mins=minimal_days * 24 * 60)
subs_icp_abp = get_patient_list(['ICP_Mean','ABP_Mean','SpO2','ECoG'], threshold_duration_mins=minimal_days * 24 * 60)
subs = list(set(subs_icp_art + subs_icp_abp))

# for sub in tqdm(subs[20:30]):
for sub in tqdm(subs):
# for sub in tqdm(['P69']):
    reader = pycns.CnsReader(data_path / sub, with_events= True, event_time_zone='Europe/Paris')
    events = pd.DataFrame(reader.events)
    if events.shape[0] != 0:
        events_sd = filter_events(events, pattern  = 'SD', mode = 'with')
        sd_datetimes = events_sd['start_time'].values
    else:
        sd_datetimes = None

    d0, d1 = get_stream_index_datetime_edges(sub)

    start_dates = np.arange(d0, d1, np.timedelta64(1, "D"))
    stop_dates = np.append(start_dates[1:], d1)

    duration_stay_s = (d1 - d0).astype(float) / 1e6
    duration_stay_hours = duration_stay_s / 3600
    duration_stay_days = duration_stay_hours / 24
    # if duration_stay_days < 1 or sub in pb_subs:
    if duration_stay_days < 1 :
        continue
    
    abp_stream_name = get_abp_stream_name_from_reader(reader)
    stream_names = ['ICP_Mean',f'{abp_stream_name}_Mean','SpO2']
    ds = reader.export_to_xarray(stream_names, resample = True, sample_rate=sr)
    
    ds['ICP_Mean'] = ds['ICP_Mean'].rolling(times=n_roll, center = True).median()
    ds[f'{abp_stream_name}_Mean'] = ds[f'{abp_stream_name}_Mean'].rolling(times=n_roll, center = True).median('times')
    ds['CPP_Mean'] = ds[f'{abp_stream_name}_Mean'] - ds['ICP_Mean']

    days = range(start_dates.size)
    for day, start_date, stop_date in zip(days, start_dates, stop_dates):
        n_sd = 0 if sd_datetimes is None else sum( (sd_datetimes > start_date) & (sd_datetimes < stop_date) )
        local_spO2 = ds['SpO2'].loc[start_date:stop_date].dropna(dim = 'times').values
        local_cpp = ds['CPP_Mean'].loc[start_date:stop_date].dropna(dim = 'times').values
        local_icp = ds['ICP_Mean'].loc[start_date:stop_date].dropna(dim = 'times').values
        percent_spo2 = sum(local_spO2 < threshs['SpO2']) / local_spO2.size * 100 if local_spO2.size > minimal_prop_duration_day * 3600 * 24 * sr else np.nan
        percent_cpp = sum(local_cpp < threshs['CPP_Mean']) / local_cpp.size * 100 if local_cpp.size > minimal_prop_duration_day * 3600 * 24 * sr else np.nan
        percent_icp = sum(local_icp > threshs['ICP_Mean']) / local_icp.size * 100 if local_icp.size > minimal_prop_duration_day * 3600 * 24 * sr else np.nan

        row = dict(sub = sub, day = day, n_sd = n_sd, spo2 = percent_spo2, icp = percent_icp, cpp = percent_cpp)
        rows.append(row)
res = pd.DataFrame(rows)
res = res.dropna(subset=['spo2','icp','cpp'], how =  'all')


In [ ]:
# res.to_excel('./df_for_stats/acsos_sd.xlsx')

In [ ]:
res

In [ ]:
res['cpp'].plot.hist()

# CO2 example for physio

In [ ]:
get_patient_list(['CO2'])

In [ ]:
sub = 'P11'
reader = pycns.CnsReader(data_path / sub)
co2_stream = reader.streams['CO2']
srate = co2_stream.sample_rate
raw_co2, times = co2_stream.get_data(with_times = True, apply_gain = True, time_as_second=True)

In [ ]:
co2_stream.index

In [ ]:
raw_co2.shape

In [ ]:
times.shape

In [ ]:
srate

In [ ]:
t0 = 1000
t1 = 1300

In [ ]:
mask = (times >= t0) & (times < t1)
raw_co2_sel = raw_co2[mask]
times_sel = times[mask]

In [ ]:
raw_co2_sel.shape

In [ ]:
%matplotlib widget

In [ ]:
fig, ax = plt.subplots()
ax.plot(times_sel, raw_co2_sel)

In [ ]:
# np.save("/crnldata/cmo/users/ValentinGhibaudo/physiotools/physio/examples/resp_CO2_4.npy", raw_co2_sel)

# BaroR in patients

## 
Method (Nevrokard, Izola, Slovenia) in which 
- sequences of three or more consecutive beats 
- where SBP and RRI change in the same direction were identified. 
- Sequences were detected using a zero-beat delay
- R2 for inclusion set at 0.85
- and using 1-mmHg and 5-ms thresholds for SBP and RRI, respectively, as previously described

In [ ]:
def _ensure_interleave(ind0, ind1, remove_first=True):
    """
    Clean ind0 so they are interleaved into ind1.
    """
    keep_inds = np.searchsorted(ind1, ind0,  side='right')
    keep = np.ones(ind0.size, dtype=bool)
    ind_bad = np.flatnonzero(np.diff(keep_inds) == 0)
    if remove_first:
        keep[ind_bad] = False
    else:
        keep[ind_bad + 1] = False
    ind0_clean = ind0[keep]
    return ind0_clean

In [ ]:
def load_ecg_abp_df(sub):
    ecg_peaks = detect_ecg_job.get(sub).to_dataframe()
    abp_peaks = detect_abp_job.get(sub).to_dataframe()
    ecg_peaks['rri_s'] = ecg_peaks['peak_time'].diff()
    ecg_peaks['rri_ms'] = ecg_peaks['rri_s']  * 1000
    ecg_peaks['rri_cs'] = ecg_peaks['rri_s']  * 100
    ecg_peaks['bpm'] = 60 / ecg_peaks['rri_s']
    ecg_peaks =  ecg_peaks.dropna()

    min_time = max([min(ecg_peaks['peak_time']),min(abp_peaks['peak_time'])])
    max_time = min([max(ecg_peaks['peak_time']),max(abp_peaks['peak_time'])])

    ecg_peaks_crop = ecg_peaks[(ecg_peaks['peak_time'] > min_time) & (ecg_peaks['peak_time'] < max_time)]
    abp_peaks_crop = abp_peaks[(abp_peaks['peak_time'] > min_time) & (abp_peaks['peak_time'] < max_time)]

    ecg_sel_times = _ensure_interleave(ecg_peaks_crop['peak_time'].values, abp_peaks_crop['peak_time'].values)
    abp_sel_times = _ensure_interleave(abp_peaks_crop['peak_time'].values, ecg_peaks_crop['peak_time'].values)

    ecg_peaks_crop2 = ecg_peaks_crop.set_index('peak_time').loc[ecg_sel_times].reset_index()#.iloc[:-1,:]
    abp_peaks_crop2 = abp_peaks_crop.set_index('peak_time').loc[abp_sel_times].reset_index()

    ecg_peaks_crop2 = ecg_peaks_crop2.rename(columns = {'peak_time':'r_peak_time'})
    abp_peaks_crop2 = abp_peaks_crop2.rename(columns = {'peak_time':'pulse_peak_time'})
    return pd.concat([ecg_peaks_crop2, abp_peaks_crop2], axis = 1)

In [ ]:
get_metadata()

In [ ]:
sub = 'P1'
df = load_ecg_abp_df(sub)

In [ ]:
(df['rri_ms'].diff() > 5).value_counts(normalize = True)

In [ ]:
(df['amplitude_at_peak'].diff() > 1).value_counts(normalize = True)

In [ ]:
((df['rri_ms'].diff() > 5) & (df['amplitude_at_peak'].diff() > 1)).value_counts(normalize = True)

In [ ]:
mask = ((df['rri_ms'].diff() > 5) & (df['amplitude_at_peak'].diff() > 1))

In [ ]:
fig, ax = plt.subplots()
ax.scatter(df[mask]['amplitude_at_peak'], df[mask]['rri_ms'], alpha = 0.1)
ax.set_ylim(400,1200)

In [ ]:
colname_ecg = 'rri_ms'
ecg_lim = (400, 2000)
fig, ax = plt.subplots()
ax.plot(df['r_peak_time'], df[colname_ecg])
ax.set_ylabel(colname_ecg)
ax.set_ylim(ecg_lim)
# ax.plot(df['r_peak_time'], df['bpm'])
ax2 = ax.twinx()
ax2.plot(df['pulse_peak_time'], df['amplitude_at_peak'], color = 'darkorange')
ax2.set_ylabel('PAS')
plt.show()

In [ ]:
def compute_baroR(df, **p):
    win_s = p['win_size_analysis_sec']
    n_roll = p['n_roll_smooth_sigs']
    n_roll_slope = p['n_roll_slope_baroR']
    
    m = 'rri_ms'
    
    df2= df.copy()
    df2['amplitude_at_peak'] = df2['amplitude_at_peak'].rolling(n_roll, center=True).median()
    df2[m] = df[m].rolling(n_roll, center=True).median()


    starts = np.arange(df['r_peak_time'].values[0], df['r_peak_time'].values[-1], win_s)

    stats = pd.DataFrame(columns = ['t','a','r','p','N'], dtype = float).set_index('t')

    for start in starts:
        stop = start + win_s
        mask = (df2['r_peak_time'] >= start) & (df2['r_peak_time'] <= stop)

        if sum(mask) == 0:
            continue

        x = df2['amplitude_at_peak'].values[mask]
        y = df2[m].values[mask]

        res = scipy.stats.linregress(df2[mask]['amplitude_at_peak'].values, df2[mask]['rri_ms'].values)
        stats.at[start , 'slope'] = res.slope
        stats.at[start , 'r'] = res.rvalue
        stats.at[start , 'r2'] = res.rvalue ** 2
        stats.at[start , 'p'] = res.pvalue
        stats.at[start, 'N'] = y.size

    stats['slope'] = stats['slope'].rolling(n_roll_slope, center = True).median()
    med_slope = stats['slope'].median()
    return stats, med_slope

    


In [ ]:
p = {
    'win_size_analysis_sec':600,
    'n_roll_smooth_sigs':30,
    'n_roll_slope_baroR':3,
}

stats, med_slope = compute_baroR(df, **p)

In [ ]:
(stats['r2'] > 0.85).value_counts()

In [ ]:
stats[stats['r2'] >= 0.85]['slope']

In [ ]:
%matplotlib inline

In [ ]:
alpha = 0.8

fig, ax = plt.subplots(figsize = (11,4))
ax.plot(df['r_peak_time'], df['rri_cs'], label = 'bpm')
ax.plot(df['r_peak_time'], df['amplitude_at_peak'], label = 'pas')
ax.legend()
ax2 = ax.twinx()
# ax2.plot(stats.index, stats['a'], color = 'g', alpha = alpha)
ax2.plot(stats.index, stats['slope'].rolling(6).median(), color = 'g', alpha = alpha)
ax2.plot(stats.index, stats['r2'], color = 'y', alpha = alpha)
# ax3 = ax.twinx()
# ax3.plot(stats.index, stats['r'], color = 'y', alpha = alpha)
# ax3.axhline(-1, color = 'y')
# ax3.axhline(1, color = 'y')

# ax2.set_ylim()
ax.set_ylim(0, 200)
ax2.set_ylim(-6, 15)
# ax3.set_ylim(0, 100)

In [ ]:
threshold_r2 = 0.85
bins = np.arange(-5,10,0.25)
p = {
    'win_size_analysis_sec':600,
    'n_roll_smooth_sigs':30,
    'n_roll_slope_baroR':1,
}

subs_abp = get_patient_list(['ECG_II','ABP'])
subs_art = get_patient_list(['ECG_II','ART'])
subs = list(set(subs_abp + subs_art))

for sub in subs:
    try:
        df = load_ecg_abp_df(sub)
        stats, _ = compute_baroR(df, **p)
        stats = stats[stats['r2'] >= threshold_r2]
        med_slope = stats['slope'].median()
        fig, ax = plt.subplots(figsize = (4,3))
        # ax.hist(stats['slope'], bins=bins)
        ax.hist(stats['slope'])
        ax.set_title(f'{sub} - Median slope : {round(med_slope, 3)} ms / mmHg')
        ax.set_xlabel('Slope (ms/mmHg)')
        ax.set_ylabel('Count')
        plt.show()
    except:
        print(sub)
        continue

In [ ]:
fig, ax = plt.subplots()
ax.hist(stats['a'], bins = np.arange(-5,12,0.2))
plt.show()

In [ ]:
stats['a'].median()

In [ ]:
stats['a'].rolling(6).median().median()

In [ ]:
fig, ax = plt.subplots()
ax.hist(stats['a'].rolling(6).median(), bins = np.arange(-5,12,0.2))
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.hist(stats['r'], bins = 100)
plt.show()

In [ ]:
start = 50070
stop = 50124
mask = (df['r_peak_time'] >= start) & (df['r_peak_time'] <= stop)

# if sum(mask) == 0:
#     continue

x = df[mask]['amplitude_at_peak'].values
y = df[mask]['bpm'].values

res = scipy.stats.linregress(x, y)
a = res.slope
r = res.rvalue

fig, ax = plt.subplots()
ax.scatter(x, y, alpha = 0.5)

# ax.set_ylim(0,100)
# ax.set_xlim(40,300)
ax.set_xlabel('Systolic Blood Pressure\n(mmHg)')
# ax.set_ylabel('R-R Interval\n(ms)')
ax.set_ylabel('IHR\n(bpm)')
ax.set_title(f'{round(start, 3)} - {round(stop, 3)} - a : {round(a, 3)} - r : {round(r, 3)}')

In [ ]:
start = 50070
stop = 50124
mask = (df['r_peak_time'] >= start) & (df['r_peak_time'] <= stop)

# if sum(mask) == 0:
#     continue

x = df[mask]['amplitude_at_peak'].values
y = df[mask]['rri_ms'].values

res = scipy.stats.linregress(x, y)
a = res.slope
r = res.rvalue
p = res.pvalue
predicted = res.intercept + x * res.slope

fig, ax = plt.subplots()
ax.scatter(x, y, alpha = 0.5)
ax.plot(x, predicted, alpha = 1, color = 'r')

# ax.set_ylim(0,100)
# ax.set_xlim(40,300)
ax.set_xlabel('Systolic Blood Pressure\n(mmHg)')
# ax.set_ylabel('R-R Interval\n(ms)')
ax.set_ylabel('RRi\n(ms)')
ax.set_title(f'{round(start, 3)} - {round(stop, 3)} - a : {round(a, 3)} - r : {round(r, 3)}')

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import IntSlider, HBox, VBox, Button, Output, Layout
import scipy.stats

# Données
n_roll = 30
t = df['r_peak_time'].values   # temps en secondes
rr = df['rri_ms'].rolling(n_roll, center = True).mean().values       # RRi (ms)
# pas = df['amplitude_at_peak'].values  # SBP (mmHg)
pas = df['amplitude_at_peak'].rolling(n_roll, center = True).mean().values  # SBP (mmHg)

# Paramètres de la fenêtre
# duree_fenetre = 60   # durée de la fenêtre d'analyse (s)
# margin = 600           # marge de contexte (s)
duree_fenetre = 600   # durée de la fenêtre d'analyse (s)
margin = 3600           # marge de contexte (s)

# Zone d'affichage pour la figure
out = Output()

# Slider en secondes
slider = IntSlider(
    min=int(t[0]),
    max=int(t[-1] - duree_fenetre),
    step=1,
    value=int(t[0]),
    description="Start (s)",
    layout=Layout(width='1000px')
)

def plot_baroreflex(start_time):
    end_time = start_time + duree_fenetre
    if end_time > t[-1]:
        end_time = t[-1]

    # Masques temporels
    mask_context = (t >= start_time - margin) & (t <= end_time + margin)
    mask_analysis = (t >= start_time) & (t <= end_time)

    # Données avec marge
    t_win = t[mask_context]
    rr_win = rr[mask_context]
    pas_win = pas[mask_context]

    # Données strictes pour analyse
    t_win_analysis = t[mask_analysis]
    rr_win_analysis = rr[mask_analysis]
    pas_win_analysis = pas[mask_analysis]

    # Régression linéaire
    slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(
        pas_win_analysis, rr_win_analysis
    )
    y_pred = intercept + slope * pas_win_analysis

    # Affichage
    with out:
        out.clear_output(wait=True)
        fig, axs = plt.subplots(ncols=2, figsize=(12, 4), sharex=False)

        # Subplot 1 : séries temporelles avec contexte
        ax = axs[0]
        ax.plot(t_win, rr_win, label="RRi (ms)", color='tab:blue')
        ax.axvspan(start_time, end_time, color='grey', alpha=0.3)
        ax.set_ylabel("RRi (ms)")
        ax.set_xlabel('Time (s)')
        ax.set_title(f"Séries temporelles RRi & SBP ({duree_fenetre}s)")

        ax2 = ax.twinx()
        ax2.plot(t_win, pas_win, label="SBP (mmHg)", color='tab:red')
        ax2.set_ylabel("SBP (mmHg)")

        # Subplot 2 : scatter + droite de regression
        ax = axs[1]
        ax.scatter(pas_win_analysis, rr_win_analysis, alpha=0.7, color='k')
        ax.plot(pas_win_analysis, y_pred, color='red')
        ax.set_xlabel("SBP (mmHg)")
        ax.set_ylabel("RRi (ms)")
        ax.set_title(f"BaroR (N = {y_pred.size})\n(Slope = {slope:.2f} ms/mmHg) (p : {round(p_value, 2)})")

        plt.tight_layout()
        plt.show()

# Callback du slider
def update_plot(change):
    plot_baroreflex(slider.value)

slider.observe(update_plot, names='value')

# Boutons navigation
btn_left = Button(description="◀️", layout=widgets.Layout(width="50px"))
btn_right = Button(description="▶️", layout=widgets.Layout(width="50px"))

def on_left(b):
    if slider.value - duree_fenetre >= slider.min:
        slider.value -= duree_fenetre

def on_right(b):
    if slider.value + duree_fenetre <= slider.max:
        slider.value += duree_fenetre

btn_left.on_click(on_left)
btn_right.on_click(on_right)

# Contrôles + affichage
controls = HBox([btn_left, slider, btn_right])

# Premier affichage
plot_baroreflex(slider.value)

display(VBox([controls, out]))


In [ ]:
n_roll = 3
win_s = 300

ecg_peaks_crop2 = ecg_peaks_crop.set_index('peak_time').loc[ecg_sel_times].reset_index().iloc[:-1,:]
abp_peaks_crop2 = abp_peaks_crop.set_index('peak_time').loc[abp_sel_times].reset_index()
abp_peaks_crop2['amplitude_at_peak'] = abp_peaks_crop2['amplitude_at_peak'].rolling(n_roll, center=True).median()
ecg_peaks_crop2['rri'] = ecg_peaks_crop2['rri'].rolling(n_roll, center=True).median()

starts = np.arange(abp_peaks_crop2['peak_time'].values[0], abp_peaks_crop2['peak_time'].values[-1], win_s)
stats = pd.DataFrame(columns = ['t','a','r'], dtype = float).set_index('t')

# for start in starts[:100]:
for start in tqdm(starts):
    stop = start + win_s
    mask = (abp_peaks_crop2['peak_time'] >= start) & (abp_peaks_crop2['peak_time'] <= stop)

    if sum(mask) == 0:
        continue

    x = abp_peaks_crop2[mask]['amplitude_at_peak'].values
    y = ecg_peaks_crop2[mask]['rri'].values

    try:
        res = scipy.stats.linregress(abp_peaks_crop2[mask]['amplitude_at_peak'].values, ecg_peaks_crop2[mask]['rri'].values)
        a = res.slope
        r = res.rvalue

        stats.at[start , 'a'] = a
        stats.at[start , 'r'] = r
    except:
        continue

fig, ax = plt.subplots()
ax.plot(ecg_peaks_crop2['peak_time'], ecg_peaks_crop2['rri'])
ax.plot(ecg_peaks_crop2['peak_time'], abp_peaks_crop2['amplitude_at_peak'])
ax2 = ax.twinx()
ax2.plot(stats.index, stats['a'], label = 'a', color = 'g')
ax3 = ax.twinx()
ax3.plot(stats.index, stats['r'], label = 'r', color = 'y')
ax3.axhline(-1, color = 'y')
ax3.axhline(1, color = 'y')

# ax2.set_ylim()
ax.set_ylim(0, 200)
ax2.set_ylim(-3, 3)
# ax3.set_ylim(0, 100)

In [ ]:
start = 305200
stop = start + win_s
mask = (abp_peaks_crop2['peak_time'] >= start) & (abp_peaks_crop2['peak_time'] <= stop)

# if sum(mask) == 0:
#     continue

x = abp_peaks_crop2[mask]['amplitude_at_peak'].values
y = ecg_peaks_crop2[mask]['rri'].values

res = scipy.stats.linregress(abp_peaks_crop2[mask]['amplitude_at_peak'].values, ecg_peaks_crop2[mask]['rri'].values)
a = res.slope
r = res.rvalue

stats.at[start , 'a'] = a
stats.at[start , 'r'] = r
fig, ax = plt.subplots()
ax.scatter(x, y * 10, alpha = 0.5)

ax.set_ylim(0,2000)
ax.set_xlim(40,300)
ax.set_xlabel('Systolic Blood Pressure\n(mmHg)')
ax.set_ylabel('R-R Interval\n(ms)')
ax.set_title(f'{round(start, 3)} - {round(stop, 3)} - a : {round(a, 3)} - r : {round(r, 3)}')

# Glasgow vs PSE

In [ ]:
import pingouin as pg

In [ ]:
def borne_value(x, c):
    return x / (x + c)

In [ ]:
win_search_margin_h = 2
c = 5
meta = get_metadata().set_index('ID_pseudo')

for sub in get_patient_list_icca_treatments('pse')[:]:
    if sub in ['P64']:
        continue
    pse_ds = icca_pse_tt_job.get(sub)
    tt_names = list(pse_ds.keys())

    gg_name = 'Glasgow, total'
    gg_da = icca_clinical_job.get(sub)[gg_name]
    if np.std(gg_da.values) == 0:
        print(sub, 'glasgow ne varie pas')
        continue
    gg_datetimes = gg_da[f'datetime_{gg_name}'].values

    entree_rea = meta.loc[sub,'entree_rea']
    days_post_entree  = (gg_datetimes - np.datetime64(entree_rea)).astype(float) / 1e9 / (3600 * 24)

    df = pd.DataFrame(index = gg_datetimes, columns = ['gg'] + tt_names)
    df.loc[:,'gg'] = gg_da

    for tt_name in tt_names:
        tt_da = pse_ds[tt_name]
        tt_da = tt_da.rename({f'datetime_{tt_name}':'datetime_tt'})

        for d in gg_datetimes:
            d0 = d - np.timedelta64(win_search_margin_h, 'h')
            d1 = d + np.timedelta64(win_search_margin_h, 'h')
            if tt_da.loc[d0:d1].size == 0:
                df.loc[d,tt_name] = 0
            else:
                # df.loc[d,tt_name] = borne_value(float(tt_da.loc[d0:d1].mean('datetime_tt')), c)
                df.loc[d,tt_name] = float(tt_da.loc[d0:d1].mean('datetime_tt'))
    df['time'] = days_post_entree
    df = df.astype(float)
    df.iloc[:,1:] = (df.iloc[:,1:] - df.iloc[:,1:].mean(axis = 0)) / df.iloc[:,1:].std(axis = 0) 
    df = df.dropna(axis = 1)

    res = pg.linear_regression(X = df.loc[:,[c for c in df.columns if not c == 'gg']], y =  df['gg'], remove_na = True, add_intercept=True, coef_only=False)
    res['significant'] = res['pval'] < 0.05
    r2 = res['r2'].values[0]
    res = res.set_index('names')

    fig, ax = plt.subplots(figsize = (8,4))
    # ax.scatter(df['gg'], borne_value(days_post_entree, c), label = 'days post entree', alpha = 0.4)
    # ax2 = ax.twiny()
    # ax2.scatter(df['gg'], df['time'], alpha = 0.1, color = 'k')
    # ax2.scatter(df['time'], df['gg'], alpha = 0.1, color = 'k')
    for tt_name in df.columns[::-1]:
        if tt_name == 'gg':
            continue
        # ax.scatter(df['gg'], df[tt_name], label = tt_name, alpha = 0.4)
        # ax.set_xlabel('Glasgow')
        # ax.set_ylabel('Scaled dose of PSE tt')
        label = '{tt_name} (a={a} , p={p})'.format(tt_name=tt_name, a = res.loc[tt_name,'coef'].round(2), p = res.loc[tt_name,'pval'].round(2))
        ax.scatter(df[tt_name], df['gg'], label = label, alpha = 0.4)
        ax.set_ylabel('Glasgow')
        ax.set_xlabel('Scaled dose of PSE tt')
    
    ax.set_title("{sub} - r2 = {r2}\n{res}".format(sub=sub, r2 = round(r2, 3), res = res.index[res['significant']].values))
    ax.legend(loc = 'upper right', fontsize = 7)
    ax.set_ylim(2, 16)
    plt.show()

In [ ]:
sub = 'P17'

pse = icca_pse_tt_job.get(sub)['Urapidil']
gg = icca_clinical_job.get(sub)['Glasgow, total']

fig , ax = plt.subplots()
ax.plot(gg['datetime_Glasgow, total'], gg.values)
ax.twinx().plot(pse['datetime_Urapidil'], pse.values, color = 'darkorange')

# HRV / RSA vs type de lésion

In [ ]:
rows = []
for sub in get_patient_list(['CO2','ECG_II']):
    rsa_sub = rsa_job.get(sub).to_dataframe()
    rsa_sub = rsa_sub['rsa_amplitude_clean'].describe().to_dict()
    row = dict(sub=sub) | rsa_sub
    rows.append(row)
df_all_rsa = pd.DataFrame(rows)


In [ ]:
df_all_rsa.dropna().set_index('sub').sort_values(by = '50%')

In [ ]:
df_all_rsa2 = df_all_rsa.dropna()
df_all_rsa2 = df_all_rsa2[~df_all_rsa2['sub'].isin(['P19'])]
# motif = get_metadata().set_index('ID_pseudo').loc[df_all_rsa2['sub'].unique(),'mRS_sortie'].to_dict()
motif = get_metadata().set_index('ID_pseudo').loc[df_all_rsa2['sub'].unique(),'motif'].to_dict()
df_all_rsa2['motif'] = df_all_rsa2['sub'].map(motif)

In [ ]:
fig, ax = plt.subplots()
sns.stripplot(data = df_all_rsa2,
            x = 'motif',
            y = '50%',
            ax=ax)
ax.set_title('Med RSA vs Motif')
ax.set_ylabel('RSA mediane (bpm)')

In [ ]:
df_all_rsa.dropna()['50%'].plot.hist(bins = np.arange(0, 8, 0.2))

In [ ]:
df_all_rsa.dropna()['75%'].plot.hist(bins = np.arange(0, 8, 0.2))

In [ ]:
df_all_rsa.dropna()['max'].plot.hist(bins = np.arange(0, 8, 0.2))

In [ ]:
meta = get_metadata().set_index('ID_pseudo')
rows = []
for sub in get_patient_list(['ECG_II']):
    if sub in ['P19','P9']:
        continue
    ecg_peaks = detect_ecg_job.get(sub).to_dataframe()
    metrics = physio.compute_ecg_metrics(ecg_peaks, min_interval_ms=400)
    row = dict(sub=sub, motif = meta.loc[sub, 'motif']) | metrics.to_dict()
    rows.append(row)
df = pd.DataFrame(rows)
df = df.dropna().reset_index(drop = True)

In [ ]:
df.columns

In [ ]:
outcomes = ['HRV_Mean', 'HRV_SD', 'HRV_Median', 'HRV_Mad', 'HRV_CV', 'HRV_MCV', 'HRV_RMSSD']


nrows = 2
ncols = 4

poss = attribute_subplots(outcomes, nrows, ncols)

fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 3, nrows * 3), constrained_layout = True)

for outcome, pos in poss.items():
    ax = axs[*pos]

    sns.stripplot(data = df,
                    x = 'motif',
                    y = outcome,
                    ax=ax)

# Ventilation effect on HRV

In [ ]:
# subs = get_patient_list(['CO2'])
# remove_sub = ['P90','P86','P16','P2','P41','SP2','P37','DV4','P10','NY15',
#     'P43','P42','P67','P76','P65','JL5','P20','P73','P6','P4','HA1',
#     'P87','P79','P57','P96','P89','P39','P21','P98','P14',
#     'P13','P66','P68','JR10'
#     ]
# for sub in subs[60:]:
#     if sub in remove_sub:
#         continue
#     resp_cycles = detect_resp_job.get(sub).to_dataframe()
#     resp_cycles['is_ventilation_controlled'] = resp_cycles['is_ventilation_controlled'].astype(bool)

#     d0_resp = resp_cycles['inspi_date'].values[0]
#     d1_resp = resp_cycles['inspi_date'].values[-1]

#     win_duration_h = 2

#     start_dates = np.arange(d0_resp, d1_resp, np.timedelta64(win_duration_h, 'h'))

#     possible_starts_machine_trig = []
#     possible_starts_patient_trig = []

#     for d0 in start_dates:
#         d1 = d0 + np.timedelta64(win_duration_h, 'h')
#         local_resp_cycles = resp_cycles[(resp_cycles['inspi_date'] > d0) & (resp_cycles['next_inspi_date'] < d1)]
#         if np.all(local_resp_cycles['is_ventilation_controlled'].values):
#             possible_starts_machine_trig.append(d0)
#         elif np.all(~local_resp_cycles['is_ventilation_controlled'].values):
#             possible_starts_patient_trig.append(d0)

#     if len(possible_starts_machine_trig) == 0 or len(possible_starts_patient_trig) == 0:
#         print(sub)
#         continue

#     d0_study_machine_trigg = possible_starts_machine_trig[0] + np.timedelta64(1, 'h')
#     d0_study_patient_trigg = possible_starts_patient_trig[0] + np.timedelta64(1, 'h')

#     figsize = (12,4)
#     fig, ax = plt.subplots(figsize = figsize)
#     ax.plot(resp_cycles['inspi_date'], resp_cycles['is_ventilation_controlled'])
#     for d in possible_starts_machine_trig:
#         ax.axvspan(d, d+np.timedelta64(win_duration_h , 'h'), color = 'r', alpha = 0.2)
#     for d in possible_starts_patient_trig:
#         ax.axvspan(d, d+np.timedelta64(win_duration_h , 'h'), color = 'g', alpha = 0.2)
#     ax.axvspan(d0_study_machine_trigg, d0_study_machine_trigg+np.timedelta64(1 , 'h'), color = 'r', alpha = 0.8)
#     ax.axvspan(d0_study_patient_trigg, d0_study_patient_trigg+np.timedelta64(1 , 'h'), color = 'g', alpha = 0.8)
#     ax.set_title(sub)

In [ ]:
def get_respi_trig_analysis_dates():
    subs = get_patient_list(['CO2'])
    remove_subs = ['P90','P86','P16','P2','P41','SP2','P37','DV4','P10','NY15',
        'P43','P42','P67','P76','P65','JL5','P20','P73','P6','P4','HA1',
        'P87','P79','P57','P96','P89','P39','P21','P98','P14',
        'P13','P66','P68','JR10'
        ]
    subs = [s for s in subs if not s in remove_subs]
    
    analysis_dates = pd.DataFrame(index = subs, columns = ['machine_trig','patient_trig'])
    for sub in subs:
        resp_cycles = detect_resp_job.get(sub).to_dataframe()
        resp_cycles['is_ventilation_controlled'] = resp_cycles['is_ventilation_controlled'].astype(bool)

        d0_resp = resp_cycles['inspi_date'].values[0]
        d1_resp = resp_cycles['inspi_date'].values[-1]

        win_duration_h = 2

        start_dates = np.arange(d0_resp, d1_resp, np.timedelta64(win_duration_h, 'h'))

        possible_starts_machine_trig = []
        possible_starts_patient_trig = []

        for d0 in start_dates:
            d1 = d0 + np.timedelta64(win_duration_h, 'h')
            local_resp_cycles = resp_cycles[(resp_cycles['inspi_date'] > d0) & (resp_cycles['next_inspi_date'] < d1)]
            if np.all(local_resp_cycles['is_ventilation_controlled'].values):
                possible_starts_machine_trig.append(d0)
            elif np.all(~local_resp_cycles['is_ventilation_controlled'].values):
                possible_starts_patient_trig.append(d0)

        if len(possible_starts_machine_trig) == 0 or len(possible_starts_patient_trig) == 0:
            print(sub)
            continue

        d0_study_machine_trigg = possible_starts_machine_trig[0] + np.timedelta64(1, 'h')
        d0_study_patient_trigg = possible_starts_patient_trig[0] + np.timedelta64(1, 'h')

        analysis_dates.loc[sub,'machine_trig'] = d0_study_machine_trigg
        analysis_dates.loc[sub,'patient_trig'] = d0_study_patient_trigg
    return analysis_dates

In [ ]:
analysis_dates = get_respi_trig_analysis_dates()
analysis_dates

In [ ]:
help(get_icca_subs)

In [ ]:
help(get_patient_list_icca_treatments)

In [ ]:
get_patient_list_icca_treatments('pse')

In [ ]:
def get_hrv_time_metrics(ecg_peaks, d0, d1):
    local_ecg_peaks = ecg_peaks[(ecg_peaks['peak_date'] > d0) & (ecg_peaks['peak_date'] < d1)]
    rri_bpm = 60 / np.diff(local_ecg_peaks['peak_time'].values)
    med_rri, mad_rri = compute_nanmedian_mad(rri_bpm)
    hrv_time = dict(med_rri=med_rri, mad_rri=mad_rri)
    return hrv_time

def get_hrv_freq_metrics(da_hrv_freq, d0, d1):
    local_da = da_hrv_freq.loc[:,d0:d1]
    total = local_da.loc[['vlf','lf','hf'],:].sum('band')
    d = {}
    for band in ['vlf','lf','hf']:
        d[band] = np.log(float(local_da.loc[band,:].median('datetime')))
        d[f'{band}n'] = float( (local_da.loc[band,:] / local_da.loc['total',:]).median('datetime'))
    d['total'] = np.log(float(total.median('datetime')))
    d['lf_hf'] = float((local_da.loc['lf',:] / local_da.loc['hf',:]).median('datetime'))
    return d

def get_hrv_med_freq(da_hrv_med_freq, d0, d1):
    return dict(hrv_med_freq = float(da_hrv_med_freq.loc['med_freq',d0:d1].median('datetime')))

def get_hrv_slope(da_hrv_slope, d0, d1):
    return dict(hrv_slope = float(da_hrv_slope.loc[d0:d1].median('datetime')))

def get_rsa(rsa_df, d0, d1):    
    local_rsa = rsa_df[(rsa_df['cycle_date'] > d0) & (rsa_df['cycle_date'] < d1)]['decay_amplitude'].values
    med_rsa = np.nanmedian(local_rsa)
    return dict(med_rsa=med_rsa)

def get_pse(pse_ds, d0, d1):
    tt_names = ['Cisatracurium','Midazolam','Norépinéphrine','Sufentanil','Propofol','Thiopenthal','Urapidil','Rémifentanil']
    d = {}
    if not pse_ds is None:
        possible_tt = list(pse_ds.keys())
        for pse_name in tt_names:
            if not pse_name in possible_tt:
                d[pse_name] = 0
            else:
                try:
                    local_tt = pse_ds[pse_name].loc[d0:d1]
                    if local_tt.size == 0:
                        d[pse_name] = 0
                    else:
                        d[pse_name] = float(local_tt.median(f'datetime_{pse_name}'))
                except:
                    d[pse_name] = 0
    else:
        d = {pse_name:0 for pse_name in tt_names}
    return d

def get_glasgow(da_glasgow, d0, d1):
    if not da_glasgow is None:
        local_gg = da_glasgow.loc[d0:d1]
        if local_gg.size == 0:
            return dict(glasgow = np.nan)
        else:
            return dict(glasgow = float(local_gg.median('datetime_Glasgow, total')))
    else:
        return dict(glasgow = np.nan)

In [ ]:
win_size_analyis_h = 1

rows = []
for sub in analysis_dates.index:
    # print(sub)
    try:
        meta = get_metadata_sub(sub)
    except:
        print(sub)
        continue
    ecg_peaks = detect_ecg_job.get(sub).to_dataframe()
    rsa_df = rsa_job.get(sub).to_dataframe()
    da_hrv_freq = hrv_freq_job.get(sub)['hrv_freq']
    da_hrv_med_freq = hr_center_frequency_job.get(sub)['hr_center_frequency']
    da_hrv_slope = hr_spectrum_slope_job.get(sub)['hr_spectrum_slope']
    try:
        da_glasgow = icca_clinical_job.get(sub)['Glasgow, total']
    except:
        da_glasgow = None
    try:
        pse_ds = icca_pse_tt_job.get(sub)
    except:
        pse_ds = None
    

    for trig_type in analysis_dates.columns:
        d0 = analysis_dates.loc[sub,trig_type]
        d1 = d0 + np.timedelta64(win_size_analyis_h, 'h') 
        
        hrv_time = get_hrv_time_metrics(ecg_peaks, d0, d1)
        hrv_freq = get_hrv_freq_metrics(da_hrv_freq, d0, d1)
        hrv_med_freq = get_hrv_med_freq(da_hrv_med_freq, d0, d1)
        hrv_slope = get_hrv_slope(da_hrv_slope, d0, d1)
        rsa = get_rsa(rsa_df, d0, d1)
        pse = get_pse(pse_ds, d0, d1)
        gg = get_glasgow(da_glasgow, d0, d1)

        meta['trig_type'] = trig_type
        row = meta | hrv_time | hrv_freq | hrv_med_freq | hrv_slope | rsa | pse | gg
        rows.append(row)

    # except:
    #     print(sub)
    #     continue

res = pd.DataFrame(rows)


    

In [ ]:
res

In [ ]:
outcomes = list(res.columns[10:])
nrows = 5
ncols = 5
poss = attribute_subplots(outcomes, nrows, ncols)

fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 3, nrows * 3), constrained_layout = True)

for outcome, pos in poss.items():
    ax = axs[*pos]
    res_stats = res.dropna(subset = ['patient_id','trig_type',outcome])
    mask_subs = res_stats['patient_id'].value_counts() == 2
    res_stats = res_stats[res_stats['patient_id'].isin(mask_subs.index[mask_subs])]
    auto_stats(df = res_stats,
               predictor = 'trig_type',
               outcome = outcome,
               design = 'within',
               subject = 'patient_id',
               ax=ax
               )

In [ ]:
pd.Series(data = res[res['trig_type'] == 'patient_trig']['med_rsa'].values - res[res['trig_type'] == 'machine_trig']['med_rsa'].values, index = res[res['trig_type'] == 'patient_trig']['patient_id'].values ).dropna().sort_values(ascending = False)

In [ ]:
res[res['patient_id'] == 'P60']

In [ ]:
res.set_index('patient_id')['med_rsa'].dropna().sort_values()

# HRV / RSA vs pronostic clinique

In [ ]:
rows = []
for sub in get_patient_list(['CO2','ECG_II']):
    rsa_sub = rsa_job.get(sub).to_dataframe()
    rsa_sub = rsa_sub['rsa_amplitude_clean'].describe().to_dict()
    row = dict(sub=sub) | rsa_sub
    rows.append(row)
df_all_rsa = pd.DataFrame(rows)


In [ ]:
df_all_rsa.dropna().set_index('sub').sort_values(by = '50%')

In [ ]:
df_all_rsa2 = df_all_rsa.dropna()
df_all_rsa2 = df_all_rsa2[~df_all_rsa2['sub'].isin(['P19'])]
# clinique = get_metadata().set_index('ID_pseudo').loc[df_all_rsa2['sub'].unique(),'mRS_sortie'].to_dict()
clinique = get_metadata().set_index('ID_pseudo').loc[df_all_rsa2['sub'].unique(),'GCS_sortie'].to_dict()
df_all_rsa2['clinique_sortie'] = df_all_rsa2['sub'].map(clinique)

In [ ]:
fig, ax = plt.subplots()
sns.stripplot(data = df_all_rsa2,
            x = 'clinique_sortie',
            y = '50%',
            ax=ax)
ax.set_title('Med RSA vs Glasgow sortie')
ax.set_ylabel('RSA mediane (bpm)')

In [ ]:
df_all_rsa.dropna()['50%'].plot.hist(bins = np.arange(0, 8, 0.2))

In [ ]:
df_all_rsa.dropna()['75%'].plot.hist(bins = np.arange(0, 8, 0.2))

In [ ]:
df_all_rsa.dropna()['max'].plot.hist(bins = np.arange(0, 8, 0.2))

In [ ]:
np.sort(df['gcs'].unique())

In [ ]:
meta = get_metadata().set_index('ID_pseudo')
rows = []
for sub in get_patient_list(['ECG_II']):
    if sub in ['P19','P9']:
        continue
    ecg_peaks = detect_ecg_job.get(sub).to_dataframe()
    metrics = physio.compute_ecg_metrics(ecg_peaks, min_interval_ms=400)
    row = dict(sub=sub, gcs = meta.loc[sub, 'GCS_sortie'], mrs = meta.loc[sub, 'mRS_sortie']) | metrics.to_dict()
    rows.append(row)
df = pd.DataFrame(rows)
df = df.dropna().reset_index(drop = True)
df['gcs'] = pd.Series(df['gcs'], dtype = pd.CategoricalDtype(categories=np.sort(df['gcs'].unique()), ordered=True))
df['mrs'] = pd.Series(df['mrs'], dtype = pd.CategoricalDtype(categories=np.sort(df['mrs'].unique()), ordered=True))

In [ ]:
df.columns

In [ ]:
outcomes = ['HRV_Mean', 'HRV_SD', 'HRV_Median', 'HRV_Mad', 'HRV_CV', 'HRV_MCV', 'HRV_RMSSD']
predictors = ['gcs','mrs']
# nrows = len(outcomes)
# ncols = len(predictors)
nrows = len(predictors)
ncols = len(outcomes)

fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 3, nrows * 3), constrained_layout = True)

for r, predictor in enumerate(predictors):
    for c, outcome in enumerate(outcomes):
        ax = axs[r,c]
        # auto_stats(df = df,
        #         predictor = predictor,
        #         outcome = outcome,
        #         design = 'between',
        #         ax=ax
        #         )
        sns.stripplot(data = df,
                      x = predictor,
                      y = outcome,
                      ax=ax)

# Preproc ABP

In [ ]:
sub = 'BJ11'

In [ ]:
reader = pycns.CnsReader(data_path / sub)
reader.streams.keys()

In [ ]:
artbp_stream_names = [stream_name for stream_name in ['ART','ABP'] if stream_name in reader.streams.keys()]
artbp_stream_names

In [ ]:
ds

In [ ]:
ds = reader.export_to_xarray(stream_names=artbp_stream_names)

In [ ]:
len(artbp_stream_names)

# Map Hospit Dates vs Recording Dates

In [ ]:
stream_names = ['ECG_II','CO2','EEG','ICP','ABP']

meta = get_metadata().dropna(subset = ['entree_rea','date_sortie_rea']).set_index('ID_pseudo')

df = pd.DataFrame(columns = ['sub','d0_hospit','d1_hospit','stream_name','d0_stream','d1_stream','hospit_duration_days','n_days_start_recording', 'n_days_stop_recording', 'recording_duration_days','proportion_recording_vs_hospit'])

ind = 0
for sub in meta.index:
    # if sub == 'P98':
    #     print(sub)
    d0_hospit = meta.at[sub,'entree_rea']
    d1_hospit = meta.at[sub,'date_sortie_rea']# + np.timedelta64(12, 'h')

    reader = pycns.CnsReader(data_path / sub)
    possible_streams = reader.streams.keys()
    stream_names = ['ECG_II','CO2','EEG','ICP','ART','ABP']

    for stream_name in stream_names:
        init_list = [sub, d0_hospit, d1_hospit, stream_name]
        init_list = init_list + [None for i in range(df.shape[1] - len(init_list))]
        df.loc[ind,:] = init_list
        if stream_name in possible_streams:
            stream = reader.streams[stream_name]
            d0_recording, d1_recording = get_datetime_edges_from_stream(stream)
            if d0_recording < d0_hospit:
                d0_hospit = d0_recording
            if d1_recording > d1_hospit:
                d1_hospit = d1_recording
            n_days_hospit = (d1_hospit - d0_hospit).total_seconds() / (3600 * 24)
            n_days_start_recording = (d0_recording - d0_hospit).total_seconds() / (3600 * 24)
            n_days_stop_recording = (d1_recording - d0_hospit).total_seconds() / (3600 * 24)
            n_days_recording = n_days_stop_recording - n_days_start_recording
            proportion_recording_vs_hospit = n_days_recording / n_days_hospit
            df.loc[ind,:] = [sub, d0_hospit, d1_hospit, stream_name, d0_recording, d1_recording, n_days_hospit, n_days_start_recording, n_days_stop_recording, n_days_recording, proportion_recording_vs_hospit]
        ind += 1

for c in df.columns[6:]:
    df[c] = df[c].astype(float)

In [ ]:
sns.boxplot(data = df,
            x = 'stream_name',
            y = 'proportion_recording_vs_hospit'
)
sns.stripplot(data = df,
            x = 'stream_name',
            y = 'proportion_recording_vs_hospit',
            color = 'k',
            alpha = 0.5,
            size = 3,
)

In [ ]:
df_si = df.set_index(['sub','stream_name'])
subs = df['sub'].unique()

figsize = (15, df_si.shape[0] * 0.2)
fig, ax = plt.subplots(figsize = figsize)
fig.tight_layout(rect=[0, 0, 1, 0.95])
sup = "Proportion yellow / black : {d}".format(d = df['proportion_recording_vs_hospit'].describe()[3:].round(2).to_dict())
fig.suptitle(sup, y = 0.95)

xlim = (-5, df_si['hospit_duration_days'].max())
ax.set_xlim(xlim)

i = 0

yticks = []
yticklabels = []

# loop_subs = subs[:5]
loop_subs = subs
for sub in loop_subs:
    i_stream = 0
    for stream_name in df['stream_name'].unique():
        x_hospit = [0, df_si.loc[(sub,stream_name), 'hospit_duration_days']]
        x_recording = [df_si.loc[(sub,stream_name), 'n_days_start_recording'], df_si.loc[(sub,stream_name), 'n_days_stop_recording']]
        y = [i,i]
        ax.plot(x_hospit, y, color = 'k', lw = 5)
        ax.plot(x_recording, y, color = 'yellow', lw = 2.5)

        ax.text(max(xlim) - 1, i, stream_name, ha = 'right', va = 'center')
        if i_stream == df['stream_name'].unique().size//2:
            s = 'mrs = {mrs} ; gcs : {gcs}'.format(mrs=meta.loc[sub,'mRS_sortie'] , gcs = meta.loc[sub,'GCS_sortie'])
            ax.text(max(x_hospit) + 1, i, s, ha = 'left', va = 'center')

        yticks.append(i)
        yticklabels.append(sub)
        i+=1
        i_stream+=1
    ax.axhline(i, color = 'k')
    i+=1
for day in range(int(xlim[1])):
    ax.axvline(day, color = 'k', alpha = 0.2)
ax.set_xlabel('Time (days)')
ax.set_ylim(min(yticks) - 2, max(yticks) + 2)
ax.set_yticks(yticks[df['stream_name'].unique().size//2::df['stream_name'].unique().size], yticklabels[df['stream_name'].unique().size//2::df['stream_name'].unique().size])
fig.savefig(base_folder / 'figures' / 'autres' / 'map_hospit_vs_recording.png', dpi = 500, bbox_inches = 'tight')
fig.show()



# Map Hospit Dates vs ICCA Dates

In [ ]:
tt_names = ['Norépinéphrine',
               'Midazolam', 'Propofol', 'Thiopenthal', 'Eskétamine', 
               'Sufentanil', 'Rémifentanil', 'Morphine','Cisatracurium','Urapidil',]


clinical_names = ['Glasgow, total']

icca_names = tt_names + clinical_names

meta = get_metadata().dropna(subset = ['entree_rea','date_sortie_rea']).set_index('ID_pseudo')
icca_subs = get_icca_subs()
bug_subs = ['P1','HA1','P73']
subs = [s for s in meta.index if s in icca_subs]
subs = [s for s in subs if not s in bug_subs]

df = pd.DataFrame(columns = ['sub','d0_hospit','d1_hospit','icca_name','d0_icca','d1_icca','hospit_duration_days','n_days_start_icca', 'n_days_stop_icca', 'icca_duration_days','proportion_icca_vs_hospit'])

ind = 0
for sub in subs:
    # if sub == 'P98':
    # print(sub)
    d0_hospit = meta.at[sub,'entree_rea']
    d1_hospit = meta.at[sub,'date_sortie_rea']# + np.timedelta64(12, 'h')

    ds_tt = icca_pse_tt_job.get(sub)
    possible_tt = list(ds_tt.keys())
    ds_clinical = icca_clinical_job.get(sub)
    possible_icca = possible_tt + clinical_names

    for icca_name in icca_names:
        # if sub == 'P64':
        #     print(icca_name)
        init_list = [sub, d0_hospit, d1_hospit, icca_name]
        init_list = init_list + [None for i in range(df.shape[1] - len(init_list))]
        df.loc[ind,:] = init_list
        if icca_name in possible_icca:
            if icca_name == 'Glasgow, total':
                da = ds_clinical[icca_name]
            else:
                da = ds_tt[icca_name]
            d0_recording, d1_recording = da[f'datetime_{icca_name}'].values[0], da[f'datetime_{icca_name}'].values[-1]
            if d0_recording < d0_hospit:
                d0_hospit = d0_recording
            if d1_recording > d1_hospit:
                d1_hospit = d1_recording
            n_days_hospit = (d1_hospit - d0_hospit).total_seconds() / (3600 * 24)
            n_days_start_recording = pd.Timedelta((d0_recording - d0_hospit)).total_seconds() / (3600 * 24)
            n_days_stop_recording = pd.Timedelta((d1_recording - d0_hospit)).total_seconds() / (3600 * 24)
            n_days_recording = n_days_stop_recording - n_days_start_recording
            proportion_recording_vs_hospit = n_days_recording / n_days_hospit
            df.loc[ind,:] = [sub, d0_hospit, d1_hospit, icca_name, d0_recording, d1_recording, n_days_hospit, n_days_start_recording, n_days_stop_recording, n_days_recording, proportion_recording_vs_hospit]
        ind += 1

for c in df.columns[6:]:
    df[c] = df[c].astype(float)

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))
sns.boxplot(data = df,
            x = 'icca_name',
            y = 'proportion_icca_vs_hospit',
            ax=ax
)
sns.stripplot(data = df,
            x = 'icca_name',
            y = 'proportion_icca_vs_hospit',
            color = 'k',
            alpha = 0.5,
            size = 3,
            ax=ax
)

In [ ]:
df_si = df.set_index(['sub','icca_name'])
subs = df['sub'].unique()

figsize = (15, df_si.shape[0] * 0.2)
fig, ax = plt.subplots(figsize = figsize)
fig.tight_layout(rect=[0, 0, 1, 0.95])
sup = "Proportion yellow / black : {d}".format(d = df['proportion_icca_vs_hospit'].describe()[3:].round(2).to_dict())
fig.suptitle(sup, y = 0.95)

xlim = (-5, df_si['hospit_duration_days'].max())
ax.set_xlim(xlim)

i = 0

yticks = []
yticklabels = []

# loop_subs = subs[:5]
loop_subs = subs
for sub in loop_subs:
    i_stream = 0
    for icca_name in df['icca_name'].unique():
        x_hospit = [0, df_si.loc[(sub,icca_name), 'hospit_duration_days']]
        x_recording = [df_si.loc[(sub,icca_name), 'n_days_start_icca'], df_si.loc[(sub,icca_name), 'n_days_stop_icca']]
        y = [i,i]
        ax.plot(x_hospit, y, color = 'k', lw = 5)
        ax.plot(x_recording, y, color = 'yellow', lw = 2.5)

        ax.text(max(xlim) - 1, i, icca_name, ha = 'right', va = 'center')
        if i_stream == df['icca_name'].unique().size//2:
            s = 'mrs = {mrs} ; gcs : {gcs}'.format(mrs=meta.loc[sub,'mRS_sortie'] , gcs = meta.loc[sub,'GCS_sortie'])
            ax.text(max(x_hospit) + 1, i, s, ha = 'left', va = 'center')

        yticks.append(i)
        yticklabels.append(sub)
        i+=1
        i_stream+=1
    ax.axhline(i, color = 'k')
    i+=1
for day in range(int(xlim[1])):
    ax.axvline(day, color = 'k', alpha = 0.2)
ax.set_xlabel('Time (days)')
ax.set_ylim(min(yticks) - 2, max(yticks) + 2)
ax.set_yticks(yticks[df['icca_name'].unique().size//2::df['icca_name'].unique().size], yticklabels[df['icca_name'].unique().size//2::df['icca_name'].unique().size])
fig.savefig(base_folder / 'figures' / 'autres' / 'map_hospit_vs_icca_v3.png', dpi = 300, bbox_inches = 'tight')
fig.show()



# HRV / RespRHV healthy subjects vs brain lesioned patients 

## mad rri

In [ ]:
def load_ecg_healthy(sub):
    data_folder_healthy = Path('/crnldata/cmo/Projets/Emosens/NBuonviso2023_Emosens3_OdeurSon_Valentin_Matthias/Raw_Data/')
    data_folder_sub = data_folder_healthy / sub / 'signaux'
    file = [f for f in data_folder_sub.glob('*baseline.vhdr')][0]
    raw_ecg, srate = physio.read_one_channel(file, format = 'brainvision', channel_name='ECG')
    return xr.DataArray(data = -raw_ecg, coords = {'time':np.arange(raw_ecg.size) / srate}, attrs = {'srate':srate})[:int(600 * srate)]

def detect_peaks_healthy(raw_ecg):
    _, ecg_peaks = physio.compute_ecg(raw_ecg, 1000, parameter_preset='human_ecg')
    return ecg_peaks

def compute_mad(ecg_peaks):
    rri = np.diff(ecg_peaks['peak_time'].values) * 1000
    rri[rri > 2000] = np.nan
    rri[rri < 400] = np.nan
    _, mad = compute_nanmedian_mad(rri)
    return mad

def compute_med(ecg_peaks):
    rri = np.diff(ecg_peaks['peak_time'].values) * 1000
    rri[rri > 2000] = np.nan
    rri[rri < 400] = np.nan
    med, _ = compute_nanmedian_mad(rri)
    return med

In [ ]:
subs = []
for i in range(31):
    sub_label = f'P0{i+1}' if i+1 < 10 else f'P{i+1}'
    subs.append(sub_label)
mads_healthy = pd.DataFrame(columns = ['mad'], index = subs, dtype = float)
for sub in subs:
    raw_ecg_da = load_ecg_healthy(sub)
    ecg_peaks = detect_peaks_healthy(raw_ecg_da.values)
    mad = compute_mad(ecg_peaks)
    mads_healthy.at[sub, 'mad'] = mad
mads_healthy['mad'].describe()

In [ ]:
meta = get_metadata().set_index('ID_pseudo')

In [ ]:
subs = get_patient_list(['ECG_II'])
# subs = get_patient_list(['ECG_II'])
mads_coma = pd.DataFrame(columns = ['mad','alive_or_dead'], index = subs, dtype = float)
for sub in subs:
    try:
        ecg_peaks = detect_ecg_job.get(sub).to_dataframe()
        # t0 = ecg_peaks['peak_time'].values[0]
        # t1 = t0 + 600
        t1 = ecg_peaks['peak_time'].values[-1]
        t0 = t1 - 600
        ecg_peaks = ecg_peaks[(ecg_peaks['peak_time'] > t0) & (ecg_peaks['peak_time'] < t1)]
        mad = compute_mad(ecg_peaks)
        alive_or_dead = 1 if meta.at[sub,'mRS_sortie'] != 6 else 0
        mads_coma.at[sub, 'mad'] = mad
        mads_coma.at[sub, 'alive_or_dead'] = alive_or_dead
    except:
        print(sub)
        continue
mads_coma['mad'].describe()

In [ ]:
print('Healthy :', mads_healthy['mad'].describe().round(2).to_dict())
print('NeuroICU patient :', mads_coma['mad'].describe().round(2).to_dict())

In [ ]:
data_healthy = mads_healthy['mad'].dropna().values.reshape(-1)
data_alive = mads_coma[mads_coma['alive_or_dead'] == 1]['mad'].dropna().values.reshape(-1)
data_dead = mads_coma[mads_coma['alive_or_dead'] == 0]['mad'].dropna().values.reshape(-1)
groups = ['healthy'] * data_healthy.size + ['alive patients'] * data_alive.size + ['dead patients'] * data_dead.size
mads = pd.DataFrame(columns = ['group','hrv mad (ms)'])
mads['group'] = groups
mads['hrv mad (ms)'] = np.concatenate([data_healthy , data_alive, data_dead])

In [ ]:
fig, ax = plt.subplots()
data_stats = mads[mads['hrv mad (ms)'] < 150]
auto_stats(df = data_stats,
            predictor = 'group',
            outcome = 'hrv mad (ms)',
            design = 'between',
            ax=ax,
            )
ax.set_title('{title}\nN : {N}'.format(N = data_stats['group'].value_counts()[['healthy','alive patients','dead patients']].to_dict(), title = ax.get_title()))
# ax.set_ylim(-10 , 200)

## med rri

In [ ]:
def load_ecg_healthy(sub):
    data_folder_healthy = Path('/crnldata/cmo/Projets/Emosens/NBuonviso2023_Emosens3_OdeurSon_Valentin_Matthias/Raw_Data/')
    data_folder_sub = data_folder_healthy / sub / 'signaux'
    file = [f for f in data_folder_sub.glob('*baseline.vhdr')][0]
    raw_ecg, srate = physio.read_one_channel(file, format = 'brainvision', channel_name='ECG')
    return xr.DataArray(data = -raw_ecg, coords = {'time':np.arange(raw_ecg.size) / srate}, attrs = {'srate':srate})[:int(600 * srate)]

def detect_peaks_healthy(raw_ecg):
    _, ecg_peaks = physio.compute_ecg(raw_ecg, 1000, parameter_preset='human_ecg')
    return ecg_peaks

def compute_mad(ecg_peaks):
    rri = np.diff(ecg_peaks['peak_time'].values) * 1000
    rri[rri > 2000] = np.nan
    rri[rri < 400] = np.nan
    _, mad = compute_nanmedian_mad(rri)
    return mad

def compute_med(ecg_peaks):
    rri = np.diff(ecg_peaks['peak_time'].values) * 1000
    rri[rri > 2000] = np.nan
    rri[rri < 400] = np.nan
    med, _ = compute_nanmedian_mad(rri)
    return med

In [ ]:
subs = []
for i in range(31):
    sub_label = f'P0{i+1}' if i+1 < 10 else f'P{i+1}'
    subs.append(sub_label)
meds_healthy = pd.DataFrame(columns = ['med'], index = subs, dtype = float)
for sub in subs:
    raw_ecg_da = load_ecg_healthy(sub)
    ecg_peaks = detect_peaks_healthy(raw_ecg_da.values)
    med = compute_med(ecg_peaks)
    meds_healthy.at[sub, 'med'] = med
meds_healthy['med'].describe()

In [ ]:
meta = get_metadata().set_index('ID_pseudo')

In [ ]:
subs = get_patient_list(['ECG_II'])
# subs = get_patient_list(['ECG_II'])
meds_coma = pd.DataFrame(columns = ['med','alive_or_dead'], index = subs, dtype = float)
for sub in subs:
    try:
        ecg_peaks = detect_ecg_job.get(sub).to_dataframe()
        # t0 = ecg_peaks['peak_time'].values[0]
        # t1 = t0 + 600
        t1 = ecg_peaks['peak_time'].values[-1]
        t0 = t1 - 600
        ecg_peaks = ecg_peaks[(ecg_peaks['peak_time'] > t0) & (ecg_peaks['peak_time'] < t1)]
        med = compute_med(ecg_peaks)
        alive_or_dead = 1 if meta.at[sub,'mRS_sortie'] != 6 else 0
        meds_coma.at[sub, 'med'] = med
        meds_coma.at[sub, 'alive_or_dead'] = alive_or_dead
    except:
        print(sub)
        continue
meds_coma['med'].describe()

In [ ]:
data_healthy = meds_healthy['med'].dropna().values.reshape(-1)
data_alive = meds_coma[meds_coma['alive_or_dead'] == 1]['med'].dropna().values.reshape(-1)
data_dead = meds_coma[meds_coma['alive_or_dead'] == 0]['med'].dropna().values.reshape(-1)
groups = ['healthy'] * data_healthy.size + ['alive patients'] * data_alive.size + ['dead patients'] * data_dead.size
meds = pd.DataFrame(columns = ['group','hrv med (ms)'])
meds['group'] = groups
meds['hrv med (ms)'] = np.concatenate([data_healthy , data_alive, data_dead])

In [ ]:
fig, ax = plt.subplots()
data_stats = meds[meds['hrv med (ms)'] < 3000]
auto_stats(df = data_stats,
            predictor = 'group',
            outcome = 'hrv med (ms)',
            design = 'between',
            ax=ax,
            )
ax.set_title('{title}\nN : {N}'.format(N = data_stats['group'].value_counts()[['healthy','alive patients','dead patients']].to_dict(), title = ax.get_title()))
# ax.set_ylim(-10 , 200)

## resphrv

In [ ]:
def load_ecg_healthy(sub):
    data_folder_healthy = Path('/crnldata/cmo/Projets/Emosens/NBuonviso2023_Emosens3_OdeurSon_Valentin_Matthias/Raw_Data/')
    data_folder_sub = data_folder_healthy / sub / 'signaux'
    file = [f for f in data_folder_sub.glob('*baseline.vhdr')][0]
    raw_ecg, srate = physio.read_one_channel(file, format = 'brainvision', channel_name='ECG')
    return xr.DataArray(data = -raw_ecg, coords = {'time':np.arange(raw_ecg.size) / srate}, attrs = {'srate':srate})[:int(600 * srate)]

def load_resp_healthy(sub):
    data_folder_healthy = Path('/crnldata/cmo/Projets/Emosens/NBuonviso2023_Emosens3_OdeurSon_Valentin_Matthias/Raw_Data/')
    data_folder_sub = data_folder_healthy / sub / 'signaux'
    file = [f for f in data_folder_sub.glob('*baseline.vhdr')][0]
    raw_resp, srate = physio.read_one_channel(file, format = 'brainvision', channel_name='RespiNasale')
    return xr.DataArray(data = -raw_resp, coords = {'time':np.arange(raw_resp.size) / srate}, attrs = {'srate':srate})[:int(600 * srate)]

def detect_peaks_healthy(raw_ecg):
    _, ecg_peaks = physio.compute_ecg(raw_ecg, 1000, parameter_preset='human_ecg')
    return ecg_peaks

def detect_resp_cycles_healthy(raw_resp):
    _, resp_cycles = physio.compute_respiration(raw_resp, 1000, parameter_preset='human_airflow')
    return resp_cycles

def compute_med_of_resphrv(ecg_peaks, resp_cycles):
    resphrv_cycles, _ = physio.compute_resphrv(resp_cycles, ecg_peaks, srate=100.0, units='bpm', limits=[30, 200], two_segment=False, points_per_cycle=50)
    return resphrv_cycles['decay_amplitude'].median()

In [ ]:
subs = []
for i in range(31):
    sub_label = f'P0{i+1}' if i+1 < 10 else f'P{i+1}'
    subs.append(sub_label)
resphrv_healthy = pd.DataFrame(columns = ['resphrv'], index = subs, dtype = float)
for sub in subs:
    raw_ecg_da = load_ecg_healthy(sub)
    raw_resp_da = load_resp_healthy(sub)
    ecg_peaks = detect_peaks_healthy(raw_ecg_da.values)
    resp_cycles = detect_resp_cycles_healthy(raw_resp_da.values)
    med_rsa = compute_med_of_resphrv(ecg_peaks, resp_cycles)
    resphrv_healthy.at[sub, 'resphrv'] = med_rsa
resphrv_healthy['resphrv'].describe()

In [ ]:
meta = get_metadata().set_index('ID_pseudo')

In [ ]:
subs = get_patient_list(['ECG_II','CO2'])
# subs = get_patient_list(['ECG_II'])
resphrv_coma = pd.DataFrame(columns = ['resphrv','alive_or_dead'], index = subs, dtype = float)
for sub in subs:
    try:
        rsa_cycles = rsa_job.get(sub).to_dataframe()
        t1 = rsa_cycles['peak_time'].values[-1]
        t0 = t1 - 600
        rsa_cycles = rsa_cycles[(rsa_cycles['peak_time'] > t0) & (rsa_cycles['peak_time'] < t1)]
        med_resphrv = rsa_cycles['decay_amplitude'].median()
        alive_or_dead = 1 if meta.at[sub,'mRS_sortie'] != 6 else 0
        resphrv_coma.at[sub, 'resphrv'] = med_resphrv
        resphrv_coma.at[sub, 'alive_or_dead'] = alive_or_dead
    except:
        print(sub)
        continue
resphrv_coma['resphrv'].describe()

In [ ]:
data_healthy = resphrv_healthy['resphrv'].dropna().values.reshape(-1)
data_alive = resphrv_coma[resphrv_coma['alive_or_dead'] == 1]['resphrv'].dropna().values.reshape(-1)
data_dead = resphrv_coma[resphrv_coma['alive_or_dead'] == 0]['resphrv'].dropna().values.reshape(-1)
groups = ['healthy'] * data_healthy.size + ['alive patients'] * data_alive.size + ['dead patients'] * data_dead.size
meds = pd.DataFrame(columns = ['group','resphrv (bpm)'])
meds['group'] = groups
meds['resphrv (bpm)'] = np.concatenate([data_healthy , data_alive, data_dead])

In [ ]:
fig, ax = plt.subplots()
data_stats = meds[meds['resphrv (bpm)'] < 3000]
auto_stats(df = data_stats,
            predictor = 'group',
            outcome = 'resphrv (bpm)',
            design = 'between',
            ax=ax,
            )
ax.set_title('{title}\nN : {N}'.format(N = data_stats['group'].value_counts()[['healthy','alive patients','dead patients']].to_dict(), title = ax.get_title()))
# ax.set_ylim(-10 , 200)

## cyclic cardiac rate

In [ ]:
def load_ecg_healthy(sub):
    data_folder_healthy = Path('/crnldata/cmo/Projets/Emosens/NBuonviso2023_Emosens3_OdeurSon_Valentin_Matthias/Raw_Data/')
    data_folder_sub = data_folder_healthy / sub / 'signaux'
    file = [f for f in data_folder_sub.glob('*baseline.vhdr')][0]
    raw_ecg, srate = physio.read_one_channel(file, format = 'brainvision', channel_name='ECG')
    return xr.DataArray(data = -raw_ecg, coords = {'time':np.arange(raw_ecg.size) / srate}, attrs = {'srate':srate})[:int(600 * srate)]

def load_resp_healthy(sub):
    data_folder_healthy = Path('/crnldata/cmo/Projets/Emosens/NBuonviso2023_Emosens3_OdeurSon_Valentin_Matthias/Raw_Data/')
    data_folder_sub = data_folder_healthy / sub / 'signaux'
    file = [f for f in data_folder_sub.glob('*baseline.vhdr')][0]
    raw_resp, srate = physio.read_one_channel(file, format = 'brainvision', channel_name='RespiNasale')
    return xr.DataArray(data = -raw_resp, coords = {'time':np.arange(raw_resp.size) / srate}, attrs = {'srate':srate})[:int(600 * srate)]

def detect_peaks_healthy(raw_ecg):
    _, ecg_peaks = physio.compute_ecg(raw_ecg, 1000, parameter_preset='human_ecg')
    return ecg_peaks

def detect_resp_cycles_healthy(raw_resp):
    _, resp_cycles = physio.compute_respiration(raw_resp, 1000, parameter_preset='human_airflow')
    return resp_cycles

def compute_cyclic_cardiac_rate(ecg_peaks, resp_cycles, points_per_cycle):
    _, cyclic_cardiac_rate = physio.compute_resphrv(resp_cycles, ecg_peaks, srate=50.0, units='bpm', limits=[30, 200], two_segment=True, points_per_cycle=points_per_cycle)
    cyclic_cardiac_rate = xr.DataArray(data= cyclic_cardiac_rate, coords = {'cycle':range(cyclic_cardiac_rate.shape[0]) , 'phase':np.linspace(0,1,points_per_cycle)})
    offset = cyclic_cardiac_rate.median(['phase','cycle'])
    cyclic_cardiac_rate = (cyclic_cardiac_rate - cyclic_cardiac_rate.median('phase')).median('cycle') + offset
    return cyclic_cardiac_rate

In [ ]:
points_per_cycle = 100

subs = []
for i in range(31):
    sub_label = f'P0{i+1}' if i+1 < 10 else f'P{i+1}'
    subs.append(sub_label)
coords = {'sub':subs , 'phase':np.linspace(0,1,points_per_cycle)}
cyclic_cardiac_rate_healthy = xr.DataArray(data = np.nan, dims = coords.keys(), coords=  coords)
coords = {'sub':subs}
cycle_ratio_healthy = xr.DataArray(data = np.nan, dims = coords.keys(), coords=  coords)
for i , sub in enumerate(subs):
    # print(sub)
    raw_ecg_da = load_ecg_healthy(sub)
    raw_resp_da = load_resp_healthy(sub)
    ecg_peaks = detect_peaks_healthy(raw_ecg_da.values)
    resp_cycles = detect_resp_cycles_healthy(raw_resp_da.values)
    t0, t1 = ecg_peaks['peak_time'].values[0], ecg_peaks['peak_time'].values[-1]
    # resp_cycles = resp_cycles[(resp_cycles['inspi_time'] > t0) & (resp_cycles['next_inspi_time'] < t1)]
    cyclic_cardiac_rate = compute_cyclic_cardiac_rate(ecg_peaks, resp_cycles, points_per_cycle)
    cyclic_cardiac_rate_healthy.loc[sub, :] = cyclic_cardiac_rate.values
    cycle_ratio_healthy.loc[sub] = resp_cycles['cycle_ratio'].median()
# resphrv_healthy['resphrv'].describe()

In [ ]:
cycle_ratio_healthy

In [ ]:
cyclic_cardiac_rate_healthy.plot.line(x = 'phase', col = 'sub', col_wrap = 5)

In [ ]:
meta = get_metadata().set_index('ID_pseudo')

In [ ]:
subs = get_patient_list(['ECG_II','CO2'])
# subs = get_patient_list(['ECG_II'])
coords = {'sub':subs , 'phase':np.linspace(0,1,points_per_cycle)}
cyclic_cardiac_rate_patients = xr.DataArray(data = np.nan, dims = coords.keys(), coords=  coords)
coords = {'sub':subs}
cycle_ratio_patients = xr.DataArray(data = np.nan, dims = coords.keys(), coords=  coords)
coords = {'sub':subs}
da_alive_or_dead = xr.DataArray(data = np.nan, dims = coords.keys(), coords=  coords)
for sub in subs:
    try:
        cyclic_cardiac_rate = heart_rate_by_resp_cycle_job.get(sub)['heart_rate_by_resp_cycle']
        med_cycle_ratio = cyclic_cardiac_rate.attrs['med_cycle_ratio']
        # t1 = cyclic_cardiac_rate['cycle_date'].values[-1] - np.timedelta64(3600 * 10, 's')
        t1 = cyclic_cardiac_rate['cycle_date'].values[-1]# - np.timedelta64(600, 's')
        t0 = t1 - np.timedelta64(600, 's')
        cyclic_cardiac_rate = cyclic_cardiac_rate.loc[t0:t1,:]
        offset = cyclic_cardiac_rate.median(['cycle_date','phase'])
        cyclic_cardiac_rate = (cyclic_cardiac_rate - cyclic_cardiac_rate.median('phase')).median('cycle_date') + offset
        cyclic_cardiac_rate_patients.loc[sub, :] = cyclic_cardiac_rate.values
        cycle_ratio_patients.loc[sub] = med_cycle_ratio
        alive_or_dead = 1 if meta.at[sub,'mRS_sortie'] != 6 else 0
        da_alive_or_dead.loc[sub] = alive_or_dead
    except:
        print(sub)
        continue

In [ ]:
subs = get_patient_list(['ECG_II','CO2'])
# subs = get_patient_list(['ECG_II'])
coords = {'sub':subs , 'phase':np.linspace(0,1,points_per_cycle)}
cyclic_cardiac_rate_patients = xr.DataArray(data = np.nan, dims = coords.keys(), coords=  coords)
coords = {'sub':subs}
cycle_ratio_patients = xr.DataArray(data = np.nan, dims = coords.keys(), coords=  coords)
coords = {'sub':subs}
da_alive_or_dead = xr.DataArray(data = np.nan, dims = coords.keys(), coords=  coords)
keep_subs = []
not_do_subs = ['P75']
for sub in subs:
    if sub in not_do_subs:
        continue
    try:
        resp_cycles = detect_resp_job.get(sub).to_dataframe()
        ecg_peaks = detect_ecg_job.get(sub).to_dataframe()

        # t1 = cyclic_cardiac_rate['cycle_date'].values[-1] - np.timedelta64(3600 * 10, 's')
        t1 = resp_cycles['next_inspi_time'].values[-1]# - np.timedelta64(600, 's')
        t0 = t1 - 600

        ecg_peaks = ecg_peaks[(ecg_peaks['peak_time'] > t0) & (ecg_peaks['peak_time'] < t1)]
        resp_cycles = resp_cycles[(resp_cycles['inspi_time'] > t0) & (resp_cycles['next_inspi_time'] < t1)]

        t0, t1 = ecg_peaks['peak_time'].values[0], ecg_peaks['peak_time'].values[-1]
        resp_cycles = resp_cycles[(resp_cycles['inspi_time'] > t0) & (resp_cycles['next_inspi_time'] < t1)]

        if resp_cycles.shape[0] < 50:
            print(sub)
            continue
        cyclic_cardiac_rate = compute_cyclic_cardiac_rate(ecg_peaks, resp_cycles, points_per_cycle)
        cyclic_cardiac_rate_patients.loc[sub, :] = cyclic_cardiac_rate.values
        cycle_ratio_patients.loc[sub] = resp_cycles['cycle_ratio'].median()
        alive_or_dead = 1 if meta.at[sub,'mRS_sortie'] != 6 else 0
        da_alive_or_dead.loc[sub] = alive_or_dead
        keep_subs.append(sub)
    except:
        print(sub)
        continue
cyclic_cardiac_rate_patients = cyclic_cardiac_rate_patients.loc[keep_subs]
cycle_ratio_patients = cycle_ratio_patients.loc[keep_subs]
da_alive_or_dead = da_alive_or_dead.loc[keep_subs]

In [ ]:
cyclic_cardiac_rate_patients.groupby(da_alive_or_dead).mean('sub').plot.line(x = 'phase', hue = 'group')

In [ ]:
(cyclic_cardiac_rate_patients - cyclic_cardiac_rate_patients.median('phase')).groupby(da_alive_or_dead).mean('sub').plot.line(x = 'phase', hue = 'group')

In [ ]:
phase = cyclic_cardiac_rate_healthy['phase'].values

fig, axs = plt.subplots(nrows = 2, ncols = 3, figsize = (9,5), constrained_layout= True)

for r, center in enumerate([False, True]):
    ax = axs[r,0] # healthy
    cyclic_cardiac_rate_healthy_sel = cyclic_cardiac_rate_healthy.copy()
    if center:
        mean_offset = cyclic_cardiac_rate_healthy_sel.mean(['sub','phase'])
        cyclic_cardiac_rate_healthy_sel = cyclic_cardiac_rate_healthy - cyclic_cardiac_rate_healthy.median('phase') 
        m = cyclic_cardiac_rate_healthy_sel.mean('sub').values
    else:
        m = cyclic_cardiac_rate_healthy_sel.mean('sub').values
        # s = cyclic_cardiac_rate_healthy_sel.std('sub').values
    ax.plot(phase, m, lw = 2, color = 'tab:blue')
    # ax.fill_between(phase, m-s, m+s, alpha = 0.5, color = 'tab:blue')
    ax.plot(phase, cyclic_cardiac_rate_healthy_sel.values.T, alpha = 0.2, color = 'k')
    ax.set_title(f'Healthy (N = {cyclic_cardiac_rate_healthy_sel.shape[0]})')
    # ax.plot()

    ax = axs[r,1] # patients alive
    cyclic_cardiac_rate_patients_sel = cyclic_cardiac_rate_patients[da_alive_or_dead == 1,:]
    if center:
        mean_offset = cyclic_cardiac_rate_patients_sel.mean(['sub','phase'])
        cyclic_cardiac_rate_patients_sel = cyclic_cardiac_rate_patients_sel - cyclic_cardiac_rate_patients_sel.median('phase') 
        m = cyclic_cardiac_rate_patients_sel.mean('sub').values
    else:
        m = cyclic_cardiac_rate_patients_sel.mean('sub').values
        # s = cyclic_cardiac_rate_patients_sel.std('sub').values
    ax.plot(phase, m, lw = 2, color = 'darkorange')
    # ax.fill_between(phase, m-s, m+s, alpha = 0.5, color = 'darkorange')

    ax.plot(phase, cyclic_cardiac_rate_patients_sel.values.T, alpha = 0.2, color = 'k')
    ax.set_title(f'Alive patients (N = {cyclic_cardiac_rate_patients_sel.shape[0]})')


    ax = axs[r,2] # patients dead
    
    cyclic_cardiac_rate_patients_sel = cyclic_cardiac_rate_patients[da_alive_or_dead == 0,:]
    if center:
        mean_offset = cyclic_cardiac_rate_patients_sel.mean(['sub','phase'])
        cyclic_cardiac_rate_patients_sel = cyclic_cardiac_rate_patients_sel - cyclic_cardiac_rate_patients_sel.median('phase') 
        m = cyclic_cardiac_rate_patients_sel.mean('sub').values
    else:
        m = cyclic_cardiac_rate_patients_sel.mean('sub').values
        # s = cyclic_cardiac_rate_patients_sel.std('sub').values
    ax.plot(phase, m, lw = 2, color = 'g')
    # ax.fill_between(phase, m-s, m+s, alpha = 0.5, color = 'r')

    ax.plot(phase, cyclic_cardiac_rate_patients_sel.values.T, alpha = 0.2, color = 'k')
    ax.set_title(f'Dead patients (N = {cyclic_cardiac_rate_patients_sel.shape[0]})')


for i, ax in enumerate(axs.flatten()):
    if i <= 2:
        ylim = (50, 120)
    else:
        ylim = (-5,5)
    ax.set_ylim(ylim)
    ax.set_ylabel('Heart rate (bpm)')
    ax.set_xlabel('Resp phase (0-1)')


In [ ]:
alpha_fillb = 0.1
phase = cyclic_cardiac_rate_healthy['phase'].values

fig, axs = plt.subplots(nrows = 2, ncols = 3, figsize = (9,5), constrained_layout= True)

for r, center in enumerate([False, True]):
    ax = axs[r,0] # healthy
    cyclic_cardiac_rate_healthy_sel = cyclic_cardiac_rate_healthy.copy()
    if center:
        mean_offset = cyclic_cardiac_rate_healthy_sel.mean(['sub','phase'])
        cyclic_cardiac_rate_healthy_sel = cyclic_cardiac_rate_healthy - cyclic_cardiac_rate_healthy.median('phase') 
        m = cyclic_cardiac_rate_healthy_sel.mean('sub').values
    else:
        m = cyclic_cardiac_rate_healthy_sel.mean('sub').values
    s = cyclic_cardiac_rate_healthy_sel.std('sub').values
    c = 'tab:blue'
    ax.plot(phase, m, lw = 2, color = c)
    ax.fill_between(phase, m-s, m+s, alpha = alpha_fillb, color = c)
    # ax.plot(phase, cyclic_cardiac_rate_healthy_sel.values.T, alpha = 0.2, color = 'k')
    ax.set_title(f'Healthy (N = {cyclic_cardiac_rate_healthy_sel.shape[0]})')
    # ax.plot()

    ax = axs[r,1] # patients alive
    cyclic_cardiac_rate_patients_sel = cyclic_cardiac_rate_patients[da_alive_or_dead == 1,:]
    if center:
        mean_offset = cyclic_cardiac_rate_patients_sel.mean(['sub','phase'])
        cyclic_cardiac_rate_patients_sel = cyclic_cardiac_rate_patients_sel - cyclic_cardiac_rate_patients_sel.median('phase') 
        m = cyclic_cardiac_rate_patients_sel.mean('sub').values
    else:
        m = cyclic_cardiac_rate_patients_sel.mean('sub').values
    s = cyclic_cardiac_rate_patients_sel.std('sub').values
    c = 'darkorange'
    ax.plot(phase, m, lw = 2, color = c)
    ax.fill_between(phase, m-s, m+s, alpha = alpha_fillb, color = c)

    # ax.plot(phase, cyclic_cardiac_rate_patients_sel.values.T, alpha = 0.2, color = 'k')
    ax.set_title(f'Alive patients (N = {cyclic_cardiac_rate_patients_sel.shape[0]})')


    ax = axs[r,2] # patients dead
    
    cyclic_cardiac_rate_patients_sel = cyclic_cardiac_rate_patients[da_alive_or_dead == 0,:]
    if center:
        mean_offset = cyclic_cardiac_rate_patients_sel.mean(['sub','phase'])
        cyclic_cardiac_rate_patients_sel = cyclic_cardiac_rate_patients_sel - cyclic_cardiac_rate_patients_sel.median('phase') 
        m = cyclic_cardiac_rate_patients_sel.mean('sub').values
    else:
        m = cyclic_cardiac_rate_patients_sel.mean('sub').values
    s = cyclic_cardiac_rate_patients_sel.std('sub').values
    c = 'g'
    ax.plot(phase, m, lw = 2, color = c)
    ax.fill_between(phase, m-s, m+s, alpha = alpha_fillb, color = c)

    # ax.plot(phase, cyclic_cardiac_rate_patients_sel.values.T, alpha = 0.2, color = 'k')
    ax.set_title(f'Dead patients (N = {cyclic_cardiac_rate_patients_sel.shape[0]})')


for i, ax in enumerate(axs.flatten()):
    # if i <= 2:
    #     ylim = (50, 120)
    # else:
    #     ylim = (-5,5)
    # ax.set_ylim(ylim)
    ax.axvline(0.4, color = 'k', alpha = 0.5, lw = 2, ls = '--')
    ax.text(0.2, 0.1, 'Inspi', ha = 'center', transform = ax.transAxes)
    ax.text(0.75, 0.1, 'Expi', ha = 'center', transform = ax.transAxes)
    ax.set_ylabel('Heart rate (bpm)')
    ax.set_xlabel('Resp phase (0-1)')


# HRV healthy subjects vs brain lesioned patients (more analysis)

In [ ]:
def load_ecg_healthy(sub):
    data_folder_healthy = Path('/crnldata/cmo/Projets/Emosens/NBuonviso2023_Emosens3_OdeurSon_Valentin_Matthias/Raw_Data/')
    data_folder_sub = data_folder_healthy / sub / 'signaux'
    file = [f for f in data_folder_sub.glob('*baseline.vhdr')][0]
    raw_ecg, srate = physio.read_one_channel(file, format = 'brainvision', channel_name='ECG')
    return xr.DataArray(data = -raw_ecg, coords = {'time':np.arange(raw_ecg.size) / srate}, attrs = {'srate':srate})[:int(600 * srate)]

def process_ecg(raw_ecg_da):
    _, ecg_peaks = physio.compute_ecg(raw_ecg_da.values, raw_ecg_da.attrs['srate'], parameter_preset='human_ecg')
    rri = np.diff(ecg_peaks['peak_time'].values)
    med, mad = physio.compute_median_mad(rri * 1000)
    ihr = physio.compute_instantaneous_rate(ecg_peaks, new_times = raw_ecg_da['time'].values)
    return xr.DataArray(data = ihr, coords = {'time':raw_ecg_da['time'].values}, attrs = {'peak_index':ecg_peaks['peak_index'].values, 'srate':raw_ecg_da.attrs['srate'], 'mad':mad})

def compute_sxx(ihr_da, win_size_s, overlap_prct = 0.5, nfft_factor = 1, apply_sqrt = True):
    srate = ihr_da.attrs['srate']
    nperseg = int(win_size_s * srate)
    noverlap = int(nperseg * overlap_prct)
    nfft = int(nfft_factor * nperseg)
    f, t, Sxx = scipy.signal.spectrogram(ihr_da.values, fs = srate, nperseg = nperseg, noverlap=noverlap, nfft=nfft, scaling = 'spectrum')
    if apply_sqrt:
        Sxx = np.sqrt(Sxx)
    return xr.DataArray(data = Sxx, coords = {'freq':f, 'time':t}).loc[:0.5,:]

def spectrum_slope(sxx_da, freq_range = (0.0033, 0.4)):
    hr_spectrum_slopes = np.apply_along_axis(compute_spectrum_log_slope, axis = 0, arr = sxx_da.values, freqs = sxx_da['freq'].values, freq_range = freq_range)
    return xr.DataArray(data = hr_spectrum_slopes, coords = {'time':sxx_da['time']})

def hrv_freqs(sxx_da, freq_bands = {'total':(0.01,0.4),'vlf':(0.01,0.04), 'lf': (0.04, 0.15), 'hf' : (0.15, .4)}):
    names = list(freq_bands.keys())
    coords = {'band':names, 'time':sxx_da['time']}
    da = xr.DataArray(data = np.nan, dims = coords.keys(), coords = coords)
    for name, (f0,f1) in freq_bands.items():
        da.loc[name, :] = sxx_da.loc[f0:f1,:].integrate('freq')
    return da

def percentage_hrv(hrv_freq_da):
    band_names = ['vlf','lf','hf']
    coords = {'band':band_names, 'time':hrv_freq_da['time']}
    percentage_da = xr.DataArray(data = np.nan, dims = coords.keys(), coords = coords)
    for band_name in band_names:
        percentage_band = (hrv_freq_da.loc[band_name,:] / hrv_freq_da.loc['total',:]) * 100
        percentage_da.loc[band_name,:] = percentage_band
    return percentage_da


In [ ]:
sub = 'P29'

raw_ecg_da = load_ecg_healthy(sub)
ihr_da = process_ecg(raw_ecg_da)
print('mad ms :', ihr_da.attrs['mad'])
sxx_da = compute_sxx(raw_ecg_da, win_size_s = 120, overlap_prct=0.9)
slope_da = spectrum_slope(sxx_da, freq_range = (0.04, 0.4))
hrv_freq_da = hrv_freqs(sxx_da)
percentage_da = percentage_hrv(hrv_freq_da)

In [ ]:
figsize = (12,3)
ihr_da.plot(figsize = figsize)

sxx_da.loc[0.04:0.5,:].plot.pcolormesh(robust = True, figsize=figsize)

sxx_da.loc[0.04:0.5,:].median('time').plot.line(figsize=figsize, yscale = 'log', xscale = 'log')

slope_da.plot(figsize=figsize)

fig, ax = plt.subplots(figsize = figsize)
colors = ["red", "green", "blue"]  # Colors for vlf, lf, hf
# Stacked area plot
ax.stackplot(
    hrv_freq_da['time'],
    hrv_freq_da.sel(band="vlf"),
    hrv_freq_da.sel(band="lf"),
    hrv_freq_da.sel(band="hf"),
    labels=["VLF", "LF", "HF"],
    colors=colors,
    alpha=0.6,
)
ax.legend(loc = 'upper left')

fig, ax = plt.subplots(figsize = figsize)
colors = ["red", "green", "blue"]  # Colors for vlf, lf, hf
# Stacked area plot
ax.stackplot(
    percentage_da['time'],
    percentage_da.sel(band="vlf"),
    percentage_da.sel(band="lf"),
    percentage_da.sel(band="hf"),
    labels=["VLF", "LF", "HF"],
    colors=colors,
    alpha=0.6,
)
ax.legend(loc = 'upper left')

In [ ]:
subs = []
for i in range(31):
    sub_label = f'P0{i+1}' if i+1 < 10 else f'P{i+1}'
    subs.append(sub_label)
mads_healthy = pd.DataFrame(columns = ['mad'], index = subs, dtype = float)
for sub in subs:
    raw_ecg_da = load_ecg_healthy(sub)
    ihr_da = process_ecg(raw_ecg_da)
    mads_healthy.at[sub, 'mad'] = ihr_da.attrs['mad']
mads_healthy['mad'].describe()

In [ ]:
meta = get_metadata().set_index('ID_pseudo')

In [ ]:
subs = get_patient_list(['ECG_II'])
# subs = get_patient_list(['ECG_II'])
mads_coma = pd.DataFrame(columns = ['mad','alive_or_dead'], index = subs, dtype = float)
for sub in subs:
    try:
        ecg_peaks = detect_ecg_job.get(sub).to_dataframe()
        # t0 = ecg_peaks['peak_time'].values[0]
        # t1 = t0 + 600
        t1 = ecg_peaks['peak_time'].values[-1]
        t0 = t1 - 600
        ecg_peaks = ecg_peaks[(ecg_peaks['peak_time'] > t0) & (ecg_peaks['peak_time'] < t1)]
        rri = np.diff(ecg_peaks['peak_time'].values) * 1000
        rri[rri > 2000] = np.nan
        rri[rri < 400] = np.nan
        _, mad = compute_nanmedian_mad(rri)
        alive_or_dead = 1 if meta.at[sub,'mRS_sortie'] != 6 else 0
        mads_coma.at[sub, 'mad'] = mad
        mads_coma.at[sub, 'alive_or_dead'] = alive_or_dead
    except:
        print(sub)
        continue
mads_coma['mad'].describe()

In [ ]:
alpha = 0.5
density = True

bins = np.arange(0,200,5)
fig, ax = plt.subplots(figsize = (8,4))
data = mads_healthy.values
ax.hist(data, bins=bins, label = f'healthy (n = {data.size})', color = 'g', alpha = alpha, density = density, edgecolor = 'k')
data = mads_coma[mads_coma['alive_or_dead'] == 1]['mad'].dropna().values
ax.hist(data, bins=bins, label = f'alive patients (n = {data.size})', color = 'darkorange', alpha = alpha, density = density, edgecolor = 'k')
data = mads_coma[mads_coma['alive_or_dead'] == 0]['mad'].dropna().values
ax.hist(data, bins=bins, label = f'dead patients (n = {data.size})', color = 'r', alpha = alpha, density = density, edgecolor = 'k')
ax.legend(loc = 'upper right')

In [ ]:
data_healthy = mads_healthy['mad'].dropna().values.reshape(-1)
data_alive = mads_coma[mads_coma['alive_or_dead'] == 1]['mad'].dropna().values.reshape(-1)
data_dead = mads_coma[mads_coma['alive_or_dead'] == 0]['mad'].dropna().values.reshape(-1)
groups = ['healthy'] * data_healthy.size + ['alive patients'] * data_alive.size + ['dead patients'] * data_dead.size
mads = pd.DataFrame(columns = ['group','hrv mad (ms)'])
mads['group'] = groups
mads['hrv mad (ms)'] = np.concatenate([data_healthy , data_alive, data_dead])

In [ ]:
fig, ax = plt.subplots()
data_stats = mads[mads['hrv mad (ms)'] < 150]
auto_stats(df = data_stats,
            predictor = 'group',
            outcome = 'hrv mad (ms)',
            design = 'between',
            ax=ax,
            )
ax.set_title('{title}\nN : {N}'.format(N = data_stats['group'].value_counts()[['healthy','alive patients','dead patients']].to_dict(), title = ax.get_title()))
# ax.set_ylim(-10 , 200)

In [ ]:
mads

In [ ]:
mads_coma[mads_coma['alive_or_dead'] == 1]['mad'].dropna().values.reshape(-1)

In [ ]:
from sklearn.metrics import roc_curve, auc

In [ ]:
mads_coma2 = mads_coma.dropna()
print(mads_coma2['alive_or_dead'].value_counts())
X = mads_coma2['mad'].values
y = mads_coma2['alive_or_dead'].values

In [ ]:
ihr_job.get('MF12')['ihr'].attrs['time'][0]

In [ ]:
test_da = hr_spectro_job.get('MF12')['hr_spectro']
test_da['datetime'] = test_da.attrs['time']
test_da = test_da.rename({'datetime':'time'})
test_da.loc[:,1800:2000].median('time')

In [ ]:
# sns.boxplot(data = mads_coma2, x=  'alive_or_dead', y = 'mad', hue = 'alive_or_dead')
auto_stats(df = mads_coma2,
           predictor='alive_or_dead',
           outcome = 'mad',
           design = 'between')

In [ ]:
fpr, tpr, thresholds = roc_curve(y, X)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')  # ligne aléatoire
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve pour la dynamique HRV')
plt.legend()
plt.grid()
plt.show()

In [ ]:
def get_hrv_time_metrics(ecg_peaks):
    min_n_peaks = 0
    if ecg_peaks.shape[0] > min_n_peaks:
        rri = np.diff(ecg_peaks['peak_time'].values) * 1000
        rri[rri > 2000] = np.nan
        med_rri, mad_rri = compute_nanmedian_mad(rri)
        hrv_time = dict(med_rri=med_rri, mad_rri=mad_rri)
    else:
        hrv_time = dict(med_rri = np.nan, mad_rri = np.nan)
    return hrv_time 

def get_hrv_freq_metrics(ecg_peaks):
    frequency_bands = dict(total = (0.005, 0.4),
                                   vlf = (0.005,0.04),
                                   lf = (0.04, 0.15),
                                   hf = (0.15, 0.4)
    )
    
    window_s = 600

    hrv_freqs = {band:np.nan for band in frequency_bands.keys()}
    hrv_freqs['vlfn'] = np.nan
    hrv_freqs['lfn'] = np.nan
    hrv_freqs['hfn'] = np.nan
    hrv_freqs['lf_hf'] = np.nan

    peak_time_s = ecg_peaks['peak_time'].values

    if peak_time_s.size > 0:
        duration_s = peak_time_s[-1] - peak_time_s[0]
        if duration_s > window_s:
            _,_,hrv_freqs = physio.compute_hrv_psd(ecg_peaks, window_s = window_s, frequency_bands=frequency_bands, min_n_cycles_lowest_freq = 2)
            hrv_freqs = hrv_freqs.to_dict()
            hrv_freqs['vlfn'] = hrv_freqs['vlf'] /  hrv_freqs['total'] * 100
            hrv_freqs['lfn'] = hrv_freqs['lf'] /  hrv_freqs['total'] * 100
            hrv_freqs['hfn'] = hrv_freqs['hf'] /  hrv_freqs['total'] * 100
            hrv_freqs['lf_hf'] = hrv_freqs['lf'] / hrv_freqs['hf']

    return  hrv_freqs


def get_hrv_freq_metrics_from_da(da):
    hrv_freqs = da.to_series().to_dict()
    hrv_freqs['vlfn'] = np.nan
    hrv_freqs['lfn'] = np.nan
    hrv_freqs['hfn'] = np.nan

    if hrv_freqs['total'] is np.nan:
        return hrv_freqs
    else:
        for b in ['ulf','vlf','lf','hf']:
            hrv_freqs[f'{b}n'] = hrv_freqs[b] /  hrv_freqs['total'] * 100
        return  hrv_freqs

def get_resp_metrics(resp_cycles):
    if resp_cycles.shape[0] > 0:
        med_freq_resp, mad_freq_resp = compute_nanmedian_mad(resp_cycles['cycle_freq_cpm'].values)
    else:
        med_freq_resp, mad_freq_resp = np.nan, np.nan
    return dict(med_freq_resp=med_freq_resp, mad_freq_resp=mad_freq_resp)

def get_rsa_metrics(rsa_cycles):
    if rsa_cycles.shape[0] > 0:
        med_rsa = np.nanmedian(rsa_cycles['rsa_amplitude_clean'].values)
    else:
        med_rsa = np.nan
    return dict(med_rsa = med_rsa)

def get_icp_metrics(icp_cycles):
    if icp_cycles.shape[0] > 0:
        med_icp = np.nanmedian(icp_cycles['icp_mean'].values)
        med_icp_pulse = np.nanmedian(icp_cycles['rise_amplitude'].values)
    else:
        med_icp, med_icp_pulse = np.nan, np.nan
    return dict(med_icp=med_icp, med_icp_pulse=med_icp_pulse)

In [ ]:
meta = get_metadata().set_index('ID_pseudo')

In [ ]:
pd.Series({sub:1 if meta.at[sub,'mRS_sortie'] != 6 else 0 for sub in get_patient_list(['ECG_II','CO2','ICP'], threshold_duration_mins = 1 * 60 * 24) if not sub in ['P19','P9']}).value_counts()

In [ ]:
# slicing_mode = 'first_day'
slicing_mode = 'all_stay'
n_segments = 12

def load_hrv_freq_sub(sub):
    da = hrv_freq_job.get(sub)['hrv_freq']
    da['datetime'] = da.attrs['time']
    da = da.rename({'datetime':'time'})
    return da

def load_hrv_slope_sub(sub):
    da = hr_spectrum_slope_job.get(sub)['hr_spectrum_slope']
    da['datetime'] = da.attrs['time']
    da = da.rename({'datetime':'time'})
    return da

def load_hrv_center_freq_sub(sub):
    da = hr_center_frequency_job.get(sub)['hr_center_frequency']
    da['datetime'] = da.attrs['time']
    da = da.rename({'datetime':'time'})
    return da


subs = get_patient_list(['ECG_II','CO2','ICP'], threshold_duration_mins = 1 * 60 * 24)
mapper_alive_or_dead = {sub:1 if meta.at[sub,'mRS_sortie'] != 6 else 0 for sub in subs if not sub in ['P19','P9']}
mapper_dead_or_alive = {sub:0 if meta.at[sub,'mRS_sortie'] != 6 else 1 for sub in subs if not sub in ['P19','P9']}
subs = list(mapper_alive_or_dead.keys())

rows = []

period_durations = pd.Series(dtype = float)

for sub in subs:
    ecg_peaks = detect_ecg_job.get(sub).to_dataframe()
    rsa = rsa_job.get(sub).to_dataframe()
    resp_cycles = detect_resp_job.get(sub).to_dataframe()
    icp_cycles = detect_icp_job.get(sub).to_dataframe()
    icp_cycles['icp_mean'] = (1/3) * icp_cycles['amplitude_at_peak'] + (2/3) * icp_cycles['amplitude_at_trough']
    # print(sub)
    hrv_freq_da = load_hrv_freq_sub(sub)
    hrv_slope_da = load_hrv_slope_sub(sub)
    hrv_center_freq_da = load_hrv_center_freq_sub(sub)

    peak_times_s = ecg_peaks['peak_time'].values
    peak_date = ecg_peaks['peak_date'].values
    tmin = peak_times_s[0]
    if slicing_mode == 'first_day':
        tmax = tmin + 24 * 3600
    elif slicing_mode == 'all_stay':
        tmax = peak_times_s[-1]
    
    time_slices = np.linspace(tmin,tmax,n_segments+1)
    t_starts = time_slices[:-1]
    t_stops = time_slices[1:]

    period_durations[sub] = np.diff(time_slices)[0]

    for i in range(t_starts.size):
        t_start = t_starts[i]
        # t_start = tmin
        t_stop = t_stops[i]

        local_ecg_peaks = ecg_peaks[(ecg_peaks['peak_time'] > t_start) & (ecg_peaks['peak_time'] <= t_stop)]
        local_rsa = rsa[(rsa['peak_time'] > t_start) & (rsa['peak_time'] <= t_stop)]
        local_resp = resp_cycles[(resp_cycles['inspi_time'] > t_start) & (resp_cycles['next_inspi_time'] <= t_stop)]
        local_icp = icp_cycles[(icp_cycles['peak_time'] > t_start) & (icp_cycles['peak_time'] <= t_stop)]
        local_hrv_freq = hrv_freq_da.loc[:,t_start:t_stop].median('time')
        local_hrv_slope = hrv_slope_da.loc[t_start:t_stop].median('time')
        local_hrv_center_freq = hrv_center_freq_da.loc[:,t_start:t_stop].median('time').loc['med_freq']

        d_hrv_time = get_hrv_time_metrics(local_ecg_peaks)
        # d_hrv_freq = get_hrv_freq_metrics(local_ecg_peaks)
        d_hrv_freq = get_hrv_freq_metrics_from_da(local_hrv_freq)
        d_hrv_slope = dict(hrv_slope = float(local_hrv_slope))
        d_hrv_center_freq = dict(center_freq = float(local_hrv_center_freq))
        d_resp = get_resp_metrics(local_resp)
        d_rsa = get_rsa_metrics(local_rsa)
        d_icp = get_icp_metrics(local_icp)

        d_meta = dict(sub = sub, period = i+1, alive_or_dead=mapper_alive_or_dead[sub], dead_or_alive=mapper_dead_or_alive[sub], d0 = peak_date[np.searchsorted(peak_times_s, t_start)], d1 =peak_date[np.searchsorted(peak_times_s, t_stop)])
        row = d_meta | d_hrv_time | d_hrv_freq | d_hrv_slope | d_hrv_center_freq | d_resp | d_rsa | d_icp
        rows.append(row)
res = pd.DataFrame(rows)

In [ ]:
pd.Series(mapper_dead_or_alive).value_counts()

In [ ]:
(period_durations / 3600).describe()[3:].round(2).to_dict()

In [ ]:
res

In [ ]:
def clean_res(df, col, n_mads = 5, verbose = False):
    df = df.dropna(subset = col)
    if (df[col] < 0).sum() < 50:
        values = np.log(df[col].values)
    else:
        values = df[col].values
    med, mad = compute_nanmedian_mad(values)
    mask = (values > med - n_mads * mad) & (values < med + n_mads * mad)
    df_clean = df.loc[mask,:]
    if verbose:
        print(df.shape[0] - df_clean.shape[0], 'rows removed among', df.shape[0], 'initially')
    if col in ['total','vlf','lf','hf']:
        df_clean.loc[:,col] = np.log(df_clean[col].values)
    return df_clean



In [ ]:
metrics = res.columns[6:]
cmap = 'viridis'
alpha = 0.5
n_mads_cleaning = 5

mapper_colname = {
    'med_rri': 'alive_or_dead',
    'mad_rri': 'alive_or_dead',
    'total': 'alive_or_dead',
    'ulf': 'alive_or_dead',
    'vlf': 'alive_or_dead',
    'lf': 'alive_or_dead',
    'hf': 'alive_or_dead',
    'ulfn': 'alive_or_dead',
    'vlfn': 'alive_or_dead',
    'lfn': 'alive_or_dead',
    'hfn': 'dead_or_alive',
    'lf_hf': 'alive_or_dead',
    'hrv_slope': 'dead_or_alive',
    'center_freq': 'dead_or_alive',
    'med_freq_resp': 'alive_or_dead',
    'mad_freq_resp': 'alive_or_dead',
    'med_rsa': 'alive_or_dead',
    'med_icp': 'dead_or_alive',
    'med_icp_pulse': 'dead_or_alive'
 }

from sklearn.metrics import roc_curve, auc

periods = res['period'].unique()
xticks_period = np.arange(2,len(periods),2)
colors = colormaps[cmap].resampled(len(periods))
colors = {element: colors(i) for i, element in enumerate(periods)}

nrows = len(metrics)
ncols = 8

fig, axs = plt.subplots(nrows= nrows, ncols = ncols, figsize=(ncols * 3, nrows * 3), constrained_layout = True)
duration_period_stats = (period_durations / 3600).describe()[3:].round(2).to_dict()
fig.suptitle(f'Duration of periods : {duration_period_stats}', weight = 'bold')

for r, metric in enumerate(metrics):
    print(metric)
    res2 = clean_res(res, metric, n_mads = n_mads_cleaning, verbose = True)
    assert res2[mapper_colname[metric]].value_counts().min() >= 1

    roc_aucs = np.zeros(periods.shape)
    stats = np.zeros(periods.shape)
    pvals = np.zeros(periods.shape)
    best_thresholds = pd.Series(index = periods, dtype = float)

    ax = axs[r,0]
    ax.set_title('ROC Curve')
    ax.plot([0, 1], [0, 1], 'k--')  # ligne aléatoire
    ax.set_xlabel('False Positive Rate')
    mapper_ylabel = '1 - dead ; 0 - alive' if mapper_colname[metric] == 'dead_or_alive' else '1 - alive ; 0 - dead'
    ax.set_ylabel(f"{metric}\n{mapper_ylabel}\nTrue Positive Rate", weight = 'bold', fontsize = 12)
    # ax.legend(fontsize = 7, loc = 'lower right')
    ax.grid()

    for i, period in enumerate(periods):
        res_period = res2[res2['period'] == period]
        X = res_period[metric].values
        y = res_period[mapper_colname[metric]].values
        fpr, tpr, thresholds = roc_curve(y, X)
        res_stats = scipy.stats.mannwhitneyu(X[y == 1], X[y==0])
        roc_aucs[i] = auc(fpr, tpr)
        stats[i] = res_stats.statistic
        pvals[i] = res_stats.pvalue
        ax.plot(fpr, tpr, label=f'Period {period} (AUC = {roc_aucs[i]:.2f})', color = colors[period], alpha=alpha)
        axs[r,1].plot(thresholds, tpr, label=f'TPR' if i == 0 else None, color = 'g', alpha = alpha)
        axs[r,1].plot(thresholds, fpr, label=f'FPR' if i == 0 else None, color = 'r', alpha = alpha)

        distances = np.sqrt((fpr)**2 + (1 - tpr)**2)
        best_idx = np.argmin(distances)
        best_threshold = thresholds[best_idx]
        best_thresholds[period] = best_threshold

    ax = axs[r,1]
    ax.legend(loc = 'upper right')
    ax.set_xlabel(f'Thresholds of {metric}')

    ax = axs[r,2]
    ax.plot(best_thresholds.index, best_thresholds.values, color = 'k')
    ax.scatter(best_thresholds.index, best_thresholds.values, color = list(colors.values()))
    ax.set_ylabel('Best threshold')
    ax.set_xlabel('Period')
    ax.set_xticks(xticks_period)

    ax = axs[r,3]
    ax.plot(range(1, roc_aucs.size+1), roc_aucs, color = 'k')
    ax.scatter(range(1, roc_aucs.size+1), roc_aucs, color = list(colors.values()))
    ax.set_ylim(0.3, 1)
    ax.set_title('ROC AUCS')
    ax.set_xlabel('Period')
    ax.set_xticks(xticks_period)

    ax = axs[r,4]
    ax.plot(range(1, stats.size+1), stats, color = 'k')
    ax.scatter(range(1, roc_aucs.size+1), stats, color = list(colors.values()))
    ax.set_ylim(150, 600)
    ax.set_title('Mann Whitney results')
    ax.set_xlabel('Period')
    ax.set_xticks(xticks_period)

    ax = axs[r,5]
    ax.scatter(stats, roc_aucs, color = list(colors.values()))
    ax.set_xlabel('Mann Whitney : stat')
    ax.set_ylabel('ROC AUC')
    ax.set_title('ROC AUC vs Mann Whitney')
    ax.set_ylim(0.3, 1)
    ax.set_xlim(150, 600)

    ax = axs[r,6]
    ax.scatter(pvals, roc_aucs, color = list(colors.values()))
    ax.set_xlabel('Mann Whitney : pval')
    ax.set_ylabel('ROC AUC')
    ax.set_title('ROC AUC vs Mann Whitney')
    ax.set_xlim(-0.1, 1)
    ax.set_ylim(0.3, 1)
    ax.axvline(0.05, color = 'r')

    ax = axs[r,7]
    sns.pointplot(data = res2,
                x = 'period',
                y = metric,
                hue = 'alive_or_dead',
                estimator = 'median',
                errorbar=("pi",50),
                ax=ax
    )

fig.savefig(base_folder / 'figures' / 'autres' / f'ROC_HRV_REA_{slicing_mode}.png', dpi = 300, bbox_inches = 'tight')
fig.show()

In [ ]:
mapper_alive_or_dead

In [ ]:
sub = 'MF12'
da = hr_spectro_job.get(sub)['hr_spectro']

In [ ]:
figsize = (12,4)
fig, ax = plt.subplots(figsize = figsize)

for sub in get_patient_list(['ECG_II']):
    da = hr_spectro_job.get(sub)['hr_spectro']
    try:
        c = 'r' if meta.loc[sub, 'mRS_sortie'] == 6 else 'g' 
        ax.plot(da['freq'], (da.cumsum('freq') / da.sum('freq')).median('datetime'), color = c, alpha = 0.2)
    except:
        continue

In [ ]:
(da.cumsum('freq') / da.sum('freq')).plot.pcolormesh(x = 'datetime', y = 'freq', figsize = (12,4), vmin = 0, vmax = 0.95)

In [ ]:
np.log(da).plot.pcolormesh(x = 'datetime', y = 'freq', robust = True)

# Cross corr ihr abp

In [ ]:
from hrv_freq_jobs import load_abp_ihr

In [ ]:
p = {
    'decimate_abp_factor':50,
    'n_cycles':(5,50),
    'f_start':0.002, 
    'f_stop':0.4, 
    'n_steps':100,
    'smooth_factor':2,
    'decimate_coherence_matrix_factor':100,
}

In [ ]:
sub = 'MF12'

In [ ]:
ds = load_abp_ihr(sub, p['decimate_abp_factor'])

In [ ]:
ds

In [ ]:
da = ds['da_time']
srate = da.attrs['srate']
srate

In [ ]:
%matplotlib widget

In [ ]:
# lowcut = 0.001
# highcut = 0.01
lowcut = 0.25
highcut = 0.4
figsize = (11,4)
ftype = 'bessel'
order = 5

ihr_filt = iirfilt(da.loc['ihr',:].values, srate, lowcut, highcut, ftype = ftype, order=order, show = True)
abp_filt = iirfilt(da.loc['abp',:].values, srate, lowcut, highcut, ftype = ftype, order=order)
# abp_filt = ihr_filt.copy()
# ihr_filt = abp_filt.copy()
# ihr_filt = get_amp(ihr_filt)
# abp_filt = get_amp(abp_filt)

mode = 'same'
corr = scipy.signal.correlate(ihr_filt,abp_filt, mode=mode)
lags = scipy.signal.correlation_lags(len(ihr_filt), len(abp_filt), mode=mode) / srate
# corr /= np.max(corr)
# corr /= corr.size

corr = corr / np.sqrt(np.dot(ihr_filt, ihr_filt) * np.dot(abp_filt, abp_filt))

print(ihr_filt.shape)
print(corr.shape)

times = da['time'].values
fig, ax = plt.subplots(figsize=figsize)
ax.plot(times, ihr_filt, color = 'r', label = 'ihr')
ax.plot(times, abp_filt, color = 'g', label = 'abp')
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=figsize)
ax.plot(lags, corr)
ax.set_xlim(-50, 50)
lag_max = lags[np.argmax(corr)]
ax.axvline(lag_max, color = 'r')
ax.set_title(f'max lag : {round(lag_max, 3)} (< 0 = ihr before abp ; >0 = abp before ihr)')

In [ ]:
p = {
    'decimate_abp_factor':50,
    # 'n_cycles':10,
        'n_cycles':(5,50),
    'f_start':0.01, 
    'f_stop':0.4, 
    'n_steps':30,
    'smooth_factor':2,
    'decimate_coherence_matrix_factor':100,
}

_, _, freqs, family = morlet_family(srate, f_start = p['f_start'], f_stop = p['f_stop'], n_steps = p['n_steps'], n_cycles = p['n_cycles'])


raw_ihr = da.loc['ihr',:].values
raw_abp = da.loc['abp',:].values

tf_ihr = signal.fftconvolve(np.tile(raw_ihr, (p['n_steps'],1)), family, mode = 'same', axes = 1)
tf_abp = signal.fftconvolve(np.tile(raw_abp, (p['n_steps'],1)), family, mode = 'same', axes = 1)

In [ ]:
freqs

In [ ]:
fig, ax = plt.subplots()
ax.pcolormesh(times[::100], freqs, np.log(np.abs(tf_ihr)[:,::100]), cmap = 'nipy_spectral')

In [ ]:
# ihr_filts = np.real(tf_ihr)
# abp_filts = np.real(tf_abp)

ihr_filts = np.abs(tf_ihr)
abp_filts = np.abs(tf_abp)

In [ ]:
max_corrs = np.zeros(freqs.shape)
for i in range(freqs.size):
    ihr_filt = ihr_filts[i,:]
    abp_filt = abp_filts[i,:]
    corr = scipy.signal.correlate(ihr_filt,abp_filt, mode='full')
    if i == 0:
        lags = scipy.signal.correlation_lags(len(ihr_filt), len(abp_filt)) / srate
    max_corrs[i] = lags[np.argmax(corr)]

In [ ]:
fig, ax = plt.subplots()
ax.plot(freqs, max_corrs)
ax.set_ylabel('max lag')
ax.set_xlabel('Freq (hz)')
ax.set_ylim(-10, 10)
ax.axhline(0, color = 'r', ls = '--')

# See spectral_causality_ihr_abp_job

In [ ]:
sub = 'P77'

In [ ]:
da = spectral_causality_ihr_abp_job.get(sub)['spectral_causality_ihr_abp']

In [ ]:
da[:-1,:,:].plot.pcolormesh(x = 'time', y ='freq', row = 'direction', figsize = (12,6), robust = True)
plt.show()
da[-1,:,:].plot.pcolormesh(x = 'time', y ='freq', figsize = (12,3), robust = True)

# See depolarisations pour Roumaïssa

## MF12

In [ ]:
sub = 'MF12'
sd_dates = get_sd_dates(sub)
sd_dates

In [ ]:
down_factor = 1000

reader = pycns.CnsReader(data_path / sub)
eeg_stream = reader.streams['EEG']
chans = eeg_stream.channel_names
ecog_chans = [ch for ch in chans if 'ECoG' in ch]
srate = eeg_stream.sample_rate
eeg_da = xr.DataArray(data = eeg_stream.get_data(apply_gain = False),
                      dims = ['datetime','chan'],
                      coords = {'datetime':eeg_stream.get_times(), 'chan':chans},
                      attrs = {'srate':srate}
)
eeg_da = eeg_da.loc[::down_factor,ecog_chans]
eeg_da.attrs['srate'] = srate / down_factor

In [ ]:
sd_detected = depolarization_cleaning_job.get(sub).to_dataframe()
sd_detected

In [ ]:
eeg_da2 = eeg_da.copy()
eeg_da2 = eeg_da2 - eeg_da2.median('datetime')

In [ ]:
fig, ax = plt.subplots(figsize = (10, 3), constrained_layout = True)

for r, ch in enumerate(ecog_chans):
    ax.plot(eeg_da['datetime'], eeg_da.loc[:,ch], label = ch)
ax.legend(loc = 'upper right')

for i, sd in sd_dates.iterrows():
    start = sd['start_time']
    start = np.datetime64(start)
    duration = int(sd['duration'])
    stop = start + np.timedelta64(duration, 's')
    ax.axvspan(start, stop, color = 'g', alpha = 0.2)

for i, sd in sd_detected.iterrows():
    ax.axvspan(sd['depol_start_date'], sd['depol_stop_date'], color = 'r', alpha = 0.2)

## GA9

In [ ]:
sub = 'GA9'
sd_dates = get_sd_dates(sub)
sd_dates

In [ ]:
down_factor = 1000

reader = pycns.CnsReader(data_path / sub)
eeg_stream = reader.streams['EEG']
chans = eeg_stream.channel_names
ecog_chans = [ch for ch in chans if 'ECoG' in ch]
srate = eeg_stream.sample_rate
eeg_da = xr.DataArray(data = eeg_stream.get_data(apply_gain = False),
                      dims = ['datetime','chan'],
                      coords = {'datetime':eeg_stream.get_times(), 'chan':chans},
                      attrs = {'srate':srate}
)
eeg_da = eeg_da.loc[::down_factor,ecog_chans]
eeg_da.attrs['srate'] = srate / down_factor

In [ ]:
sd_detected = depolarization_cleaning_job.get(sub).to_dataframe()
sd_detected

In [ ]:
eeg_da2 = eeg_da.copy()
eeg_da2 = eeg_da2 - eeg_da2.median('datetime')

In [ ]:
fig, ax = plt.subplots(figsize = (10, 3), constrained_layout = True)

for r, ch in enumerate(ecog_chans):
    ax.plot(eeg_da['datetime'], eeg_da.loc[:,ch], label = ch)
ax.legend(loc = 'upper right')

for i, sd in sd_dates.iterrows():
    start = sd['start_time']
    start = np.datetime64(start)
    duration = int(sd['duration'])
    stop = start + np.timedelta64(duration, 's')
    ax.axvspan(start, stop, color = 'g', alpha = 0.2)

for i, sd in sd_detected.iterrows():
    ax.axvspan(sd['depol_start_date'], sd['depol_stop_date'], color = 'r', alpha = 0.2)

# Machine learning HRV * mRS

## Subs selection

In [ ]:
subs = get_patient_list(['ECG_II'])
mrs = get_metadata().set_index('ID_pseudo').loc[:,'mRS_sortie'].to_dict()
outcome = {sub:'alive' if score < 6 else 'dead' for sub,score in mrs.items()}

In [ ]:
sub = 'MF12'
hrv_freqs_da = hrv_freq_job.get(sub)['hrv_freq']

In [ ]:

hrv_freqs = None
for sub in subs:
    hrv_freqs_sub = hrv_freq_job.get(sub)['hrv_freq']
    hrv_freqs_sub = hrv_freqs_sub / hrv_freqs_sub.sel(band = 'total')
    hrv_freqs_sub = hrv_freqs_sub.median('datetime')
    coords = {'sub':subs, 'band':hrv_freqs_sub['band']}
    if hrv_freqs is None:
        hrv_freqs = xr.DataArray(data = np.nan,
                                 dims = coords.keys(),
                                 coords = coords,
                                 )
    hrv_freqs.loc[sub,:] = hrv_freqs_sub



In [ ]:
df = hrv_freqs.to_dataframe(name = 'power').reset_index()
df['outcome'] = df['sub'].map(outcome)

In [ ]:
df

In [ ]:
sns.boxplot(data = df[~df['band'].isin(['ulf','lf_hf','total'])],
            x = 'band',
            y = 'power',
            hue = 'outcome')


In [ ]:
sel_bands = ['vlf','lf','hf']
df2 = df.pivot(index = 'sub', columns = 'band', values = 'power')
df2 = df2.loc[:,sel_bands]
mrs_and_df2_subs = [s for s in df2.index if s in outcome.keys()]
df2 = df2.loc[mrs_and_df2_subs,:]

In [ ]:
X = df2.values
y = np.array([1 if outcome[s] == 'alive' else 0 for s in df2.index])

In [ ]:
X

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
clf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs = 10)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

print(accuracy_score(y_test, y_pred))

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
# cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
# disp = ConfusionMatrixDisplay(confusion_matrix=cm_percent )
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
disp = ConfusionMatrixDisplay(confusion_matrix=cm )
disp.plot(cmap='Blues')

importances = clf.feature_importances_

# Affichage
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sel_bands, importances)
ax.set_xlabel("band")
# ax.set_xticks(range(len(cols_features)), labels= cols_features, rotation = 45)
ax.set_ylabel("Importance")
ax.grid(True)
fig.show()

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

In [ ]:
cv = KFold(n_splits=10)
cross_val_score(clf, X, y, cv=cv)

In [ ]:
cross_val_score(clf, X, y, cv=10, )

# Correlation HRV ABp spectrograms

In [ ]:
def load_abp_ihr(sub, decimate_abp_factor):
    reader = pycns.CnsReader(data_path / sub)
    raw_abp, abp_datetimes = reader.streams['ABP'].get_data(with_times = True, apply_gain = True, time_as_second = False)
    abp_times = reader.streams['ABP'].get_times(as_second = True)
    abp_srate = reader.streams['ABP'].sample_rate

    ihr_da = ihr_job.get(sub)['ihr']
    ihr_sig = ihr_da.values
    ihr_srate = ihr_da.attrs['srate']
    ihr_datetimes = ihr_da['datetime'].values
    ihr_times = ihr_da.attrs['time']

    min_date = max([abp_datetimes[0], ihr_datetimes[0]])
    max_date = min([abp_datetimes[-1], ihr_datetimes[-1]])

    mask_abp = (abp_datetimes > min_date) & (abp_datetimes < max_date)
    abp_datetimes = abp_datetimes[mask_abp]
    abp_times = abp_times[mask_abp]
    raw_abp = raw_abp[mask_abp]

    mask_ihr = (ihr_datetimes > min_date) & (ihr_datetimes < max_date)
    ihr_datetimes = ihr_datetimes[mask_ihr]
    ihr_times = ihr_times[mask_ihr]
    ihr_sig = ihr_sig[mask_ihr]

    f = scipy.interpolate.interp1d(ihr_times, ihr_sig, kind='linear', axis=0, bounds_error=False, fill_value='extrapolate')
    ihr_sig_up = f(abp_times)

    raw_abp = scipy.signal.decimate(raw_abp, q = decimate_abp_factor)
    raw_ihr = scipy.signal.decimate(ihr_sig_up, q = decimate_abp_factor)
    times = abp_times[::decimate_abp_factor]
    datetimes = abp_datetimes[::decimate_abp_factor].astype('datetime64[ns]')
    srate = abp_srate / decimate_abp_factor

    raw_data_datetime = xr.DataArray(data = np.concatenate([raw_abp[None,:], raw_ihr[None,:]], axis = 0),
                            coords = {'raw_sig':['abp','ihr'], 'datetime':datetimes},
                            attrs = {'srate':srate})
    raw_data_time = xr.DataArray(data = np.concatenate([raw_abp[None,:], raw_ihr[None,:]], axis = 0),
                            coords = {'raw_sig':['abp','ihr'], 'time':times},
                            attrs = {'srate':srate})
    
    ds = xr.Dataset()
    ds['da_time'] = raw_data_time
    ds['da_datetime'] = raw_data_datetime
    return ds

In [ ]:
sub = 'P4'
get_patient_dates(sub)

In [ ]:
t0 = '2022-09-10T17:16'
t1 = '2022-09-14T17:16'

In [ ]:
decimate_abp_factor = 50
raw_data_ds = load_abp_ihr(sub, decimate_abp_factor=decimate_abp_factor)

In [ ]:
raw_data_ds

In [ ]:
raw_data_ds['da_time']

In [ ]:
raw_data_ds['da_datetime']

In [ ]:
def morlet_power2(sig, srate, f_start, f_stop, n_steps, n_cycles, amplitude_exponent=2, duration_s = 'auto', return_mode = 'real'):
    """
    Compute time-frequency matrix by convoluting wavelets on a signal
    
    ------------------------------
    Inputs =
    - sig : the signal (1D np vector)
    - srate : sampling rate
    - f_start : lowest frequency of the wavelet family
    - f_stop : highest frequency of the wavelet family
    - n_steps : number of frequencies from f_start to f_stop
    - n_cycles : number of waves in the wavelet
    - amplitude_exponent : amplitude values extracted from the length of the complex vector will be raised to this exponent factor (default = 2 = V**2 as unit)

    Outputs = 
    - freqs : frequency 1D np vector
    - power : 2D np array , axis 0 = freq, axis 1 = time

    """
    _, _, freqs, family = morlet_family(srate, f_start = f_start, f_stop = f_stop, n_steps = n_steps, n_cycles = n_cycles, duration_s=duration_s)
    sigs = np.tile(sig, (n_steps,1))
    tf = signal.fftconvolve(sigs, family, mode = 'same', axes = 1)
    if return_mode == 'power':
        return_arr = np.abs(tf) ** amplitude_exponent
    elif return_mode == 'real':
        return_arr = np.real(tf)
    return xr.DataArray(data = return_arr, coords = {'freq':freqs, 'time':np.arange(sig.size) / srate})

In [ ]:
srate = raw_data_ds['da_time'].attrs['srate']
f_start = 0.002
f_stop = 0.40
n_steps = 100
n_cycles = (5, 50)
return_mode = 'power'
amplitude_exponent = 1

real_da_abp = morlet_power2(sig=raw_data_ds['da_time'].loc['abp'].values,
                            srate=srate,
                            f_start=f_start,
                            f_stop=f_stop,
                            n_steps=n_steps,
                            n_cycles=n_cycles,
                            return_mode=return_mode,
                            amplitude_exponent=amplitude_exponent,
                            )
real_da_ihr = morlet_power2(sig=raw_data_ds['da_time'].loc['ihr'].values,
                            srate=srate,
                            f_start=f_start,
                            f_stop=f_stop,
                            n_steps=n_steps,
                            n_cycles=n_cycles,
                            return_mode=return_mode,
                            amplitude_exponent=amplitude_exponent,
                            )
real_da = xr.concat([real_da_abp, real_da_ihr], dim = 'sig_type')
real_da.coords['sig_type'] = ['abp','ihr']
real_da['time'] = raw_data_ds['da_datetime']['datetime'].values

In [ ]:
real_da['time']

In [ ]:
t0 = "2020-03-11T16:35"
t1 = "2020-03-11T16:40"
real_da.loc[:,0.28:0.32,t0:t1].plot.line(x = 'time', hue = 'sig_type', row = 'freq', figsize = (12,20), ylim=(-1, 8))

In [ ]:
t0 = "2020-03-12T16:30"
t1 = "2020-03-12T16:35"
real_da.loc[:,0.28:0.32,t0:t1].plot.line(x = 'time', hue = 'sig_type', row = 'freq', figsize = (12,20))

In [ ]:
raw_data_ds['da_time'].differentiate('time').loc[:,1000:1120].plot.line(x = 'time')
# raw_data_ds['da_time'].differentiate('time').loc[:,:].plot.line(x = 'time')

In [ ]:
raw_local = raw_data_ds['da_datetime'].loc[:,t0:t1]
raw_abp = raw_local.loc['abp'].values
raw_ihr = raw_local.loc['ihr'].values
srate = raw_data_ds['da_datetime'].attrs['srate']
local_datetimes = raw_local['datetime'].values

n_cycles = (5,50)
da_mw = morlet_coherence(sig1= raw_abp, sig2=raw_ihr, srate=srate, f_start = 0.002, f_stop = 0.4, n_steps = 100, n_cycles = n_cycles, smooth_factor=2)
da_mw['time'] = local_datetimes


In [ ]:
da_mw.loc['Cxy',:,:][:,::100].plot(cmap = 'nipy_spectral', vmin = 0, vmax = 1, figsize = (12,4))

In [ ]:
def compute_granger_causality(sig1, sig2, lags, test_name_chosen = 'lrtest', apply_diff = True):
    data = pd.DataFrame()
    if apply_diff:
        data['sig1'] = np.diff(sig1)
        data['sig2'] = np.diff(sig2)
    else:
        data['sig1'] = sig1
        data['sig2'] = sig2

    res = grangercausalitytests(data, maxlag=lags, verbose = False)
    rows = []
    for lag in lags:
        LR, p, df = res[lag][0][test_name_chosen]
        d = dict(
            lag=lag,
            stat = LR,
            p_value=p,
            df=df
        )
        rows.append(d)
    res = pd.DataFrame(rows)
    return res

In [ ]:
%matplotlib widget

def norm(sig):
    med, mad = physio.compute_median_mad(sig)
    return (sig - med) / mad

raw_local = raw_data_ds['da_datetime'].loc[:,t0:t1]
raw_abp = raw_local.loc['abp'].values
raw_ihr = raw_local.loc['ihr'].values
srate = raw_data_ds['da_datetime'].attrs['srate']
local_datetimes = raw_local['datetime'].values

ihr_granger = raw_ihr.copy()
abp_granger = raw_abp.copy()
lowcut = 0.001
highcut = 0.05
ihr_granger = iirfilt(ihr_granger, srate, lowcut=lowcut, highcut=highcut)
abp_granger = iirfilt(abp_granger, srate, lowcut=lowcut, highcut=highcut)
ihr_granger = norm(ihr_granger)
abp_granger = norm(abp_granger)

fig, ax = plt.subplots()
ax.plot(local_datetimes, ihr_granger, color = 'r', label = 'ihr')
ax.plot(local_datetimes, abp_granger, color = 'g', label = 'abp')
ax.legend()

# lags_s = np.array([10, 20])
lags_s = np.arange(1,10)
lags = (lags_s * srate).astype(int)

res_abp_ihr = compute_granger_causality(ihr_granger, abp_granger, lags)
res_ihr_ab = compute_granger_causality(abp_granger, ihr_granger, lags)

fig, ax = plt.subplots()
ax.plot(res_abp_ihr['lag'], res_abp_ihr['stat'], label = 'abp -> ihr')
ax.plot(res_abp_ihr['lag'], res_ihr_ab['stat'], label = 'ihr -> abp')
ax.legend()

fig, ax = plt.subplots()
ax.plot(res_abp_ihr['lag'], res_ihr_ab['stat'] - res_abp_ihr['stat'], label = 'ihr-abp - abp-ihr')
ax.legend()

In [ ]:
def compute_granger_causality_short(target, predictor, lags, test_name_chosen = 'lrtest', apply_diff = False):
    data = pd.DataFrame()
    if apply_diff:
        data['target'] = np.diff(target)
        data['predictor'] = np.diff(predictor)
    else:
        data['target'] = target
        data['predictor'] = predictor

    res = grangercausalitytests(data, maxlag=lags, verbose = False)
    statistics = [res[i][0][test_name_chosen][0] for i in lags]
    pvalues = [res[i][0][test_name_chosen][1] for i in lags]
    return statistics, pvalues

def morlet_causality(dict_data, srate, f_start, f_stop, n_steps, n_cycles, lags, apply_diff = True, test_name_chosen = 'lrtest', sigma_s = 'auto', smooth_factor = None, conv_feature = 'power', zscore = False, progress_bar = False):
    _, n_cycles_vector, freqs, family = morlet_family(srate, f_start = f_start, f_stop = f_stop, n_steps = n_steps, n_cycles = n_cycles, duration_s='auto')
    
    lags_s = lags / srate
    label_a, label_b = dict_data.keys()
    sig_a, sig_b = dict_data.values()
    times = np.arange(dict_data[label_a].size) / srate
    coords = {'sig':[label_a,label_b], 
              'freq':freqs, 
              'time':times
              }
    data = xr.DataArray(data = np.nan , dims = coords.keys(), coords = coords)
    a_tf = scipy.signal.fftconvolve(np.tile(sig_a, (n_steps,1)), family, mode = 'same', axes = 1)
    b_tf = scipy.signal.fftconvolve(np.tile(sig_b, (n_steps,1)), family, mode = 'same', axes = 1)

    if conv_feature == 'power':
        conv_a = np.abs(a_tf)**2
        conv_b = np.abs(b_tf)**2
    elif conv_feature == 'real':
        conv_a = np.real(a_tf)
        conv_b = np.real(b_tf)
    elif conv_feature == 'angle':
        conv_a = np.angle(a_tf)
        conv_b = np.angle(b_tf)

    if smooth_factor is None:
        data.loc[label_a,:,:] = conv_a
        data.loc[label_b,:,:] = conv_b
    else:
        sigma_s = smooth_factor * n_cycles_vector / freqs
        data.loc[label_a,:,:] = smooth_per_freq(conv_a, srate, sigma_s)
        data.loc[label_b,:,:] = smooth_per_freq(conv_b,srate, sigma_s)

    if zscore:
        data = (data - data.mean('time')) / data.std('time')

    coords = {'direction':[f'{label_a}_cause_{label_b}',f'{label_b}_cause_{label_a}'], 'freq':freqs, 'gc_feature':['statistic','pvalue'],'lag':lags}
    da_granger = xr.DataArray(data = np.nan, dims = coords.keys(), coords = coords)
    da_granger.attrs['lags_s'] = lags_s
    
    loop = range(freqs.size)
    if progress_bar:
        loop = tqdm(loop)
    for i in loop:
        f = freqs[i]
        for direction in da_granger['direction'].values:
            letter_causal, letter_consequence = direction.split('_cause_')
            # try:
            statistics, pvalues = compute_granger_causality_short(data.loc[letter_consequence,f,:].values, data.loc[letter_causal,f,:].values, lags=lags, apply_diff = apply_diff, test_name_chosen=test_name_chosen)
            da_granger.loc[direction,f,'statistic',:] = statistics
            da_granger.loc[direction,f,'pvalue',:] = pvalues
            # except:
            #     print(direction, f)
            #     continue

    return data, da_granger


In [ ]:
%matplotlib widget

def norm(sig):
    med, mad = physio.compute_median_mad(sig)
    return (sig - med) / mad

t0 = '2022-09-12T17:16'
t1 = '2022-09-14T17:16'

raw_local = raw_data_ds['da_datetime'].loc[:,t0:t1]
raw_abp = raw_local.loc['abp'].values
raw_ihr = raw_local.loc['ihr'].values
srate = raw_data_ds['da_datetime'].attrs['srate']
local_datetimes = raw_local['datetime'].values

ihr_granger = raw_ihr.copy()
abp_granger = raw_abp.copy()
# lowcut = 0.001
# highcut = 0.05
# ihr_granger = iirfilt(ihr_granger, srate, lowcut=lowcut, highcut=highcut)
# abp_granger = iirfilt(abp_granger, srate, lowcut=lowcut, highcut=highcut)
# ihr_granger = norm(ihr_granger)
# abp_granger = norm(abp_granger)

fig, ax = plt.subplots()
ax.plot(local_datetimes, ihr_granger, color = 'r', label = 'ihr')
ax.plot(local_datetimes, abp_granger, color = 'g', label = 'abp')
ax.legend()

In [ ]:
raw_local

In [ ]:
from mne_connectivity import spectral_connectivity_epochs

fmin = 0.002
fmax = 0.40
n_freqs = 100

gc_n_lags_s = 5

n_jobs = 20

n_cycles_start = 5
n_cycles_stop = 50

indices_ihr_abp = ([[1]], [[0]])
indices_abp_ihr = ([[0]], [[1]])

cwt_freqs = np.linspace(fmin,fmax,n_freqs)
cwt_n_cycles = np.linspace(n_cycles_start,n_cycles_stop,n_freqs)
# cwt_n_cycles = 5
gc_n_lags = int(gc_n_lags_s * srate)

gc_ihr_abp = spectral_connectivity_epochs(data = raw_local.values[None,:,:],
                                                        names = raw_local['raw_sig'].values,
                                                        method = 'gc',
                                                        indices = indices_ihr_abp,
                                                        sfreq = srate,
                                                        mode = 'cwt_morlet',
                                                        fmin = cwt_freqs[0],
                                                        fmax = cwt_freqs[-1],
                                                        cwt_n_cycles = cwt_n_cycles,
                                                        cwt_freqs=cwt_freqs,
                                                        gc_n_lags = gc_n_lags,
                                                        n_jobs=n_jobs,
                                                        verbose = False
                                                        )

gc_abp_ihr = spectral_connectivity_epochs(data = raw_local.values[None,:,:],
                                                        names = raw_local['raw_sig'].values,
                                                        method = 'gc',
                                                        indices = indices_abp_ihr,
                                                        sfreq = srate,
                                                        mode = 'cwt_morlet',
                                                        fmin = cwt_freqs[0],
                                                        fmax = cwt_freqs[-1],
                                                        cwt_n_cycles = cwt_n_cycles,
                                                        cwt_freqs=cwt_freqs,
                                                        gc_n_lags = gc_n_lags,
                                                        n_jobs=n_jobs,
                                                        verbose = False
                                                        )

coords = {'direction':['ihr_predict_abp','abp_predict_ihr','ihrabp_minus_abp_ihr'], 'freq':gc_ihr_abp.freqs, 'time':gc_ihr_abp.times}
da_gc = xr.DataArray(data = np.nan, 
                     dims = coords.keys(),
                     coords = coords)
da_gc.loc['ihr_predict_abp'] = gc_ihr_abp.get_data()[0]
da_gc.loc['abp_predict_ihr'] = gc_abp_ihr.get_data()[0]
da_gc.loc['ihrabp_minus_abp_ihr'] = da_gc.loc['ihr_predict_abp'] - da_gc.loc['abp_predict_ihr']

In [ ]:
da_gc.sel(direction = ['ihr_predict_abp','abp_predict_ihr']).plot.pcolormesh(x = 'time', y ='freq', row = 'direction', robust=True, figsize = (12,8))
plt.show()
da_gc.loc['ihrabp_minus_abp_ihr',:,:].plot.pcolormesh(x = 'time', y ='freq', figsize = (12,4), robust=True)

In [ ]:
da_gc.sel(direction = ['ihr_predict_abp','abp_predict_ihr']).plot.pcolormesh(x = 'time', y ='freq', row = 'direction', robust=True, figsize = (12,8))
plt.show()
da_gc.loc['ihrabp_minus_abp_ihr',:,:].plot.pcolormesh(x = 'time', y ='freq', figsize = (12,4))

In [ ]:
da_gc.sel(direction = ['ihr_predict_abp','abp_predict_ihr']).plot.pcolormesh(x = 'time', y ='freq', row = 'direction', robust=True, figsize = (12,8))
plt.show()
da_gc.loc['ihrabp_minus_abp_ihr',:,:].plot.pcolormesh(x = 'time', y ='freq', figsize = (12,4))

In [ ]:
da_gc.sel(direction = ['ihr_predict_abp','abp_predict_ihr']).plot.pcolormesh(x = 'time', y ='freq', row = 'direction', robust=True, figsize = (12,8))
plt.show()
da_gc.loc['ihrabp_minus_abp_ihr',:,:].plot.pcolormesh(x = 'time', y ='freq', figsize = (12,4))

In [ ]:
%matplotlib inline
plt.close('all')

In [ ]:
srate

In [ ]:
dict_data = {'ihr':ihr_granger, 'abp':abp_granger}
srate=srate
f_start = 0.05
f_stop = 0.45
n_steps = 25
n_cycles = (5,50)
lags_s = np.arange(1,6,1)
lags = (lags_s * srate).astype(int)
print(lags_s)
print(lags)
apply_diff = False
smooth_factor = None
conv_feature = 'real'
progress_bar = True

data, da_granger = morlet_causality(dict_data=dict_data, srate=srate, f_start=f_start, f_stop=f_stop, n_steps=n_steps, n_cycles=n_cycles, lags=lags, apply_diff=apply_diff, smooth_factor=smooth_factor, conv_feature=conv_feature, zscore = False, progress_bar=progress_bar)

In [ ]:
conv_feature = 'angle'

_, n_cycles_vector, freqs, family = morlet_family(srate, f_start = f_start, f_stop = f_stop, n_steps = n_steps, n_cycles = n_cycles, duration_s='auto')

dict_data = {'ihr':ihr_granger, 'abp':abp_granger}

lags_s = lags / srate
label_a, label_b = dict_data.keys()
sig_a, sig_b = dict_data.values()
times = np.arange(dict_data[label_a].size) / srate
coords = {'sig':[label_a,label_b], 
            'freq':freqs, 
            'time':times
            }
data = xr.DataArray(data = np.nan , dims = coords.keys(), coords = coords)
a_tf = scipy.signal.fftconvolve(np.tile(sig_a, (n_steps,1)), family, mode = 'same', axes = 1)
b_tf = scipy.signal.fftconvolve(np.tile(sig_b, (n_steps,1)), family, mode = 'same', axes = 1)

if conv_feature == 'power':
    conv_a = np.abs(a_tf)**2
    conv_b = np.abs(b_tf)**2
elif conv_feature == 'real':
    conv_a = np.real(a_tf)
    conv_b = np.real(b_tf)
elif conv_feature == 'angle':
    conv_a = np.angle(a_tf)
    conv_b = np.angle(b_tf)

phase_diff_raw = conv_a - conv_b 
phase_diff = np.unwrap(phase_diff_raw, axis = 1)
mean_phase_diff = np.mean(phase_diff, axis = 1)
time_lag_sec = mean_phase_diff / (2 * np.pi * freqs)

print(freqs)
print(time_lag_sec)

fig, ax  = plt.subplots()
ax.plot(freqs, time_lag_sec)

In [ ]:
window_length = int(srate * 30)  # fenêtre de 30 secondes (ou moins)
step = int(window_length / 2)    # chevauchement 50%

time_lags_windows = []

for start in range(0, conv_a.shape[1] - window_length, step):
    phase_diff_win = phase_diff[:, start:start+window_length]
    mean_phase_diff_win = np.mean(phase_diff_win, axis=1)
    time_lag_sec_win = mean_phase_diff_win / (2 * np.pi * freqs)
    time_lags_windows.append(time_lag_sec_win)

time_lags_windows = np.array(time_lags_windows)  # shape (n_windows, n_freqs)

# Moyenne et écart-type par fréquence
mean_lag = np.mean(time_lags_windows, axis=0)
std_lag = np.std(time_lags_windows, axis=0)

print("Décalage moyen (s) par fréquence :", mean_lag)
print("Écart-type (s) par fréquence :", std_lag)


In [ ]:
conv_a

In [ ]:
conv_b

In [ ]:
time_lag_sec = mean_phase_diff / (2 * np.pi * freqs)

In [ ]:
fig, ax  = plt.subplots()
ax.plot(freqs, time_lag_sec)

In [ ]:
mean_phase_diff

In [ ]:
data.plot.pcolormesh(x = 'time', y = 'freq', row = 'sig', robust=  True, cmap = 'viridis', figsize = (12,4))

In [ ]:
da_granger.sel(gc_feature = 'statistic').plot.pcolormesh(x = 'lag', y = 'freq', col = 'direction', robust=  False, cmap = 'viridis')
plt.show()
(da_granger.sel(gc_feature = 'pvalue') < 0.05).plot.pcolormesh(x = 'lag', y = 'freq', col = 'direction', robust=  False, cmap = 'RdBu')
plt.show()
(da_granger.sel(gc_feature = 'statistic', direction = 'ihr_cause_abp') - da_granger.sel(gc_feature = 'statistic', direction = 'abp_cause_ihr')).plot.pcolormesh(x = 'lag', y = 'freq', robust=  True)

In [ ]:
da_granger.sel(gc_feature = 'statistic').plot.pcolormesh(x = 'lag', y = 'freq', col = 'direction', robust=  False, cmap = 'viridis')
plt.show()
(da_granger.sel(gc_feature = 'pvalue') < 0.05).plot.pcolormesh(x = 'lag', y = 'freq', col = 'direction', robust=  False, cmap = 'RdBu')
plt.show()
(da_granger.sel(gc_feature = 'statistic', direction = 'ihr_cause_abp') - da_granger.sel(gc_feature = 'statistic', direction = 'abp_cause_ihr')).plot.pcolormesh(x = 'lag', y = 'freq', robust=  True)

In [ ]:
da_granger

In [ ]:
res = grangercausalitytests(data.loc[['ihr','abp'],0.45,:].values.T, maxlag = 5, verbose = True)

In [ ]:
res[1]

In [ ]:
res2 = compute_granger_causality_short(target = data.loc['abp',0.45,:].values, predictor = data.loc['ihr',0.45,:].values, lags = np.arange(1,11,1))

In [ ]:
res2

In [ ]:
da_granger

In [ ]:
%matplotlib inline

In [ ]:
plt.figure()
data.loc[:,:,100:150].plot.line(x = 'time', hue = 'sig', col = 'freq')
plt.show()

In [ ]:
# data.plot.pcolormesh(col = 'sig', robust = True)
data.differentiate('time').plot.pcolormesh(col = 'sig', robust = True)
plt.show()

da_granger.plot.line(x = 'freq', hue = 'direction')
plt.show()

(da_granger.loc['ihr_cause_abp'] - da_granger.loc['abp_cause_ihr']).plot.line()


In [ ]:
(da_granger.loc['ihr_cause_abp'] - da_granger.loc['abp_cause_ihr']).plot.line()

In [ ]:
duration = 300

fig, ax = plt.subplots(figsize = (12,4))
ax.plot(abp_times, np.gradient(ihr_sig_up), color = 'r')
ax.plot(abp_times, np.gradient(raw_abp), color = 'g')
# ax2 = ax.twinx()
# ax2.plot(abp_times, np.gradient(raw_abp), color = 'g')
ax.set_xlim(abp_times[0], abp_times[0] + duration)
ax.set_ylim(-5,5)
# ax.set_ylim(55,75)

# ax2.set_ylim(100,130)

In [ ]:
n_cycles = (5,50)
freqs, Cxy = morlet_coherence(raw_abp, ihr_sig_up, abp_srate, f_start = 0.002, f_stop = 0.4, n_steps = 100, n_cycles = n_cycles, smooth_factor=2)
fig, ax = plt.subplots(figsize = (12,4))
ax.pcolormesh(abp_times, freqs, Cxy, vmin= 0, vmax = 1, cmap = 'nipy_spectral')
ax.set_title(n_cycles)

In [ ]:
n_cycles = (5,50)
freqs, Cxy = morlet_coherence(raw_abp, ihr_sig_up, abp_srate, f_start = 0.002, f_stop = 0.4, n_steps = 100, n_cycles = n_cycles, smooth_factor=2)
fig, ax = plt.subplots(figsize = (12,4))
ax.pcolormesh(abp_times, freqs, Cxy, vmin= 0, vmax = 1, cmap = 'nipy_spectral')
ax.set_title(n_cycles)

In [ ]:
wsize_s = 300

nperseg = int(wsize_s * abp_srate)
f, Cxy = scipy.signal.coherence(ihr_sig_up, raw_abp, fs = abp_srate, nperseg=nperseg)

In [ ]:
f, Pxx_abp = scipy.signal.welch(raw_abp, fs = abp_srate, nperseg=nperseg)
f, Pxx_ihr = scipy.signal.welch(ihr_sig_up, fs = abp_srate, nperseg=nperseg)

In [ ]:
fmin = 0.04
fmax = 0.5

f_mask = (f >= fmin) & (f <= fmax)

fig, axs = plt.subplots(nrows = 2, figsize = (12,6))

ax = axs[0]
ax.plot(f[f_mask], Pxx_abp[f_mask], color = 'g')
ax2 = ax.twinx()
ax2.plot(f[f_mask], Pxx_ihr[f_mask], color = 'r')

ax = axs[1]
ax.plot(f[f_mask], Cxy[f_mask], color = 'k')
ax.set_ylim(0,1)

In [ ]:
f, t, Sxx_abp = scipy.signal.spectrogram(raw_abp, fs = abp_srate, nperseg=nperseg)
f, t, Sxx_ihr = scipy.signal.spectrogram(ihr_sig_up, fs = abp_srate, nperseg=nperseg)

In [ ]:
fmin = 0
fmax = 0.5
q = 0.025
cmap = 'viridis'
# cmap = 'nipy_spectral'

f_mask = (f >= fmin) & (f <= fmax)

def compute_vlim(data, q):
    return np.quantile(data, q), np.quantile(data, 1-q)

fig, axs = plt.subplots(nrows= 2, figsize=  (10,6))
ax = axs[0]
ax.set_title('ABP')
data = np.log(Sxx_abp[f_mask,:])
vmin, vmax = compute_vlim(data, q)
ax.pcolormesh(t, f[f_mask],  data, vmin=vmin, vmax=vmax, cmap = cmap)
ax = axs[1]
ax.set_title('HRV')
data = np.log(Sxx_ihr[f_mask,:])
vmin, vmax = compute_vlim(data, q)
ax.pcolormesh(t, f[f_mask],  data, vmin=vmin, vmax=vmax, cmap = cmap)
# ax.set_xlim(0,0.5)

In [ ]:
def discretize_signal(signal, bins=10):
    return np.digitize(signal, np.histogram_bin_edges(signal, bins=bins))
def compute_mi(sig1, sig2, bins = 'auto'):
    if bins == 'auto':
        bins = int(np.sqrt(sig1.size))
    sig1 = discretize_signal(sig1, bins=bins)
    sig2 = discretize_signal(sig2, bins=bins)
    return mutual_info_score(sig1, sig2)

def compute_pearsonr(sig1, sig2):
    res = scipy.stats.pearsonr(sig1, sig2)
    return res.statistic

In [ ]:
mis = []
pearsonr = []

f_masked = f[f_mask]
Sxx_ihr_masked = Sxx_ihr[f_mask,:]
Sxx_abp_masked = Sxx_abp[f_mask,:]
for i, f_current in enumerate(f_masked):
    # print(f_current)
    sig1 = Sxx_ihr_masked[i,:]
    sig2 = Sxx_abp_masked[i,:]
    mis.append(compute_mi(sig1, sig2))
    pearsonr.append(compute_pearsonr(sig1, sig2))

In [ ]:
fig, axs = plt.subplots(nrows = 2, figsize = (12,6))
ax = axs[0]
ax.plot(f[f_mask], mis)
ax = axs[1]
ax.plot(f[f_mask], pearsonr)

In [ ]:
fig, ax = plt.subplots()
ax.plot(f, Cxy)
ax.set_xlim(0,0.5)

In [ ]:
def coherence_spectrogram(x, y, fs=1.0, window='hann', nperseg=256, noverlap=None, factor_nperseg = 2):
    assert x.size == y.size

    if noverlap is None:
        noverlap = nperseg // 2

    step = nperseg - noverlap
    starts = np.arange(0, x.size, step)
    stops = starts + nperseg * factor_nperseg
    mask_stops = stops < x.size
    starts = starts[mask_stops]
    stops = stops[mask_stops]

    inds = range(starts.size)

    times = starts / fs

    Cxy_spectrogram = None

    for i , start, stop in zip(inds, starts, stops):
        x_seg = x[start:stop]
        y_seg = y[start:stop]

        f, Cxy = scipy.signal.coherence(x_seg, y_seg, fs=fs, window=window, nperseg=nperseg, noverlap=noverlap)

        if Cxy_spectrogram is None:
            Cxy_spectrogram = np.zeros((f.size, stops.size)) * np.nan

        Cxy_spectrogram[:, i] = Cxy

        if i ==0:
            fig, axs = plt.subplots(nrows = 2)
            ax = axs[0]
            ax.plot(x_seg)
            ax.plot(y_seg)
            ax = axs[1]
            ax.plot(f, Cxy)
            ax.set_ylim(-0.1, 1.1)
            ax.set_xlim(0,1)
        

    return f, times, Cxy_spectrogram


In [ ]:
wsize_s = 300

nperseg = int(wsize_s * abp_srate)
print(nperseg)

In [ ]:
f, t, Cxys = coherence_spectrogram(ihr_sig_up, raw_abp, fs = abp_srate, nperseg=nperseg, factor_nperseg=3)

In [ ]:
fmin = 0
fmax = 0.5

f_mask = (f >= fmin) & (f <= fmax)
fig, axs = plt.subplots(nrows = 2, figsize = (12,6))
ax = axs[0]
ax.pcolormesh(t, f[f_mask] , Cxys[f_mask,:], vmin = 0, vmax = 1, cmap = 'viridis')

ax = axs[1]
ax.plot(f[f_mask] , np.nanmedian(Cxys[f_mask,:], axis = 1))

In [ ]:
freqs, power = morlet_power(raw_abp, abp_srate, f_start = 0.05, f_stop = 0.5, n_steps = 100, n_cycles = 5, duration_s='auto')

In [ ]:
qvlim = 0.025
data =  np.log(power)
vmin = np.quantile(data, qvlim)
vmax = np.quantile(data, 1 - qvlim)
fig, ax = plt.subplots(figsize = (12,6))
ax.pcolormesh(abp_times, freqs ,data, vmin=vmin,vmax=vmax, cmap = 'viridis')

In [ ]:
def morlet_coherence_bad(sig1, sig2, srate, f_start, f_stop, n_steps, n_cycles, duration_s = 'auto'):
    _, n_cycles_vector, freqs, family = morlet_family(srate, f_start = f_start, f_stop = f_stop, n_steps = n_steps, n_cycles = n_cycles, duration_s=duration_s)
    x_tf = scipy.signal.fftconvolve(np.tile(sig1, (n_steps,1)), family, mode = 'same', axes = 1)
    y_tf = scipy.signal.fftconvolve(np.tile(sig2, (n_steps,1)), family, mode = 'same', axes = 1)
    Sxx = np.abs(x_tf)**2
    Syy = np.abs(y_tf)**2
    Sxy = x_tf * np.conj(y_tf)
    Cxy = np.abs(Sxy)**2 / (Sxx * Syy + 1e-10)  # + epsilon to prevent 0 division
    return freqs, Cxy

In [ ]:
freqs, Cxy_morlet = morlet_coherence_bad(raw_abp, ihr_sig_up, abp_srate, f_start = 0.05, f_stop = 0.5, n_steps = 100, n_cycles = 5, duration_s=abp_times[-1])

In [ ]:
f_start = 0.05
f_stop = 0.5
n_steps = 100
n_cycles = 5
duration_s=abp_times[-1]
srate = abp_srate

sig1 = raw_abp.copy()
sig2 = ihr_sig_up.copy()

_, n_cycles_vector, freqs, family = morlet_family(srate, f_start = f_start, f_stop = f_stop, n_steps = n_steps, n_cycles = n_cycles, duration_s=duration_s)
x_tf = scipy.signal.fftconvolve(np.tile(sig1, (n_steps,1)), family, mode = 'same', axes = 1)
y_tf = scipy.signal.fftconvolve(np.tile(sig2, (n_steps,1)), family, mode = 'same', axes = 1)
Sxx = np.abs(x_tf)**2
Syy = np.abs(y_tf)**2
Sxy = x_tf * np.conj(y_tf)
Cxy = np.abs(Sxy)**2 / (Sxx * Syy + 1e-10)  # + epsilon to prevent 0 division

In [ ]:
np.abs(Sxy) ** 2

In [ ]:
(Sxx * Syy) == np.abs(Sxy) ** 2

In [ ]:
decim_plot = 10
qvlim = 0.05

data_plot =  np.abs(Sxy)[:,::decim_plot]
# data_plot = np.log(data_plot)
vmin = np.quantile(data_plot, qvlim)
vmax = np.quantile(data_plot, 1 - qvlim)
fig, ax = plt.subplots(figsize = (12,6))
ax.pcolormesh(abp_times[::decim_plot], freqs ,data_plot, vmin=vmin, vmax=vmax, cmap = 'viridis')

In [ ]:
decim_plot = 10
qvlim = 0.05

data_plot =  np.abs(x_tf)[:,::decim_plot]
# data_plot = np.log(data_plot)
vmin = np.quantile(data_plot, qvlim)
vmax = np.quantile(data_plot, 1 - qvlim)
fig, ax = plt.subplots(figsize = (12,6))
ax.pcolormesh(abp_times[::decim_plot], freqs ,data_plot, vmin=vmin, vmax=vmax, cmap = 'viridis')

In [ ]:
decim_plot = 10
qvlim = 0.05

data_plot =  np.abs(y_tf)[:,::decim_plot]
# data_plot = np.log(data_plot)
vmin = np.quantile(data_plot, qvlim)
vmax = np.quantile(data_plot, 1 - qvlim)
fig, ax = plt.subplots(figsize = (12,6))
ax.pcolormesh(abp_times[::decim_plot], freqs ,data_plot, vmin=vmin, vmax=vmax, cmap = 'viridis')

In [ ]:
data =  Cxy_morlet
vmin = 0
vmax = 1
fig, ax = plt.subplots(figsize = (12,6))
ax.pcolormesh(abp_times, freqs , data, vmin=vmin,vmax=vmax, cmap = 'viridis')

In [ ]:
# from scipy.ndimage import uniform_filter1d
# def smooth_data(data, size):
#     size = min(size, data.shape[1] - 1)  # Empêche size > longueur du signal
#     return uniform_filter1d(data, size=size, axis=1, mode='nearest')

def smooth_data(trace, srate, sigma_s=5.0, axis = 1):
    size = int(srate * sigma_s)
    times = np.arange(- 5 * size, 5 * size + 1)
    kernel = np.exp(- times ** 2 / size ** 2)
    kernel /= np.sum(kernel)
    trace_smooth = scipy.signal.fftconvolve(trace, kernel[None,:], mode='same', axes=axis)
    return trace_smooth

def smooth_data2d(trace, srate, sigma_s=10):
    """
    Lisse un tableau 2D (freqs x temps) avec un noyau gaussien en temps (axis=1).
    
    trace : ndarray (n_freqs, n_time)
    srate : fréquence d'échantillonnage
    sigma_s : largeur du noyau gaussien en secondes
    """
    sigma_samples = sigma_s * srate
    kernel_half = int(4 * sigma_samples)
    times = np.arange(-kernel_half, kernel_half + 1)
    
    kernel = np.exp(-0.5 * (times / sigma_samples)**2)
    kernel /= kernel.sum()

    smoothed = scipy.signal.fftconvolve(trace, kernel[None, :], mode='same', axes=1)
    return smoothed

# window_samples = int(300 * srate)
# Sxx_smooth = smooth_data(Sxx, window_samples)
# Syy_smooth = smooth_data(Syy, window_samples)
# Sxy_real = smooth_data(Sxy.real, window_samples)
# Sxy_imag = smooth_data(Sxy.imag, window_samples)
# Sxy_smooth = Sxy_real + 1j * Sxy_imag

sigma_s = 300
Sxx_smooth = smooth_data2d(Sxx, srate, sigma_s)
Syy_smooth = smooth_data2d(Syy, srate, sigma_s)
Sxy_real = smooth_data2d(Sxy.real,  srate, sigma_s)
Sxy_imag = smooth_data2d(Sxy.imag,  srate, sigma_s)
Sxy_smooth = Sxy_real + 1j * Sxy_imag # reconstruction du spectre croisé après lissage séparé des parties réelle et imaginaire



Cxy = np.abs(Sxy_smooth)**2 / (Sxx_smooth * Syy_smooth + 1e-10)


In [ ]:
data =  Cxy.copy()
vmin = 0
vmax = 1

fig, ax = plt.subplots(figsize = (12,6))
im = ax.pcolormesh(abp_times[::10], freqs , data[:,::10], vmin=vmin,vmax=vmax, cmap = 'viridis')
fig.colorbar(im, ax=ax)

In [ ]:
a = np.array([1])

In [ ]:
type(a)

In [ ]:
isinstance(a, np.ndarray)

In [ ]:
def smooth_data2d(arr_2d, srate, sigma_s=10):
    """
    Lisse un tableau 2D (freqs x temps) avec un noyau gaussien en temps (axis=1).
    
    arr : ndarray (n_freqs, n_time)
    srate : fréquence d'échantillonnage
    sigma_s : largeur du noyau gaussien en secondes
    """
    if isinstance(sigma_s, np.ndarray):
        sigma_samples = sigma_s * srate
        kernel_half = (4 * sigma_samples).astype(int)
        times = np.arange(-kernel_half, kernel_half + 1)

        smoothed = scipy.signal.fftconvolve(arr_2d, kernel, mode='same', axes=1)
    else:
        sigma_samples = sigma_s * srate
        kernel_half = int(4 * sigma_samples)
        times = np.arange(-kernel_half, kernel_half + 1)
        kernel = np.exp(-0.5 * (times / sigma_samples)**2)
        kernel /= kernel.sum()
        kernel = kernel[None, :]

    smoothed = scipy.signal.fftconvolve(arr_2d, kernel, mode='same', axes=1)
    return smoothed

def morlet_coherence2(sig1, sig2, srate, f_start, f_stop, n_steps, n_cycles, sigma_s = 'auto', smooth_factor = 5):
    _, n_cycles_vector, freqs, family = morlet_family(srate, f_start = f_start, f_stop = f_stop, n_steps = n_steps, n_cycles = n_cycles, duration_s='auto')
    x_tf = scipy.signal.fftconvolve(np.tile(sig1, (n_steps,1)), family, mode = 'same', axes = 1)
    y_tf = scipy.signal.fftconvolve(np.tile(sig2, (n_steps,1)), family, mode = 'same', axes = 1)
    Sxx = np.abs(x_tf)**2
    Syy = np.abs(y_tf)**2
    Sxy = x_tf * np.conj(y_tf)
    if sigma_s == 'auto':
        # if isinstance(n_cycles, int):
        #     sigma_s = smooth_factor * n_cycles / freqs[0]
        # elif isinstance(n_cycles, tuple):
        #     sigma_s = smooth_factor * n_cycles / freqs[0]
        sigma_s = smooth_factor * n_cycles_vector[0] / freqs[0]
    Sxx_smooth = smooth_data2d(Sxx, srate, sigma_s)
    Syy_smooth = smooth_data2d(Syy, srate, sigma_s)
    Sxy_real = smooth_data2d(Sxy.real,  srate, sigma_s)
    Sxy_imag = smooth_data2d(Sxy.imag,  srate, sigma_s)
    Sxy_smooth = Sxy_real + 1j * Sxy_imag # reconstruction du spectre croisé après lissage séparé des parties réelle et imaginaire
    Cxy = np.abs(Sxy_smooth)**2 / (Sxx_smooth * Syy_smooth + 1e-10)
    return freqs, Cxy

def smooth_per_freq(matrix, srate, sigma_s_per_freq):
    """Lissage ligne par ligne avec un sigma_s différent pour chaque fréquence."""

    n_freqs, n_times = matrix.shape
    output = np.zeros_like(matrix, dtype=matrix.dtype)
    
    for i in range(n_freqs):
        sigma_samples = sigma_s_per_freq[i] * srate
        half = int(4 * sigma_samples)
        t = np.arange(-half, half + 1)
        kernel = np.exp(-0.5 * (t / sigma_samples) ** 2)
        kernel /= kernel.sum()
        output[i] = scipy.signal.fftconvolve(matrix[i], kernel, mode='same')
    
    return output

def morlet_coherence3(sig1, sig2, srate, f_start, f_stop, n_steps, n_cycles, sigma_s = 'auto', smooth_factor = 5):
    _, n_cycles_vector, freqs, family = morlet_family(srate, f_start = f_start, f_stop = f_stop, n_steps = n_steps, n_cycles = n_cycles, duration_s='auto')
    x_tf = scipy.signal.fftconvolve(np.tile(sig1, (n_steps,1)), family, mode = 'same', axes = 1)
    y_tf = scipy.signal.fftconvolve(np.tile(sig2, (n_steps,1)), family, mode = 'same', axes = 1)
    Sxx = np.abs(x_tf)**2
    Syy = np.abs(y_tf)**2
    Sxy = x_tf * np.conj(y_tf)
    sigma_s = smooth_factor * n_cycles_vector / freqs
    Sxx_smooth = smooth_per_freq(Sxx, srate, sigma_s)
    Syy_smooth = smooth_per_freq(Syy, srate, sigma_s)
    Sxy_smooth = smooth_per_freq(Sxy.real, srate, sigma_s) + 1j * smooth_per_freq(Sxy.imag, srate, sigma_s)
    Cxy = np.abs(Sxy_smooth)**2 / (Sxx_smooth * Syy_smooth + 1e-10)
    return freqs, Cxy

def morlet_family4(srate, f_start, f_stop, n_steps):
    """
    Create a family of morlet wavelets
    
    ------------------------------
    srate : sampling rate
    f_start : lowest frequency of the wavelet family
    f_stop : highest frequency of the wavelet family
    n_steps : number of frequencies from f_start to f_stop
    n_cycles : number of waves in the wavelet. If it is an integer, all wavelets have this same number of cycles, else if a tuple, number of cycles increase linearly from the first to the second element.
    duration_s : duration in seconds of the kernel
    """
    freqs = np.linspace(f_start,f_stop,n_steps)
    duration_kernel_s = 1 / freqs[0] * 5
    n_cycles_vector = freqs * duration_kernel_s
    duration_s = 2 * n_cycles_vector[0] * (srate / freqs[0]) 
    tmw = np.arange(-duration_s/2,duration_s/2,1/srate)
    mw_family = np.zeros((freqs.size, tmw.size), dtype = 'complex')
    for i, fi in enumerate(freqs):
        mw_family[i,:] = complex_mw(tmw, n_cycles = n_cycles_vector[i], freq = fi)
    return tmw, n_cycles_vector, freqs, mw_family

def morlet_coherence4(sig1, sig2, srate, f_start, f_stop, n_steps, n_cycles = 'auto', sigma_s = 'auto', smooth_factor = 5):
    _, n_cycles_vector, freqs, family = morlet_family4(srate, f_start = f_start, f_stop = f_stop, n_steps = n_steps)
    x_tf = scipy.signal.fftconvolve(np.tile(sig1, (n_steps,1)), family, mode = 'same', axes = 1)
    y_tf = scipy.signal.fftconvolve(np.tile(sig2, (n_steps,1)), family, mode = 'same', axes = 1)
    Sxx = np.abs(x_tf)**2
    Syy = np.abs(y_tf)**2
    Sxy = x_tf * np.conj(y_tf)
    sigma_s = smooth_factor * n_cycles_vector / freqs
    Sxx_smooth = smooth_per_freq(Sxx, srate, sigma_s)
    Syy_smooth = smooth_per_freq(Syy, srate, sigma_s)
    Sxy_smooth = smooth_per_freq(Sxy.real, srate, sigma_s) + 1j * smooth_per_freq(Sxy.imag, srate, sigma_s)
    Cxy = np.abs(Sxy_smooth)**2 / (Sxx_smooth * Syy_smooth + 1e-10)
    return freqs, Cxy

In [ ]:
n_cycles = 10
freqs, Cxy = morlet_coherence2(raw_abp, ihr_sig_up, abp_srate, f_start = 0.05, f_stop = 0.5, n_steps = 50, n_cycles = n_cycles)
fig, ax = plt.subplots(figsize = (12,4))
ax.pcolormesh(abp_times, freqs, Cxy, vmin= 0, vmax = 1)
ax.set_title(n_cycles)

In [ ]:
n_cycles = (10,15)
freqs, Cxy = morlet_coherence2(raw_abp, ihr_sig_up, abp_srate, f_start = 0.05, f_stop = 0.5, n_steps = 50, n_cycles = n_cycles)
fig, ax = plt.subplots(figsize = (12,4))
ax.pcolormesh(abp_times, freqs, Cxy, vmin= 0, vmax = 1)
ax.set_title(n_cycles)

In [ ]:
n_cycles = 10
freqs, Cxy = morlet_coherence3(raw_abp, ihr_sig_up, abp_srate, f_start = 0.05, f_stop = 0.5, n_steps = 50, n_cycles = n_cycles)
fig, ax = plt.subplots(figsize = (12,4))
ax.pcolormesh(abp_times, freqs, Cxy, vmin= 0, vmax = 1)
ax.set_title(n_cycles)

In [ ]:
n_cycles = (5,50)
freqs, Cxy = morlet_coherence3(raw_abp, ihr_sig_up, abp_srate, f_start = 0.005, f_stop = 0.4, n_steps = 50, n_cycles = n_cycles, smooth_factor=4)
fig, ax = plt.subplots(figsize = (12,4))
ax.pcolormesh(abp_times, freqs, Cxy, vmin= 0, vmax = 1, cmap = 'nipy_spectral')
ax.set_title(n_cycles)

In [ ]:
freqs, Cxy = morlet_coherence4(raw_abp, ihr_sig_up, abp_srate, f_start = 0.005, f_stop = 0.4, n_steps = 50, n_cycles = 'auto', smooth_factor=1)
fig, ax = plt.subplots(figsize = (12,4))
ax.pcolormesh(abp_times, freqs, Cxy, vmin= 0, vmax = 1, cmap = 'nipy_spectral')
ax.set_title(n_cycles)

In [ ]:
tmw, n_cycles_vector, freqs, mw_family = morlet_family4(abp_srate, f_start=0.05, f_stop=0.5, n_steps=10)
fig, ax = plt.subplots()
# ax.plot(tmw, mw_family.real.T)
ax.pcolormesh(tmw, freqs, mw_family.real)

# Ventilation control effect on RSA

In [ ]:
keep_cols = ['sub','is_ventilation_controlled','rsa_amplitude_clean']

In [ ]:
concat = []
for sub in get_patient_list(['CO2','ECG_II']):
    # print(sub)
    try:
        resp_cycles = detect_resp_job.get(sub).to_dataframe()
        rsa_cycles = rsa_job.get(sub).to_dataframe()
        rsa_cycles['sub'] = sub
        assert resp_cycles.shape[0] == rsa_cycles.shape[0]
        rsa_resp_cycles = pd.concat([resp_cycles, rsa_cycles], axis = 1)
        rsa_resp_cycles = rsa_resp_cycles.loc[:,keep_cols]
        concat.append(rsa_resp_cycles)
    except:
        print(sub)
df = pd.concat(concat)
df = df.dropna(subset = ['rsa_amplitude_clean']).reset_index(drop = True)

In [ ]:

bins = np.arange(0,1,0.05)
df['rsa_amplitude_clean'].plot.hist(bins=bins)

In [ ]:
sub = 'MF12'
bins = np.arange(0,1,0.01)
df[df['sub'] == sub]['rsa_amplitude_clean'].plot.hist(bins=bins)

In [ ]:
df['rsa_amplitude_clean'].isna().sum()

In [ ]:
data_stats_between = df.groupby(['sub','is_ventilation_controlled'])['rsa_amplitude_clean'].median(True).reset_index()

In [ ]:
auto_stats(df = data_stats_between,
           predictor = 'is_ventilation_controlled',
           outcome = 'rsa_amplitude_clean',
           design = 'between',
          )

In [ ]:
keep_subs_within = data_stats['sub'].value_counts().index[data_stats['sub'].value_counts() > 1]
data_stats_within = data_stats_between[data_stats_between['sub'].isin(keep_subs_within)]

In [ ]:
auto_stats(df = data_stats_within,
           predictor = 'is_ventilation_controlled',
           outcome = 'rsa_amplitude_clean',
           design = 'within',
           subject = 'sub'
          )

In [ ]:
resp_cycles

In [ ]:
fig, ax = plt.subplots(figsize = (10, 4))
ax.plot(resp_cycles['inspi_date'], resp_cycles['is_ventilation_controlled'])

# Test machine learning cycles respi

In [ ]:
sub = 'MF12'
points_per_cycle = 40
n_cycles_prct = 100

resp_cycles = detect_resp_job.get(sub).to_dataframe()
if n_cycles_prct != 100:
    resp_cycles = resp_cycles.loc[:int(n_cycles_prct / 100 * resp_cycles.shape[0]),:]

reader = pycns.CnsReader(data_path / sub)
co2_stream = reader.streams['CO2']
raw_co2, co2_times = co2_stream.get_data(with_times = True, apply_gain = True, time_as_second = True)

cycle_times = resp_cycles[['inspi_time','next_inspi_time']].values

co2_2d = physio.deform_traces_to_cycle_template(data = raw_co2, 
                                                times = co2_times,
                                                cycle_times= cycle_times,
                                                points_per_cycle=points_per_cycle,
                                               )

phase = np.linspace(0, 1, points_per_cycle)

print(co2_2d.shape)

In [ ]:
# co2_2d = co2_2d - co2_2d[:,0][:,None]

m, s = np.mean(co2_2d, axis=1), np.std(co2_2d, axis = 1)
s[s == 0] = np.mean(s)
co2_2d_norm = co2_2d - m[:,None]
# co2_2d_norm = co2_2d - co2_2d[:,0][:,None]
co2_2d_norm = co2_2d_norm / s[:,None]

In [ ]:
m = co2_2d_norm.mean(axis = 0)
s = co2_2d_norm.std(axis = 0)

fig, ax = plt.subplots(figsize = (10,5))
ax.plot(phase, m, color = 'k')
ax.fill_between(phase, m-s, m+s, color = 'r', alpha = 0.2)

In [ ]:
fig, ax = plt.subplots(figsize = (10,5))
ax.plot(phase, s)

## Supervisé

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report, accuracy_score

In [ ]:
cols_features = ['cycle_duration','inspi_duration', 'expi_duration', 'cycle_freq', 'cycle_ratio', 'cycle_freq_cpm','sliding_variability_cpm']
cols_features

In [ ]:
X = co2_2d_norm.copy()
# X = resp_cycles[cols_features].values
y = resp_cycles['is_ventilation_controlled'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape

In [ ]:
clf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs = 10)
clf.fit(X_train, y_train)

In [ ]:
y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['assistée', 'contrôlée']))

In [ ]:
acc = accuracy_score(y_test, y_pred)
acc

In [ ]:
# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
disp = ConfusionMatrixDisplay(confusion_matrix=cm_percent, display_labels=["assistée", "contrôlée"], )
disp.plot(cmap='Blues')

In [ ]:
importances = clf.feature_importances_

# Affichage
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(importances)
ax.set_title("Importance des features classification")
ax.set_xlabel("Index temporel dans le cycle (0 à 199)")
# ax.set_xticks(range(len(cols_features)), labels= cols_features, rotation = 45)
ax.set_ylabel("Importance")
ax.grid(True)
fig.show()

In [ ]:
predictions = clf.predict(X)

In [ ]:
%matplotlib widget

In [ ]:
fig, ax = plt.subplots(figsize = (11,5))
ax.plot(resp_cycles['inspi_date'], predictions, alpha = 0.5)
ax.plot(resp_cycles['inspi_date'], y, alpha = 0.5)
ax.plot(resp_cycles['inspi_date'], pd.Series(predictions).rolling(100, center = True).mean())

## Test modèle sur autre patient

In [ ]:
sub = 'FC13'
reader = pycns.CnsReader(data_path / sub)
co2_stream = reader.streams['CO2']
raw_co2, co2_times = co2_stream.get_data(with_times = True, apply_gain = True, time_as_second = True)

resp_cycles = detect_resp_job.get(sub).to_dataframe()
cycle_times = resp_cycles[['inspi_time','next_inspi_time']].values

co2_2d = physio.deform_traces_to_cycle_template(data = raw_co2, 
                                                times = co2_times,
                                                cycle_times= cycle_times,
                                                points_per_cycle=points_per_cycle,
                                               )

phase = np.linspace(0, 1, points_per_cycle)

print(co2_2d.shape)

In [ ]:
# co2_2d = co2_2d - co2_2d[:,0][:,None]

# med_along_phase, mad_along_phase = physio.compute_median_mad(co2_2d.T, axis = 0)
m, s = np.mean(co2_2d, axis=1), np.std(co2_2d, axis = 1)
s[s == 0] = np.mean(s)
co2_2d_norm = co2_2d - m[:,None]
co2_2d_norm = co2_2d_norm / s[:,None]

In [ ]:
m = co2_2d_norm.mean(axis = 0)
s = co2_2d_norm.std(axis = 0)

# m, s = physio.compute_median_mad(co2_2d_norm, axis = 0)

fig, ax = plt.subplots(figsize = (10,5))
ax.plot(phase, m, color = 'k')
ax.fill_between(phase, m-s, m+s, color = 'r', alpha = 0.2)

In [ ]:
X = co2_2d_norm.copy()
y = resp_cycles['is_ventilation_controlled'].values
predictions = clf.predict(X)

In [ ]:
%matplotlib widget

In [ ]:
fig, ax = plt.subplots(figsize = (11,5))
ax.plot(resp_cycles['inspi_date'], predictions, alpha = 0.5)
ax.plot(resp_cycles['inspi_date'], y, alpha = 0.5)
ax.plot(resp_cycles['inspi_date'], pd.Series(predictions).rolling(50, center = True).mean())

## Non supervisé

In [ ]:

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# 1. Normaliser
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # X = tes cycles


# 2. Clustering
kmeans = KMeans(n_clusters=2, random_state=42)
clusters = kmeans.fit_predict(X_scaled)

# 3. Évaluer la séparation
score = silhouette_score(X_scaled, clusters)
print(f"Silhouette score : {score:.2f} (plus proche de 1 = meilleur)")

# 4. Visualisation (optionnel)
pca = PCA(n_components=2)
X_2D = pca.fit_transform(X_scaled)
plt.scatter(X_2D[:, 0], X_2D[:, 1], c=clusters, cmap='viridis')
plt.title("Clusters non supervisés des cycles")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Pour chaque cluster trouvé
for cluster_label in np.unique(clusters):
    idx = np.where(clusters == cluster_label)[0]
    
    # Prendre 5 cycles aléatoires de ce cluster
    samples = np.random.choice(idx, size=min(5, len(idx)), replace=False)
    
    plt.figure(figsize=(10, 4))
    for i, sample_idx in enumerate(samples):
        plt.plot(X[sample_idx], label=f'cycle {sample_idx}')
    
    plt.title(f'Exemples de cycles du cluster {cluster_label}')
    plt.xlabel('Temps (points)')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
for cluster_label in np.unique(clusters):
    mean_cycle = X[clusters == cluster_label].mean(axis=0)
    plt.plot(mean_cycle, label=f'Cluster {cluster_label}')
    
plt.title("Moyenne des cycles par cluster")
plt.xlabel("Temps")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()


# Stats resp cycles ventilation mode

In [ ]:
sub = 'MF12'
resp_cycles = detect_resp_job.get(sub).to_dataframe()
resp_cycles['sliding_variability_cpm'].plot.hist(bins = np.arange(0, 1,0.01))

In [ ]:
bins = np.arange(0, 1,0.01)
sns.histplot(data = resp_cycles,
            x = 'sliding_variability_cpm',
            hue = 'is_ventilation_controlled',
             bins=bins,
            )

In [ ]:
resp_cycles.columns

In [ ]:
# bins = np.arange(0, 1,0.01)
bins = np.arange(0, 1,0.01)
sns.histplot(data = resp_cycles,
            x = 'cycle_freq',
            hue = 'is_ventilation_controlled',
             bins=bins,
            )

In [ ]:
bins = np.arange(0, 1,0.025)
count, bins = np.histogram(resp_cycles['sliding_variability_cpm'], bins)
d1 = np.gradient(count)

In [ ]:
minimums = np.where((d1[:-1] < 0) & ((d1[1:] >= 0)))[0]
first_minimum = minimums[0]
thresh = bins[1:][first_minimum]
thresh

In [ ]:
fig, ax = plt.subplots()
ax.hist(resp_cycles['sliding_variability_cpm'], bins)
ax.plot(bins[1:], count, color = 'k')

ax2=ax.twinx()
ax2.plot(bins[1:], d1, color=  'darkorange')
ax.scatter(bins[1:][minimums], count[minimums], color = 'g')
ax.scatter(bins[1:][first_minimum], count[first_minimum], color = 'r')
ax.axvline(thresh, color = 'r')
ax2.axhline(0, color = 'k', ls = '--')

# Rep hugo ardaillon 16/06/2025

In [ ]:
sub = 'MF12'
reader = pycns.CnsReader(data_path / sub)
reader

stream_name = 'Temp'
stream = reader.streams[stream_name]
raw_temp, datetimes = stream.get_data(apply_gain=True, with_times= True)
srate = stream.sample_rate

fig, ax = plt.subplots()
ax.plot(datetimes, raw_temp)

In [ ]:
ds_medication = icca_medication_tt_job.get('MF12')
da_paracetamol = ds_medication['Paracétamol']

# Icca to moberg ABP / ICP

In [ ]:
df = detect_abp_job.get('MF12').to_dataframe()
df

In [ ]:
pam = (df['amplitude_at_peak'] + 2*df['amplitude_at_trough']) / 3
pam

In [ ]:
def process_abp(abp_detections):
    

# Connectivity metrics between 2 time series

In [ ]:
def discretize_signal(signal, bins=10):
    return np.digitize(signal, np.histogram_bin_edges(signal, bins=bins))

In [ ]:
def compute_mi(sig1, sig2, bins = 'auto'):
    if bins == 'auto':
        bins = int(np.sqrt(sig1.size))
    sig1 = discretize_signal(sig1, bins=bins)
    sig2 = discretize_signal(sig2, bins=bins)
    return mutual_info_score(sig1, sig2)

In [ ]:
duration = 10
freq1 = 2
freq2 = 2
phi1 = 0 * np.pi
phi2 = 1/2 * np.pi
amp1 = 1
amp2 = 1

srate = 1000

time = np.arange(0, duration, 1  / srate)

sig1 = amp1 * np.sin(2 * np.pi * freq1 * time + phi1)
sig2 = amp2 * np.sin(2 * np.pi * freq2 * time + phi2) #+ np.random.randn(sig1.size) * 0.1

fig, axs = plt.subplots(nrows = 2, figsize = (9,6), constrained_layout = True)

ax = axs[0]
ax.plot(time, sig1)
ax.plot(time, sig2)

ax = axs[1]
rpearson = scipy.stats.pearsonr(sig1, sig2).statistic
rspearman = scipy.stats.spearmanr(sig1, sig2).statistic
mi = compute_mi(sig1, sig2, bins = 'auto')
ax.scatter(sig1, sig2)
ax.set_title(f'pearson r = {round(rpearson, 3)} ; spearman r = {round(rspearman, 3)} ; mi = {round(mi, 3)}')

In [ ]:
# amps = np.arange(1 , 6)
amps = np.arange(0. , 2.1, 0.2)
freqs = np.arange(2 , 10, 1)
phis = np.linspace(0 , 2 * np.pi, 50)
metrics = ['r_pearson','r_spearman','mi']
da = xr.DataArray(data = np.nan, dims = ['phi','amp','freq','metric'], coords = {'phi':phis, 'amp':amps, 'freq':freqs,'metric':metrics})
for phi in phis:
    for amp in amps:
        for freq in freqs:
            # sig_target = amp * np.sin(2 * np.pi * freq * time + phi)
            sig_target = np.sin(2 * np.pi * freq * time + phi) + np.random.randn(sig1.size) * amp
            da.loc[phi, amp, freq,'r_pearson'] = scipy.stats.pearsonr(sig1, sig_target).statistic
            da.loc[phi, amp, freq,'r_spearman'] = scipy.stats.spearmanr(sig1, sig_target).statistic
            da.loc[phi, amp, freq,'mi'] = compute_mi(sig1, sig_target, bins = 'auto')

In [ ]:
q = 0.
nrows = len(freqs)
ncols = len(metrics)
fig, axs = plt.subplots(nrows = nrows, ncols = ncols, constrained_layout = True, figsize = (ncols * 4,nrows * 3))
for c, metric in enumerate(metrics):
    da_metric = da.sel(metric = metric)
    # vmin = da_metric.min()
    # vmax = da_metric.max()
    for r, freq in enumerate(freqs):
        da_freq = da_metric.sel(freq=freq)
        vmin = da_freq.min()
        vmax = da_freq.max()
        ax = axs[r,c]
        data = da_freq.values.T
        vmin = np.quantile(data, q)
        vmax = np.quantile(data, 1-q)
        if -1 in np.sign(data):
            cmap = 'seismic'
        else:
            cmap = 'viridis'
        im = ax.pcolormesh(da_freq['phi'], da_freq['amp'], data, vmin=vmin , vmax=vmax, cmap = cmap)
        ax.set_xlabel('phi')
        ax.set_ylabel('amp noise')
        ax.set_title(f'{metric} - {freq} hz')
        clb = fig.colorbar(im, ax=ax)
        clb.ax.set_ylabel(metric)
# fig.savefig('/crnldata/tiger/Valentin_Ghibaudo/simulation_sinusoids.png', bbox_inches = 'tight', dpi = 200)

In [ ]:
da.sel(freq = 2, amp = 0., metric = 'mi').plot()

# HRV vs TREATMENTS

In [ ]:
def mad_func(x, axis):
    return scipy.stats.median_abs_deviation(x, scale = 'normal', axis=axis)

In [ ]:
def merge_icca_to_moberg(moberg_da, icca_da):
    moberg_datetimes = moberg_da['datetime'].values
    icca_datetimes = icca_da[icca_da.dims[0]].values
    inds_to_loc_moberg_from_icca_datetimes = np.searchsorted(moberg_datetimes, icca_datetimes)
    inds_to_loc_moberg_from_icca_datetimes_unique = np.unique(inds_to_loc_moberg_from_icca_datetimes)
    inds_to_loc_moberg_from_icca_datetimes_unique = inds_to_loc_moberg_from_icca_datetimes_unique[(inds_to_loc_moberg_from_icca_datetimes_unique != 0) & (inds_to_loc_moberg_from_icca_datetimes_unique != moberg_datetimes.size)]
    datetimes_when_moberg_can_be_loc = moberg_datetimes[inds_to_loc_moberg_from_icca_datetimes_unique]
    moberg_when_moberg_can_be_loc = moberg_da[inds_to_loc_moberg_from_icca_datetimes_unique]
    
    inds_datetimes_when_moberg_have_been_loc_in_icca = np.searchsorted(icca_datetimes, datetimes_when_moberg_can_be_loc) - 1
    icca_datetimes_localized = icca_datetimes[inds_datetimes_when_moberg_have_been_loc_in_icca]
    icca_when_localized = icca_da[inds_datetimes_when_moberg_have_been_loc_in_icca]
    da_merge = xr.DataArray(data = np.nan,
                            dims = ['dtype','datetime'], 
                            coords = {'dtype':['moberg','icca'], 'datetime':icca_datetimes_localized},
                           )
    da_merge.loc['moberg',:] = moberg_when_moberg_can_be_loc.values
    da_merge.loc['icca',:] = icca_when_localized.values
    return da_merge

In [ ]:
def plot_moberg_to_icca(sub, moberg_trace_name, moberg_trace_job, icca_trace_name, icca_trace_job, win_size_s = 60, estimator = 'pos', color_moberg = 'r', figsize = (8,10)):
    moberg_da = moberg_trace_job.get(sub)[moberg_trace_name]
    if estimator == 'pos':
        moberg_da = moberg_da.rolling(datetime = int(win_size_s * moberg_da.attrs['srate']), center = True).median().bfill('datetime').ffill('datetime')
    elif estimator == 'var':
        # moberg_da = moberg_da.rolling(datetime = int(win_size_s * moberg_da.attrs['srate']), center = True).reduce(mad_func).bfill('datetime').ffill('datetime')
        moberg_da = moberg_da.rolling(datetime = int(win_size_s * moberg_da.attrs['srate']), center = True).std().bfill('datetime').ffill('datetime')
    icca_ds = icca_trace_job.get(sub)
    print(list(icca_ds.keys()))
    icca_da = icca_ds[icca_trace_name]
    da_merge = merge_icca_to_moberg(moberg_da, icca_da)
    
    fig, axs = plt.subplots(nrows = 3, figsize = figsize, constrained_layout = True)
    icca_axlabel = '{icca_trace_name} ({unit})'.format(icca_trace_name = icca_trace_name, unit = icca_da.attrs['unit'])
    moberg_axlabel = f'{moberg_trace_name} ({estimator})'
    
    ax = axs[0]
    ax.plot(icca_da[f'datetime_{icca_trace_name}'], icca_da, color = 'k')
    ax.scatter(icca_da[f'datetime_{icca_trace_name}'], icca_da, color = 'b')
    ax.scatter(da_merge['datetime'], da_merge.loc['icca'], color = 'darkorange', marker = 'x')
    ax.set_ylabel(icca_axlabel)
    ax2 = ax.twinx()
    ax2.plot(moberg_da['datetime'], moberg_da, color = color_moberg, alpha = 0.6)
    ax2.set_ylabel(moberg_axlabel, color = color_moberg)

    ax = axs[1]
    ax.plot(da_merge['datetime'], da_merge.loc['icca'], color = 'k')
    ax.scatter(da_merge['datetime'], da_merge.loc['icca'], color = 'b')
    ax.scatter(da_merge['datetime'], da_merge.loc['icca'], color = 'darkorange', marker = 'x', alpha = 0.8)
    ax.set_ylabel(icca_axlabel)
    ax2 = ax.twinx()
    ax2.plot(da_merge['datetime'], da_merge.loc['moberg'], color = color_moberg, alpha = 0.6)
    ax2.set_ylabel(moberg_axlabel, color = color_moberg)
    
    ax = axs[2]
    df = pd.DataFrame()
    df['icca'] = da_merge.loc['icca'].values
    df['moberg'] = da_merge.loc['moberg'].values
    stats_quantitative(df, 'icca', 'moberg', ax=ax)
    # ax.scatter(ds_merge['icca'], ds_merge['moberg'], color = 'k')
    ax.set_xlabel(icca_axlabel)
    ax.set_ylabel(moberg_axlabel , color = color_moberg)

In [ ]:
subs = get_patient_list(['ECG_II','CO2'])
np.array(subs)

In [ ]:
def icca_moberg_sub_to_df(sub, moberg_trace_name, icca_trace_name, win_size_s_moberg_roll = 60):
    if moberg_trace_name == 'ihr':
        moberg_trace_job = ihr_job
    elif moberg_trace_name == 'irr':
        moberg_trace_job = irr_job
    elif moberg_trace_name == 'irsa':
        moberg_trace_job = irsa_job

    if icca_trace_name in ['Propofol','Thiopenthal','Midazolam', 'Sufentanil', 'Urapidil', 'Norépinéprhine','Rémifentanil','Nicardipinine chlor.','Cisatracurium']:
        icca_trace_job = icca_pse_tt_job
    elif icca_trace_name in ['Glasgow, total']:
        icca_trace_job = icca_clinical_job

    moberg_da = moberg_trace_job.get(sub)[moberg_trace_name]
    moberg_da_pos = moberg_da.rolling(datetime = int(win_size_s_moberg_roll * moberg_da.attrs['srate']), center = True).median().bfill('datetime').ffill('datetime')
    moberg_da_var = moberg_da.rolling(datetime = int(win_size_s_moberg_roll * moberg_da.attrs['srate']), center = True).std().bfill('datetime').ffill('datetime')
    icca_da = icca_trace_job.get(sub)[icca_trace_name]
    da_merge_pos = merge_icca_to_moberg(moberg_da_pos, icca_da)
    da_merge_var = merge_icca_to_moberg(moberg_da_var, icca_da)

    df = pd.DataFrame()
    df[f'{icca_trace_name}'] = da_merge_pos.loc['icca',:].values
    df[f'{moberg_trace_name}_pos'] = da_merge_pos.loc['moberg',:].values
    df[f'{moberg_trace_name}_var'] = da_merge_var.loc['moberg',:].values
    df['sub'] = sub
    return df

In [ ]:
concat = []
for sub in subs:
    try:
        concat.append(icca_moberg_sub_to_df(sub, 'ihr', 'Propofol'))
    except:
        print(sub)
        continue

In [ ]:
propofol_ihr_df = pd.concat(concat).reset_index(drop = True)

In [ ]:
propofol_ihr_df

In [ ]:
sns.lmplot(data = propofol_ihr_df, 
           x = 'Propofol',
           y = 'ihr_pos',
          )

In [ ]:
sns.lmplot(data = propofol_ihr_df, 
           x = 'Propofol',
           y = 'ihr_pos',
           hue = 'sub',
           ci = None
          )

In [ ]:
propofol_ihr_df['sub'].value_counts()

In [ ]:
sns.scatterplot(data = propofol_ihr_df, 
           x = 'Propofol',
           y = 'ihr_pos',
           hue = 'sub'
          )

In [ ]:
sns.pairplot(data = propofol_ihr_df[['Propofol','ihr_pos']])

In [ ]:
sub = 'LD16'

# icca_trace_name = 'Glasgow, total'
# icca_trace_job = icca_clinical_job

icca_trace_name = 'Propofol'
icca_trace_job = icca_pse_tt_job

plot_moberg_to_icca(sub, 'ihr', ihr_job, icca_trace_name, icca_trace_job, color_moberg = 'r', estimator = 'pos')
plot_moberg_to_icca(sub, 'ihr', ihr_job, icca_trace_name, icca_trace_job, color_moberg = 'r', estimator = 'var')

plot_moberg_to_icca(sub, 'irr', irr_job, icca_trace_name, icca_trace_job, color_moberg = 'b', estimator = 'pos')
plot_moberg_to_icca(sub, 'irr', irr_job, icca_trace_name, icca_trace_job, color_moberg = 'b', estimator = 'var')

plot_moberg_to_icca(sub, 'irsa', irsa_job, icca_trace_name, icca_trace_job, color_moberg = 'g', estimator = 'pos')



# DC VIEW JOB

In [ ]:
get_subs_with_sd()

In [ ]:
sub = 'RM6'
sd_dates = get_sd_dates(sub)
sd_dates.shape

In [ ]:
dc = decimate_ecog_job.get(sub)['decimate_ecog']
chans = dc['chan'].values
datetimes = dc['datetime'].values
srate = 1 / np.median(np.diff(datetimes).astype(float)) * 1e9
cmap = 'viridis'
colors = colormaps[cmap].resampled(len(chans))
colors = {element: colors(i) for i, element in enumerate(chans)}

In [ ]:
def upsampled_mask_quality(mask_quality, dates_mask_quality, dates_to_target):
    mask_quality_upsampled = np.full(dates_to_target.size, np.nan) # prepare Nan vector ready to be filled with quality boolean values
    inds_insertion = np.searchsorted(dates_to_target, dates_mask_quality) # search indices where boolean quality values have to be inserted in the Nan vector
    mask_quality_upsampled[inds_insertion] = mask_quality # insert boolean quality values at the right place
    mask_quality_upsampled = pd.Series(mask_quality_upsampled).ffill().values.astype('bool') # fill nan forward to upsample and transtype to boolean = get an upsampled quality mask with size of original date vector
    return xr.DataArray(data = mask_quality_upsampled, dims = ['datetime'], coords = {'datetime':dates_to_target})

In [ ]:
ecog_quality_by_chan = eeg_quality_job.get(sub)['eeg_quality'].sel(chan = chans, feature = 'slope_or_power')
ecog_quality_by_chan[:] = ecog_quality_by_chan.values
ecog_quality_averaged = eeg_quality_averaged_job.get(sub)['eeg_quality_averaged'].sel(mode = 'ecog')
ecog_quality_averaged[:] = ecog_quality_averaged.values
ecog_quality_averaged = ecog_quality_averaged.astype('bool')
ecog_good_quality = upsampled_mask_quality(ecog_quality_averaged.values, ecog_quality_averaged['datetime'].values, datetimes)
ecog_bad_quality = ~ecog_good_quality

In [ ]:
ecog_quality_by_chan.sum('datetime') /  ecog_quality_by_chan.count('datetime')

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))
bins = np.linspace(dc.min(), dc.max(), 100)
for ch in chans:
    ax.hist(dc.sel(chan=ch).values, bins = bins, label = ch, alpha = 0.6, color = colors[ch])
ax.legend(loc = 'upper right')

In [ ]:
fig, ax = plt.subplots(figsize = (12,4))
for ch in chans:
    ax.plot(datetimes, dc.sel(chan=ch).values, label = ch, alpha = 0.6, color = colors[ch])
ax.legend(loc = 'upper right')
ax.fill_between(datetimes, y1 = 0, y2 = 1 , where = ecog_bad_quality, color= 'k', alpha = 0.2, transform=ax.get_xaxis_transform())

In [ ]:
win_size_mins = 5

win_size_s = win_size_mins * 60
win_size = int(srate * win_size_s)
dc_baselines = dc.rolling(datetime = win_size).median().bfill('datetime').ffill('datetime')
dc_baseline_corrected = dc - dc_baselines
dc_baseline_corrected = dc_baseline_corrected - dc_baseline_corrected.median('chan')

fig, ax = plt.subplots(figsize = (12,4))
for ch in chans:
    ax.plot(datetimes, dc_baselines.sel(chan=ch).values, label = ch, alpha = 0.6, color = colors[ch])
ax.legend(loc = 'upper right')

fig, ax = plt.subplots(figsize = (12,4))
for ch in chans:
    ax.plot(datetimes, dc_baseline_corrected.sel(chan=ch).values, label = ch, alpha = 0.6, color = colors[ch])
ax.legend(loc = 'upper right')

fig, ax = plt.subplots(figsize = (12,4))
q = 0.01
bins = np.linspace(dc_baseline_corrected.quantile(q), dc_baseline_corrected.quantile(1-q), 100)
for ch in chans:
    ax.hist(dc_baseline_corrected.sel(chan=ch).values, bins = bins, label = ch, alpha = 0.6, color = colors[ch])
ax.legend(loc = 'upper right')

In [ ]:
%matplotlib widget

In [ ]:
def get_ylim(da, q):
    return da.quantile(q), da.quantile(1-q)

In [ ]:
def mad_func(x, axis):
    return scipy.stats.median_abs_deviation(x, scale = 'normal', axis=axis)

In [ ]:
dc_gfp_std = dc_baseline_corrected.std('chan')
dc_gfp_mad = dc_baseline_corrected.reduce(mad_func, dim='chan')
dc_gfp = xr.DataArray(data = np.nan,
                      dims = ['method','datetime'],
                      coords = {'method':['std','mad'], 'datetime':dc_gfp_mad['datetime']}
                     )
dc_gfp.loc['std',:] = dc_gfp_std
dc_gfp.loc['mad',:] = dc_gfp_mad


fig, axs = plt.subplots(nrows = 2 , figsize = (12,9), sharex = True)
fig.subplots_adjust(hspace = 0)

ax = axs[0]
da_plot = -dc_baseline_corrected
ylims = get_ylim(da_plot, 0.001)
for chan in chans:
    ax.plot(datetimes, da_plot.sel(chan = chan).values, label = chan, alpha = 0.6, lw = 0.8, color = colors[chan])
ax.set_ylim(*ylims)
ax.legend(loc = 'upper right')

ax.fill_between(datetimes, y1 = 0, y2 = 1 , where = ecog_bad_quality, color= 'k', alpha = 0.2, transform=ax.get_xaxis_transform())

ax = axs[1]
ax.plot(datetimes, dc_gfp.loc['std'].values, color = 'r', lw = 2, alpha = 0.8, label = 'std')
ax.plot(datetimes, dc_gfp.loc['mad'].values, color = 'y', lw = 2, alpha = 0.8, label = 'mad')
ax.plot(datetimes, dc_gfp.mean('method').values, color = 'k', lw = 2, alpha = 0.8, label = 'both_m')
ax.legend(loc = 'upper right')
ax.set_ylim(*get_ylim(dc_gfp, 0.001))

for ax in axs:
    for i, row in sd_dates.iterrows():
        start = row['start_time']
        stop = start + np.timedelta64(int(row['duration']), 's')
        ax.axvspan(start, stop, color = 'r', lw = 2, alpha = 0.2)

ax.fill_between(datetimes, y1 = 0, y2 = 1 , where = ecog_bad_quality, color= 'k', alpha = 0.2, transform=ax.get_xaxis_transform())

plt.show()

In [ ]:
dc_amps = dc_baseline_corrected.copy()
dc_amps[:] = get_amp(dc_amps.values, axis = 0)

In [ ]:
dc_gfp_std = dc_amps.std('chan')
dc_gfp_mad = dc_amps.reduce(mad_func, dim='chan')
dc_gfp = xr.DataArray(data = np.nan,
                      dims = ['method','datetime'],
                      coords = {'method':['std','mad'], 'datetime':dc_gfp_mad['datetime']}
                     )
dc_gfp.loc['std',:] = dc_gfp_std
dc_gfp.loc['mad',:] = dc_gfp_mad


fig, axs = plt.subplots(nrows = 2 , figsize = (12,9), sharex = True)
fig.subplots_adjust(hspace = 0)

ax = axs[0]
da_plot = dc_amps.copy()
ylims = get_ylim(da_plot, 0.001)
for chan in chans:
    ax.plot(datetimes, da_plot.sel(chan = chan).values, label = chan, alpha = 0.6, lw = 0.8, color = colors[chan])
ax.set_ylim(*ylims)
ax.legend(loc = 'upper right')

ax.fill_between(datetimes, y1 = 0, y2 = 1 , where = ecog_bad_quality, color= 'k', alpha = 0.2, transform=ax.get_xaxis_transform())

ax = axs[1]
ax.plot(datetimes, dc_gfp.loc['std'].values, color = 'r', lw = 2, alpha = 0.8, label = 'std')
ax.plot(datetimes, dc_gfp.loc['mad'].values, color = 'y', lw = 2, alpha = 0.8, label = 'mad')
ax.plot(datetimes, dc_gfp.mean('method').values, color = 'k', lw = 2, alpha = 0.8, label = 'both_m')
ax.legend(loc = 'upper right')
ax.set_ylim(*get_ylim(dc_gfp, 0.001))

for ax in axs:
    for i, row in sd_dates.iterrows():
        start = row['start_time']
        stop = start + np.timedelta64(int(row['duration']), 's')
        ax.axvspan(start, stop, color = 'r', lw = 2, alpha = 0.2)

ax.fill_between(datetimes, y1 = 0, y2 = 1 , where = ecog_bad_quality, color= 'k', alpha = 0.2, transform=ax.get_xaxis_transform())

plt.show()

In [ ]:
vlims = get_ylim(dc_amps, 0.01)

fig, ax = plt.subplots(figsize = (9,4))
ax.pcolormesh(datetimes, dc_amps['chan'], dc_amps.T, vmin=vlims[0], vmax = vlims[1])
ax.fill_between(datetimes, y1 = 0, y2 = 1 , where = ecog_bad_quality, color= 'k', alpha = 0.2, transform=ax.get_xaxis_transform())
for i, row in sd_dates.iterrows():
    start = row['start_time']
    stop = start + np.timedelta64(int(row['duration']), 's')
    ax.axvspan(start, stop, color = 'r', lw = 2, alpha = 0.6)
    # ax.axvline(start, color = 'r')
plt.show()

# DC VIEW

In [ ]:
sub = 'MF12'
reader = CnsReader(data_path / sub)
eeg_stream = reader.streams['EEG']
d_start = eeg_stream.index['datetime'][0]
d_stop = eeg_stream.index['datetime'][-1]
duration_min = 300
lag_start_min = 0
d_start_load = d_start + np.timedelta64(lag_start_min, 'm')
d_stop_load = d_start_load + np.timedelta64(duration_min, 'm')
chans = eeg_stream.channel_names
srate = eeg_stream.sample_rate
ecog_chans = [ch for ch in chans if 'ECoG' in ch]

In [ ]:
# sigs, datetimes = eeg_stream.get_data(isel = slice (10, 110), apply_gain = True)
sigs, datetimes = eeg_stream.get_data(sel = slice (d_start_load, d_stop_load), apply_gain = True, with_times = True)
datetimes = datetimes.astype('datetime64[ns]')

da = xr.DataArray(data = sigs.T, dims = ['chan','time'], coords = {'chan':chans, 'time':datetimes})

In [ ]:
cutoff_Hz = 0.1
new_sr_targetted = cutoff_Hz * 4
decimate_factor = int(srate / new_sr_targetted)
win_size_s = 1 / cutoff_Hz
win_size = int(win_size_s * srate)
da_decim = da.rolling(time = win_size, center=True).median().bfill('time').ffill('time')[:,::decimate_factor]

In [ ]:
data2_decim_iir = scipy.signal.decimate(da.values, q = decimate_factor, axis = 1, zero_phase=True)

In [ ]:
da_both = xr.DataArray(data = np.nan, dims = ['method','chan','time'], coords = {'method':['med','iir'], 'chan':da_decim['chan'], 'time':da_decim['time']})

In [ ]:
da_both.loc['med',:,:] = da_decim.values
da_both.loc['iir',:,:] = data2_decim_iir

In [ ]:
%matplotlib widget

In [ ]:
plt.figure()
da_both.loc[:,'ECoGA3',:].plot.line(x = 'time')
plt.show()

In [ ]:
da_ecog = da_full.sel(chan = ecog_chans).astype('float64')

In [ ]:
da_ecog *= eeg_stream.gain

In [ ]:
da_ecog

In [ ]:
da_ecog.rolling(time = 10, center = True).median()

# CSD PHYSIO NEW SEE

In [ ]:
da = concat_erp_csd_hr_job.get('concat')['concat_erp_csd_hr']

subs = da['sub'].values
features = da['feature'].values
methods = da['average_event_method'].values
times = da['time_mins'].values
for sub in subs:
    try:
        da = erp_csd_hr_job.get(sub)['erp_csd_hr']
        N = da.shape[0]
        ncols = len(features)
        fig, axs = plt.subplots(ncols = ncols, figsize = (ncols * 5, 4), constrained_layout = True)
        fig.suptitle(f'{sub} (N = {N})')
        for c, feature in enumerate(features):
            ax = axs[c]
            da_feature = da.sel(feature=feature)
            m = da_feature.median('event').values
            ax.plot(times, m, color = 'r')
            ax.plot(times, da_feature.values.T, color = 'r', alpha = 0.2, lw = 0.5)
            ax.set_xlabel('Time (min)')
            ax.set_ylabel(feature)
            ax.axvline(0, color = 'k')
        plt.show()
    except:
        continue


In [ ]:
da = concat_erp_csd_rr_job.get('concat')['concat_erp_csd_rr']

# subs = da['sub'].values
# features = da['feature'].values
# methods = da['average_event_method'].values
# times = da['time_mins'].values
# for sub in subs:
#     da = concat_erp_csd_rr_job.get('concat')['concat_erp_csd_rr']
#     try:
#         N = erp_csd_rr_job.get(sub)['erp_csd_rr'].shape[0]
#     except:
#         N = 0
#     ncols = len(features)
#     fig, axs = plt.subplots(ncols = ncols, figsize = (ncols * 5, 4), constrained_layout = True)
#     fig.suptitle(f'{sub} (N = {N})')
#     for c, feature in enumerate(features):
#         ax = axs[c]
#         m = da.loc[sub, feature, 'm',:].values
#         s = da.loc[sub, feature, 's',:].values
#         ax.plot(times, m, color = 'b')
#         ax.fill_between(times, m-s, m+s, color = 'b', alpha = 0.2)
#         ax.set_xlabel('Time (min)')
#         ax.set_ylabel(feature)
#         ax.axvline(0, color = 'k')
#     plt.show()

subs = da['sub'].values
features = da['feature'].values
methods = da['average_event_method'].values
times = da['time_mins'].values
for sub in subs:
    try:
        da = erp_csd_rr_job.get(sub)['erp_csd_rr']
        N = da.shape[0]
        ncols = len(features)
        fig, axs = plt.subplots(ncols = ncols, figsize = (ncols * 5, 4), constrained_layout = True)
        fig.suptitle(f'{sub} (N = {N})')
        for c, feature in enumerate(features):
            ax = axs[c]
            da_feature = da.sel(feature=feature)
            m = da_feature.median('event').values
            ax.plot(times, m, color = 'b')
            ax.plot(times, da_feature.values.T, color = 'b', alpha = 0.2, lw = 0.5)
            ax.set_xlabel('Time (min)')
            ax.set_ylabel(feature)
            ax.axvline(0, color = 'k')
        plt.show()
    except:
        continue

In [ ]:
da = concat_erp_csd_rsa_job.get('concat')['concat_erp_csd_rsa']

# subs = da['sub'].values
# features = da['feature'].values
# methods = da['average_event_method'].values
# times = da['time_mins'].values
# for sub in subs:
#     da = concat_erp_csd_rr_job.get('concat')['concat_erp_csd_rr']
#     try:
#         N = erp_csd_rr_job.get(sub)['erp_csd_rr'].shape[0]
#     except:
#         N = 0
#     ncols = len(features)
#     fig, axs = plt.subplots(ncols = ncols, figsize = (ncols * 5, 4), constrained_layout = True)
#     fig.suptitle(f'{sub} (N = {N})')
#     for c, feature in enumerate(features):
#         ax = axs[c]
#         m = da.loc[sub, feature, 'm',:].values
#         s = da.loc[sub, feature, 's',:].values
#         ax.plot(times, m, color = 'b')
#         ax.fill_between(times, m-s, m+s, color = 'b', alpha = 0.2)
#         ax.set_xlabel('Time (min)')
#         ax.set_ylabel(feature)
#         ax.axvline(0, color = 'k')
#     plt.show()

subs = da['sub'].values
features = da['feature'].values
methods = da['average_event_method'].values
times = da['time_mins'].values
for sub in subs:
    try:
        da = erp_csd_rsa_job.get(sub)['erp_csd_rsa']
        N = da.shape[0]
        ncols = len(features)
        fig, ax = plt.subplots(figsize = (5, 4), constrained_layout = True)
        fig.suptitle(f'{sub} (N = {N})')
        da_feature = da.sel(feature=feature)
        m = da_feature.median('event').values
        ax.plot(times, m, color = 'g')
        ax.plot(times, da_feature.values.T, color = 'g', alpha = 0.2, lw = 0.5)
        ax.set_xlabel('Time (min)')
        ax.set_ylabel(feature)
        ax.axvline(0, color = 'k')
        plt.show()
    except:
        continue


# CSD DATES

In [ ]:
def filter_sd(events_df):
    try:
        return events_df[events_df['name'].apply(lambda x:True if 'SD' in x else False)].reset_index(drop = True)
    except:
        return events_df

def load_events_sd_sub(sub):
    reader = CnsReader(data_path / sub, with_events = True, event_time_zone='Europe/Paris')
    events_df = pd.DataFrame(reader.events)
    events_df['sub'] = sub
    events_df = filter_sd(events_df)
    return events_df

def get_sd_dates(sub = None):
    if not sub is None:
        events_df = load_events_sd_sub(sub)
    else:
        events_df = pd.concat([load_events_sd_sub(sub) for sub in get_patients_list_raw()]).reset_index(drop = True)
    return events_df   

In [ ]:
all_sds = get_sd_dates()

In [ ]:
all_sds['sub'].value_counts()

In [ ]:
%matplotlib widget

In [ ]:
def decimate_da(da, q):
    datetimes = da['datetime'].values
    sr = da.attrs['srate']
    new_sr = sr / q
    new_datetimes = datetimes[::q]
    new_trace = scipy.signal.decimate(da.values, q=q)
    return xr.DataArray(data = new_trace, dims = ['datetime'], coords = {'datetime':new_datetimes}, attrs = {'srate':new_sr})

In [ ]:
sub = 'GA9'
q = 3
ihr_da = ihr_job.get(sub)['ihr']
ihr_da = decimate_da(ihr_da, q)
datetimes = ihr_da['datetime'].values
srate = ihr_da.attrs['srate']

In [ ]:
n_secs = 60

def mad_roll_func(x, axis):
    return scipy.stats.median_abs_deviation(x, scale = 'normal', axis=axis)

med_roll = ihr_da.rolling(datetime = int(srate * n_secs), center = True).median().bfill('datetime').ffill('datetime')
mad_roll = ihr_da.rolling(datetime = int(srate * n_secs), center = True).reduce(mad_roll_func).bfill('datetime').ffill('datetime')

In [ ]:
fig, ax = plt.subplots(figsize = (10, 4))
ax.plot(datetimes, med_roll.values, color = 'r', alpha = 0.6)
ax.twinx().plot(datetimes, mad_roll.values, color = 'b', alpha = 0.6)
for t in get_sd_dates(sub)['start_time'].values:
    ax.axvline(t, color = 'k')

In [ ]:
%matplotlib inline

In [ ]:
n_sec = 10
da_sel = ihr_da[30000:31000]
da_sel.rolling(datetime = int(srate * n_sec), center = True).reduce(mad).plot()
da_sel.rolling(datetime = int(srate * n_sec)).reduce(mad).plot()
da_sel[::-1].rolling(datetime = int(srate * n_sec)).reduce(mad)[::-1].rolling(datetime = int(srate * n_sec)).reduce(mad).plot()

In [ ]:
win_size_mins = 10

win_size_len = int(win_size_mins * 60 * srate)
win_size_len_half = win_size_len // 2

da_erp = None

datetimes = med_roll['datetime'].values
ev_datetimes = get_sd_dates(sub)['start_time'].values
ev_inds = np.searchsorted(datetimes, ev_times)
for i, ev_ind in enumerate(ev_inds):
    start_ind = ev_ind - win_size_len_half
    stop_ind = start_ind + win_size_len
    local_med = med_roll[start_ind:stop_ind].values
    local_mad = mad_roll[start_ind:stop_ind].values
    if da_erp is None:
        time_mins = np.arange(local_med.size)/srate/60
        time_mins = time_mins - time_mins[-1] / 2
        da_erp = xr.DataArray(data = np.nan,
                              dims = ['event','feature','time_mins'],
                              coords = {'event':range(ev_inds.size), 'feature':['med','mad'], 'time_mins':time_mins}
                             )
    da_erp.loc[i, 'med',:] = local_med
    da_erp.loc[i, 'mad',:] = local_mad

In [ ]:
for feature in da_erp['feature'].values:
    da_erp_feature = da_erp.sel(feature=feature)
    m = da_erp_feature.median('event')
    s = da_erp_feature.reduce(mad, dim = 'event')
    fig, ax = plt.subplots()
    ax.plot(da_erp_feature['time_mins'], m, color = 'r')
    ax.fill_between(da_erp_feature['time_mins'], m-s , m+s, color ='r', alpha = 0.2)

In [ ]:
da_erp.median('event').sel(feature = 'med').plot.line(x = 'time_mins')

In [ ]:
da_erp.median('event').sel(feature = 'mad').plot.line(x = 'time_mins')

In [ ]:
med_roll['datetime'].sel(datetime = ev_times[0], method = 'nearest')

# VARIABLES CSD SNA

In [ ]:
ecg = concat_ecg_csd_job.get(global_key).to_dataframe()
print(ecg.columns)
ecg.shape

In [ ]:
ecg.iloc[:,40:]

In [ ]:
ecg['glasgow_score'].value_counts()

In [ ]:
# sns.histplot(data=  ecg, x = 'csd_duration_mins', hue = 'win_label', bins = np.arange(0,30,0.5))
fig, ax = plt.subplots()
sns.histplot(data=  ecg, x = 'win_duration_mins', hue = 'win_label', ax=ax)
ax.set_xlim(0,30)

In [ ]:
fig, ax = plt.subplots()
bins = np.arange(0,50,1)
ax.hist(ecg[ecg['win_label'] == 'per_1']['win_duration_mins'], bins = bins)
ax.legend()

In [ ]:
ecg[ecg['win_label'] == 'per_1']['win_duration_mins'].describe()

# META STATS hidayat

In [ ]:
subs = get_patient_list(['ECG_II','CO2'])

In [ ]:
meta = get_metadata()
meta

In [ ]:
meta = meta[meta['ID_pseudo'].isin(subs)]
meta

In [ ]:
meta['durée'].describe()

# HRV EMD

In [ ]:
sub = 'LD16'


ihr_da = ihr_job.get(sub)['ihr']
sig = ihr_da.values
srate = ihr_da.attrs['srate']
datetimes = ihr_da['datetime'].values
times = ihr_da.attrs['time']

duration = 600
tmin = 120000
tmax = tmin + duration

mask_t = (times > tmin) & (times < tmax)
sig = sig[mask_t]
time_vect = times[mask_t]

In [ ]:
imf = emd.sift.sift(sig)

In [ ]:
print(imf.shape)

In [ ]:
IP, IF, IA = emd.spectra.frequency_transform(imf, srate, 'hilbert')

In [ ]:
%matplotlib widget

In [ ]:
plt.figure()
emd.plotting.plot_imfs(imf, time_vect=time_vect)
plt.show()

In [ ]:
# Define frequency range (low_freq, high_freq, nsteps, spacing)
freq_range = (0.001, 0.6, 100, 'linear')
f, hht = emd.spectra.hilberthuang(IF, IA, freq_range, sum_time=False)

print(f.shape)
print(hht.shape)

In [ ]:
fig = plt.figure(figsize=(10, 6))
emd.plotting.plot_hilberthuang(hht, 
                               time_vect,
                               f,
                               fig=fig, 
                               log_y=False,
                               cmap='viridis',
                               )
plt.show()

In [ ]:
time_vect.shape

In [ ]:
fig, ax = plt.subplots()
ax.pcolormesh(time_vect, np.arange(hht.shape[0]), hht)
plt.show()

In [ ]:
class hrv_process_emd:
    def __init__(self, sub, figsize = (20,5)):
        ihr_da = ihr_job.get(sub)['ihr']
        self.sig = ihr_da.values
        self.srate = ihr_da.attrs['srate']
        self.datetimes = ihr_da['datetime'].values
        self.times = ihr_da.attrs['time']
        self.figsize = figsize

        imfs = emd.sift.sift(self.sig).T
        n_imfs = imfs.shape[0]
        print(n_imfs, 'IMFs were detected')
        self.da_emd = xr.DataArray(data=imfs, dims = ['n_IMF','time'], coords = {'n_IMF':range(imfs.shape[0]), 'time':self.times})

    def plot_raw(self):
        fig, ax = plt.subplots(figsize = self.figsize)
        ax.plot(self.times, self.sig)
        ax.set_ylabel('Heart rate (bpm)')
        ax.set_xlabel('Time (s)')
        plt.show()

    def plot_imf_time(self,i_imf, n_mads_ylim = 5):
        da_emd = self.da_emd
        imf = da_emd.loc[i_imf, :].values
        med, mad = physio.compute_median_mad(imf)
        ylim = (med-n_mads_ylim*mad, med + n_mads_ylim)

        fig, ax = plt.subplots(figsize=(self.figsize))
        fig.suptitle(f"IMF N° {i_imf} / {da_emd['n_IMF'].values[-1]}")
        
        ax.plot(self.times, imf)
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Amplitude')
        ax.set_ylim(*ylim)


    def plot_imf_spectrum(self, i_imf, win_size_s_welch = 120, flim = (0, 0.5), plot_scale = 'semilogy'):
        da_emd = self.da_emd
        imf = da_emd.loc[i_imf, :].values
        
        f, Pxx = scipy.signal.welch(imf, self.srate, nperseg = int(win_size_s_welch * self.srate), scaling = 'spectrum')
        f_mask = (f >= flim[0]) & (f < flim[1])
        f_plot = f[f_mask]
        Pxx_plot = Pxx[f_mask]
        
        fig, ax = plt.subplots(figsize=(self.figsize))
        fig.suptitle(f"IMF N° {i_imf} / {da_emd['n_IMF'].values[-1]}")
        if plot_scale == 'semilogy':
            ax.semilogy(f_plot, Pxx_plot)
        elif plot_scale == 'loglog':
            ax.loglog(f_plot, Pxx_plot)
        elif plot_scale == 'plot':
            ax.plot(f_plot, Pxx_plot)
        ax.set_ylabel('Amplitude ** 2')
        ax.set_xlabel('Frequency (Hz)')
        
        

In [ ]:
sub = 'LD16'
hrv_emd = hrv_process_emd(sub)

In [ ]:
hrv_emd.plot_raw()

In [ ]:

i_imf = 11
hrv_emd.plot_imf_time(i_imf=i_imf)
hrv_emd.plot_imf_spectrum(i_imf=i_imf, win_size_s_welch=36000, flim= (0, 0.7), plot_scale='semilogy')

# HRV filter by band and plot in time domain

In [ ]:
class hrv_process:
    def __init__(self, sub, figsize = (20,5)):
        ihr_da = ihr_job.get(sub)['ihr']
        self.sig = ihr_da.values
        self.srate = ihr_da.attrs['srate']
        self.datetimes = ihr_da['datetime'].values
        self.times = ihr_da.attrs['time']
        self.figsize = figsize

    def plot_raw(self):
        fig, ax = plt.subplots(figsize = self.figsize)
        ax.plot(self.times, self.sig)
        ax.set_ylabel('Heart rate (bpm)')
        ax.set_xlabel('Time (s)')
        plt.show()

    def psd(self, win_size_welch_mins, f_sel = (0, 0.6), scale = 'log'):
        nperseg = int(win_size_welch_mins * 60 * self.srate)
        f, Pxx = scipy.signal.welch(self.sig, self.srate, nperseg = nperseg, average = 'median', scaling = 'spectrum')
        f_mask = (f >= f_sel[0]) & (f <= f_sel[1])
        fig, ax = plt.subplots(figsize = (20,5))
        if scale == 'linear':
            ax.plot(f[f_mask], Pxx[f_mask])
        elif scale == 'log':
            ax.loglog(f[f_mask], Pxx[f_mask])
        ax.set_xlabel('Frequency (Hz)')
        plt.show()

    def spectrogram(self, win_size_welch_mins, f_sel = (0, 0.6), scale = 'linear', robust_q = 0.025, prop_overlap = 0.5):
        nperseg = int(win_size_welch_mins * 60 * self.srate)
        noverlap = int(nperseg * prop_overlap)
        f, t, Sxx = scipy.signal.spectrogram(self.sig, self.srate, nperseg = nperseg, noverlap = noverlap, scaling = 'spectrum')
        f_mask = (f >= f_sel[0]) & (f < f_sel[1])

        if scale == 'linear':
            Sxx = Sxx[f_mask,:]
        elif scale == 'log':
            Sxx = 10 * np.log10(Sxx[f_mask,:])

        if robust_q == 0:
            vmin, vmax = None, None
        else:
            vmin = np.quantile(Sxx, robust_q)
            vmax = np.quantile(Sxx, 1-robust_q)

        fig, ax = plt.subplots(figsize = self.figsize)
        ax.pcolormesh(t, f[f_mask], Sxx, vmin=vmin, vmax=vmax)
        ax.set_ylabel('Frequency (Hz)')
        ax.set_xlabel('Time (sec)')
        plt.show()

    def filter_bands(self, dict_bands = {'vlf':(0.0033, 0.04), 'lf':(0.04,0.15), 'hf':(0.15,0.4)}, ftype = 'bessel', order = 2):
        fig, ax = plt.subplots(figsize = self.figsize)
        for name, band in dict_bands.items():
            sig_filtered = iirfilt(self.sig, self.srate, lowcut = band[0], highcut = band[1], order = order, ftype = ftype)
            ax.plot(self.times, sig_filtered, label = name, alpha = 0.8)
        ax.legend(loc = 'upper right')
        plt.show()
        
        

In [ ]:
sub = 'LD16'
hrv = hrv_process(sub)

In [ ]:
hrv.plot_raw()

In [ ]:
hrv.psd(60)

In [ ]:
hrv.spectrogram(60, robust_q=0.01, prop_overlap=0.9, scale = 'linear', f_sel = (0.01, 0.6))

In [ ]:
dict_bands = {'ulf':(None,0.005),'vlf':(0.0033, 0.04), 'lf':(0.04,0.15), 'hf':(0.15,0.4)}

hrv.filter_bands(dict_bands)

# HRV spectro csd

In [ ]:
bug_subs = 'MR21'

sub = 'LJ8'

da = hrv_spectro_csd_job.get(sub)['hrv_spectro_csd']
da.median('num_csd').plot.pcolormesh(x = 'time', y = 'f', robust=  True)

In [ ]:
da_concat = None

subs = ['LJ8','JL5','BM3','SP2','GA9','JR10','PL20','LA19','RM6','HA1','FC13','MF12','P62','P69','P73','P70']

for sub in subs:
    da = hrv_spectro_csd_job.get(sub)['hrv_spectro_csd']
    da = da.median('num_csd')
    # print(da.shape)

    if da_concat is None:
        da_concat = init_da({'sub':subs, 'f':da['f'].values, 'time':da['time'].values})
    try:
        da_concat.loc[sub, :,:] = da.values
    except:
        continue

In [ ]:
da_concat.median('sub').plot(robust = True)

# Times RSA

In [ ]:
sub = 'MF12'
rsa_df = rsa_job.get(sub).to_dataframe()
rsa_df[['peak_time','trough_time','cycle_time']]

# Filter event not SD

In [ ]:
sub = 'RM6'
reader = pycns.CnsReader(data_path / sub, with_events=True, event_time_zone='Europe/Paris')
events = pd.DataFrame(reader.events)
events

In [ ]:
def filter_events(events_df, pattern, mode = 'without', output_mode = 'df'):
    """
    Filter events dataframe based one a string pattern used to keep or reject it.
    Parameters:
        events_df : pd.DataFrame , containing the events with a column 'name' on which the patterns will be filtered
        pattern : str , set the pattern that will be used to filter the column 'name'
        mode : str , 'with' or 'without' to keep or reject the events corresponding to the pattern, resepectively. Default = 'without' = rejection mode.
        output_mode : str , set the output of the function as being a dataframe already filtered if set to 'df' or a boolean mask if set to 'df'. Default = 'df'
    """
    assert mode in ['with','without'], f"'{mode}' search mode not possible.'mode' parameter should be set by 'with' or 'without'"
    assert output_mode in ['df','mask'], f"'{output_mode}' output_mode not possible.'output_mode' parameter should be set by 'df' or 'mask'"
    if mode == 'without':
        mask = events_df['name'].apply(lambda x:False if pattern in x else True)
    elif mode == 'with':
        mask = events_df['name'].apply(lambda x:True if pattern in x else False)
    if output_mode == 'df':
        return events_df[mask].reset_index(drop = True)
    elif output_mode == 'mask':
        return mask

In [ ]:
events_filtered = filter_events(events, 'SD', mode = 'with', output_mode='mask')

In [ ]:
events_filtered

In [ ]:
pattern_1 = 'SD'
pattern_2 = 'artefact'
mask1 = filter_events(events, pattern_1, mode = 'without', output_mode='mask')
mask2 = filter_events(events, pattern_2, mode = 'without', output_mode='mask')
events_filtered = events[mask1 & mask2] #

# Instant center frequency of HRV spectrogam

In [ ]:
sub = 'MF12'

In [ ]:
hr_spectro_da = hr_spectro_job.get(sub)['hr_spectro']

In [ ]:
f = hr_spectro_da['freq'].values
psd = hr_spectro_da[:,100].values


In [ ]:
f.shape

In [ ]:
psd.shape

In [ ]:
data.shape

In [ ]:
def icf(f, psd):
    if psd.ndim == 1:
        return np.sum(f * psd) / np.sum(psd)
    else:
        return np.apply_along_axis(lambda psd:np.sum(f * psd) / np.sum(psd), arr = psd, axis = 0)
    
def edge_freq(freqs, Sxx):
    relative_cumsum = np.cumsum(Sxx, axis = 0) / np.sum(Sxx, axis = 0)
    spectral_edge_freq = np.apply_along_axis(lambda x:freqs[np.searchsorted(x, 0.95)], arr = relative_cumsum, axis = 0)
    return spectral_edge_freq

def med_freq(freqs, Sxx):
    relative_cumsum = np.cumsum(Sxx, axis = 0) / np.sum(Sxx, axis = 0)
    median_freq = np.apply_along_axis(lambda x:freqs[np.searchsorted(x, 0.5)], arr = relative_cumsum, axis = 0)
    return median_freq

In [ ]:
fig, ax = plt.subplots()
ax.plot(f, np.cumsum(psd))
ax.axvline(icf(f, psd), color = 'r')
ax.axvline(edge_freq(f, psd), color=  'g')
ax.axvline(med_freq(f, psd), color = 'y')

In [ ]:
fig, ax = plt.subplots()
ax.plot(f, psd)
ax.axvline(icf(f, psd), color = 'g')
ax.axvline(edge_freq(f, psd), color = 'r')

In [ ]:
datetimes = hr_spectro_da['datetime'].values
data = hr_spectro_da.values

fig, ax = plt.subplots()
ax.pcolormesh(datetimes, f, data)
ax.plot(datetimes, icf(f, data), color = 'g')
ax.plot(datetimes, edge_freq(f, data), color = 'r')

# HRV Stats vs Clinical

In [ ]:
clinical_criteria = 'mRS_sortie'

meta = get_metadata()
meta = meta[meta[clinical_criteria].notna()]
subs = get_patient_list(['ECG_II'])
subs = [s for s in subs if s in meta['ID_pseudo'].unique()]
subs = [s for s in subs if not '_fin' in s]
meta = meta[meta['ID_pseudo'].isin(subs)]
dict_sub_class = meta.set_index('ID_pseudo').loc[subs,clinical_criteria].astype(int).to_dict()
sub_class_list = {}
for mrs in np.sort(meta[clinical_criteria].unique()):
    try:
        sub_class_list[mrs] = meta.set_index(clinical_criteria).loc[mrs,'ID_pseudo'].tolist()
    except:
        sub_class_list[mrs] = [meta.set_index(clinical_criteria).loc[mrs,'ID_pseudo']]
print({sub_class:len(elements) for sub_class, elements in sub_class_list.items()})

fig, ax = plt.subplots(figsize = (4,4))
pd.Series({sub_class:len(elements) for sub_class, elements in sub_class_list.items()}).plot.bar(ax=ax)
for bar in ax.containers:
    ax.bar_label(bar)
ax.set_ylabel('N patients')
ax.set_xlabel(clinical_criteria)

In [ ]:
def get_meta_info(meta, sub):
    glasgow_bins = (7, 12)
    meta_search = ['motif','GCS_sortie','mRS_sortie','mRS_1mois','mRS_3mois','mRS_6mois']
    meta_return = {}
    meta_set = meta.set_index('ID_pseudo')
    for meta_metric in meta_search:
        try:
            meta_return[meta_metric] = meta_set.loc[sub,meta_metric]
        except:
            meta_return[meta_metric] = None
    if meta_return['GCS_sortie'] <= glasgow_bins[0]:
        meta_return['gcs_interpretation'] = 'bad'
    elif meta_return['GCS_sortie'] > glasgow_bins[1]:
        meta_return['gcs_interpretation'] = 'good'
    else:
        meta_return['gcs_interpretation'] = 'intermediate'

    if meta_return['mRS_sortie'] is np.nan:
        meta_return['mrs_interpretation'] = np.nan
    elif meta_return['mRS_sortie'] == 6:
        meta_return['mrs_interpretation'] = 'death'
    elif meta_return['mRS_sortie'] == 5:
        meta_return['mrs_interpretation'] = 'severe'
    else:
        meta_return['mrs_interpretation'] = 'moderate'
    return meta_return

In [ ]:
n_hours_before_end = 5

meta = get_metadata()
subs_meta = list(meta['ID_pseudo'].unique())

metrics = ['ulf','vlf','lf','hf','lf_hf','rsa_freq','slope']

rows = []

subs = get_patient_list(['ECG_II'])
subs = [s for s in subs if s in subs_meta]

for sub in subs:
    meta_dict = get_meta_info(meta, sub)

    hrv_freq_da = hrv_freq_job.get(sub)['hrv_freq']
    times = hrv_freq_da.attrs['time']
    sr = np.median(np.diff(times))
    n_inds_before_end = int(n_hours_before_end * 3600 * sr)
    rsa_freq_da = rsa_freq_job.get(sub)['rsa_freq']
    try:
        rsa_time = rsa_job.get(sub).to_dataframe()['rsa_amplitude_clean'].values
    except:
        rsa_time = np.nan
    hr_slope_da = hr_spectrum_slope_job.get(sub)['hr_spectrum_slope']
    hr_med_freq_da = hr_center_frequency_job.get(sub)['hr_center_frequency'].loc['med_freq',:]
    ecg_peaks = detect_ecg_job.get(sub).to_dataframe()

    traces = {'total':hrv_freq_da.loc['total'].values,
              'ulf':hrv_freq_da.loc['ulf'].values,
              'vlf':hrv_freq_da.loc['vlf'].values, 
            'lf':hrv_freq_da.loc['lf'].values, 
            'hf':hrv_freq_da.loc['hf'].values,
            'lf_hf':hrv_freq_da.loc['lf_hf'].values,
            'rsa_freq':rsa_freq_da.values,
            'rsa_time':rsa_time,
            'slope':hr_slope_da.values,
            'med_freq':hr_med_freq_da.values
            }
    traces['ulf_n'] = 100 * traces['ulf'] / traces['total']
    traces['vlf_n'] = 100 * traces['vlf'] / traces['total']
    traces['lf_n'] = 100 * traces['lf'] / traces['total']
    traces['hf_n'] = 100 * traces['hf'] / traces['total']
    traces['rsa_freq_n'] = traces['rsa_freq'] / traces['total']
    
    # results = {metric:np.nanmedian(trace[-n_inds_before_end:]) for metric, trace in traces.items()}
    results = {metric:np.nanmedian(trace) for metric, trace in traces.items()}
    med_rri, mad_rri = physio.compute_median_mad(60 / np.diff(ecg_peaks['peak_time']))
    results['heart_rate_bpm'] = med_rri
    results['hrv_mad_bpm'] = mad_rri 

    row = {'sub':sub} | meta_dict | results
    rows.append(row)

res = pd.DataFrame(rows)

In [ ]:
res

In [ ]:
outcomes_clean = {
    'heart_rate_bpm':('Heart rate','bpm'),
    'hrv_mad_bpm':('HRV MAD','bpm'),
    'rsa_time':('RSA time','bpm'),
    'ulf':('ULF','bpm'),
    'vlf':('VLF','bpm'),
    'lf':('LF','bpm'),
    'hf':('HF','bpm'),
    'total':('Total "power"','bpm'),
    'ulf_n':('ULF norm',None),
    'vlf_n':('VLF norm',None),
    'lf_n':('LF norm',None),
    'hf_n':('HF norm',None),
    'lf_hf':('LF / HF',None),
    'slope':('Spectum Slope',None),
    'rsa_freq':('RSA freq','bpm'),
    'rsa_freq_n':('RSA freq norm',None),
                  }

In [ ]:
predictors = list(meta_dict.keys())
outcomes = list(outcomes_clean.keys())

nrows = len(outcomes)
ncols = len(predictors)

fig, axs = plt.subplots(nrows = nrows, ncols=ncols, figsize = (ncols * 3, nrows * 3), constrained_layout = True)

for c, predictor in enumerate(predictors):
    for r, outcome in enumerate(outcomes):
        ax = axs[r,c]
        res_kruskal = pg.kruskal(data=res, dv=outcome, between=predictor)
        stat_res = res_kruskal.loc['Kruskal','H'].round(3)
        p = res_kruskal.loc['Kruskal','p-unc'].round(3)
        signif = get_significance(p)
 
        # sns.pointplot(data = res,
        #               x = predictor,
        #               y = outcome,
        #               ax=ax,
        #               estimator = 'median',
        #               )

        sns.boxplot(data = res,
                      x = predictor,
                      y = outcome,
                      ax=ax,
                      )


        ax.set_title(f'H : {stat_res} - p : {p} ({signif})', weight = 'bold' if p < 0.05 else 'normal')

fig.savefig(base_folder / 'figures' / 'hrv_vs_clinic' / 'huge_boxplot_matrix.png', dpi = 500, bbox_inches = 'tight')
plt.show()

In [ ]:
res.columns

In [ ]:
fig, ax = plt.subplots(figsize = (3,3))
xs = [0,1,2]
levels = ['death','severe','moderate']
for i, level, c in zip(xs,levels,['r','b','g']): 
    ax.bar(i, height = res['mrs_interpretation'].value_counts()[level], color = c)

for bar in ax.containers:
    ax.bar_label(bar)
ax.set_xticks(xs,levels)
ax.set_ylabel('N')

In [ ]:
predictor = 'mrs_interpretation'
outcomes = list(outcomes_clean.keys())

nrows = 4
ncols = 4
poss = attribute_subplots(outcomes, nrows, ncols)

fig, axs = plt.subplots(nrows = nrows, ncols=ncols, figsize = (ncols * 3, nrows * 3), constrained_layout = True)

for outcome, pos in poss.items():
    ax = axs[*pos]
    res_kruskal = pg.kruskal(data=res, dv=outcome, between=predictor)
    stat_res = res_kruskal.loc['Kruskal','H'].round(3)
    p = res_kruskal.loc['Kruskal','p-unc'].round(3)
    signif = get_significance(p)


    auto_stats(df = res, 
               predictor = predictor, 
               outcome = outcome, 
               design = 'between', 
               ax=ax, 
               xtick_info=False,
               force_post_hoc=False,
               outcome_clean_label=outcomes_clean[outcome][0],
               outcome_unit=outcomes_clean[outcome][1],
               order=['death','severe','moderate'],
               palette = {'death':'r','severe':'b','moderate':'g'},
               with_title = False,
               )
    ax.set_title(f'Kruskall-Wallis :\nH : {stat_res} - p : {p} ({signif})', weight = 'bold' if p < 0.05 else 'normal')
    ax.set_xlabel('mRS Interpretation')

fig.savefig(base_folder / 'figures' / 'hrv_vs_clinic' / 'mrs_boxplots.png', dpi = 500, bbox_inches = 'tight')
plt.show()

In [ ]:
stats_quali_quali(data = res, predictor = 'gcs_interpretation', outcome = 'mrs_interpretation')

In [ ]:
stats_quantitative(df = res, xlabel = 'rsa_time', ylabel = 'rsa_freq')

In [ ]:
plt.figure()
sns.clustermap(res[outcomes].corr('spearman'), cmap ='seismic', vmin = -1, vmax = 1, annot = True)
plt.savefig(base_folder / 'figures' / 'hrv_vs_clinic' / 'clustermap_metrics.png', dpi = 500, bbox_inches = 'tight')
plt.show()

In [ ]:
mapper_rename = {name:outcomes_clean[name][0] for name in outcomes_clean.keys()}
fig, ax = plt.subplots(figsize = (8,6))
sns.heatmap(res[outcomes].rename(columns = mapper_rename).corr('spearman'), cmap ='seismic', vmin = -1, vmax = 1, annot = True, ax=ax)
fig.savefig(base_folder / 'figures' / 'hrv_vs_clinic' / 'heatmap_metrics.png', dpi = 500, bbox_inches = 'tight')
plt.show()

In [ ]:
plt.figure()
sns.pairplot(data = res[outcomes])
plt.savefig(base_folder / 'figures' / 'hrv_vs_clinic' / 'pairplot_metrics.png', dpi = 500, bbox_inches = 'tight')
plt.show()

# HRV Spectrum slope

In [ ]:
sub = 'MF12' #

In [ ]:
p = {
    'wsize_welch_mins':30,
    'power_or_amplitude':'amplitude',
    'overlap_prct':0.5,
    'nfft_factor':1,
    }

In [ ]:
# hr_spectro_da = hr_spectro_job.get(sub)['hr_spectro']
hr_spectro_da = hr_spectro(sub, **p)['hr_spectro']
fmin = 0.01
fmax = 0.3
spectro_slopes = np.apply_along_axis(compute_spectrum_log_slope, axis = 0, arr = hr_spectro_da.values, freqs = hr_spectro_da['freq'].values, freq_range = [fmin, fmax])
spectro_slopes_da = xr.DataArray(data = spectro_slopes, dims = ['datetime'], coords = {'datetime':hr_spectro_da['datetime']})

In [ ]:
compute_spectrum_log_slope(hr_spectro_da[:,20], freqs = hr_spectro_da['freq'].values, freq_range = [0.0033, 0.4], show = True)

In [ ]:
hr_spectro_da.median('datetime').plot(xscale = 'log', yscale = 'log')

# Correlation hrv freq band rsa freq

In [ ]:
sub = 'LA19'

hrv_freq_da = hrv_freq_job.get(sub)['hrv_freq']
rsa_freq_da = rsa_freq_job.get(sub)['rsa_freq']
hr_slope_da = hr_spectrum_slope_job.get(sub)['hr_spectrum_slope']

In [ ]:
df = pd.DataFrame()
df['total'] = hrv_freq_da.loc['total'].values
df['vlf'] = hrv_freq_da.loc['vlf'].values
df['lf'] = hrv_freq_da.loc['lf'].values
df['hf'] = hrv_freq_da.loc['hf'].values
df['rsa_freq'] = rsa_freq_da.values
df['hr_slope'] = hr_slope_da.values

In [ ]:
pairplot_homemade(data = df, kind = 'scatter')

# IHR to HRV spectrum percentage plot, effect of nperseg

In [ ]:
def ihr_to_stackplot(sub, wsize_welch_mins, power_or_amplitude):
    hr_spectro_params = {
    'wsize_welch_mins':wsize_welch_mins,
    'power_or_amplitude':power_or_amplitude,
    'overlap_prct':0.5,
    'nfft_factor':1,
    }

    hrv_freq_params = {
        'hr_spectro_params':hr_spectro_params,
        'freq_bands':{'total':(0.005,0.4),'vlf':(0.005,0.04), 'lf': (0.04, 0.15), 'hf' : (0.15, .4)},
    }


    p = hr_spectro_params
    ihr_da = ihr_job.get(sub)['ihr']
    times = ihr_da.attrs['time']
    srate = ihr_da.attrs['srate']
    datetimes = ihr_da['datetime'].values

    nperseg = int(p['wsize_welch_mins'] * 60 * srate)
    noverlap = int(nperseg * p['overlap_prct'])
    nfft = int(nperseg * p['nfft_factor'])

    freqs, times_spectrum_s, Sxx = scipy.signal.spectrogram(ihr_da.values, fs = srate, nperseg =  nperseg, noverlap = noverlap, nfft = nfft, scaling = 'spectrum')
    if p['power_or_amplitude'] == 'amplitude': 
        Sxx = np.sqrt(Sxx)
    times_spectrum_s = times_spectrum_s + times[0]
    inds_times_spectrum = np.searchsorted(times, times_spectrum_s)
    datetimes_spectrum = datetimes[inds_times_spectrum]

    f_mask = (freqs < 1)
    Sxx = Sxx[f_mask,:]

    hr_spectro_da = xr.DataArray(data = Sxx, dims = ['freq','datetime'], coords = {'freq':freqs[f_mask], 'datetime':datetimes_spectrum})
    hr_spectro_da.attrs['time'] = times_spectrum_s



    p = hrv_freq_params
    dict_fband = p['freq_bands']
    names = list(dict_fband.keys())
    names_full = names + ['lf_hf']
    hrv_freq_da = init_da({'band':names_full, 'datetime':hr_spectro_da['datetime'].values})
    for name, (f0,f1) in dict_fband.items():
        hrv_freq_da.loc[name, :] = hr_spectro_da.loc[f0:f1,:].integrate('freq')
    hrv_freq_da.loc['lf_hf',:] = hrv_freq_da.loc['lf',:] / hrv_freq_da.loc['hf',:]
    hrv_freq_da.attrs['time'] = hr_spectro_da.attrs['time']


    fig, axs = plt.subplots(nrows = 2, figsize = (10, 8), constrained_layout = True)
    fig.suptitle(f'{sub} - {wsize_welch_mins} mins - {power_or_amplitude}', fontsize = 20)

    ax = axs[0]
    colors = ["red", "green", "blue"]  # Colors for vlf, lf, hf
    ax.stackplot(
        hrv_freq_da['datetime'],
        hrv_freq_da.sel(band="vlf"),
        hrv_freq_da.sel(band="lf"),
        hrv_freq_da.sel(band="hf"),
        labels=["VLF", "LF", "HF"],
        colors=colors,
        alpha=0.6,
    )
    ax.set_ylim(0, 0.2)
    ax.set_ylabel('Amplitude')
    ax.set_title('HRV spectro total amplitude')
    ax.legend(loc = 'upper left')

    ax = axs[1]
    band_names = ['vlf','lf','hf']
    percentage_da = None
    for band_name in band_names:
        percentage_band = (hrv_freq_da.loc[band_name,:] / hrv_freq_da.loc['total',:]) * 100
        if percentage_da is None:
            percentage_da = init_da({'band':band_names, 'datetime':hrv_freq_da['datetime'].values})
        percentage_da.loc[band_name,:] = percentage_band
    colors = ["red", "green", "blue"]  # Colors for vlf, lf, hf

    # Stacked area plot
    ax.stackplot(
        percentage_da['datetime'],
        percentage_da.sel(band="vlf"),
        percentage_da.sel(band="lf"),
        percentage_da.sel(band="hf"),
        labels=["VLF", "LF", "HF"],
        colors=colors,
        alpha=0.6,
    )
    ax.legend(loc = 'upper left')
    ax.set_title("HRV spectro % amplitude by band")
    ax.set_title('Percentage HRV bands')
    ax.set_ylabel('Percentage (%)')

    plt.show()

In [ ]:
ihr_to_stackplot('MF12', 30, 'amplitude')

In [ ]:
ihr_to_stackplot('MF12', 30, 'power')

# HRV spectrum percentage plot

In [ ]:
sub = 'MF12'
hrv_freq_da = hrv_freq_job.get(sub)['hrv_freq']

In [ ]:
hrv_freq_da

In [ ]:
band_names = ['vlf','lf','hf']

percentage_da = None
for band_name in band_names:
    percentage_band = (hrv_freq_da.loc[band_name,:] / hrv_freq_da.loc['total',:]) * 100
    if percentage_da is None:
        percentage_da = init_da({'band':band_names, 'datetime':hrv_freq_da['datetime'].values})
    percentage_da.loc[band_name,:] = percentage_band


In [ ]:
percentage_da.rolling(datetime = 10).median().plot.line(x = 'datetime')

In [ ]:
percentage_da_smooth = percentage_da.rolling(datetime = 10).median()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ["blue", "green", "red"]  # Colors for vlf, lf, hf

# Stacked area plot
ax.stackplot(
    percentage_da['datetime'],
    percentage_da.sel(band="vlf"),
    percentage_da.sel(band="lf"),
    percentage_da.sel(band="hf"),
    labels=["VLF", "LF", "HF"],
    colors=colors,
    alpha=0.6,
)

plt.show()

# HRV spectrum vs Clinical State

In [ ]:
clinical_criteria = 'mrs_interpretation'

mapper_mrs = {6:'death',5:'severe',4:'moderate'}

meta = get_metadata()
meta['mrs_interpretation'] = meta['mRS_sortie'].map(mapper_mrs)
meta = meta[meta['mRS_sortie'] >= 4]
subs = get_patient_list(['ECG_II'])
subs = [s for s in subs if s in meta['ID_pseudo'].unique()]
subs = [s for s in subs if not '_fin' in s]
meta = meta[meta['ID_pseudo'].isin(subs)]
dict_sub_class = meta.set_index('ID_pseudo').loc[subs,clinical_criteria].to_dict()
sub_class_list = {mrs:meta.set_index(clinical_criteria).loc[mrs,'ID_pseudo'].tolist() for mrs in mapper_mrs.values()}
print({sub_class:len(elements) for sub_class, elements in sub_class_list.items()})

In [ ]:
# clinical_criteria = 'mRS_sortie'

# meta = get_metadata()
# meta = meta[meta[clinical_criteria].notna()]
# subs = get_patient_list(['ECG_II'])
# subs = [s for s in subs if s in meta['ID_pseudo'].unique()]
# subs = [s for s in subs if not '_fin' in s]
# meta = meta[meta['ID_pseudo'].isin(subs)]
# dict_sub_class = meta.set_index('ID_pseudo').loc[subs,clinical_criteria].astype(int).to_dict()
# sub_class_list = {mrs:meta.set_index(clinical_criteria).loc[mrs,'ID_pseudo'].tolist() for mrs in np.sort(meta[clinical_criteria].unique())}

# print({sub_class:len(elements) for sub_class, elements in sub_class_list.items()})

In [ ]:
n_mins_welch_win = 10

n_periods = 5

win_size_welch_s = 60 * n_mins_welch_win 

hrv_freq_da = None

for sub in subs:
    ihr_da = ihr_job.get(sub)['ihr']
    srate = ihr_da.attrs['srate']
    ihr_sig = ihr_da.values

    inds_slice = np.linspace(0, ihr_sig.size, n_periods + 1).astype(int)
    start_inds =  inds_slice[:-1]
    stop_inds = inds_slice[1:]

    for period_i in range(n_periods):
        start_ind = start_inds[period_i]
        stop_ind = stop_inds[period_i]

        ihr_sel = ihr_sig[start_ind:stop_ind]

        nperseg = int(win_size_welch_s * srate)
        nfft = nperseg

        f, Pxx = scipy.signal.welch(ihr_sel, srate, nperseg = nperseg, nfft=nfft, scaling = 'spectrum', average = 'median')

        if hrv_freq_da is None:
            hrv_freq_da = init_da({'sub':subs, 'period':np.arange(n_periods)+1,'f':f})
        hrv_freq_da.loc[sub, period_i+1, :] = Pxx
        

In [ ]:
midx = [dict_sub_class.keys(), dict_sub_class.values()]
midx = pd.MultiIndex.from_arrays(midx, names=["sub_name", "mrs_interpretation"])
hrv_freq_da_midx = hrv_freq_da.copy()
hrv_freq_da_midx = hrv_freq_da_midx.assign_coords(sub = midx)

In [ ]:
hrv_freq_da.loc['P17',:,:].plot.line(x='f', hue = 'period', xscale = 'log', yscale = 'log', alpha = 0.5)

In [ ]:
# hrv_freq_da.loc[:,:,0.01:0.5].plot.line(x='f', hue = 'period', xscale = 'log', yscale = 'log', alpha = 0.5, col = 'sub', col_wrap = 10)

In [ ]:
hrv_freq_da_midx.loc[:,:,0:0.5].groupby('mrs_interpretation').median().plot.line(hue = 'mrs_interpretation', xscale = 'log', yscale = 'log', col = 'period')

In [ ]:
np.unique(hrv_freq_da_midx['mrs_interpretation'])

In [ ]:
nrows = 5
fig, axs = plt.subplots(nrows = nrows, figsize = (9, nrows * 3), constrained_layout = True)

colors_mrs = {'death':'r', 'severe':'b', 'moderate':'g'}

hrv_freq_da_midx_plot = hrv_freq_da_midx.sel(f = slice(0,0.5))
# hrv_freq_da_midx_plot[:] = np.log(hrv_freq_da_midx_plot.values)

q = 0.05
vmin = hrv_freq_da_midx_plot.quantile(q)
vmax = hrv_freq_da_midx_plot.quantile(1-q)

for i, period in enumerate(hrv_freq_da_midx_plot['period'].values):
    ax = axs[i]
    ax.set_title(f'Period : {period}')
    ax.set_xlabel('Frequency (Hz)')
    # ax.set_ylim(vmin, vmax)
    for mrs in np.unique(hrv_freq_da_midx_plot['mrs_interpretation']):
        for sub in hrv_freq_da_midx_plot.sel(mrs_interpretation = mrs)['sub_name'].values:
            ax.semilogy(hrv_freq_da_midx_plot['f'], hrv_freq_da_midx_plot.loc[sub, period, :][0,:].values, alpha = 0.2, color = colors_mrs[mrs])
        



In [ ]:
nrows = 5
fig, axs = plt.subplots(nrows = nrows, figsize = (9, nrows * 3), constrained_layout = True)

colors_mrs = {'death':'r', 'severe':'b', 'moderate':'g'}

hrv_freq_da_midx_plot = hrv_freq_da_midx.sel(f = slice(0,0.5))
hrv_freq_da_midx_plot_estimators = None
periods = hrv_freq_da_midx_plot['period'].values
mrss = np.unique(hrv_freq_da_midx_plot['mrs_interpretation'])
for period in periods:
    for mrs in mrss:
        hrv_mrs_period = hrv_freq_da_midx_plot.sel(mrs_interpretation = mrs, period = period)
        m = np.median(hrv_mrs_period, axis = 0)
        s = scipy.stats.median_abs_deviation(hrv_mrs_period, axis = 0, scale = 'normal')
        if hrv_freq_da_midx_plot_estimators is None:
            hrv_freq_da_midx_plot_estimators = init_da({'period':periods, 'mrs_interpretation':mrss, 'estimator':['m','s','low','high'], 'f':hrv_freq_da_midx_plot['f'].values})
        hrv_freq_da_midx_plot_estimators.loc[period, mrs, 'm' ,:] = m
        hrv_freq_da_midx_plot_estimators.loc[period, mrs, 's' ,:] = s
        hrv_freq_da_midx_plot_estimators.loc[period, mrs, 'low' ,:] = m-s
        hrv_freq_da_midx_plot_estimators.loc[period, mrs, 'high' ,:] = m+s
# hrv_freq_da_midx_plot_estimators_log[:] = np.log(hrv_freq_da_midx_plot_estimators.values)

q = 0.05
# vmin = hrv_freq_da_midx_plot.quantile(q)
# vmax = hrv_freq_da_midx_plot.quantile(1-q)
vmin = hrv_freq_da_midx_plot_estimators.min()
vmax = hrv_freq_da_midx_plot_estimators.max()

for i, period in enumerate(hrv_freq_da_midx_plot_estimators['period'].values):
    ax = axs[i]
    ax.set_title(f'Period : {period}')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylim(vmin, vmax)
    for mrs in np.unique(hrv_freq_da_midx_plot_estimators['mrs_interpretation']):
        hrv_mrs_period = hrv_freq_da_midx_plot_estimators.sel(mrs_interpretation = mrs, period = period)
        ax.plot(hrv_mrs_period['f'], hrv_mrs_period.sel(estimator = 'm'), lw = 2, color = colors_mrs[mrs], label = mrs)
        ax.fill_between(hrv_mrs_period['f'], hrv_mrs_period.sel(estimator = 'low'), hrv_mrs_period.sel(estimator = 'high'), color = colors_mrs[mrs], alpha = 0.2)
    ax.legend(loc = 'upper right')


        # ax.semilogy(hrv_freq_da_midx_plot['f'], hrv_freq_da_midx_plot.loc[sub, period, :][0,:].values, alpha = 0.2, color = colors_mrs[mrs])
        



In [ ]:

colors_mrs = {'death':'r', 'severe':'b', 'moderate':'g'}

hrv_freq_da_midx_plot = hrv_freq_da_midx.sel(f = slice(0.0033,0.5))
hrv_freq_da_midx_plot_estimators_log = None
periods = hrv_freq_da_midx_plot['period'].values
mrss = np.unique(hrv_freq_da_midx_plot['mrs_interpretation'])
for period in periods:
    for mrs in mrss:
        hrv_mrs_period = hrv_freq_da_midx_plot.sel(mrs_interpretation = mrs, period = period)
        m = np.median(np.log(hrv_mrs_period), axis = 0)
        s = scipy.stats.median_abs_deviation(np.log(hrv_mrs_period), axis = 0, scale = 'normal')
        if hrv_freq_da_midx_plot_estimators_log is None:
            hrv_freq_da_midx_plot_estimators_log = init_da({'period':periods, 'mrs_interpretation':mrss, 'estimator':['m','s','low','high'], 'f':hrv_freq_da_midx_plot['f'].values})
        hrv_freq_da_midx_plot_estimators_log.loc[period, mrs, 'm' ,:] = m
        hrv_freq_da_midx_plot_estimators_log.loc[period, mrs, 's' ,:] = s
        hrv_freq_da_midx_plot_estimators_log.loc[period, mrs, 'low' ,:] = m-s
        hrv_freq_da_midx_plot_estimators_log.loc[period, mrs, 'high' ,:] = m+s

q = 0.05
# vmin = hrv_freq_da_midx_plot_estimators_log.quantile(q)
# vmax = hrv_freq_da_midx_plot_estimators_log.quantile(1-q)
vmin = hrv_freq_da_midx_plot_estimators_log.min()
vmax = hrv_freq_da_midx_plot_estimators_log.max()

nrows = 5
fig, axs = plt.subplots(ncols = nrows, figsize = (nrows * 3, 4), constrained_layout = True, sharey = True)
fig.suptitle('HRV log-log spectrum according to the period of stay (divided in 5 equal parts) and clinical outcome at end', fontsize=  20)

for i, period in enumerate(hrv_freq_da_midx_plot_estimators_log['period'].values):
    ax = axs[i]
    ax.set_title(f'Period : {period}')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylim(vmin, 2)
    if i == 0:
        ax.set_ylabel('HRV (log)')
    for mrs in np.unique(hrv_freq_da_midx_plot_estimators_log['mrs_interpretation']):
        hrv_mrs_period = hrv_freq_da_midx_plot_estimators_log.sel(mrs_interpretation = mrs, period = period)
        n_subs = len(sub_class_list[mrs])
        ax.semilogx(hrv_mrs_period['f'], hrv_mrs_period.sel(estimator = 'm'), lw = 2, color = colors_mrs[mrs], label = f'outcome : {mrs} (n = {n_subs})')
        ax.fill_between(hrv_mrs_period['f'], hrv_mrs_period.sel(estimator = 'low'), hrv_mrs_period.sel(estimator = 'high'), color = colors_mrs[mrs], alpha = 0.2)
    ax.legend(loc = 'lower left', fontsize = 8)


        # ax.semilogy(hrv_freq_da_midx_plot['f'], hrv_freq_da_midx_plot.loc[sub, period, :][0,:].values, alpha = 0.2, color = colors_mrs[mrs])
        
fig.savefig(base_folder / 'figures' / 'hrv_vs_clinic' / 'hrv_spectrum_by_period_and_mrs.png', dpi = 300, bbox_inches = 'tight')

plt.show()


In [ ]:
hrv_freq_da_midx.loc[:,:,0.0033:0.5].integrate('f').groupby('mrs_interpretation').median().plot.line(x = 'period', hue = 'mrs_interpretation')

In [ ]:
hrv_freq_da_midx.loc[:,:,0.0033:0.04].integrate('f').groupby('mrs_interpretation').median().plot.line(x = 'period', hue = 'mrs_interpretation')

In [ ]:
hrv_freq_da_midx.loc[:,:,0.04:0.15].integrate('f').groupby('mrs_interpretation').median().plot.line(x = 'period', hue = 'mrs_interpretation')

In [ ]:
hrv_freq_da_midx.loc[:,:,0.15:0.5].integrate('f').groupby('mrs_interpretation').median().plot.line(x = 'period', hue = 'mrs_interpretation')

In [ ]:
meds = hrv_freq_da_midx.loc[:,:,:0.6].groupby("mrs_interpretation").median()
meds.plot.line(x = 'f', xscale = 'log', yscale = 'log', alpha = 0.5, col = 'period')

In [ ]:
def mad_func(a):
    mad = scipy.stats.median_abs_deviation(a, scale='normal', axis=0)
    return xr.DataArray(mad, coords={'period':np.arange(n_periods) +1,'f':a['f']}, dims=['period','f'])
mads = hrv_freq_da_midx.loc[:,:, :0.6].groupby("mrs_interpretation").apply(mad_func)
mads.plot.line(x = 'f', xscale = 'log', yscale = 'log', alpha = 0.5, col = 'period')


In [ ]:
# fig, ax = plt.subplots()

# for i_alive, c in zip([4,5,6], ['g','b','r']):
#     f = meds['f'].values
#     med = meds.loc[i_alive,:].values
#     mad = mads.loc[i_alive,:].values
#     ax.plot(f, med , color = c)
#     ax.fill_between(f, med-mad, med+mad, color = c, alpha = 0.2)

# plt.show()


In [ ]:
import mne

In [ ]:
def stat_fun(*X):
    res = scipy.stats.kruskal(*X)
    return res.statistic

In [ ]:
hrv_freq_da_for_stats = hrv_freq_da.copy()
hrv_freq_da_for_stats = np.sqrt(hrv_freq_da_for_stats)
hrv_freq_da_for_stats = hrv_freq_da_for_stats.sel(period = 5)
levels = [4,5,6]
X = [hrv_freq_da_for_stats.loc[sub_class_list[level]].values for level in [4,5,6]]
F_obs, clusters, cluster_pv, H0 = mne.stats.permutation_cluster_test(X, n_permutations=100, stat_fun = stat_fun, out_type = 'mask', n_jobs = 5)
# F_obs, clusters, cluster_pv, H0 = mne.stats.permutation_cluster_test(X, n_permutations=100, stat_fun = None, n_jobs = 5)

res_stats = xr.DataArray(np.zeros(F_obs.shape), dims=['F','p'])
print((cluster_pv < 0.05).sum() , 'cluster significant')


In [ ]:
f_mask = (hrv_freq_da['f'] > 0.005) & (hrv_freq_da['f'] < 0.6)
fig, ax = plt.subplots()
ax.plot(hrv_freq_da['f'][f_mask], F_obs[f_mask])
plt.show()

In [ ]:
for period in hrv_freq_da['period'].values:

    hrv_freq_da_for_stats = hrv_freq_da.copy()
    # hrv_freq_da_for_stats = np.sqrt(hrv_freq_da_for_stats)
    hrv_freq_da_for_stats = hrv_freq_da_for_stats.sel(period = period)
    levels = [4,5,6]
    X = [hrv_freq_da_for_stats.loc[sub_class_list[level]].values for level in [4,5,6]]

    res = scipy.stats.kruskal(*X, axis = 0)

    f_mask = (hrv_freq_da['f'] > 0.005) & (hrv_freq_da['f'] < 0.6)

    x = hrv_freq_da['f'].values[f_mask]
    y = res.statistic[f_mask]
    ps = res.pvalue[f_mask]

    fig, ax = plt.subplots()
    ax.plot(x, y)
    ax.fill_between(x, y, where=(ps)  < 0.05, color="r", alpha=0.3)
    ax.set_title(period)
    plt.show()

In [ ]:
res = scipy.stats.friedmanchisquare(*[hrv_freq_da.loc[:,i,:].values for i in hrv_freq_da['period'].values], axis = 0)

In [ ]:
res.statistic.shape

In [ ]:
f_mask = (hrv_freq_da['f'] > 0.005) & (hrv_freq_da['f'] < 0.6)

x = hrv_freq_da['f'].values[f_mask]
y = res.statistic[f_mask]
ps = res.pvalue[f_mask]

fig, ax = plt.subplots()
ax.plot(x, y)
ax.fill_between(x, y, where=(ps)  < 0.05, color="r", alpha=0.3)
# ax.set_title(period)
plt.show()

# Drugs doses distributions

In [ ]:
sub = 'MF12'

In [ ]:
pse_ds = icca_pse_tt_job.get(sub)
pse_ds

In [ ]:
for drug in pse_ds.data_vars.keys():
    doses = pse_ds[drug].values
    fix, ax = plt.subplots()
    ax.hist(doses, bins = 50)
    ax.set_title(doses.size)
    plt.show()

In [ ]:
rows = []
for sub in get_icca_subs():
    try:
        pse_ds = icca_pse_tt_job.get(sub)
        for drug in pse_ds.data_vars.keys():
            doses = pse_ds[drug].values
            row = dict(sub=sub, drug=drug, N = doses.size, min_ = min(doses), max_ = max(doses))
            rows.append(row)
    except:
        continue

res = pd.DataFrame(rows)

In [ ]:
mins = {drug:min(res[res['drug'] == drug]['min_']) for drug in res['drug'].unique()}
maxs = {drug:max(res[res['drug'] == drug]['max_']) for drug in res['drug'].unique()}

In [ ]:
# drugs = res['drug'].unique().tolist()
# no_pse_subs = ['P1','P73','HA1']
# subs = res['sub'].unique().tolist()
# colors = colormaps['nipy_spectral']
# colors_inds = np.linspace(0,1,len(subs))
# colors = {sub:colors(i) for  sub,i in zip(subs,colors_inds)}
# for drug in drugs:
#     fig, ax = plt.subplots()
#     ax.set_title(drug)
#     bins = np.linspace(mins[drug], maxs[drug], 30)
#     unit = None
#     for sub in subs:
#         pse_ds = icca_pse_tt_job.get(sub)
#         if drug in pse_ds.data_vars.keys():
#             if unit is None:
#                 unit = pse_ds[drug].attrs['unit']
#             doses = pse_ds[drug].values
#             ax.hist(doses, bins=bins, edgecolor = 'k', color = colors[sub], label = f'{sub} (N : {doses.size})', alpha = 0.5)
#     ax.set_xlabel(unit)
#     ax.legend(loc = 'upper left', fontsize = 5)
#     plt.show()

In [ ]:
drugs = res['drug'].unique().tolist()
subs = res['sub'].unique().tolist()

drug_doses = {}
for drug in drugs:
    pool = {'values':[], 'unit' : None}
    for sub in subs:
        pse_ds = icca_pse_tt_job.get(sub)
        if drug in pse_ds.data_vars.keys():
            if pool['unit'] is None:
                pool['unit'] = pse_ds[drug].attrs['unit']
            doses = pse_ds[drug].values
            pool['values'].extend(doses)
    drug_doses[drug] = pool

In [ ]:
nrows = 7
ncols = 5
poss = attribute_subplots(list(drug_doses.keys()), nrows, ncols)

fig, axs = plt.subplots(nrows, ncols, constrained_layout = True, figsize = (nrows * 2, ncols * 2))

for drug_name, pos in poss.items():
    ax = axs[*pos]
    d_drug = drug_doses[drug_name]
    unit = d_drug['unit']
    values = np.array(d_drug['values'])
    bins = np.linspace(np.nanquantile(values, 0.05), np.nanquantile(values, 0.95), 30)
    ax.hist(values, bins = bins, edgecolor = 'k')
    ax.set_title(f'{drug_name} - N : {values.size}')
    ax.set_xlabel(unit)
fig.savefig(base_folder / 'figures' / 'icca_figs' / 'pse_dose.png', dpi = 500, bbox_inches = 'tight')
plt.show()
    

In [ ]:
rows = []
for sub in get_icca_subs():
    try:
        ds = icca_medication_tt_job.get(sub)
        for drug in ds.data_vars.keys():
            doses = ds[drug].values
            row = dict(sub=sub, drug=drug, N = doses.size, min_ = min(doses), max_ = max(doses))
            rows.append(row)
    except:
        continue

res = pd.DataFrame(rows)

In [ ]:
drug_sel = res['drug'].value_counts().index[res['drug'].value_counts() > 20].tolist()
len(drug_sel)

In [ ]:
mins = {drug:min(res[res['drug'] == drug]['min_']) for drug in drug_sel}
maxs = {drug:max(res[res['drug'] == drug]['max_']) for drug in drug_sel}

In [ ]:
subs = res['sub'].unique().tolist()

drug_doses = {}
for drug in drug_sel:
    pool = {'values':[], 'unit' : None}
    for sub in subs:
        ds = icca_medication_tt_job.get(sub)
        if drug in ds.data_vars.keys():
            if pool['unit'] is None:
                pool['unit'] = ds[drug].attrs['unit']
            doses = ds[drug].values
            pool['values'].extend(doses)
    drug_doses[drug] = pool

In [ ]:
nrows = 5
ncols = 4
poss = attribute_subplots(list(drug_doses.keys()), nrows, ncols)

fig, axs = plt.subplots(nrows, ncols, constrained_layout = True, figsize = (nrows * 2, ncols * 2))

for drug_name, pos in poss.items():
    ax = axs[*pos]
    d_drug = drug_doses[drug_name]
    unit = d_drug['unit']
    values = np.array(d_drug['values'])
    bins = np.linspace(np.nanquantile(values, 0.05), np.nanquantile(values, 0.95), 30)
    ax.hist(values, bins = bins, edgecolor = 'k')
    ax.set_title(f'{drug_name} - N : {values.size}')
    ax.set_xlabel(unit)
fig.savefig(base_folder / 'figures' / 'icca_figs' / 'medication_dose.png', dpi = 500, bbox_inches = 'tight')
plt.show()
    

# IHR spectrogram

In [ ]:
da = ihr_job.get('RM6')['ihr']
srate = da.attrs['srate']

In [ ]:
da

In [ ]:
da[int(srate * 72000):int(srate * 72150)].plot()

# RAQ freq domain

In [ ]:
sub = 'MF12'
#

In [ ]:
raq_time_domain = raq_icp_job.get(sub)['raq_icp']

In [ ]:
raq_time_domain

In [ ]:
raq_time_domain.rolling(cycle_date=30).median().plot(size = 4)


In [ ]:
reader = CnsReader(data_path / sub)
icp_stream = reader.streams['ICP']
srate = icp_stream.sample_rate
i0 = 0
i1 = int(srate * 3600)
raw_icp, times = icp_stream.get_data(isel = slice(i0, i1), with_times = True, time_as_second=True, apply_gain=True)

In [ ]:
fig, ax = plt.subplots()
ax.plot(times, raw_icp)
ax.set_xlim(10, 30)

In [ ]:
freqs, power = morlet_power(raw_icp , srate, f_start=0.5, f_stop=1.5, n_steps=50, n_cycles=5)

In [ ]:
t_mask = (times < 30)
fig, ax = plt.subplots()
ax.pcolormesh(times[t_mask], freqs, power[:,t_mask])
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(freqs, np.mean(power, axis = 1))

In [ ]:
f, Pxx_raw = scipy.signal.welch(raw_icp, srate, nperseg = int(srate * 300))

fig, ax = plt.subplots()
ax.plot(f, Pxx_raw)
ax.set_xlim(0,2)

In [ ]:
heart_rate = f[np.argmax(Pxx_raw)]
print(heart_rate, 'Hz')
print(heart_rate * 60, 'bpm')

d = 0.2
icp_highpass_heart = iirfilt(raw_icp, srate, lowcut = heart_rate - d, highcut =  heart_rate + d, ftype = 'butter', order = 5)
icp_heart_amp = np.abs(scipy.signal.hilbert(icp_highpass_heart))

In [ ]:
fig, ax = plt.subplots()
ax.plot(times, icp_highpass_heart)
ax.plot(times, icp_heart_amp)
ax.set_xlim(10, 50)

In [ ]:
f, Pxx_raw = scipy.signal.welch(raw_icp, srate, nperseg = int(srate * 300), scaling = 'spectrum')
f, Pxx_amp = scipy.signal.welch(icp_heart_amp, srate, nperseg = int(srate * 300), scaling = 'spectrum')


f_mask = (f < 0.3)

fig, ax = plt.subplots()
ax.semilogy(f[f_mask], Pxx_raw[f_mask])
ax.semilogy(f[f_mask], Pxx_amp[f_mask], color='darkorange')
ax.semilogy(f[f_mask], Pxx_raw[f_mask]/Pxx_amp[f_mask], color='r', ls = '--')

# ax.plot(f[f_mask], Pxx_raw[f_mask])
# ax.plot(f[f_mask], Pxx_amp[f_mask], color='darkorange')
# ax.plot(f[f_mask], Pxx_raw[f_mask]/Pxx_amp[f_mask], color='r', ls = '--')
# ax.set_xlim(0,0.5)

In [ ]:
f, Pxx_raw_med = scipy.signal.welch(raw_icp, srate, nperseg = int(srate * 120), average='median')
f, Pxx_amp_mean = scipy.signal.welch(raw_icp, srate, nperseg = int(srate * 120), average='mean')

fig, ax = plt.subplots()
ax.plot(f, Pxx_raw_med)
ax.plot(f, Pxx_amp_mean)
ax.set_xlim(0,1)

In [ ]:
f, Pxx_raw_spec = scipy.signal.welch(raw_icp, srate, nperseg = int(srate * 120), scaling='spectrum')
f, Pxx_raw_dens = scipy.signal.welch(raw_icp, srate, nperseg = int(srate * 120), scaling='density')

# Phase locking resp ICP

In [ ]:
n_phase_bins = 36 # 10 per bin
save_folder = base_folder / 'figures' / 'phase_histogram_respi'

d_jobs = {'Heart_Rate':heart_rate_by_resp_cycle_job,
          'ABP':abp_by_resp_cycle_job,
          'ABP_pulse':abp_pulse_by_resp_cycle_job,
          'ICP':icp_by_resp_cycle_job,
          'ICP_pulse':icp_pulse_by_resp_cycle_job,
          }

d_colors = {'Heart_Rate':'r',
          'ABP':'m',
          'ABP_pulse':'m',
          'ICP':'y',
          'ICP_pulse':'y',
          }

d_da_name = {'Heart_Rate':'heart_rate_by_resp_cycle',
          'ABP':'abp_by_resp_cycle',
          'ABP_pulse':'abp_pulse_by_resp_cycle',
          'ICP':'icp_by_resp_cycle',
          'ICP_pulse':'icp_pulse_by_resp_cycle',
          }

sub_list = get_patient_list(['CO2','ECG_II','ICP','ABP'])

nrows = len(d_da_name)
# for sub in sub_list[:1]:
for sub in sub_list:
    fig, axs = plt.subplots(nrows = nrows, figsize = (8,nrows * 2), sharex = True, constrained_layout = True)
    fig.suptitle(f'{sub} - Respi phase histogram', weight = 'bold')
    for i, dtype_ in enumerate(d_jobs):
        ax = axs[i]
        cycle_matrix = d_jobs[dtype_].get(sub)[d_da_name[dtype_]]
        phase_max = cycle_matrix.idxmax('phase')
        ax.hist(phase_max, bins = n_phase_bins, edgecolor = 'k', color = d_colors[dtype_])
        ax.set_ylabel(f'{dtype_}\n(N resp cycles)', weight = 'bold')
        if i == len(d_da_name) - 1:
            ax.set_xlabel('Resp phase [0-1]')
        try:
            c_ratio = cycle_matrix.attrs['cycle_ratio']
        except:
            c_ratio = cycle_matrix.attrs['med_cycle_ratio']
        ax.axvline(c_ratio, color = 'r', label = 'i -> e')
        ax.legend(loc = 'upper right')
    fig.savefig(save_folder / f'{sub}.png', dpi = 500, bbox_inches = 'tight')
    plt.close('all')
    # plt.show()
# 

In [ ]:
n_phase_bins = 100 # 20° per bin
remove_edge_phase = 0.08

d_jobs = {'Heart_Rate':heart_rate_by_resp_cycle_job,
          'ABP':abp_by_resp_cycle_job,
          'ABP_pulse':abp_pulse_by_resp_cycle_job,
          'ICP':icp_by_resp_cycle_job,
          'ICP_pulse':icp_pulse_by_resp_cycle_job,
          }

d_colors = {'Heart_Rate':'r',
          'ABP':'m',
          'ABP_pulse':'m',
          'ICP':'y',
          'ICP_pulse':'y',
          }

d_da_name = {'Heart_Rate':'heart_rate_by_resp_cycle',
          'ABP':'abp_by_resp_cycle',
          'ABP_pulse':'abp_pulse_by_resp_cycle',
          'ICP':'icp_by_resp_cycle',
          'ICP_pulse':'icp_pulse_by_resp_cycle',
          }

sub_list = get_patient_list(['CO2','ECG_II','ICP','ABP'])

res = []

for sub in sub_list:
    for i, dtype_ in enumerate(d_jobs):
        cycle_matrix = d_jobs[dtype_].get(sub)[d_da_name[dtype_]]
        phase_max = cycle_matrix.idxmax('phase')
        count, bins = np.histogram(phase_max.values, bins = np.linspace(0, 1, n_phase_bins))
        bins = bins[1:]
        phase_mode = bins[np.argmax(count[(bins > remove_edge_phase) & (bins < 1 - remove_edge_phase)])]
        try:
            c_ratio = cycle_matrix.attrs['cycle_ratio']
        except:
            c_ratio = cycle_matrix.attrs['med_cycle_ratio']
        res.append({'sub':sub, 'dtype':dtype_, 'phase_mode':phase_mode, 'phase_resp_of_mode':'inspi' if phase_mode < c_ratio else 'expi'})
phase_modes = pd.DataFrame(res)
gby = phase_modes.groupby('dtype')['phase_resp_of_mode'].value_counts(normalize = True)

nrows = len(d_da_name)
fig, axs = plt.subplots(nrows = nrows, figsize = (8,nrows * 2), sharex = True, constrained_layout = True)
fig.suptitle("Concat phase histogram of phase modes (N patients = {N})".format(N = phase_modes['sub'].unique().size), weight = 'bold')
for i, dtype_ in enumerate(d_jobs):
    ax = axs[i]
    ress = phase_modes[phase_modes['dtype'] == dtype_]['phase_mode'].values
    ax.hist(ress, bins = 18, edgecolor = 'k', color = d_colors[dtype_])
    prct_i = gby.loc[(dtype_,'inspi')] * 100
    prct_e = gby.loc[(dtype_,'expi')] * 100
    ax.set_title('% patients with max in inspi = {insp} , vs expi = {exp}'.format(insp = int(prct_i), exp = int(prct_e)))
    ax.set_ylabel(f'{dtype_}\n(N patients)', weight = 'bold')
    if i == len(d_da_name) - 1:
        ax.set_xlabel('Resp phase [0-1]')
fig.savefig(save_folder / f'phase_histo_of_phase_modes.png', dpi = 500, bbox_inches = 'tight')
plt.show()

In [ ]:
d_jobs = {'Heart_Rate':heart_rate_by_resp_cycle_job,
          'ABP':abp_by_resp_cycle_job,
          'ABP_pulse':abp_pulse_by_resp_cycle_job,
          'ICP':icp_by_resp_cycle_job,
          'ICP_pulse':icp_pulse_by_resp_cycle_job,
          }

d_colors = {'Heart_Rate':'r',
          'ABP':'m',
          'ABP_pulse':'m',
          'ICP':'y',
          'ICP_pulse':'y',
          }

d_da_name = {'Heart_Rate':'heart_rate_by_resp_cycle',
          'ABP':'abp_by_resp_cycle',
          'ABP_pulse':'abp_pulse_by_resp_cycle',
          'ICP':'icp_by_resp_cycle',
          'ICP_pulse':'icp_pulse_by_resp_cycle',
          }

sub_list = get_patient_list(['CO2','ECG_II','ICP','ABP'])

phases = None
for sub in sub_list:
    for i, dtype_ in enumerate(d_jobs):
        cycle_matrix = d_jobs[dtype_].get(sub)[d_da_name[dtype_]]
        cycle_matrix = cycle_matrix.median('cycle_date')
        if phases is None:
            phases = init_da({'sub':sub_list, 'dtype_':list(d_da_name.keys()), 'phase':cycle_matrix['phase'].values})
        phases.loc[sub, dtype_ ,:] = cycle_matrix.values

In [ ]:
# phases.plot.line(x = 'phase' , col = 'dtype_', row = 'sub')

In [ ]:
(phases - phases.median('phase')).median('sub').plot.line(x = 'phase', col = 'dtype_')

# RAQ distrib

In [ ]:
data = concat_slow_icp_features('compliance')

In [ ]:
data['RAQ_2'].describe()

In [ ]:
data[data['RAQ_2'] > 20]#['RAQ_2'].describe()

In [ ]:
sub = 'LJ8'
raq_trace = raq_icp_job.get(sub)['raq_icp']

In [ ]:
raq_trace.plot()

In [ ]:
med = np.nanmedian(raq_trace)
mad = scipy.stats.median_abs_deviation(raq_trace, nan_policy='omit', scale= 0.67449)

print(med - 10 * mad, med + 10 * mad)

In [ ]:
raq_trace2 = raq_trace.copy()
sig = raq_trace2.values
mask = (sig < med - 5 * mad) | (sig > med + 5 * mad)
sig[mask] = np.nan
raq_trace2[:] = sig

In [ ]:
raq_trace2.plot()

In [ ]:
np.nanstd(raq_trace)

# RAQ intersect

In [ ]:
sub = 'MF12'
icp_resp = icp_by_resp_cycle_job.get(sub)['icp_by_resp_cycle']
icp_pulse_resp = icp_pulse_by_resp_cycle_job.get(sub)['icp_pulse_by_resp_cycle']

In [ ]:
icp_resp.shape

In [ ]:
icp_pulse_resp.shape

In [ ]:
icp_resp['cycle_date']

In [ ]:
cycles_intersect_date= np.intersect1d(icp_pulse_resp['cycle_date'], icp_resp['cycle_date'])

In [ ]:
cycles_intersect_date

In [ ]:
icp_resp.loc[cycles_intersect_date]

In [ ]:
icp_resp.loc[cycles_intersect_date] / icp_pulse_resp.loc[cycles_intersect_date]

In [ ]:
icp_pulse_resp.loc[cycles_intersect_date]

# Complete Metadata Nory

## Lesion side

In [ ]:
bonus_nory_meta = pd.read_excel(base_folder / 'documents_valentin' / 'liste_pour_valentin_data_manquantes.xlsx')
bonus_nory_meta['sexe'] = bonus_nory_meta['sexe'].map({'H':'M','F':'F'})
old_nory_meta = pd.read_excel(base_folder / 'documents_valentin' / 'liste_monio_multi_Nory_pour_Valentin.xlsx').drop_duplicates('ID_pseudo').reset_index(drop = True)
old_nory_meta['sexe'] = old_nory_meta['sexe'].map({1:'M',0:'F'})
old_subs = old_nory_meta['ID_pseudo'].unique().tolist()

In [ ]:
mapper_cols_old_bonus = {'ID_pseudo':'ID_pseudo', 
                         'id_moberg':'id_moberg', 
                         'sexe':'sexe', 
                         'age':'age', 
                         'motif':'diagnostic_initial', 
                         'glasgow_initial':'glasgow_initial',
                         'fisher':'fisher', 
                         'wfns':'wfns', 
                         'lesion_place':'territoire_lesion', 
                         'lesion_side':'lateralite_lesion'
}

bonus_subs = bonus_nory_meta['ID_pseudo'].tolist()
old_subs = old_nory_meta['ID_pseudo'].tolist()
all_subs = list(set(bonus_subs + old_subs))

In [ ]:
rows_new_meta = []
for sub in all_subs:
    d_new_meta = {}
    for new_col_name, old_col_name in mapper_cols_old_bonus.items():
        if new_col_name == 'ID_pseudo':
            d_new_meta['ID_pseudo'] = sub
        else:
            if sub in bonus_subs and not sub in old_subs:
                d_new_meta[new_col_name] = bonus_nory_meta.set_index('ID_pseudo').loc[sub,new_col_name]
            elif not sub in bonus_subs and sub in old_subs:
                d_new_meta[new_col_name] = old_nory_meta.set_index('ID_pseudo').loc[sub,old_col_name]
            elif sub in bonus_subs and sub in old_subs:
                old_value = old_nory_meta.set_index('ID_pseudo').loc[sub,old_col_name]
                bonus_value = bonus_nory_meta.set_index('ID_pseudo').loc[sub,new_col_name]
                d_new_meta[new_col_name] = bonus_value if not bonus_value is None else old_value
    rows_new_meta.append(d_new_meta)
new_meta = pd.DataFrame(rows_new_meta)
new_meta['lesion_side'] = new_meta['lesion_side'].map({'droit':'droit','gauche':'gauche','bilatéral':'bilatéral','diffus':'bilatéral'})

In [ ]:
# new_meta.to_excel(base_folder / 'documents_valentin' / 'metadata_nory_clean.xlsx')

# TEst hrv freq

In [ ]:
sub = 'MF12'
reader = CnsReader(data_path / sub)
ecg_stream = reader.streams['ECG_II']
srate = ecg_stream.sample_rate
raw_ecg, times = ecg_stream.get_data(isel = slice(0, int(3600 * srate)), apply_gain = True, time_as_second = True, with_times = True)

In [ ]:
ecg, ecg_peaks = physio.compute_ecg(raw_ecg, srate)

In [ ]:
rows = []
freq_bands = {'vlf':(0.0033, 0.04), 'lf': (0.04, .15), 'hf' : (0.15, .4)}
min_freq = min(freqs[0] for freqs in freq_bands.values())
duration_s_lowest_period = 1/min_freq

for scaling in ['spectrum','density']:
    for win_size_s in [60, 120, 300, 600, 900, 1200, 1800, 2400, 3000, 3599]:

        fig, ax = plt.subplots(figsize = (9,3))
        ax.set_title('scaling : {scaling} - win_size_s : {win_size_s}'.format(win_size_s=win_size_s, scaling=scaling))
        for interp_method in ['linear','cubic']:
            ihr = physio.compute_instantaneous_rate(ecg_peaks, times, interpolation_kind='cubic')
            f, Pxx = scipy.signal.welch(ihr, srate, nperseg = int(srate * win_size_s), scaling = scaling)

            mask = (f < 1)
            
            ax.plot(f[mask], Pxx[mask], label = interp_method)
            
            for dx_spacing in ['with_dx','no_dx']:
                if dx_spacing == 'with_dx':
                    dx = np.median(np.diff(f))
                elif dx_spacing == 'no_dx':
                    dx = 1.0
                
                vlf = np.trapz(Pxx[(f > freq_bands['vlf'][0]) & (f <= freq_bands['vlf'][1])], dx=dx)
                lf = np.trapz(Pxx[(f > freq_bands['lf'][0]) & (f <= freq_bands['lf'][1])], dx=dx)
                hf = np.trapz(Pxx[(f > freq_bands['hf'][0]) & (f <= freq_bands['hf'][1])], dx=dx)
                lf_hf = lf / hf
                sum_all_powers = vlf + lf + hf
                prop_vlf, prop_lf, prop_hf = vlf / sum_all_powers, lf / sum_all_powers, hf / sum_all_powers, 
                row = {
                    'scaling':scaling,
                    'win_size_s':win_size_s, 
                    'duration_s_lowest_period':duration_s_lowest_period,
                    'n_lowest_freq':win_size_s/duration_s_lowest_period,
                    'interp_method':interp_method, 
                    'dx_spacing':dx_spacing,
                    'vlf':vlf,
                        'lf':lf ,
                        'hf':hf , 
                        'lf_hf':lf_hf,
                        'prop_vlf':prop_vlf,
                        'prop_lf':prop_lf,
                        'prop_hf':prop_hf,
                        'n_lowest_freq_period_band_vlf':win_size_s / (1/freq_bands['vlf'][0]),
                        'n_lowest_freq_period_band_lf':win_size_s / (1/freq_bands['lf'][0]),
                        'n_lowest_freq_period_band_hf':win_size_s / (1/freq_bands['hf'][0]),
                        }
                rows.append(row)
            
        ax.legend(loc = 'upper right', fontsize = 9)
        plt.show()

        res = pd.DataFrame(rows)
        res

In [ ]:
metrics = ['vlf','lf','hf','lf_hf','prop_vlf','prop_lf','prop_hf']
scalings = res['scaling'].unique()
nrows = len(metrics)
ncols = len(scalings)
fig, axs = plt.subplots(nrows=nrows, ncols=ncols, figsize = (9, nrows * 3), constrained_layout = True)
for r, metric in enumerate(metrics):
    for c , scaling in enumerate(scalings):
        ax = axs[r,c]
        sns.pointplot(data = res[res['scaling'] == scaling], 
                    x = 'win_size_s',
                    y = metric, 
                    hue = 'dx_spacing',
                    ax=ax,
        )
                    # log_scale = True)
        ax.set_title(f'{metric} - {scaling}')

In [ ]:
for scaling in res['scaling'].unique():
    rownames = ['vlf','lf','hf']
    colnames = ['with_dx','no_dx']
    fig, axs = plt.subplots(nrows = len(rownames), ncols = len(colnames), figsize = (4 * len(colnames), 3 * len(rownames)), constrained_layout = True)
    fig.suptitle(scaling, fontsize = 20)
    for r, rowname in enumerate(rownames):
        for c, colname in enumerate(colnames):
            ax = axs[r,c]
            res_mask = (res['scaling'] == scaling) & (res['interp_method'] == 'linear') & (res['dx_spacing'] == colname)
            ax.scatter(res[res_mask][f'n_lowest_freq_period_band_{rowname}'].values, res[res_mask][f'{rowname}'].values)
            ax.set_xlabel(f'n_lowest_freq_period_band_{rowname}')
            ax.set_ylabel(f'{rowname}')
            ax.set_title(f'{rowname} - {colname}')
    plt.show()

In [ ]:
for scaling in res['scaling'].unique():
    rownames = ['vlf','lf','hf']
    colnames = ['with_dx','no_dx']
    fig, axs = plt.subplots(nrows = len(rownames), ncols = len(colnames), figsize = (4 * len(colnames), 3 * len(rownames)), constrained_layout = True)
    fig.suptitle(scaling, fontsize = 20)
    for r, rowname in enumerate(rownames):
        for c, colname in enumerate(colnames):
            ax = axs[r,c]
            res_mask = (res['scaling'] == scaling) & (res['interp_method'] == 'linear') & (res['dx_spacing'] == colname)
            ax.scatter(res[res_mask][f'win_size_s'].values, res[res_mask][f'{rowname}'].values)
            ax.set_xlabel(f'win_size_s')
            ax.set_ylabel(f'{rowname}')
            ax.set_title(f'{rowname} - {colname}')
    plt.show()

In [ ]:
# freqency_bands = {'vlf':(0.0033, 0.04), 'lf': (0.04, .15), 'hf' : (0.15, .4)}
freqency_bands = {'lf': (0.04, .15), 'hf' : (0.15, .4)}
psd_freqs, psd, metrics = physio.compute_hrv_psd(ecg_peaks, sample_rate=500., limits=None, units='bpm',
                                        freqency_bands = freqency_bands,
                                        window_s=250., interpolation_kind='cubic')

In [ ]:
metrics

In [ ]:
mask_f = (psd_freqs < 1) & (psd_freqs > 0.001)
fig, ax = plt.subplots()
ax.semilogy(psd_freqs[mask_f], psd[mask_f])
plt.show()

In [ ]:
mask_f = (psd_freqs < 1) & (psd_freqs > 0.001)
fig, ax = plt.subplots()
ax.loglog(psd_freqs, psd)

# Sample interval by segment

In [ ]:
rows = []
for sub in get_patients_list_raw():
    reader = CnsReader(data_path / sub)
    stream_names = list(reader.streams.keys())
    for stream_name in stream_names:
        stream = reader.streams[stream_name]
        sr_pycns = stream.sample_rate
        si_us = stream.index['sample_interval_integer']
        min_si_us = np.min(si_us)
        max_si_us = np.max(si_us)
        sr_max = 1 / (min_si_us / 1e6)
        sr_min = 1 / (max_si_us / 1e6)
        delta_si_us = max_si_us - min_si_us
        d = dict(sub=sub, 
                 stream_name=stream_name, 
                 sr_pycns=sr_pycns, 
                 med_si_us=np.median(si_us), 
                 min_si_us = min_si_us, 
                 max_si_us = max_si_us,
                 delta_si_us = delta_si_us,
                 sr_max = sr_max,
                 sr_min = sr_min
                 )
        rows.append(d)
        
res = pd.DataFrame(rows)

stream_sel = ['ICP','ABP','ECG_II','CO2','ART','EEG']
res_sel = res[res['stream_name'].isin(stream_sel)]
res_sel
        

In [ ]:
res['stream_name'].unique()

In [ ]:
threshold_prct_ptp = 5
si_time_series = {}
for sub in get_patients_list_raw():
    si_time_series_stream = {}
    reader = CnsReader(data_path / sub)
    stream_names = list(reader.streams.keys())
    for stream_name in stream_names:
        stream = reader.streams[stream_name]
        datetimes = stream.index['datetime']
        si_us = stream.index['sample_interval_integer']
        ptp_us = np.ptp(si_us)
        prct_ptp_us_baseline = (ptp_us / np.median(si_us)) * 100
        warning_si_us = True if prct_ptp_us_baseline > threshold_prct_ptp else False
        si_time_series_stream[stream_name] = {'si':si_us, 'datetime':datetimes, 'sr_pycns':stream.sample_rate, 'ptp_us':ptp_us, 'prct_ptp_us_baseline':prct_ptp_us_baseline, 'warning_si_us':warning_si_us}
    si_time_series[sub] = si_time_series_stream
        

In [ ]:

rows = []
for sub, d_sub in si_time_series.items():
    for stream, res_sub_stream in d_sub.items():
            si_us = res_sub_stream['si']
            med_si_us = np.median(si_us)
            s = threshold_prct_ptp / 100 * med_si_us
            n_samples = si_us.size
            n_samples_out_of_median = si_us[(si_us < (med_si_us - s)) | (si_us > (med_si_us + s))].size
            prct_samples_out_of_median = (n_samples_out_of_median / n_samples) * 100
            med_si_us = np.median(si_us)
            prct_ptp_us_baseline = res_sub_stream['prct_ptp_us_baseline']
            warning = res_sub_stream['warning_si_us']
            rows.append(dict(sub=sub, 
                             stream=stream, 
                             prct_ptp_us_baseline=prct_ptp_us_baseline, 
                             warning=warning,
                             n_samples_out_of_median=n_samples_out_of_median,
                             prct_samples_out_of_median=prct_samples_out_of_median,
                             ))
res_global = pd.DataFrame(rows)


In [ ]:
stream_sel = ['ICP','ABP','ECG_II','CO2','ART','EEG','PLETH']
count_by_stream = res_global['stream'].value_counts().to_dict()
count_by_stream_sel = {k:v for k,v in count_by_stream.items() if k in stream_sel}
fig, axs = plt.subplots(nrows = 4, figsize = (8,10), constrained_layout = True)
fig.suptitle('Quels streams sont les plus impactés par des problèmes de sampling interval ?', fontsize = 15, weight = 'bold')
ax = axs[0]
ax.set_title('prct_ptp_us_baseline = (np.ptp(si_us) / np.median(si_us)) * 100  (si_us = sample interval series in µs)')
sns.boxplot(data = res_global[res_global['stream'].isin(stream_sel)],
              x=  "stream",
              y = 'prct_ptp_us_baseline',
              order = stream_sel,
              # estimator='median',
            ax=ax,
)
sns.stripplot(data = res_global[res_global['stream'].isin(stream_sel)],
              x=  "stream",
              y = 'prct_ptp_us_baseline',
            #   estimator='median'
            order = stream_sel,
            ax=ax,
            color = 'k'
)

ax = axs[1]
ax.set_title('warning = True if prct_ptp_us_baseline > 5 else False')
sns.countplot(data = res_global[res_global['stream'].isin(stream_sel)], x = 'stream', hue = 'warning', ax=ax,order = stream_sel,)
ax.set_ylabel('Nombre de patients')
for bar in ax.containers:
    ax.bar_label(bar)
ax.set_xticks(range(len(count_by_stream_sel)), [f'{k}\nN={v}' for k,v in count_by_stream_sel.items()])

ax = axs[2]
ax.set_title('n_samples_out_of_median = si_us[(si_us < (med_si_us - med_si_us * 5/100)) | (si_us > (med_si_us + med_si_us * 5/100))].size')
sns.barplot(data = res_global[res_global['stream'].isin(stream_sel)],
              x=  "stream",
              y = 'n_samples_out_of_median',
              order = stream_sel,
              # estimator='median',
            ax=ax,
)
# ax.set_ylim(-2,10)

ax = axs[3]
ax.set_title('prct_samples_out_of_median = (n_samples_out_of_median / n_samples) * 100')
sns.barplot(data = res_global[res_global['stream'].isin(stream_sel)],
              x=  "stream",
              y = 'prct_samples_out_of_median',
              order = stream_sel,
              # estimator='median',
            ax=ax,
)
# ax.set_ylim(-5,40)

fig.savefig(base_folder / 'figures' / 'stream_index_exploration' / 'ampleur_dégats.png', dpi = 200, bbox_inches = 'tight')
plt.show()


In [ ]:
# sub = 'P94'
# stream_names = si_time_series[sub].keys()
# nrows = len(stream_names)
# fig, axs = plt.subplots(nrows = nrows, figsize = (9, nrows * 3), constrained_layout = True)
# for i, stream_name in enumerate(stream_names):
#     ax = axs[i]
#     res_sub_stream = si_time_series[sub][stream_name]
#     ax.scatter(res_sub_stream['datetime'], res_sub_stream['si'], color = 'k')
#     ax.set_title(f'{sub} - {stream_name}')
#     ax.set_ylabel('Sample interval (µs)')
#     ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 5)
# plt.show()

In [ ]:
stream_names = ['ICP','ABP','ECG_II','CO2','ART','EEG']
save_folder = base_folder / 'figures' / 'stream_index_exploration'
nrows = len(stream_names)
for sub in tqdm(si_time_series.keys()):
    fig, axs = plt.subplots(nrows = nrows, figsize = (9, nrows * 2), constrained_layout = True)
    for i, stream_name in enumerate(stream_names):
        ax = axs[i]
        
        try:
            res_sub_stream = si_time_series[sub][stream_name]
            prct_ptp_us_baseline = res_sub_stream['prct_ptp_us_baseline']
            warning = res_sub_stream['warning_si_us']
            sr_pycns = res_sub_stream['sr_pycns']
            ax.set_title(f'{sub} - {stream_name} - sr pycns : {round(sr_pycns , 2)} Hz - si % max error : {round(prct_ptp_us_baseline, 2)}')
        except:
            ax.set_title(f'{sub} - {stream_name} (not available)')
            continue
        ax.axhline(res_sub_stream['sr_pycns'], color = 'r', ls = 'dotted' , alpha = 0.5)
        hz_time_series = 1 / (res_sub_stream['si'] / 1e6)
        delta_hz = np.ptp(hz_time_series)
        ax.plot(res_sub_stream['datetime'], hz_time_series, color = 'k', alpha = 1)
        ax.scatter(res_sub_stream['datetime'], hz_time_series, color = 'k', alpha = 1)
        if warning:
            ax.set_ylabel('Srate (Hz)', color='red', weight='bold')
        else:
            ax.set_ylabel('Srate (Hz)')
        ax2 = ax.twinx()
        ax2.plot(res_sub_stream['datetime'], res_sub_stream['si'], color = 'tab:blue', alpha = 0.8, ls = '--')
        ax2.scatter(res_sub_stream['datetime'], res_sub_stream['si'], color = 'tab:blue', alpha = 0.8)
        ax2.set_ylabel('Sample interval (µs)')

        for ax in [ax, ax2]:
            yticks = ax.get_yticks()
            ax.set_yticks(yticks)
            ax.set_yticklabels([f'{round(tick, 5)}' for tick in yticks])
            ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 5)
    fig.savefig(save_folder / f'{sub}.png', dpi = 200, bbox_inches = 'tight')
    plt.close(fig)

# Computation requiremetns sam

## Stats

In [ ]:
computers = ['cluster','samnode','local_laptop']

concat = []
for computer in computers:
    df = pd.read_excel(base_folder / 'figures' / 'slow_icp_rises_figs' / 'resource_requirements' / f'res_requirements_{computer}.xlsx', index_col = 0)
    df['computer'] = computer
    concat.append(df)
res = pd.concat(concat).reset_index(drop = True)

In [ ]:
res.columns

In [ ]:
metrics = ['duration', 'cpu', 'mem_GB']
nrows = len(metrics)
fig, axs = plt.subplots(nrows = nrows, figsize = (10, nrows * 3), constrained_layout = True)
for i, metric in enumerate(metrics):
    ax = axs[i]
    sns.pointplot(data = res, x = 'metric', y = metric, hue = 'computer', ax = ax, errorbar = 'ci', estimator = 'median')
    # sns.boxplot(data = res, x = 'computer', y = metric, hue = 'metric', ax = ax)
    ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 5)
fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'resource_requirements' / 'pointplots.png', dpi = 200, bbox_inches = 'tight')
plt.show()

## Dvp

In [ ]:
def load_raw_icp(sub, duration_hours):
    raw_folder = data_path / sub
    cns_reader = pycns.CnsReader(raw_folder)
    icp_stream = cns_reader.streams['ICP']
    srate_icp = icp_stream.sample_rate
    i0 = 0
    i1 = int(srate_icp * 3600 * duration_hours)
    raw_icp = icp_stream.get_data(isel = slice(i0, i1), apply_gain=True)
    return raw_icp, srate_icp

def icp_pulse_amplitude_time_domain(raw_icp, srate):
    p =  {
    'lowcut':0.1,
    'highcut':10,
    'order':4,
    'ftype':'butter',
    'peak_prominence' : 0.5,
    'h_distance_s' : 0.3,
    'rise_amplitude_limits' : (0,20),
    'amplitude_at_trough_low_limit' : -10,
    }
    icp_features = compute_icp(raw_icp, srate, lowcut = p['lowcut'], highcut = p['highcut'], order = p['order'], ftype = p['ftype'], peak_prominence = p['peak_prominence'], h_distance_s = p['h_distance_s'], rise_amplitude_limits=p['rise_amplitude_limits'], amplitude_at_trough_low_limit = p['amplitude_at_trough_low_limit'])
    return icp_features['rise_amplitude'].values

def icp_pulse_amplitude_freq_domain(raw_icp, srate):
    p = {
    'spectrogram_win_size_secs':60,
    'heart_fband':(0.8,2.5),
    'rolling_N_time_spectrogram':5,
    }
    nperseg = int(p['spectrogram_win_size_secs'] * srate)
    nfft = int(nperseg)

    # Compute spectro ICP
    freqs, times_spectrum_s, Sxx_icp = scipy.signal.spectrogram(raw_icp, fs = srate, nperseg =  nperseg, nfft = nfft)
    Sxx_icp = np.sqrt(Sxx_icp)
    da = xr.DataArray(data = Sxx_icp, dims = ['freq','time'], coords = {'freq':freqs, 'time':times_spectrum_s})
    heart_fband = p['heart_fband']
    rolling_N_time = p['rolling_N_time_spectrogram']
    heart_amplitude = da.loc[heart_fband[0]:heart_fband[1],:].max('freq').rolling(time = rolling_N_time).median().bfill('time').ffill('time')
    return heart_amplitude.values

def test_computation_requirements(sub, duration_hours):
    raw_icp, srate = load_raw_icp(sub, duration_hours)

    t1 = time.perf_counter()
    _ = icp_pulse_amplitude_time_domain(raw_icp, srate)
    t2 = time.perf_counter()
    print('time domain :', t2-t1, ' seconds')

    t1 = time.perf_counter()
    _ = icp_pulse_amplitude_freq_domain(raw_icp, srate)
    t2 = time.perf_counter()
    print('freq domain :', t2-t1, ' seconds')




In [ ]:
sub = 'MF12'
duration_hours = 5

test_computation_requirements(sub, duration_hours)

In [ ]:
from cpu_usage import start_measurement


# df = 

functions = {
    'icp_pulse_amplitude_time_domain' : icp_pulse_amplitude_time_domain,
    'icp_pulse_amplitude_freq_domain' : icp_pulse_amplitude_freq_domain,

}


sub_list = ['MF12','NY15']

rows = []

for sub in sub_list:
    raw_icp, srate = load_raw_icp(sub, duration_hours)

    for func_name, func in functions.items():

        info = start_measurement(func, (raw_icp, srate))
        info['duration_hours'] = duration_hours
        info['patient'] = sub
        rows.append(info)

computation_requirements = pd.DataFrame(rows)
computation_requirements

In [ ]:
import psutil
print(psutil.cpu_count())

# Show to Hugo

In [ ]:
sub = 'P2'
# read the data folder
cns_reader = CnsReader(data_path / sub)

# export some streams to Xarray Dataset, containing one DataArray per stream with various sampling rates.
# Note that gain is automatically applied
stream_names = ['ECG_II', 'CO2']
start = '2020-02-17T10:10:15'
stop = '2022-02-17T10:20:52'
ds = cns_reader.export_to_xarray(stream_names, start=start, stop=stop, resample=True, sample_rate=500)

In [ ]:
srate_sig = ds['ECG_II'].attrs['sample_rate']
sig = ds['ECG_II'].values
down_factor = 10
sig_down = sig[::down_factor]
new_srate = srate_sig / down_factor

sig_down_clean = scipy.signal.decimate(sig, q = down_factor)

In [ ]:
srate_sig

In [ ]:
new_srate

In [ ]:
%matplotlib widget

In [ ]:
fig, axs = plt.subplots(nrows = 3)
ax = axs[0]
ax.plot(np.arange(sig.size) / srate_sig, sig)
ax.set_xlim(0, 15)
ax = axs[1]
ax.plot(np.arange(sig_down.size) / new_srate, sig_down)
ax.set_xlim(0, 15)
ax = axs[2]
ax.plot(np.arange(sig_down.size) / new_srate, sig_down_clean)
ax.set_xlim(0, 15)
plt.show()

# ICP filtered

In [ ]:
sub = 'BM3'
p = {'highcut': 0.02, 'order':4, 'ftype':'butter'}

cns_reader = pycns.CnsReader(data_path / sub)
stream_names = list(cns_reader.streams)

icp_stream = cns_reader.streams['ICP']
srate_icp = icp_stream.sample_rate
raw_icp = icp_stream.get_data(isel = slice(int(srate_icp * 0), int(srate_icp * 5 * 3600)), with_times = False, apply_gain = True)
icp_filtered = iirfilt(raw_icp, srate_icp, highcut = p['highcut'], order = p['order'], ftype = p['ftype'])

In [ ]:
t = np.arange(raw_icp.size) / srate_icp
fig, ax = plt.subplots()
ax.plot(t, raw_icp)
ax.plot(t, icp_filtered)
plt.show()

# ICP or ABP filtered and cyclically deformed by resp (dephasage ?)

In [ ]:
def from_sub_to_fig_decalage_phase(sub, target_stream, n_hours = 5, fig=None, ax=None, save=False):
    cns_reader = pycns.CnsReader(data_path / sub)
    stream_names = list(cns_reader.streams)

    if target_stream == 'ABP':
        p = {'highcut': 0.5, 'order':4, 'ftype':'butter','segmentation_deformation':'bi'}
        assert 'ABP' in stream_names or 'ART' in stream_names
        if 'ABP' in stream_names and not 'ART' in stream_names:
            abp_stream_name = 'ABP'
        elif 'ART' in stream_names and not 'ABP' in stream_names:
            abp_stream_name = 'ART'
        else:
            abp_test_stream = cns_reader.streams['ABP']
            art_test_stream = cns_reader.streams['ART']
            abp_stream_name = 'ABP' if abp_test_stream.shape[0] > art_test_stream.shape[0] else 'ART'
        abp_stream = cns_reader.streams[abp_stream_name]
        srate_sig = abp_stream.sample_rate
        raw_abp = abp_stream.get_data(isel = slice(int(srate_sig * 0), int(srate_sig * n_hours * 3600)), with_times = False, apply_gain = True)
        sig_filtered = iirfilt(raw_abp, srate_sig, highcut = p['highcut'], order = p['order'], ftype = p['ftype'])
    
    elif target_stream == 'ICP':
        p = {'highcut': 0.5, 'order':4, 'ftype':'butter','segmentation_deformation':'bi'}

        cns_reader = pycns.CnsReader(data_path / sub)
        stream_names = list(cns_reader.streams)

        icp_stream = cns_reader.streams['ICP']
        srate_sig = icp_stream.sample_rate
        raw_icp = icp_stream.get_data(isel = slice(int(srate_sig * 0), int(srate_sig * n_hours *  3600)), with_times = False, apply_gain = True)
        sig_filtered = iirfilt(raw_icp, srate_sig, highcut = p['highcut'], order = p['order'], ftype = p['ftype'])

    times_sig = np.arange(sig_filtered.size) / srate_sig
    resp_cycles = detect_resp_job.get(sub).to_dataframe()
    resp_cycles = resp_cycles[(resp_cycles['inspi_time'] > times_sig[0]) & (resp_cycles['next_inspi_time'] < times_sig[-1])]
    cycles_dates = resp_cycles['inspi_date'].values

    if p['segmentation_deformation'] == 'mono':
        cycle_times = resp_cycles[['inspi_time','next_inspi_time']].values
        segment_ratios = None
    elif p['segmentation_deformation'] == 'bi':
        cycle_times = resp_cycles[['inspi_time','expi_time','next_inspi_time']].values
        segment_ratios = 0.4

    nbins = 100
    sig_by_resp_cycle = physio.deform_traces_to_cycle_template(data = sig_filtered, 
                                                            times = times_sig,
                                                            cycle_times = cycle_times,
                                                            segment_ratios = segment_ratios,
                                                            points_per_cycle = nbins
                                                            )
    # ptp = np.abs(np.ptp(abp_by_resp_cycle, axis = 1))

    if ax is None:
        fig, ax = plt.subplots()
    phase = np.linspace(0,1, nbins)
    # ax.pcolormesh(phase, np.arange(icp_by_resp_cycle.shape[0]), icp_by_resp_cycle)
    # ax.pcolormesh(phase, resp_cycles['inspi_time'], icp_by_resp_cycle)
    data = sig_by_resp_cycle - np.median(sig_by_resp_cycle, axis = 1)[:,None]
    q = 0.05
    vmin = np.quantile(data, q)
    vmax = np.quantile(data, 1-q)
    n_hours_plot = n_hours
    # n_hours_plot = 0.5
    tmax = 3600 * n_hours_plot
    c_mask = resp_cycles['inspi_time'].values < tmax
    dates = resp_cycles[c_mask]['inspi_date']
    im = ax.pcolormesh(dates, phase, data[c_mask,:].T, vmin=vmin ,vmax=vmax)
    ax.axhline(0.4, color = 'g')
    clb = fig.colorbar(im, ax=ax)
    ax.set_title(f'{target_stream} modulated by resp')
    clb.set_label('mmHg')
    ax.set_ylabel('Resp Phase')
    ax.set_xlabel('Datetime')
    ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 45)
    return ax

In [ ]:
abp_subs = list(np.unique(np.concatenate([np.array(get_patient_list(['ABP'])), np.array(get_patient_list(['ART']))])))
icp_subs = get_patient_list(['ICP'])
subs = list(set(abp_subs + icp_subs))

In [ ]:
%matplotlib inline

In [ ]:
# sub = 'P70'
n_hours = 10
# for sub in subs[:1]:
for sub in subs[12:]:
# for sub in subs:
    try:
        fig, axs = plt.subplots(nrows = 2, constrained_layout = True, figsize = (8, 5))
        fig.suptitle(f'{sub} - duration hours : {n_hours}', fontsize = 15)
        ax = axs[0]
        target_stream ='ABP'
        from_sub_to_fig_decalage_phase(sub=sub, target_stream=target_stream, n_hours=n_hours, ax=ax, fig=fig)
        ax = axs[1]
        target_stream ='ICP'
        from_sub_to_fig_decalage_phase(sub=sub, target_stream=target_stream, n_hours=n_hours, ax=ax, fig=fig)
        fig.savefig(base_folder / 'figures' / 'decalage_phase_resp' / f'{sub}.png', dpi = 200, bbox_inches = 'tight')
        plt.close(fig)
    except:
        continue
    # plt.show()

In [ ]:
%matplotlib widget

In [ ]:
n_hours = 10

In [ ]:
sub = 'P70'

In [ ]:
p = {'highcut': 0.5, 'order':4, 'ftype':'butter','segmentation_deformation':'bi'}

cns_reader = pycns.CnsReader(data_path / sub)
stream_names = list(cns_reader.streams)
assert 'ABP' in stream_names or 'ART' in stream_names
if 'ABP' in stream_names and not 'ART' in stream_names:
    abp_stream_name = 'ABP'
elif 'ART' in stream_names and not 'ABP' in stream_names:
    abp_stream_name = 'ART'
else:
    abp_test_stream = cns_reader.streams['ABP']
    art_test_stream = cns_reader.streams['ART']
    abp_stream_name = 'ABP' if abp_test_stream.shape[0] > art_test_stream.shape[0] else 'ART'

abp_stream = cns_reader.streams[abp_stream_name]
srate_abp = abp_stream.sample_rate
raw_abp = abp_stream.get_data(isel = slice(int(srate_abp * 0), int(srate_abp * n_hours * 3600)), with_times = False, apply_gain = True)
abp_filtered = iirfilt(raw_abp, srate_abp, highcut = p['highcut'], order = p['order'], ftype = p['ftype'])

In [ ]:
times_abp = np.arange(abp_filtered.size) / srate_abp

In [ ]:
# fig, ax = plt.subplots()
# ax.plot(times_abp, abp_filtered)
# ax.plot(times_abp, raw_abp)
# plt.show()

In [ ]:
resp_cycles = detect_resp_job.get(sub).to_dataframe()
resp_cycles = resp_cycles[(resp_cycles['inspi_time'] > times_abp[0]) & (resp_cycles['next_inspi_time'] < times_abp[-1])]
cycles_dates = resp_cycles['inspi_date'].values

if p['segmentation_deformation'] == 'mono':
    cycle_times = resp_cycles[['inspi_time','next_inspi_time']].values
    segment_ratios = None
elif p['segmentation_deformation'] == 'bi':
    cycle_times = resp_cycles[['inspi_time','expi_time','next_inspi_time']].values
    segment_ratios = 0.4

nbins = 100
abp_by_resp_cycle = physio.deform_traces_to_cycle_template(data = abp_filtered, 
                                                        times = times_abp,
                                                        cycle_times = cycle_times,
                                                        segment_ratios = segment_ratios,
                                                        points_per_cycle = nbins
                                                        )
ptp = np.abs(np.ptp(abp_by_resp_cycle, axis = 1))

In [ ]:
fig, ax = plt.subplots()
ax.plot(resp_cycles['inspi_time'], ptp)
plt.show()

In [ ]:
fig, ax = plt.subplots()
# ax.plot(abp_by_resp_cycle.T, color = 'k', alpha = 0.1)

m = np.median(abp_by_resp_cycle, axis = 0)
s = abp_by_resp_cycle.std(axis = 0)
x = np.linspace(0,1, nbins)
ax.plot(x, m, color = 'r')
ax.axvline(0.4)
# ax.fill_between(x, m-s, m+s, color = 'k', alpha = 0.2)
plt.show()

In [ ]:
fig, ax = plt.subplots()
phase = np.linspace(0,1, nbins)
# ax.pcolormesh(phase, np.arange(icp_by_resp_cycle.shape[0]), icp_by_resp_cycle)
# ax.pcolormesh(phase, resp_cycles['inspi_time'], icp_by_resp_cycle)
data = abp_by_resp_cycle - np.median(abp_by_resp_cycle, axis = 1)[:,None]
q = 0.05
vmin = np.quantile(data, q)
vmax = np.quantile(data, 1-q)
n_hours_plot = n_hours
# n_hours_plot = 0.5
tmax = 3600 * n_hours_plot
c_mask = resp_cycles['inspi_time'].values < tmax
dates = resp_cycles[c_mask]['inspi_date']
im = ax.pcolormesh(phase, dates, data[c_mask,:], vmin=vmin ,vmax=vmax)
ax.axvline(0.4, color = 'g')
clb = fig.colorbar(im, ax=ax)
ax.set_title('ABP modulated by resp')
clb.set_label('mmHg')
plt.show()

In [ ]:
# i = 300
# fig, ax = plt.subplots()
# ax.plot(abp_by_resp_cycle[i])
# current_t = resp_cycles['inspi_time'].values[i]
# ax.set_title(f'{ptp[i]} - {current_t}')
# plt.show()

In [ ]:

p = {'highcut': 0.5, 'order':4, 'ftype':'butter','segmentation_deformation':'bi'}

cns_reader = pycns.CnsReader(data_path / sub)
stream_names = list(cns_reader.streams)

icp_stream = cns_reader.streams['ICP']
srate_icp = icp_stream.sample_rate
raw_icp = icp_stream.get_data(isel = slice(int(srate_icp * 0), int(srate_icp * n_hours *  3600)), with_times = False, apply_gain = True)
icp_filtered = iirfilt(raw_icp, srate_icp, highcut = p['highcut'], order = p['order'], ftype = p['ftype'])

In [ ]:
times_icp = np.arange(icp_filtered.size) / srate_icp

In [ ]:
# fig, ax = plt.subplots()
# ax.plot(times_icp, icp_filtered)
# ax.plot(times_icp, raw_icp)
# plt.show()

In [ ]:
resp_cycles = detect_resp_job.get(sub).to_dataframe()
resp_cycles = resp_cycles[(resp_cycles['inspi_time'] > times_icp[0]) & (resp_cycles['next_inspi_time'] < times_icp[-1])]
cycles_dates = resp_cycles['inspi_date'].values

if p['segmentation_deformation'] == 'mono':
    cycle_times = resp_cycles[['inspi_time','next_inspi_time']].values
    segment_ratios = None
elif p['segmentation_deformation'] == 'bi':
    cycle_times = resp_cycles[['inspi_time','expi_time','next_inspi_time']].values
    segment_ratios = 0.4

nbins = 100
icp_by_resp_cycle = physio.deform_traces_to_cycle_template(data = icp_filtered, 
                                                        times = times_icp,
                                                        cycle_times = cycle_times,
                                                        segment_ratios = segment_ratios,
                                                        points_per_cycle = nbins
                                                        )
ptp = np.abs(np.ptp(icp_by_resp_cycle, axis = 1))

In [ ]:
# fig, ax = plt.subplots()
# ax.plot(resp_cycles['inspi_time'], ptp)
# plt.show()

In [ ]:
print('icp', cns_reader.streams['ICP'].sample_rate)
print('abp', cns_reader.streams['ABP'].sample_rate)

In [ ]:
fig, ax = plt.subplots()
phase = np.linspace(0,1, nbins)
ax.plot(phase, icp_by_resp_cycle.T, color = 'k', alpha = 0.1)
ax.plot(phase, icp_by_resp_cycle.mean(axis = 0), color = 'r')
ax.axvline(0.4, color = 'g')
plt.show()

In [ ]:
fig, ax = plt.subplots()
phase = np.linspace(0,1, nbins)
# ax.pcolormesh(phase, np.arange(icp_by_resp_cycle.shape[0]), icp_by_resp_cycle)
# ax.pcolormesh(phase, resp_cycles['inspi_time'], icp_by_resp_cycle)
data = icp_by_resp_cycle - np.median(icp_by_resp_cycle, axis = 1)[:,None]
q = 0.05
vmin = np.quantile(data, q)
vmax = np.quantile(data, 1-q)
n_hours_plot = n_hours
# n_hours_plot = 0.1
tmax = 3600 * n_hours_plot
c_mask = resp_cycles['inspi_time'].values < tmax
dates = resp_cycles[c_mask]['inspi_date']
im = ax.pcolormesh(phase, dates, data[c_mask,:], vmin=vmin ,vmax=vmax)
ax.axvline(0.4, color = 'g')
clb = fig.colorbar(im, ax=ax)
ax.set_title('ICP modulated by Respi')
ax.set_ylabel('Date')
clb.set_label('mmHg')
plt.show()

In [ ]:
fig, ax = plt.subplots()
# ax.plot(abp_by_resp_cycle.T, color = 'k', alpha = 0.1)

m = np.median(icp_by_resp_cycle, axis = 0)
s = icp_by_resp_cycle.std(axis = 0)
x = np.linspace(0,1, nbins)
ax.plot(x, m, color = 'r')
ax.axvline(0.4)
# ax.fill_between(x, m-s, m+s, color = 'k', alpha = 0.2)
plt.show()

In [ ]:
i = 300
fig, ax = plt.subplots()
ax.plot(icp_by_resp_cycle[i])
current_t = resp_cycles['inspi_time'].values[i]
ax.set_title(f'{ptp[i]} - {current_t}')
plt.show()

# Distrib ABP sig

In [ ]:
%matplotlib widget

In [ ]:
sub = 'MF12'
cns_reader = pycns.CnsReader(data_path / sub)
abp_stream = cns_reader.streams['ABP']
srate = abp_stream.sample_rate
start = 300
duration = 36000
stop = start + int(duration * srate)
raw_abp, datetimes = abp_stream.get_data(isel=slice(start,stop), with_times = True, apply_gain = True)
lowcut = 0.5
highcut = 10
order = 1
ftype = 'bessel'
abp_filt = iirfilt(raw_abp, srate, lowcut = lowcut, highcut = highcut, order = order, ftype = ftype)

In [ ]:
peak_inds, properties_peaks = scipy.signal.find_peaks(abp_filt, distance = int(srate * 0.4), prominence = 15)
trough_inds,properties_troughs = scipy.signal.find_peaks(-abp_filt, distance = int(srate * 0.4), prominence = 15)

fig, ax = plt.subplots(figsize = (20,4))
ax.plot(datetimes, abp_filt)
ax.scatter(datetimes[trough_inds], abp_filt[trough_inds], color = 'r')
ax.scatter(datetimes[peak_inds], abp_filt[peak_inds], color = 'g')
plt.show()

# Sample size compliance project

In [ ]:
n_hours = 10
n_mins = 60*n_hours
subs1 = [s for s in get_patient_list(['ICP','CO2'], threshold_duration_mins=n_mins) if not 'fin' in s]
subs1=  [s for s in subs1 if not s in ['P19', 'P95']]
len(subs1)

In [ ]:
n_hours = 10
n_mins = 60*n_hours
subs_abp = [s for s in get_patient_list(['ICP','CO2','ABP'], threshold_duration_mins=n_mins) if not 'fin' in s]
subs_art = [s for s in get_patient_list(['ICP','CO2','ART'], threshold_duration_mins=n_mins) if not 'fin' in s]
subs2 = list(set(subs_abp + subs_art))
subs2=  [s for s in subs2 if not s in ['P19', 'P95']]
len(subs2)

In [ ]:
np.array(subs1)

In [ ]:
print([s for s in subs2 if not s in subs1])
print([s for s in subs1 if not s in subs2])

# ABP or ART stream availability --> detection problem 

In [ ]:
np.array(get_patient_list(['ABP']))

In [ ]:
np.array(get_patient_list(['ART']))

In [ ]:
np.array(get_patient_list(['ABP','ART']))

In [ ]:
list(np.unique(np.concatenate([np.array(get_patient_list(['ABP'])), np.array(get_patient_list(['ART']))])))

In [ ]:
%matplotlib widget

In [ ]:
# sub = 'MF12'
# sub = 'BJ11'
sub = 'DV4'
# sub = 'NY15'
# sub = 'MJ18'

cns_reader = pycns.CnsReader(data_path / sub)

stream_name = 'ABP'
stream = cns_reader.streams[stream_name]
srate = stream.sample_rate
raw_sig, dates = stream.get_data(apply_gain = True, with_times = True)

# detections = compute_abp(raw_sig, srate, date_vector = dates, show = True, lowcut=0.5)

In [ ]:
lowcut = 0.3
highcut = 10
order = 1
ftype = 'bessel'
sig_filtered = iirfilt(raw_sig, srate, lowcut = lowcut, highcut = highcut, order = order, ftype = ftype)

In [ ]:
normalize = True


sig_sel = sig_filtered[:int(srate * 300)]
# sig_sel = raw_sig[:int(srate * 300)]
raw_sig_sel = raw_sig[:int(srate * 300)]
raw_sig_sel_raw = raw_sig[:int(srate * 300)]

if normalize:
    med_sig, mad_sig = physio.compute_median_mad(sig_sel)
    med_raw, mad_raw = physio.compute_median_mad(raw_sig_sel)
    sig_sel = (sig_sel - med_sig) / mad_sig
    raw_sig_sel = (raw_sig_sel - med_raw) / mad_raw

firt_derivarive_sig = np.gradient(sig_sel)
second_derivative_sig = np.gradient(firt_derivarive_sig)
zero_1st = detect_cross(firt_derivarive_sig, thresh = 0)
zero_2nd = detect_cross(second_derivative_sig, thresh = 0)

t = np.arange(sig_sel.size) / srate

fig, ax = plt.subplots()

ax.plot(t, sig_sel, color = 'k', lw = 1.5, ls = '-', alpha = 0.5)
ax.scatter(t[zero_1st['rises']], sig_sel[zero_1st['rises']], color = 'r')
ax.scatter(t[zero_1st['decays']], sig_sel[zero_1st['decays']], color = 'g')

ax.axhline(2, color = 'g')
ax.axhline(-1, color = 'r')

plt.show()
# ax.plot(t, raw_sig_sel, color = 'k', lw = 1.5, ls = '-')
# ax.scatter(t[zero_1st['rises']], raw_sig_sel[zero_1st['rises']], color = 'r')
# ax.scatter(t[zero_1st['decays']], raw_sig_sel[zero_1st['decays']], color = 'g')

# ax2 = ax.twinx()
# ax2.plot(t, firt_derivarive_sig, color = 'm')
# ax2.tick_params(colors='m', which='both')
# ax3 = ax.twinx()
# ax3.plot(t, second_derivative_sig, color = 'g')
# ax3.tick_params(colors='g', which='both')

In [ ]:
threshold_peaks_normalized = 2
peak_index = zero_1st['decays'][sig_sel[zero_1st['decays']] > threshold_peaks_normalized].values

In [ ]:
trough_index_raw = zero_1st['rises'].values
trough_index = trough_index_raw[np.searchsorted(trough_index_raw, peak_index) - 1]

In [ ]:
detection = pd.DataFrame()
detection['trough_ind'] = trough_index
detection['trough_time'] =  detection['trough_ind'] / srate
next_trough_inds = trough_index[1:]
next_trough_inds = np.append(next_trough_inds, np.nan)
detection['next_trough_ind'] = next_trough_inds.astype(int)
detection['next_trough_time'] =  detection['next_trough_ind'] / srate
detection['peak_ind'] = peak_index
detection['peak_time'] =  detection['peak_ind'] / srate
detection = detection.iloc[:-1,:]
detection['rise_duration'] = detection['peak_time'] - detection['trough_time']
detection['decay_duration'] = detection['next_trough_time'] - detection['peak_time']
detection['total_duration'] = detection['rise_duration'] + detection['decay_duration']

detection['amplitude_at_trough'] = raw_sig_sel_raw[detection['trough_ind']]
detection['amplitude_at_peak'] = raw_sig_sel_raw[detection['peak_ind']]
detection['amplitude_at_next_trough'] = raw_sig_sel_raw[detection['next_trough_ind']]

detection['rise_amplitude'] = detection['amplitude_at_peak'] - detection['amplitude_at_trough']
detection['decay_amplitude'] = detection['amplitude_at_peak'] - detection['amplitude_at_next_trough']

In [ ]:
n_mads_cleaning = 6

detection_clean = detection.copy()
# metrics = ['rise_duration','decay_duration','total_duration','amplitude_at_trough', 'amplitude_at_peak',
#        'amplitude_at_next_trough', 'rise_amplitude', 'decay_amplitude']
metrics = ['rise_amplitude', 'decay_amplitude']
masks = np.zeros((detection_clean.shape[0], len(metrics)), dtype = 'bool')
concat_mask = []
for i, metric in enumerate(metrics):
    values = detection_clean[metric].values
    med, mad = physio.compute_median_mad(values)
    masks[:,i] = (values > med - n_mads_cleaning * mad) & (values < med + n_mads_cleaning * mad)
mask = np.all(masks, axis = 1)
detection_clean = detection_clean[mask]

In [ ]:
def compute_abp2(raw_abp, srate, date_vector = None, lowcut = 0.3, highcut = 10, order = 1, ftype = 'bessel', threshold_peaks_normalized = 1.5, show = False, verbose = False):
    assert not np.any(np.isnan(raw_abp)), 'Nans in ABP sig'
    abp_filt = iirfilt(raw_abp, srate, lowcut = lowcut, highcut = highcut, order = order, ftype = ftype)
    med_sig, mad_sig = physio.compute_median_mad(abp_filt)
    abp_filt_norm = (abp_filt - med_sig) / mad_sig

    firt_derivarive_sig = np.gradient(abp_filt_norm)
    minimums, = np.where((firt_derivarive_sig[:-1] <=0) & (firt_derivarive_sig[1:] >0)) # detect where sign inversion from - to +
    maximums, = np.where((firt_derivarive_sig[:-1] >=0) & (firt_derivarive_sig[1:] <0)) # detect where sign inversion from + to -
    if minimums[0] > maximums[0]: # first point detected has to be a minimum
        maximums = maximums[1:] # so remove the first maximum if is before first minimum
    if maximums[-1] > minimums[-1]: # last point detected has to be a minimum
        maximums = maximums[:-1] # so remove the last maximum if is after last minimum


    peak_index = maximums[abp_filt_norm[maximums] > threshold_peaks_normalized]
    trough_index = minimums[np.searchsorted(minimums, peak_index) - 1]

    detection = pd.DataFrame()
    detection['trough_ind'] = trough_index
    detection['trough_time'] =  detection['trough_ind'] / srate
    next_trough_inds = trough_index[1:]
    next_trough_inds = np.append(next_trough_inds, np.nan)
    detection['next_trough_ind'] = next_trough_inds
    detection['next_trough_time'] =  detection['next_trough_ind'] / srate
    detection['peak_ind'] = peak_index
    detection['peak_time'] =  detection['peak_ind'] / srate
    detection = detection.iloc[:-1,:]
    detection['next_trough_ind'] = detection['next_trough_time'].astype(int)
    detection['rise_duration'] = detection['peak_time'] - detection['trough_time']
    detection['decay_duration'] = detection['next_trough_time'] - detection['peak_time']
    detection['total_duration'] = detection['rise_duration'] + detection['decay_duration']

    detection['amplitude_at_trough'] = raw_abp[detection['trough_ind']]
    detection['amplitude_at_peak'] = raw_abp[detection['peak_ind']]
    detection['amplitude_at_next_trough'] = raw_abp[detection['next_trough_ind']]

    detection['rise_amplitude'] = detection['amplitude_at_peak'] - detection['amplitude_at_trough']
    detection['decay_amplitude'] = detection['amplitude_at_peak'] - detection['amplitude_at_next_trough']

    if not date_vector is None:
        detection['trough_date'] =  date_vector[detection['trough_ind']]
        detection['peak_date'] =  date_vector[detection['peak_ind']]
    
    #cleaning
    detection_clean = detection.copy()
    detection_clean = detection_clean[(detection_clean['amplitude_at_trough'] >= 20) & (detection_clean['amplitude_at_trough'] <= np.median(raw_abp))]
    detection_clean = detection_clean[(detection_clean['rise_amplitude'] >= 15) & (detection_clean['rise_amplitude'] <= 200) ]

    if verbose:
        print("{n_removed} abp cycles were removed by cleaning".format(n_removed = detection.shape[0] - detection_clean.shape[0]))

    if show:
        t = np.arange(raw_abp.size) / srate
        fig, ax = plt.subplots()
        ax.plot(t, raw_abp)
        ax.scatter(t[detection_clean['trough_ind']], raw_abp[detection_clean['trough_ind']], color = 'r')
        ax.scatter(t[detection_clean['peak_ind']], raw_abp[detection_clean['peak_ind']], color = 'g')
        plt.show()

    return detection_clean.reset_index(drop = True)

In [ ]:
detection = compute_abp2(raw_sig, srate, date_vector=dates, show = True, verbose = True)

In [ ]:
detection['rise_amplitude'].describe()

In [ ]:
detection[detection['rise_amplitude'] < 0]

In [ ]:
fig, ax = plt.subplots()
ax.plot(t, raw_sig_sel_raw, color = 'k', lw = 1.5, ls = '-', alpha = 0.5)
ax.scatter(t[detection['trough_ind']], raw_sig_sel_raw[detection['trough_ind']], color = 'r')
ax.scatter(t[detection['peak_ind']], raw_sig_sel_raw[detection['peak_ind']], color = 'g')
plt.show()

fig, ax = plt.subplots()
ax.plot(t, raw_sig_sel_raw, color = 'k', lw = 1.5, ls = '-', alpha = 0.5)
ax.scatter(t[detection_clean['trough_ind']], raw_sig_sel_raw[detection_clean['trough_ind']], color = 'r')
ax.scatter(t[detection_clean['peak_ind']], raw_sig_sel_raw[detection_clean['peak_ind']], color = 'g')
plt.show()

In [ ]:
bins = 100

fig, ax = plt.subplots()
ax.hist(sig_sel[zero_1st['rises']], bins=bins, color = 'r')
ax.axvline(-1, color = 'k')
plt.show()

fig, ax = plt.subplots()
ax.hist(sig_sel[zero_1st['decays']], bins=bins, color = 'g')
ax.axvline(2, color = 'k')
plt.show()

In [ ]:
bins = 100

fig, ax = plt.subplots()
ax.hist(sig_sel[zero_1st['rises']], bins=bins, color = 'r')
ax.axvline(-1, color = 'k')
plt.show()

fig, ax = plt.subplots()
ax.hist(sig_sel[zero_1st['decays']], bins=bins, color = 'g')
ax.axvline(2, color = 'k')
plt.show()

In [ ]:
zero_1st

# ICP WAVEFORM ACCORDING TO WIN LABEL WINDOW

In [ ]:
df_compliance_all = concat_slow_icp_features('compliance')
print(df_compliance_all['patient'].unique())
print(df_compliance_all['win_duration_mins'].min() * 60)

In [ ]:
# sub = 'MF12'
sub = 'JR10'

# params
win_size_s = 1.5
ev = 0
n_samples = 200
random_sampling = False

#code
cns_reader = pycns.CnsReader(data_path / sub)
icp_stream = cns_reader.streams['ICP']
raw_icp, datetimes = icp_stream.get_data(with_times = True, apply_gain = True)
srate = icp_stream.sample_rate
window_size_ind = int(srate * win_size_s)
half_window_size_ind = window_size_ind // 2

compliance_df = slow_icp_detection_compliance_features_job.get(sub).to_dataframe()
event_nums = compliance_df['n_event'].unique().tolist()
# n_event_sample = int(((compliance_df['win_duration_mins'].min()) * 60) - 100)
n_event_sample = n_samples
icp_detections = detect_icp_job.get(sub).to_dataframe()

win_labels = compliance_df['win_label'].unique().tolist()

dims = ['event','win_label','estimator','time']
data_init = np.zeros((len(event_nums), len(win_labels), 2, window_size_ind))
t_vector = range(window_size_ind)/srate
t_vector = t_vector - np.max(t_vector) / 2
coords = {'event':event_nums, 'win_label':win_labels, 'estimator':['m','s'], 'time':t_vector}
waveforms = xr.DataArray(data = data_init, dims = dims, coords = coords)

for ev in event_nums:
    compliance_df_ev = compliance_df[compliance_df['n_event'] == ev].set_index('win_label')
    for win_label in win_labels:
        d1 = compliance_df_ev.loc[win_label, 'start_win_date']
        d2 = compliance_df_ev.loc[win_label, 'stop_win_date']
        local_icp_detections = icp_detections[(icp_detections['peak_date'] > d1) & (icp_detections['peak_date'] < d2)].reset_index(drop = True)
        if local_icp_detections.shape[0] < n_samples:
            continue
        if random_sampling:
            local_icp_detections = local_icp_detections.sample(n_event_sample).reset_index(drop = True)
        else:
            # local_icp_detections = local_icp_detections.loc[:n_event_sample,:].reset_index(drop = True)
            local_icp_detections = local_icp_detections.iloc[-n_event_sample:,:].reset_index(drop = True)
        epochs = np.zeros((local_icp_detections.shape[0], window_size_ind))
        for i, row in local_icp_detections.iterrows():
            peak_ind = row['peak_ind']
            start_ind = peak_ind - half_window_size_ind
            stop_ind = start_ind + window_size_ind
            icp_epoch = raw_icp[start_ind:stop_ind]
            epochs[i,:] = icp_epoch
        m = np.mean(epochs, axis = 0)
        s = np.std(epochs, axis = 0)
        waveforms.loc[ev, win_label,'m',:] = m
        waveforms.loc[ev, win_label,'s',:] = s
waveforms.attrs['n_samples'] = n_event_sample

t = waveforms['time'].values
min_, max_ = waveforms.min(), waveforms.max()

nrows = waveforms['event'].size
ncols = waveforms['win_label'].size
fig, axs = plt.subplots(nrows = nrows, ncols = ncols, figsize = (3 * ncols, nrows * 4), sharey = True)
n_samples = waveforms.attrs['n_samples']
fig.suptitle(f'N pulses randomly sampled : {n_samples}', y = 1.01)
fig.subplots_adjust(wspace = 0)
for r, ev in enumerate(waveforms['event'].values):
    for c, win_label in enumerate(waveforms['win_label'].values):
        ax = axs[r,c]
        m = waveforms.loc[ev, win_label,'m',:].values
        s = waveforms.loc[ev, win_label,'s',:].values
        ax.plot(t, m, color = 'k')
        ax.fill_between(t, m-s, m+s, color = 'k', alpha = 0.2)
        ax.set_title(win_label)
        # ax.set_ylim(5 , 40)
        ax.set_ylim(min_ - min_ / 10, max_ + max_ / 10)
        if c == 0:
            ax.set_ylabel(f'Ev n°{int(ev)}\nICP (mmHg)')
plt.show()
    

# SLOW ICP FEATURES 

In [ ]:
true_subs = get_compliance_subs()
detect_subs = get_compliance_files()
print(len(detect_subs))
n_detections = {sub:slow_icp_rise_detection_job.get(sub).to_dataframe().shape[0] for sub in detect_subs}

In [ ]:
ser = pd.Series(n_detections).sort_values()
fig, ax = plt.subplots(figsize = (10,3))
ser.plot.bar(ax=ax)
for bar in ax.containers:
    ax.bar_label(bar)
ax.set_ylabel('Number of slow ICP rises detected')
ax.set_xlabel('Patient')
ax.set_title(f'N patient : {len(true_subs)} - N files : {len(detect_subs)} - N detections total : {ser.sum()} = {int(ser.mean())} by patient')
plt.show()


In [ ]:
# detections_slow_icp_rises = []
# for s in detect_subs:
#     df_sub = slow_icp_rise_detection_job.get(s).to_dataframe()
#     df_sub['patient'] = s
#     df_sub['total_duration_min'] = df_sub['rise_duration_min'] + df_sub['decay_duration_min']
#     detections_slow_icp_rises.append(df_sub)
# detections_slow_icp_rises = pd.concat(detections_slow_icp_rises)
# detections_slow_icp_rises.columns


## compliance

In [ ]:
df_compliance = concat_slow_icp_features('compliance')
df_compliance

In [ ]:
df_compliance[df_compliance['patient'] == 'P40']

In [ ]:
df_compliance.columns

In [ ]:
df_compliance[[ 'icp_resp_modulated','icp_pulse_resp_modulated',]].describe()

In [ ]:
%matplotlib inline

In [ ]:
prct_missing_values = df_compliance.isna().sum() / df_compliance.shape[0] * 100
fig, ax = plt.subplots(figsize = (10,5))
bars = ax.bar(prct_missing_values.index, prct_missing_values, width = 0.5)
bar_labels = ax.bar_label(bars, labels=[str(int(round(value, 0))) for value in prct_missing_values.values], padding = 5)
ax.set_ylim(0, 120)
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
ax.set_ylabel("% of missing values")
for label in bar_labels:
    label.set_rotation(90)  # Rotate labels by 45 degrees
plt.show()

In [ ]:
ser = (df_compliance['patient'].value_counts() / 12).sort_values()
fig, ax = plt.subplots(figsize = ( 9,4), constrained_layout = True)
ser.plot.bar(ax=ax, color = 'r')
for bar in ax.containers:
    ax.bar_label(bar)
ax.set_ylabel('Number of slow ICP rises use for compliance')
ax.set_xlabel('Patient')
ax.set_title(f'N patient : {len(ser)} - N detections total : {ser.sum()} = {int(ser.mean())} by patient')
plt.show()

In [ ]:
event_features_col = ['rise_duration_min','decay_duration_min','trough_amplitude_mmHg','peak_amplitude_mmHg']
df_event_features = df_compliance[df_compliance['win_label'] == 'baseline_before'][event_features_col]
fig, axs = plt.subplots(ncols = len(event_features_col), figsize = (12, 4), constrained_layout = True)
fig.suptitle(f'Event features : N = {df_event_features.shape[0]}')
for i, col in enumerate(event_features_col):
    ax = axs[i]
    sns.boxplot(data = df_event_features, y = col, ax=ax)
    if 'mmHg' in col:
        ax.set_ylim(-5, 100)
plt.show()

In [ ]:
df_compliance['icp_pulse_amplitude_mmHg'].describe()

In [ ]:
df_compliance.columns

In [ ]:
metrics = ['median_icp_mmHg','heart_in_icp_spectrum','icp_pulse_amplitude_mmHg','resp_in_icp_spectrum','icp_resp_modulated',
           'ratio_heart_resp_in_icp_spectrum', 'icp_pulse_resp_modulated','median_abp_mmHg','median_cpp_mmHg','abp_pulse_amplitude_mmHg',
           'abp_pulse_resp_modulated','abp_resp_modulated','RAQ_2','RAQ_ABP','P2P1_ratio', 'PSI','PRx','heart_rate_bpm','resp_rate_cpm']
nrows = 5
ncols = 4
poss = attribute_subplots(metrics, nrows, ncols)

for estimator in ['mean','median']:
       fig, axs = plt.subplots(nrows, ncols, figsize = (ncols * 3.5,nrows * 3.5), constrained_layout = True)
       fig.suptitle(f'Estimator : {estimator}')

       for m, (r,c) in poss.items():
              ax = axs[r,c]  
              sns.pointplot(data = df_compliance,
                            x = 'win_label',
                            y = m,
                            ax=ax,
                            # errorbar='sd'
                            errorbar=('ci',95),
                            estimator=estimator,
                            )
              ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
              # ax.legend(fontsize = 8, loc = 'upper right')
              ax.set_xlabel('')
       fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / f'win_label_effect_estimator_{estimator}.png', dpi = 300, bbox_inches = 'tight')
       plt.show()

In [ ]:
possible_hues = ['diagnosis','glasgow_end','lesion_place','lesion_side','Mobilisation']

In [ ]:
# metrics = ['median_icp_mmHg',
#        'heart_in_icp_spectrum', 'resp_in_icp_spectrum','ratio_heart_resp_in_icp_spectrum', 'icp_pulse_resp_modulated',
#        'P2P1_ratio', 'PSI']
# nrows = len(metrics)
# for hue in possible_hues:
#        try:
#               fig, axs = plt.subplots(nrows, figsize = (9,nrows * 4), constrained_layout = True)

#               for r, m in enumerate(metrics):
#                      ax = axs[r]  
#                      sns.pointplot(data = df_compliance,
#                                    x = 'win_label',
#                                    y = m,
#                                    hue = hue,
#                                    ax=ax,
#                                    # errorbar='sd'
#                                    errorbar=('ci',95)
#                                    )
#                      ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
#                      # ax.legend(fontsize = 8, loc = 'upper right')
#               # fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'win_label_effect.png', dpi = 300, bbox_inches = 'tight')
#               plt.show()
#        except:
#               continue

In [ ]:
df_compliance.columns

In [ ]:
metrics = ['Norépinéphrine', 'Midazolam', 'Propofol',
       'Kétamine', 'Thiopental', 'Eskétamine', 'Sufentanil', 'Rémifentanil',
       'Morphine', 'Dextro (mmol_l)']
nrows = 2
ncols = 5
poss = attribute_subplots(metrics, nrows, ncols)

for estimator in ['mean','median']:
       fig, axs = plt.subplots(nrows, ncols, figsize = (ncols * 3.5,nrows * 3.5), constrained_layout = True)
       fig.suptitle(f'Estimator : {estimator}')

       for m, (r,c) in poss.items():
              ax = axs[r,c]  
              sns.lineplot(data = df_compliance.groupby(['patient','win_label']).median(True).reset_index(),
                            x = 'win_label',
                            y = m,
                            hue = 'patient',
                            ax=ax,
                            # errorbar='sd'
                            # errorbar=('ci',95),
                            estimator=None,
                            legend = False,
                            )
              ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
              # ax.legend(fontsize = 8, loc = 'upper right')
              ax.set_xlabel('')
    #    fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / f'win_label_effect_estimator_{estimator}.png', dpi = 300, bbox_inches = 'tight')
       plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (30,20))
sns.heatmap(df_compliance.corr(numeric_only=True, method = 'spearman'), annot = True, cmap = 'seismic', vmin = -1, vmax = 1, ax=ax)
fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'corr_matrix.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (30,20))
sns.heatmap(df_compliance.groupby(['patient','win_label']).median(True).corr(numeric_only=True, method = 'spearman'), annot = True, cmap = 'seismic', vmin = -1, vmax = 1, ax=ax)
# fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'corr_matrix.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (12,8))
sns.heatmap(df_compliance.groupby(['patient','win_label']).median(True)[metrics].corr(numeric_only=True, method = 'spearman'), annot = True, cmap = 'seismic', vmin = -1, vmax = 1, ax=ax)
fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'corr_matrix_short_grouped.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (12,8))
sns.heatmap(df_compliance[metrics].corr(numeric_only=True, method = 'spearman'), annot = True, cmap = 'seismic', vmin = -1, vmax = 1, ax=ax)
fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'corr_matrix_short.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
plt.figure()
sns.clustermap(df_compliance[metrics].corr(numeric_only=True, method = 'spearman'),figsize=(12, 12), annot = True, cmap = 'seismic', vmin = -1, vmax = 1)
plt.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'clustermap.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
# plt.figure()
# sns.pairplot(data = df_compliance[df_compliance['pulse_amplitude_mmHg'] < 100][metrics + ['win_label']], kind = 'hist')
# plt.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'pairplot.png', dpi = 300, bbox_inches = 'tight')
# plt.show()

In [ ]:
df_compliance.columns

In [ ]:
sel_metrics = ['icp_pulse_amplitude_mmHg','abp_pulse_amplitude_mmHg','icp_pulse_resp_modulated','abp_pulse_resp_modulated', 'icp_resp_modulated', 'abp_resp_modulated']
plt.figure()
sns.pairplot(data = df_compliance[df_compliance['icp_pulse_amplitude_mmHg'] < 100].groupby(['patient','win_label']).median(True)[sel_metrics], kind = 'hist')
# plt.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'pairplot_grouped_sel_metrics.png', dpi = 300, bbox_inches = 'tight')
plt.show()

fig, ax = plt.subplots(figsize = (12,8))
sns.heatmap(data = df_compliance[df_compliance['icp_pulse_amplitude_mmHg'] < 100].groupby(['patient','win_label']).median(True)[sel_metrics].corr(numeric_only=True, method = 'spearman'), annot = True, cmap = 'seismic', vmin = -1, vmax = 1, ax=ax)
fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'corr_matrix_short_short.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
# plt.figure()
# # sns.pairplot(data = df_compliance[df_compliance['icp_pulse_amplitude_mmHg'] < 100].groupby(['patient','win_label']).median(True)[metrics], kind = 'scatter', plot_kws = {'alpha': 0.2})
# sns.pairplot(data = df_compliance[df_compliance['icp_pulse_amplitude_mmHg'] < 100].groupby(['patient','win_label']).median(True)[metrics], kind = 'hist')
# # sns.pairplot(data = df_compliance[df_compliance['icp_pulse_amplitude_mmHg'] < 100].groupby(['patient','win_label']).median(True).reset_index()[metrics + ['win_label']], kind = 'hist', hue = 'win_label')
# plt.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'pairplot_grouped.png', dpi = 300, bbox_inches = 'tight')
# plt.show()

In [ ]:
plt.figure()
# sns.pairplot(data = df_compliance[df_compliance['icp_pulse_amplitude_mmHg'] < 100].groupby(['patient','win_label']).median(True)[metrics], kind = 'scatter', plot_kws = {'alpha': 0.2})
sns.pairplot(data = df_compliance[df_compliance['icp_pulse_amplitude_mmHg'] < 100].groupby('win_label').median(True)[metrics], kind = 'scatter')
# sns.pairplot(data = df_compliance[df_compliance['icp_pulse_amplitude_mmHg'] < 100].groupby(['patient','win_label']).median(True).reset_index()[metrics + ['win_label']], kind = 'hist', hue = 'win_label')
# plt.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'compliance' / 'pairplot_grouped.png', dpi = 300, bbox_inches = 'tight')
plt.show()

## eeg

### concat bipol

In [ ]:
df_eeg_bipol = concat_slow_icp_features('eeg_bipolar')
df_eeg_bipol

In [ ]:
df_eeg_bipol['win_label'].unique()

In [ ]:
print(df_eeg_bipol.columns)
print('Number of patients :', df_eeg_bipol['patient'].unique().size)
# print('Number of patients :', df_eeg_bipol['patient'].unique().size)

In [ ]:
prct_missing_values = df_eeg_bipol.isna().sum() / df_eeg_bipol.shape[0] * 100
fig, ax = plt.subplots(figsize = (10,5))
bars = ax.bar(prct_missing_values.index, prct_missing_values, width = 0.5)
bar_labels = ax.bar_label(bars, labels=[str(int(round(value, 0))) for value in prct_missing_values.values], padding = 5)
ax.set_ylim(0, 120)
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
ax.set_ylabel("% of missing values")
for label in bar_labels:
    label.set_rotation(90)  # Rotate labels by 45 degrees
plt.show()

In [ ]:
ser = df_eeg_bipol.drop_duplicates('n_event_sub')['patient'].value_counts()
fig, ax = plt.subplots(figsize = ( 9,4), constrained_layout = True)
ser.plot.bar(ax=ax, color = 'r')
for bar in ax.containers:
    ax.bar_label(bar)
ax.set_ylabel('Number of slow ICP rises used for qEEG bipolar')
ax.set_xlabel('Patient')
ax.set_title(f'N patient : {len(ser)} - N detections total : {ser.sum()} = {int(ser.mean())} by patient')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (30,20))
sns.heatmap(df_eeg_bipol.groupby(['patient','win_label']).median(True).corr(numeric_only=True, method = 'spearman'), annot = True, cmap = 'seismic', vmin = -1, vmax = 1, ax=ax)
# fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'qEEG' / 'corr_matrix.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:

fig, ax = plt.subplots(figsize =(12, 4))
sns.pointplot(data = df_eeg_bipol,
                        x = 'win_label',
                                   y = 'median_icp_mmHg',
                                   estimator = 'median',
                                   ax=ax)
plt.show()


In [ ]:
metrics = [
       'delta_power_spectrum_norm', 'theta_power_spectrum_norm','alpha_power_spectrum_norm','adr_norm_spectrum', 'spectrum_slope',
       'spectrum_entropy', 'suppression']
hues = ['is_side_of_lesion','lobe','channel_name']

for estimator in ['mean','median']:
       nrows = len(metrics)
       ncols = len(hues)
       fig, axs = plt.subplots(nrows, ncols, figsize = (15,nrows * 4), constrained_layout = True)
       fig.suptitle(f'Bipolar montage ({estimator} estimator pointplot)', fontsize = 20)
       for r, m in enumerate(metrics):
              for c, hue in enumerate(hues):
                     ax = axs[r,c]  
                     sns.pointplot(data = df_eeg_bipol,
                                   x = 'win_label',
                                   y = m,
                                   hue = hue,
                                   ax=ax,
                                   estimator = 'mean',
                                   )
                     ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
                     ax.legend(fontsize = 8, loc = 'upper right')
       fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'qEEG' / f'win_label_effect_bipol_{estimator}_new.png', dpi = 300, bbox_inches = 'tight')
       plt.show()
       

In [ ]:
fig, ax = plt.subplots(figsize = (30,20))
sns.heatmap(df_eeg_bipol.corr(numeric_only=True, method = 'spearman'), annot = True, cmap = 'seismic', vmin = -1, vmax = 1, ax=ax)
fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'qEEG' / 'corr_matrix_bipolar_new.png', dpi = 300, bbox_inches = 'tight')
plt.show()

#### nory :
- des données manquantes sur les canaux (variable channel_name) : C3-Fp1, C3-1, C4-Fp2, C4-O2. Avais-tu ce même problème ?
- Sur les canaux Fz-Cz et Cz-Pz, je n'ai que des données "injured" et "midline", ce qui n'est pas étonnant car ce sont les dérivations de la ligne médiane. En revanche, j'ai la même chose pour le montage C3-Cz alors que ce n'est pas le cas pour Cz-C4. Effet du hasard ou erreur de données ?

In [ ]:
# des données manquantes sur les canaux (variable channel_name) : C3-Fp1, C3-1, C4-Fp2, C4-O2. Avais-tu ce même problème ?
df_eeg_bipol.columns

In [ ]:
df_eeg_bipol[['channel_name','suppression']].groupby('channel_name').describe()

In [ ]:
df_eeg_bipol.drop_duplicates(subset = ['patient','channel_name'])['patient'].value_counts()

In [ ]:
sns.pointplot(data = df_eeg_bipol[df_eeg_bipol['patient'] == 'P2'],
              x = 'channel_name',
              y = 'suppression')

In [ ]:
sns.pointplot(data = df_eeg_bipol[df_eeg_bipol['patient'] == 'P2'],
              x = 'is_side_of_lesion',
              y = 'suppression')

In [ ]:
# for chan in df_eeg_bipol['channel_name'].unique():
#     print(chan)
#     fig, ax = plt.subplots()
#     sns.pointplot(data = df_eeg_bipol[df_eeg_bipol['channel_name'] == chan],
#                   x = 'win_label',
#                   y = 'suppression',
#                  ax=ax
#                  )
#     ax.set_title(chan)
#     plt.show()

### concat monopol

In [ ]:
df_eeg_mono = concat_slow_icp_features('eeg_monopolar')
df_eeg_mono

In [ ]:
print(df_eeg_mono.columns)
print('Number of patients :', df_eeg_mono['patient'].unique().size)
# print('Number of patients :', df_eeg_mono['patient'].unique().size)

In [ ]:
prct_missing_values = df_eeg_mono.isna().sum() / df_eeg_mono.shape[0] * 100
fig, ax = plt.subplots(figsize = (10,5))
bars = ax.bar(prct_missing_values.index, prct_missing_values, width = 0.5)
bar_labels = ax.bar_label(bars, labels=[str(int(round(value, 0))) for value in prct_missing_values.values], padding = 5)
ax.set_ylim(0, 120)
ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
ax.set_ylabel("% of missing values")
for label in bar_labels:
    label.set_rotation(90)  # Rotate labels by 45 degrees
plt.show()

In [ ]:
ser = df_eeg_mono.drop_duplicates('n_event_sub')['patient'].value_counts()
fig, ax = plt.subplots(figsize = ( 9,4), constrained_layout = True)
ser.plot.bar(ax=ax, color = 'r')
for bar in ax.containers:
    ax.bar_label(bar)
ax.set_ylabel('Number of slow ICP rises used for qEEG monopolar')
ax.set_xlabel('Patient')
ax.set_title(f'N patient : {len(ser)} - N detections total : {ser.sum()} = {int(ser.mean())} by patient')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (30,20))
sns.heatmap(df_eeg_mono.groupby(['patient','win_label']).median(True).corr(numeric_only=True, method = 'spearman'), annot = True, cmap = 'seismic', vmin = -1, vmax = 1, ax=ax)
# fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'qEEG' / 'corr_matrix.png', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
metrics = [
       'delta_power_spectrum_norm', 'theta_power_spectrum_norm','alpha_power_spectrum_norm','adr_norm_spectrum', 'spectrum_slope',
       'spectrum_entropy', 'suppression']
hues = ['is_side_of_lesion','lobe','channel_name']

for estimator in ['mean','median']:
       nrows = len(metrics)
       ncols = len(hues)
       fig, axs = plt.subplots(nrows, ncols, figsize = (15,nrows * 4), constrained_layout = True)
       fig.suptitle(f'Bipolar montage ({estimator} estimator pointplot)', fontsize = 20)
       for r, m in enumerate(metrics):
              for c, hue in enumerate(hues):
                     ax = axs[r,c]  
                     sns.pointplot(data = df_eeg_mono,
                                   x = 'win_label',
                                   y = m,
                                   hue = hue,
                                   ax=ax,
                                   estimator = 'mean',
                                   )
                     ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
                     ax.legend(fontsize = 8, loc = 'upper right')
       fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'qEEG' / f'win_label_effect_mono_{estimator}_new.png', dpi = 300, bbox_inches = 'tight')
       plt.show()
       

In [ ]:
fig, ax = plt.subplots(figsize = (30,20))
sns.heatmap(df_eeg_mono.corr(numeric_only=True, method = 'spearman'), annot = True, cmap = 'seismic', vmin = -1, vmax = 1, ax=ax)
fig.savefig(base_folder / 'figures' / 'slow_icp_rises_figs' / 'qEEG' / 'corr_matrix_monopolar_new.png', dpi = 300, bbox_inches = 'tight')
plt.show()

### lacking gender + wfns + fischer + lesion place + lesion side

In [ ]:
for col in ['gender','wfns','fisher','lesion_place','lesion_side']:
    # print(col, df_eeg_mono[col].unique())
    print(col , 'information is missing for :', df_eeg_mono['patient'][df_eeg_mono[col].isna()].unique())

# DVP VENTILATION MDOE DETECTOR

In [ ]:
from scipy.stats import median_abs_deviation

In [ ]:
sub = 'MF12'
resp = detect_resp_job.get(sub).to_dataframe()
resp['cycle_freq_cpm'] = resp['cycle_freq'] * 60

In [ ]:
N = 30
rolling_mad = resp['cycle_freq_cpm'].rolling(N).apply(median_abs_deviation)
# rolling_sd = resp['cycle_freq_cpm'].rolling(N).std()

In [ ]:
threshold = 0.4
bins = np.arange(0, 2, 0.01)

fig, ax = plt.subplots()
ax.hist(rolling_mad, bins=bins)
ax.axvline(threshold, color = 'r')
plt.show()


In [ ]:
fig, ax = plt.subplots()
ax.hist(rolling_sd, bins=bins)
ax.axvline(threshold, color = 'r')
plt.show()

In [ ]:
resp['sliding_variability_cpm_mad'] = rolling_mad
resp['sliding_variability_cpm_sd'] = rolling_sd
resp['is_full_controlled'] = resp['sliding_variability_cpm_mad'] < threshold

In [ ]:
%matplotlib widget

In [ ]:
fig, ax = plt.subplots()
ax.plot(resp['cycle_freq_cpm'])
ax2 = ax.twinx()
ax2.plot(resp['sliding_variability_cpm_mad'], color ='darkorange')
ax2.plot(resp['sliding_variability_cpm_sd'], color ='g')
ax2.axhline(0.4, color = 'm')
ax2.plot(resp['is_full_controlled'] * 5, color = 'k')
plt.show()

# STATS PHYSIO CSD

## ECG

In [ ]:
ecg_csd = concat_ecg_csd_job.get(global_key).to_dataframe()

In [ ]:
save_folder = base_folder / 'figures' / 'physio_csd_valentin'

In [ ]:
col_names = ecg_csd.columns
remove_cols = ['win_start','win_stop','Kétamine','Thiopental','Eskétamine','Morphine','Mobilisation',
               'hr_med_ms','hr_mad_ms','hr_mean_ms','hr_freq_mean_bpm','hr_sd_ms','hr_sd_bpm','keep_window',
               'is_window_in_csd','N','num_csd']
remove_cols = remove_cols + list(col_names[ecg_csd.dtypes == 'object'])
col_sel = [c for c in col_names if not c in remove_cols]
fig, ax = plt.subplots(figsize = (15,10))
sns.heatmap(data = ecg_csd[col_sel].corr('spearman'),
            annot = True,
            cmap = 'seismic',
            vmin = -1, 
            vmax = 1,
            ax=ax
            )
fig.savefig(save_folder / 'corr_matrix_ecg.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
sns.clustermap(data = ecg_csd[col_sel].corr('spearman'),
            annot = True,
            cmap = 'seismic',
            vmin = -1, 
            vmax = 1,
            )

In [ ]:
import ghibtools as gh

In [ ]:
%matplotlib inline

In [ ]:
# x = 'length_of_stay'
# x = 'Sufentanil'
x = 'win_duration_mins'
y = 'hr_mad_bpm'
# y = 'Norépinéphrine'
gh.stats_quantitative(df = ecg_csd[[x,y]].dropna(), xlabel = x, ylabel = y, corr_method = 'spearman')

In [ ]:
rows = ['Norépinéphrine','Midazolam','Propofol','Sufentanil','Rémifentanil','Dextro (mmol_l)']
cols = ['hr_freq_med_bpm','hr_mad_bpm']

nrows = len(rows)
ncols = len(cols)

fig, axs = plt.subplots(nrows, ncols, figsize = (ncols * 4, nrows * 3), constrained_layout = True)
for r, x in enumerate(rows):
    for c, y in enumerate(cols):
        ax = axs[r,c]
        gh.stats_quantitative(df = ecg_csd[[x,y]].dropna(), xlabel = x, ylabel = y, corr_method = 'spearman', ax=ax)
fig.savefig(save_folder / 'ecg_vs_treatments_and_dextro.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
metrics = ['hr_freq_med_bpm','hr_mad_bpm']
ylims = {'hr_freq_med_bpm':(40, 130), 'hr_mad_bpm':(0,15)}
n = len(metrics) 
fig, axs = plt.subplots(ncols = n , figsize = (n * 4, 5), constrained_layout = True)
for r, metric in enumerate(metrics):
    ax = axs[r]
    sns.lineplot(data = ecg_csd,
                x = 'win_label',
                y = metric,
                estimator = None,
                units='patient_num_csd',
                alpha = 0.1,
                color = 'k',
                ax=ax
                )
    sns.pointplot(data = ecg_csd,
                x = 'win_label',
                y = metric,
                ax=ax,
                color = 'r',
                )
    ax.set_xticks(ax.get_xticks(), labels=ax.get_xticklabels(), rotation = 90)
    ax.set_ylim(*ylims[metric])
fig.savefig(save_folder / 'ecg_window.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
ecg_csd.columns

In [ ]:
metrics = ['hr_freq_med_bpm', 'hr_mad_bpm']
# metrics = ['hr_freq_mean_bpm', 'hr_sd_bpm']
fig, axs = plt.subplots(ncols = len(metrics), constrained_layout = True)
for i, metric in enumerate(metrics):
    ax = axs[i]
    gh.auto_stats(df = ecg_csd,
                predictor = 'Mobilisation',
                outcome = metric,
                design = 'between',
                ax=ax)

## RESP

In [ ]:
resp_csd = concat_resp_csd_job.get(global_key).to_dataframe()

In [ ]:
save_folder = base_folder / 'figures' / 'physio_csd_valentin'

In [ ]:
col_names = resp_csd.columns
remove_cols = ['win_start','win_stop','Kétamine','Thiopental','Eskétamine','Morphine','Mobilisation',
               'respi_med_secs','respi_mad_secs','respi_mean_secs','respi_sd_secs','rr_mean_cpm','rr_std_cpm','keep_window',
               'is_window_in_csd','N','num_csd','win_duration_mins']
remove_cols = remove_cols + list(col_names[resp_csd.dtypes == 'object'])
col_sel = [c for c in col_names if not c in remove_cols]
fig, ax = plt.subplots(figsize = (15,10))
sns.heatmap(data = resp_csd[col_sel].corr('spearman'),
            annot = True,
            cmap = 'seismic',
            vmin = -1, 
            vmax = 1,
            ax=ax
            )
fig.savefig(save_folder / 'corr_matrix_resp.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
import ghibtools as gh

In [ ]:
%matplotlib widget

In [ ]:
# x = 'percentage_of_controlled_ventilation'
x = 'rr_mad_cpm'
# x = 'Sufentanil'
y = 'Midazolam'
# y = 'mrs_6'
gh.stats_quantitative(df = resp_csd[[x,y]].dropna(), xlabel = x, ylabel = y, corr_method = 'spearman')

In [ ]:
rows = ['Norépinéphrine','Midazolam','Propofol','Sufentanil','Rémifentanil','Dextro (mmol_l)']
cols = ['rr_med_cpm','rr_mad_cpm']

nrows = len(rows)
ncols = len(cols)

fig, axs = plt.subplots(nrows, ncols, figsize = (ncols * 4, nrows * 3), constrained_layout = True)
for r, x in enumerate(rows):
    for c, y in enumerate(cols):
        ax = axs[r,c]
        gh.stats_quantitative(df = resp_csd[[x,y]].dropna(), xlabel = x, ylabel = y, corr_method = 'spearman', ax=ax)
fig.savefig(save_folder / 'resp_vs_treatments_and_dextro.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
metrics = ['rr_med_cpm','rr_mad_cpm']
ylims = {'rr_med_cpm':(5, 25), 'rr_mad_cpm':(0, 5)}
n = len(metrics) 
fig, axs = plt.subplots(ncols = n , figsize = (n * 4, 5), constrained_layout = True)
for r, metric in enumerate(metrics):
    ax = axs[r]
    sns.lineplot(data = resp_csd,
                x = 'win_label',
                y = metric,
                estimator = None,
                units='patient_num_csd',
                alpha = 0.1,
                color = 'k',
                ax=ax
                )
    sns.pointplot(data = resp_csd,
                x = 'win_label',
                y = metric,
                ax=ax,
                color = 'tab:blue',
                )
    ax.set_xticks(ax.get_xticks(), labels=ax.get_xticklabels(), rotation = 90)
    ax.set_ylim(*ylims[metric])
fig.savefig(save_folder / 'resp_window.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
metrics = ['rr_med_cpm','rr_mad_cpm','percentage_of_controlled_ventilation']
ylims = {'rr_med_cpm':(5, 25), 'rr_mad_cpm':(0, 5)}
n = len(metrics) 
fig, axs = plt.subplots(ncols = n , figsize = (n * 4, 5), constrained_layout = True)
for r, metric in enumerate(metrics):
    ax = axs[r]
    sns.lineplot(data = resp_csd,
                x = 'win_label',
                y = metric,
                estimator = None,
                units='patient_num_csd',
                alpha = 0.1,
                color = 'k',
                ax=ax
                )
    sns.pointplot(data = resp_csd,
                x = 'win_label',
                y = metric,
                ax=ax,
                color = 'tab:blue',
                )
    ax.set_xticks(ax.get_xticks(), labels=ax.get_xticklabels(), rotation = 90)
    ax.set_ylim(*ylims[metric])
# fig.savefig(save_folder / 'resp_window.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
metrics = ['rr_med_cpm', 'rr_mad_cpm','percentage_of_controlled_ventilation']
# metrics = ['hr_freq_mean_bpm', 'hr_sd_bpm']
fig, axs = plt.subplots(ncols = len(metrics), constrained_layout = True)
for i, metric in enumerate(metrics):
    ax = axs[i]
    gh.auto_stats(df = resp_csd,
                predictor = 'Mobilisation',
                outcome = metric,
                design = 'between',
                ax=ax)

## RSA

In [ ]:
rsa_csd = concat_rsa_csd_job.get(global_key).to_dataframe()

In [ ]:
rsa_csd.columns

In [ ]:
save_folder = base_folder / 'figures' / 'physio_csd_valentin'

In [ ]:
col_names = rsa_csd.columns
remove_cols = ['win_start','win_stop','Kétamine','Thiopental','Eskétamine','Morphine','Mobilisation','keep_window',
               'is_window_in_csd','N','num_csd','win_duration_mins']
remove_cols = remove_cols + list(col_names[rsa_csd.dtypes == 'object'])
col_sel = [c for c in col_names if not c in remove_cols]
fig, ax = plt.subplots(figsize = (15,10))
sns.heatmap(data = rsa_csd[col_sel].corr('spearman'),
            annot = True,
            cmap = 'seismic',
            vmin = -1, 
            vmax = 1,
            ax=ax
            )
# fig.savefig(save_folder / 'corr_matrix_rsa.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
import ghibtools as gh

In [ ]:
%matplotlib inline

In [ ]:
x = 'Norépinéphrine'
# x = 'Sufentanil'
y = 'rsa_med_bpm'
# y = 'mrs_6'
gh.stats_quantitative(df = rsa_csd[[x,y]].dropna(), xlabel = x, ylabel = y, corr_method = 'spearman')

In [ ]:
rows = ['Norépinéphrine','Midazolam','Propofol','Sufentanil','Rémifentanil','Dextro (mmol_l)']
cols = ['rsa_med_bpm','rsa_mean_bpm']

nrows = len(rows)
ncols = len(cols)

fig, axs = plt.subplots(nrows, ncols, figsize = (ncols * 4, nrows * 3), constrained_layout = True)
for r, x in enumerate(rows):
    for c, y in enumerate(cols):
        ax = axs[r,c]
        gh.stats_quantitative(df = rsa_csd[[x,y]].dropna(), xlabel = x, ylabel = y, corr_method = 'spearman', ax=ax)
fig.savefig(save_folder / 'rsa_vs_treatments_and_dextro.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
metrics = ['rsa_med_bpm','rsa_mean_bpm']
ylims = {'rsa_med_bpm':(0, 10), 'rsa_mean_bpm':(0, 10)}
n = len(metrics) 
fig, axs = plt.subplots(ncols = n , figsize = (n * 4, 5), constrained_layout = True)
for r, metric in enumerate(metrics):
    ax = axs[r]
    sns.lineplot(data = rsa_csd,
                x = 'win_label',
                y = metric,
                estimator = None,
                units='patient_num_csd',
                alpha = 0.1,
                color = 'k',
                ax=ax
                )
    sns.pointplot(data = rsa_csd,
                x = 'win_label',
                y = metric,
                ax=ax,
                color = 'g',
                )
    ax.set_xticks(ax.get_xticks(), labels=ax.get_xticklabels(), rotation = 90)
    ax.set_ylim(*ylims[metric])
fig.savefig(save_folder / 'rsa_window.png', dpi = 200, bbox_inches = 'tight')
plt.show()

In [ ]:
metrics = ['rsa_med_bpm', 'rsa_mean_bpm']
# metrics = ['hr_freq_mean_bpm', 'hr_sd_bpm']
fig, axs = plt.subplots(ncols = len(metrics), constrained_layout = True)
for i, metric in enumerate(metrics):
    ax = axs[i]
    gh.auto_stats(df = rsa_csd,
                predictor = 'Mobilisation',
                outcome = metric,
                design = 'between',
                ax=ax)

# LOAD ALL TT

In [ ]:
all_treatments = pd.read_csv(base_folder / 'icca_review' / 'all_treatments.csv', index_col = 0)

In [ ]:
pattern = 'stigmine'
search = all_treatments[all_treatments['short_label'].apply(lambda x:True if pattern in x else False)]
search
# search[search['short_label'] == search['short_label'].unique()[0]]
# search['short_label'].unique()

In [ ]:
all_treatments_set = all_treatments.set_index('intervention Id')

In [ ]:
id = 4508
id_tts = all_treatments_set.loc[id]
id_tts
# id_tts['Patient'].unique()
# id_tts

In [ ]:
sub = 'NN7'
id_tts[id_tts['Patient'] == sub]

# DETECT ICP INCREASES

In [ ]:
from icp_rises import load_table_nory

In [ ]:
manual_all = load_table_nory()
manual_all['ID_pseudo'].value_counts()

In [ ]:
sub = 'MF12'
cns_reader = CnsReader(data_path / sub)
stream_name = 'ICP'
stream = cns_reader.streams['ICP']

In [ ]:
manual = manual_all[manual_all['ID_pseudo'] == sub]

In [ ]:
dates_events = []
for i, row in manual.iterrows():
    day = pd.to_datetime(row['date_debut_ev'])
    date = day + pd.Timedelta(str(row['heure_debut_ev']))
    dates_events.append(date)
dates_events = pd.to_datetime(dates_events).tz_localize('Europe/Paris').tz_convert('GMT')

In [ ]:
srate = stream.sample_rate

In [ ]:
sig, dates = stream.get_data(with_times = True, apply_gain=True)

In [ ]:
down_factor = 100
sig2_down = sig[::down_factor]
dates_down = dates[::down_factor]
new_srate  = srate/down_factor

sig3 = sig[::down_factor]

In [ ]:
def detect_slow_icp_rises(raw_icp, 
                          srate, 
                          datetimes, 
                          params_detection = {'min_rise_duration_min':15,'min_rise_amplitude_mmHg':5, 'min_peak_amplitude_mmHg':20,'max_peak_amplitude_raw_mmHg':80,'min_decay_amplitude_mmHg':2},
                          params_filtering = {'half_period_filter_min':15,'ftype':'bessel','order':10},
                          verbose = False
                          ):
    period_min = params_filtering['half_period_filter_min'] * 2
    period_s = period_min * 60
    highcut = 1 / period_s
    if verbose:
        print(f'Highcut = {highcut} Hz')
    icp_filtered = iirfilt(raw_icp, srate, highcut = highcut, ftype = params_filtering['ftype'], order = params_filtering['order'])
    firt_derivarive_sig = np.gradient(icp_filtered)
    detection = detect_cross(firt_derivarive_sig, thresh = 0)
    detection.columns = ['trough_ind','peak_ind']
    detection['next_trough_ind'] = np.append(detection.loc[1:,'trough_ind'].values, np.nan)
    detection = detection.dropna()
    detection = detection.astype(int)
    detection['trough_time_s'] = detection['trough_ind'] / new_srate
    detection['peak_time_s'] = detection['peak_ind'] / new_srate
    detection['next_trough_time_s'] = detection['next_trough_ind'] / new_srate
    detection['trough_date'] = datetimes[detection['trough_ind']]
    detection['peak_date'] = datetimes[detection['peak_ind']]
    detection['next_trough_date'] = datetimes[detection['next_trough_ind']]
    detection['rise_duration_s'] = detection['peak_time_s'] - detection['trough_time_s']
    detection['rise_duration_min'] = detection['rise_duration_s'] / 60
    detection['decay_duration_s'] = detection['next_trough_time_s'] - detection['peak_time_s']
    detection['decay_duration_min'] = detection['decay_duration_s'] / 60
    detection['trough_amplitude_mmHg'] = icp_filtered[detection['trough_ind']]
    detection['peak_amplitude_mmHg'] = icp_filtered[detection['peak_ind']]
    detection['next_trough_amplitude_mmHg'] = icp_filtered[detection['next_trough_ind']]
    detection['rise_amplitude_mmHg'] = detection['peak_amplitude_mmHg'] - detection['trough_amplitude_mmHg']
    detection['peak_amplitude_raw_mmHg'] = raw_icp[detection['peak_ind']]
    detection['decay_amplitude_mmHg'] = detection['peak_amplitude_mmHg'] - detection['next_trough_amplitude_mmHg']
    
    params_detection = {k:v for k,v in params_detection.items() if not v is None}
    if len(params_detection) != 0:
        masking = pd.DataFrame(index = detection.index, columns = params_detection.keys(), dtype = bool)
        masking[:] = True
        for param_name, threshold in params_detection.items():
            metric_name = param_name[4:]
            if 'min' in param_name:
                masking.loc[:,param_name] = detection[metric_name] > threshold
            elif 'max' in param_name:
                masking.loc[:,param_name] = detection[metric_name] < threshold
        mask = masking.all(axis = 1)
        detection = detection[mask].reset_index(drop = True)

    return icp_filtered, detection

In [ ]:
params_detection = {'min_rise_duration_min':15,
                        'min_rise_amplitude_mmHg':5, 
                        'min_peak_amplitude_mmHg':20,
                        'max_peak_amplitude_raw_mmHg':80,
                        'min_decay_amplitude_mmHg':2
                        }
# params_detection = {'min_rise_duration_min':15,
#                         'min_rise_amplitude_mmHg':None, 
#                         'min_peak_amplitude_mmHg':None,
#                         'max_peak_amplitude_raw_mmHg':None,
#                         'min_decay_amplitude_mmHg':2
#                         }
params_filtering = {'half_period_filter_min':15,
                    'ftype':'bessel',
                    'order':10
                    }
sig2_down, manual_rise_detection = detect_slow_icp_rises(sig3, new_srate, dates_down, params_detection=params_detection, params_filtering=params_filtering)
print(manual_rise_detection.shape)

In [ ]:
%matplotlib widget

In [ ]:
manual_rise_detection.head(5)

In [ ]:
manual_rise_detection.shape

In [ ]:
fig , ax = plt.subplots()
t = np.arange(sig2_down.size) / new_srate
ax.plot(dates_down, sig2_down)
ax.plot(dates_down, sig3, alpha = 0.5)
# ax2 = ax.twinx()
# ax2.plot(dates_down, np.gradient(sig2_down), color = 'k', lw = 2)
ax.scatter(manual_rise_detection['trough_date'], sig2_down[manual_rise_detection['trough_ind']], color = 'g', marker = 'x', s=50)
ax.scatter(manual_rise_detection['peak_date'], sig2_down[manual_rise_detection['peak_ind']], color = 'r')
ax.scatter(manual_rise_detection['next_trough_date'], sig2_down[manual_rise_detection['next_trough_ind']], color = 'm', marker = '^')
# ax.scatter(dates_down[peaks], sig2_down[peaks], color = 'r')
ax.axhline(20, color = 'r', ls = '--')
ax.set_ylim(-5, 60)
for d in dates_events:
    ax.axvline(d, color = 'g')
for i, row in manual_rise_detection.iterrows():
    ax.axvspan(row['trough_date'], row['peak_date'] , color = 'r', alpha = 0.2)
for i, row in manual_rise_detection.iterrows():
    ax.axvspan(row['peak_date'], row['next_trough_date'] , color = 'g', alpha = 0.2)
# contour_heights = sig2_down[peaks] - res['prominences']
# ax.vlines(x=dates_down[peaks], ymin=contour_heights, ymax=sig2_down[peaks], color = 'k')
plt.show()

# TEST ICCA JOBS

In [ ]:
sub = 'MF12'
# ds = icca_pse_tt_job.get(sub)
# ds = icca_medication_tt_job.get(sub)
# ds = icca_bio_job.get(sub)
# ds = icca_csf_job.get(sub)
ds = icca_clinical_job.get(sub)
ds

In [ ]:
%matplotlib widget

In [ ]:
plt.figure()
name = 'Glasgow, total'
ds[name].plot.line()
ds[name].plot.scatter(color = 'k')

# name = 'Midazolam'
# ds[name].plot.line()
# ds[name].plot.scatter(color = 'k')
plt.show()

In [ ]:
name = 'Sufentanil'
ds[name].plot.line(color = 'g')
ds[name].plot.scatter(color = 'k')

In [ ]:
ds['Taille Gauche'].plot()

# TEST ICCA TOOLS

In [ ]:
icca_subs = get_icca_subs()

In [ ]:
sub = 'MF12'
dtype = 'treatment'
icca_raw = load_icca_raw(sub, dtype)

In [ ]:
res = load_treatment(sub, name = 'Sufentanil', administration_type='PSE',  use_human_weight=False, verbose = True, show = True)

In [ ]:
get_PSE_conc_unit_mapping(compute=False)

In [ ]:
res

In [ ]:
icca_raw

In [ ]:
sub = icca_subs[10]
# sub = 'MF12'
test_ds = load_clinical_in_dataset(sub)

In [ ]:
test_ds['Taille Gauche'].plot()

In [ ]:
test_ds['Taille Droite'].plot()

In [ ]:
sub = icca_subs[40]
# sub = 'MF12'
test_ds = load_PSE_treatment_in_dataset(sub)

In [ ]:
test_ds

In [ ]:
for var in test_ds.data_vars:
    test_ds[var].plot()
    plt.show()

# READ ICCA

## Raw function

In [ ]:
def load_icca_raw(sub, dtype, time_zone = 'Europe/Paris'):
    """
    sub : str
    dtype : str - clinical or treatment or biology or csf
    """
    df = pd.read_excel(icca_path / sub / f'{sub}_ICCA_{dtype}_anonymous.xlsx')
    df['date_gmt'] = df['Date'].dt.tz_localize(time_zone, ambiguous='NaT').dt.tz_convert('GMT').values
    return df

In [ ]:
sub = 'MF12'
dtype = 'treatment'
df = load_icca_raw(sub, dtype)
df.head(5)

## Treatment function

In [ ]:
def load_treatment(sub, name = None, administration_type = None, use_human_weight = True, verbose = False, show = False):
    df = load_icca_raw(sub, 'treatment')
    df['short_label'] = df['short Label'].apply(lambda x:x.split(':')[-1][1:])
    df['administration_type'] = df['short Label'].apply(lambda x:'PSE' if 'PSE' in x else 'medication')
    if verbose:
        print('labels', df['short_label'].unique().tolist())
    clinical = load_icca_raw(sub, 'clinical')
    median_weight = clinical[clinical['Unite'] == 'kg']['Valeur'].apply(lambda x:float(x.replace(',','.'))).median()
    if name is None:
        return df
    else:
        df = df[df['short_label'] == name]
        n_param_court = df['Parametre court'].unique().size
        if n_param_court == 1:
            possible_administration_types = ['medication']
        elif n_param_court == 3:
            possible_administration_types = ['PSE','medication']
        elif n_param_court == 2:
            possible_administration_types = ['PSE']
        if verbose:
            print('possible administrations', possible_administration_types)
        if administration_type is None:
            return df
        elif administration_type == 'PSE':
            assert administration_type in possible_administration_types, f'"{administration_type}" not available for this treatment'
            df = df[df['administration_type'] == administration_type]
            df = df.sort_values(by = 'date_gmt')
            unit = df[df['Parametre court'] == 'Conc']['Unite'].unique()[0].split('/')[0]
            t_unit = df[df['Parametre court'] == 'Débit adm']['Unite'].unique()[0].split('/')[1]
            unit = f'{unit}/{t_unit}'
            concentrations = df[df['Parametre court'] == 'Conc']['value Number'].values
            flows = df[df['Parametre court'] == 'Débit adm']['value Number'].values
            if np.unique(concentrations).size == 1: # if concentration do not vary
                concentrations = concentrations[0] # take the first concentration to compute doses
            else: # else check than same number of concentrations and flows values are available
                assert concentrations.size == flows.size, f'Not the same amount of contration and flow values (conc : {concentrations.size}, flow : {flows.size}, datetimes : {datetimes.size})'
            datetimes = df[df['Parametre court'] == 'Débit adm']['date_gmt'].to_numpy()
            doses = concentrations * flows
            if use_human_weight:
                doses = doses / median_weight
                unit = f'{unit}/kg'
            if show:
                fig, ax = plt.subplots()
                ax.plot(datetimes, doses)
                ax.scatter(datetimes, doses, color = 'k')
                ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
                ax.set_ylabel(f'{name} - {administration_type} ({unit})')
                plt.show()
            return {'datetimes':datetimes,'values':doses, 'unit':unit}
        elif administration_type == 'medication':
            assert administration_type in possible_administration_types, f'"{administration_type}" not available for this treatment'
            df = df[df['administration_type'] == administration_type]
            df = df.sort_values(by = 'date_gmt')
            datetimes = df['date_gmt'].values
            unit = df['Unite'].unique()[0]
            doses = df['value Number'].values
            if use_human_weight:
                doses = doses / median_weight
                unit = f'{unit}/kg'
            if show:
                fig, ax = plt.subplots()
                ax.plot(datetimes, doses)
                ax.scatter(datetimes, doses, color = 'k')
                ax.set_ylabel(f'{name} - {administration_type} ({unit})')
                ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
                plt.show()
            return {'datetimes':datetimes,'values':doses, 'unit':unit}

In [ ]:
# dict_res = load_treatment(sub, verbose = True, name = 'Sufentanil', administration_type='PSE', use_human_weight=False, show = True)
dict_res = load_treatment(sub, verbose = True, name = 'Norépinéphrine', administration_type= 'PSE', show = True, use_human_weight=True)
# dict_res = load_treatment(sub, verbose = True, name = 'Thiopenthal', administration_type= 'medication', show = True, use_human_weight=True)

## Clinical function

In [ ]:
sub = 'MF12'
kind = 'clinical'
clin = load_icca_raw(sub, kind)
clin.head(5)

In [ ]:
def load_clinical(sub, name = None, verbose = False, show = False):
    df = load_icca_raw(sub, 'clinical')
    df['short_label'] = df['Parametre court'].copy()
    if verbose:
        print(df['short_label'].unique().tolist())
    if name is None:
        return df
    else:
        if name == 'Pupille Gauche':
            name = 'Gauche'
        elif name == 'Pupille Droite':
            name = 'Droite'
        elif name == 'Taille Pupille Gauche':
            name = 'Taille Gauche'
        elif name == 'Taille Pupille Droite':
            name = 'Taille Droite'
        df = df[df['short_label'] == name]
        df = df.sort_values(by = 'date_gmt')
        unit = df['Unite'].unique()[0]
        df_sel = df[['date_gmt','Valeur']].dropna()
        datetimes = df_sel['date_gmt'].values
        values = df_sel['Valeur']
        # print(values)
        try:
            values = values.values.astype(float)
        except:
            if 'Dextro' in name or 'Poids' in name:
                values = values.apply(lambda x:float(x.replace(',','.'))).values
            else:
                values = values.values

        if show:
            fig, ax = plt.subplots()
            ax.plot(datetimes, values)
            ax.scatter(datetimes, values, color = 'k')
            ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
            ax.set_ylabel(f'{name} ({unit})')
            plt.show()

        return {'datetimes':datetimes,'values':values, 'unit':unit}

In [ ]:
def load_clinical_in_dataset(sub):
    df_start = load_icca_raw(sub, 'clinical')
    df_start['short_label'] = df_start['Parametre court'].copy()
    names = df_start['short_label'].unique().tolist()
    ds = xr.Dataset()
    for name in names:
        df = df_start[df_start['short_label'] == name]
        df = df.sort_values(by = 'date_gmt')
        unit = df['Unite'].unique()[0]
        df_sel = df[['date_gmt','Valeur']].dropna()
        datetimes = df_sel['date_gmt'].values
        values = df_sel['Valeur']
        try:
            values = values.values.astype(float)
        except:
            if 'Dextro' in name or 'Poids' in name:
                values = values.apply(lambda x:float(x.replace(',','.'))).values
            else:
                values = values.values
        da_name = xr.DataArray(data = values, dims = [f'datetime_{name}'], coords = {f'datetime_{name}':datetimes}, attrs = {'unit':unit})
        ds[name] = da_name
    return ds

In [ ]:
test_ds_clin = load_clinical_in_dataset('MF12')

In [ ]:
test = load_clinical(sub)

In [ ]:
for clin_name in ['Moyenne', 'Systolique', 'Diastolique', 'FC', 'Dextro (mmol/l)', 
                  'PPC', 'PIC', 'Pupille Gauche', 'Taille Pupille Gauche', 'Pupille Droite', 'Taille Pupille Droite',
                    'Glasgow, MSG', 'Glasgow, yeux', 'Glasgow, MIG', 'Glasgow, MSD',
                      'Glasgow, verbal', 'Glasgow, total', 'Glasgow, MID', 'SpO2', 'Mobilisation', 'Poids']:
    print(clin_name)
    dict_res = load_clinical(sub, name = clin_name, show = True, verbose = False)

## Biology function

In [ ]:
bio = load_icca_raw(sub, 'biology')

In [ ]:
def load_biology(sub, name = None, verbose = False, show = False):
    df = load_icca_raw(sub, 'biology')
    df['short_label'] = df['Parametre long'].apply(lambda x:x.split('.')[-1])
    df['unit'] = df['Parametre long'].apply(lambda x:x.split('.')[0].split(' ')[-1])
    if verbose:
        print(df['short_label'].unique().tolist())
    if name is None:
        return df
    else:
        df = df[df['short_label'] == name]
        df = df.sort_values(by = 'date_gmt')
        unit = df['unit'].unique()[0]
        df_sel = df[['date_gmt','Valeur num']].dropna()
        datetimes = df_sel['date_gmt'].values
        values = df_sel['Valeur num'].values
  
        if show:
            fig, ax = plt.subplots()
            ax.plot(datetimes, values)
            ax.scatter(datetimes, values, color = 'k')
            ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
            ax.set_ylabel(f'{name} {unit}')
            plt.show()

        return {'datetimes':datetimes,'values':values, 'unit':unit}

In [ ]:
def load_biology_in_dataset(sub):
    df_start = load_icca_raw(sub, 'biology')
    df_start['short_label'] = df_start['Parametre long'].apply(lambda x:x.split('.')[-1])
    df_start['unit'] = df_start['Parametre long'].apply(lambda x:x.split('.')[0].split(' ')[-1])
    names = df_start['short_label'].unique().tolist()
    ds = xr.Dataset()
    for name in names:
        df = df_start[df_start['short_label'] == name]
        df = df.sort_values(by = 'date_gmt')
        unit = df['unit'].unique()[0]
        df_sel = df[['date_gmt','Valeur num']].dropna()
        datetimes = df_sel['date_gmt'].values
        values = df_sel['Valeur num'].values
        da_name = xr.DataArray(data = values, dims = [f'datetime_{name}'], coords = {f'datetime_{name}':datetimes}, attrs = {'unit':unit})
        ds[name] = da_name
    return ds

In [ ]:
test_ds_bio = load_biology_in_dataset('MF12')

In [ ]:
for bio_name in ['pCO2 artériel','pH artériel','Urémie','pO2 artériel','Acide lactique',
                 'PNN','Potassium (mmol/l)','Sodium (mmol/l)','Magnésium','Troponine',
                 'CreatNum','Glucose']:
    bio_res = load_biology(sub, name = bio_name, verbose = True, show = True)

## Smart function

In [ ]:
def load_icca(sub, dtype, name, administration_type = None, use_human_weight = True, verbose = False, show = False):
    if dtype == 'treatment':
        res = load_treatment(sub, name, administration_type, use_human_weight, verbose, show)
    elif dtype == 'biology':
        res = load_biology(sub, name, verbose, show)
    elif dtype == 'clinical':
        res = load_clinical(sub, name, verbose, show)
    return res

In [ ]:
res_bio = load_icca(sub, dtype = 'biology', name = 'PNN', verbose = True, show =True)
res_tt = load_icca(sub, dtype = 'treatment', name = 'Norépinéphrine', administration_type = 'PSE', use_human_weight=True, verbose = True, show =True)
res_clin = load_icca(sub, dtype = 'clinical', name = 'Glasgow, total', verbose = True, show =True)

print(res_bio)
print(res_tt)
print(res_clin)

## Dvp PSE concentration * unit mapping function

In [ ]:
def get_icca_subs():
    icca_subs = [str(sub).split('/')[-1] for sub in icca_path.iterdir()]
    icca_subs = [sub for sub in icca_subs if not '.txt' in sub]
    return icca_subs

In [ ]:
def get_PSE_conc_unit_mapping(compute = False):
    if compute:
        icca_subs = get_icca_subs()
        dtype = 'treatment'
        concat = []
        for sub in icca_subs:
            df_sub = load_icca_raw(sub, dtype)
            df_sub['admin_type'] = df_sub['short Label'].apply(lambda x:'PSE' if 'PSE' in x else 'medication')
            keep_mask = (df_sub['admin_type'] == 'PSE') & (df_sub['Parametre court'] == 'Conc')
            df_sub = df_sub[keep_mask]
            res = df_sub.drop_duplicates(subset = ['short Label','value Number','Unite'])[['short Label','value Number','Unite']].reset_index(drop = True)
            res['sub'] = sub
            concat.append(res)
        res_all = pd.concat(concat)
        pse_possibilities = res_all[res_all['short Label'] != 'Adm. PSE']
        df_return = pse_possibilities.drop_duplicates(['short Label','value Number','Unite']).drop(columns = ['sub']).reset_index(drop = True).drop_duplicates(['value Number','Unite'])
        df_return['short_label'] = df_return['short Label'].apply(lambda x:x.split(':')[-1][1:])
    else:
        df_return = pd.read_excel(base_folder / 'icca_review' / 'PSE_conc_unit_mapping.xlsx', index_col = 0)
    return df_return

In [ ]:
mapping_conc_unit = get_PSE_conc_unit_mapping(compute = True)

In [ ]:
sub = 'RM6'
dtype = 'treatment'
df_sub = load_icca_raw(sub, dtype)

In [ ]:
df_sub['admin_type'] = df_sub['short Label'].apply(lambda x:'PSE' if 'PSE' in x else 'other')

In [ ]:
mask = (df_sub['admin_type'] == 'PSE') & (df_sub['Parametre court'] == 'Conc')
df_sub_pse = df_sub[mask].reset_index(drop = True)
for i, row in df_sub_pse.iterrows():
    df_sub_pse.loc[i,'PSE_Label'] = mapping_conc_unit.loc[(row['Unite'],row['value Number']),'short Label']


In [ ]:
df_sub_pse

## Load PSE Treatments

In [ ]:
def load_PSE_treatment(sub, name = None, use_human_weight = True, verbose = False, show = False):
    df = load_icca_raw(sub, 'treatment')
    df['administration_type'] = df['short Label'].apply(lambda x:'PSE' if 'PSE' in x else 'medication')
    df = df[df['administration_type'] == 'PSE'].reset_index(drop = True)
    assert df.shape[0] != 0, f'no PSE treatment available in subject {sub}'

    mapping_conc_unit = get_PSE_conc_unit_mapping(compute = False).set_index(['Unite','value Number'])

    for i, row in df.iterrows():
        if row['Parametre court'] == 'Conc':
            df.loc[i,'short_label'] = mapping_conc_unit.loc[(row['Unite'],row['value Number']),'short_label']
    for i, row in df.iterrows():
        if row['Parametre court'] == 'Débit adm':
            date_deb = row['date_gmt']
            if i-1 >= 0:
                if df.loc[i-1,'Parametre court'] == 'Conc' and df.loc[i-1,'date_gmt'] == date_deb:
                    df.loc[i,'short_label'] = df.loc[i-1,'short_label']
            if i+1 <= df.index[-1]:
                if df.loc[i+1,'Parametre court'] == 'Conc' and df.loc[i+1,'date_gmt'] == date_deb:
                    df.loc[i,'short_label'] = df.loc[i+1,'short_label']
    assert df['short_label'].isna().sum() == 0

    if verbose:
        print('possible names :', df['short_label'].unique().tolist())
    
    if name is None:
        return df
    else:
        assert name in df['short_label'].tolist(), f'name {name} not available'
        df = df[df['short_label'] == name].reset_index(drop = True)
        df = df.sort_values(by = 'date_gmt')
        unit = df[df['Parametre court'] == 'Conc']['Unite'].unique()[0].split('/')[0]
        t_unit = df[df['Parametre court'] == 'Débit adm']['Unite'].unique()[0].split('/')[1]
        unit = f'{unit}/{t_unit}'
        concentrations = df[df['Parametre court'] == 'Conc']['value Number'].values
        flows = df[df['Parametre court'] == 'Débit adm']['value Number'].values
        if np.unique(concentrations).size == 1: # if concentration do not vary
            concentrations = concentrations[0] # take the first concentration to compute doses
        else: # else check than same number of concentrations and flows values are available
            assert concentrations.size == flows.size, f'Not the same amount of contration and flow values (conc : {concentrations.size}, flow : {flows.size}, datetimes : {datetimes.size})'
        datetimes = df[df['Parametre court'] == 'Débit adm']['date_gmt'].to_numpy()
        doses = concentrations * flows
        if use_human_weight:
            clinical = load_icca_raw(sub, 'clinical')
            median_weight = clinical[clinical['Unite'] == 'kg']['Valeur'].apply(lambda x:float(x.replace(',','.'))).median()
            doses = doses / median_weight
            unit = f'{unit}/kg'
        if show:
            fig, ax = plt.subplots()
            ax.plot(datetimes, doses)
            ax.scatter(datetimes, doses, color = 'k')
            ax.set_xticks(ax.get_xticks(), ax.get_xticklabels(), rotation = 90)
            ax.set_ylabel(f'{name} - PSE ({unit})')
            plt.show()
        return {'datetimes':datetimes,'values':doses, 'unit':unit}

In [ ]:
def load_PSE_treatments_in_dataset(sub, use_human_weight = True):
    df = load_icca_raw(sub, 'treatment')
    df['administration_type'] = df['short Label'].apply(lambda x:'PSE' if 'PSE' in x else 'medication')
    df = df[df['administration_type'] == 'PSE'].reset_index(drop = True)
    assert df.shape[0] != 0, f'no PSE treatment available in subject {sub}'

    clinical = load_icca_raw(sub, 'clinical')
    median_weight = clinical[clinical['Unite'] == 'kg']['Valeur'].apply(lambda x:float(x.replace(',','.'))).median()

    mapping_conc_unit = get_PSE_conc_unit_mapping(compute = False).set_index(['Unite','value Number'])

    for i, row in df.iterrows():
        if row['Parametre court'] == 'Conc':
            df.loc[i,'short_label'] = mapping_conc_unit.loc[(row['Unite'],row['value Number']),'short_label']
    for i, row in df.iterrows():
        if row['Parametre court'] == 'Débit adm':
            date_deb = row['date_gmt']
            if i-1 >= 0:
                if df.loc[i-1,'Parametre court'] == 'Conc' and df.loc[i-1,'date_gmt'] == date_deb:
                    df.loc[i,'short_label'] = df.loc[i-1,'short_label']
            if i+1 <= df.index[-1]:
                if df.loc[i+1,'Parametre court'] == 'Conc' and df.loc[i+1,'date_gmt'] == date_deb:
                    df.loc[i,'short_label'] = df.loc[i+1,'short_label']
    assert df['short_label'].isna().sum() == 0

    names = df['short_label'].unique().tolist()
    ds = xr.Dataset()
    for name in names:
        df_name = df[df['short_label'] == name].reset_index(drop = True)
        df_name = df_name.sort_values(by = 'date_gmt')
        unit = df_name[df_name['Parametre court'] == 'Conc']['Unite'].unique()[0].split('/')[0]
        t_unit = df_name[df_name['Parametre court'] == 'Débit adm']['Unite'].unique()[0].split('/')[1]
        unit = f'{unit}/{t_unit}'
        concentrations = df_name[df_name['Parametre court'] == 'Conc']['value Number'].values
        flows = df_name[df_name['Parametre court'] == 'Débit adm']['value Number'].values
        if np.unique(concentrations).size == 1: # if concentration do not vary
            concentrations = concentrations[0] # take the first concentration to compute doses
        else: # else check than same number of concentrations and flows values are available
            assert concentrations.size == flows.size, f'Not the same amount of contration and flow values (conc : {concentrations.size}, flow : {flows.size}, datetimes : {datetimes.size})'
        datetimes = df_name[df_name['Parametre court'] == 'Débit adm']['date_gmt'].to_numpy()
        doses = concentrations * flows
        if use_human_weight:
            doses = doses / median_weight
            unit = f'{unit}/kg'
        da_name = xr.DataArray(data = doses, dims = [f'datetime_{name}'], coords = {f'datetime_{name}':datetimes}, attrs = {'unit':unit})
        ds[name] = da_name
    return ds

In [ ]:
test_ds = load_PSE_treatments_in_dataset('BM3')

In [ ]:
test_ds

In [ ]:
sub = 'MF12'
res = load_PSE_treatment(sub, name = 'Urapidil', verbose = True, show = True, use_human_weight=True)

In [ ]:
df2 = df.copy()
for i, row in df2.iterrows():
    if row['Parametre court'] == 'Conc':
        df2.loc[i,'short_label'] = mapping_conc_unit.loc[(row['Unite'],row['value Number']),'short_label']
for i, row in df2.iterrows():
    if row['Parametre court'] == 'Débit adm':
        date_deb = row['date_gmt']
        if df2.loc[i-1,'Parametre court'] == 'Conc' and df2.loc[i-1,'date_gmt'] == date_deb:
            df2.loc[i,'short_label'] = df2.loc[i-1,'short_label']
        elif df2.loc[i+1,'Parametre court'] == 'Conc' and df2.loc[i+1,'date_gmt'] == date_deb:
            df2.loc[i,'short_label'] = df2.loc[i+1,'short_label']
assert df2['short_label'].isna().sum() == 0

## Load treatments in dataset

In [ ]:
def load_treatments_in_dataset(sub, use_human_weight = True):
    df = load_icca_raw(sub, 'treatment')
    df['administration_type'] = df['short Label'].apply(lambda x:'PSE' if 'PSE' in x else 'medication')
    df = df[df['administration_type'] == 'PSE'].reset_index(drop = True)
    assert df.shape[0] != 0, f'no PSE treatment available in subject {sub}'

    clinical = load_icca_raw(sub, 'clinical')
    median_weight = clinical[clinical['Unite'] == 'kg']['Valeur'].apply(lambda x:float(x.replace(',','.'))).median()

    mapping_conc_unit = get_PSE_conc_unit_mapping(compute = False).set_index(['Unite','value Number'])

    for i, row in df.iterrows():
        if row['Parametre court'] == 'Conc':
            df.loc[i,'short_label'] = mapping_conc_unit.loc[(row['Unite'],row['value Number']),'short_label']
    for i, row in df.iterrows():
        if row['Parametre court'] == 'Débit adm':
            date_deb = row['date_gmt']
            if i-1 >= 0:
                if df.loc[i-1,'Parametre court'] == 'Conc' and df.loc[i-1,'date_gmt'] == date_deb:
                    df.loc[i,'short_label'] = df.loc[i-1,'short_label']
            if i+1 <= df.index[-1]:
                if df.loc[i+1,'Parametre court'] == 'Conc' and df.loc[i+1,'date_gmt'] == date_deb:
                    df.loc[i,'short_label'] = df.loc[i+1,'short_label']
    assert df['short_label'].isna().sum() == 0

    names = df['short_label'].unique().tolist()
    ds = xr.Dataset()
    for name in names:
        df_name = df[df['short_label'] == name].reset_index(drop = True)
        df_name = df_name.sort_values(by = 'date_gmt')
        unit = df_name[df_name['Parametre court'] == 'Conc']['Unite'].unique()[0].split('/')[0]
        t_unit = df_name[df_name['Parametre court'] == 'Débit adm']['Unite'].unique()[0].split('/')[1]
        unit = f'{unit}/{t_unit}'
        concentrations = df_name[df_name['Parametre court'] == 'Conc']['value Number'].values
        flows = df_name[df_name['Parametre court'] == 'Débit adm']['value Number'].values
        if np.unique(concentrations).size == 1: # if concentration do not vary
            concentrations = concentrations[0] # take the first concentration to compute doses
        else: # else check than same number of concentrations and flows values are available
            assert concentrations.size == flows.size, f'Not the same amount of contration and flow values (conc : {concentrations.size}, flow : {flows.size}, datetimes : {datetimes.size})'
        datetimes = df_name[df_name['Parametre court'] == 'Débit adm']['date_gmt'].to_numpy()
        doses = concentrations * flows
        if use_human_weight:
            doses = doses / median_weight
            unit = f'{unit}/kg'
        da_name = xr.DataArray(data = doses, dims = [f'datetime_{name}'], coords = {f'datetime_{name}':datetimes}, attrs = {'unit':unit})
        ds[name] = da_name
    return ds

In [ ]:
sub = 'MF12'
df = load_icca_raw(sub, 'treatment')
df['administration_type'] = df['short Label'].apply(lambda x:'PSE' if 'PSE' in x else 'medication')
# df = df[df['administration_type'] == 'PSE'].reset_index(drop = True)
df.value_counts(subset = ['intervention Id','short Label'])

In [ ]:
sub = 'NN7'
df = load_icca_raw(sub, 'treatment')
df['administration_type'] = df['short Label'].apply(lambda x:'PSE' if 'PSE' in x else 'medication')
# df = df[df['administration_type'] == 'PSE'].reset_index(drop = True)
df.value_counts(subset = ['intervention Id','short Label'])

In [ ]:
sub = 'MJ18'
df = load_icca_raw(sub, 'treatment')
df['administration_type'] = df['short Label'].apply(lambda x:'PSE' if 'PSE' in x else 'medication')
# df = df[df['administration_type'] == 'PSE'].reset_index(drop = True)
df.value_counts(subset = ['intervention Id','short Label'])

In [ ]:
from tqdm import tqdm

In [ ]:
def classify_function(x):
    if 'PSE' in x:
        if ':' in x:
            return 'PSE_labeled'
        else:
            return 'PSE_alone'
    elif 'médicaments' in x:
        if ':' in x:
            return 'medication_labeled'
        else:
            return 'medication_alone'
    else:
        return 'other'

icca_subs_for_fig_for_fig = get_icca_subs()
rows_abs = {}
rows_rel = {}
rows_param_court = {}
first_date_dict = {}
pb_subs = []
for sub in tqdm(icca_subs):
    try:
        df = load_icca_raw(sub, 'treatment')
    except:
        pb_subs.append(sub)
        continue
    first_date_dict[sub] = df['Date'][0]
    rows_param_court[sub] = df['Parametre court'].value_counts().to_dict()
    labelisation = df['short Label'].apply(classify_function)
    rows_abs[sub] = labelisation.value_counts().to_dict()
    rows_rel[sub] = labelisation.value_counts(normalize=True).to_dict()

In [ ]:
pb_subs

In [ ]:
test = pd.DataFrame(rows_abs).T[['PSE_labeled','PSE_alone','medication_labeled','medication_alone','other']]
test['total_lines'] = test.sum(axis = 1)
test = pd.concat([test, pd.DataFrame(rows_param_court).T[['Conc','Débit adm']]], axis = 1)
test = test.rename(columns = {'Conc':'N Conc','Débit adm':'N Flow'})
test['Equal_N_of_Conc_and_Flow'] = test['N Conc'] == test['N Flow']
test['First_Date_TT'] = first_date_dict.values()
test = test.sort_values(by = 'First_Date_TT')
# test.to_excel(base_folder / 'icca_review' / 'short_label_col_exploration.xlsx')
test

In [ ]:
test.sort_values(by = 'First_Date_TT')

In [ ]:
test.sort_values(by = 'First_Date_TT')

In [ ]:
a = ((test[['PSE_alone','medication_alone']].isna().all(axis = 1)) & (test['Equal_N_of_Conc_and_Flow']))
a.index[a].to_list()

In [ ]:
a = ((test[['PSE_alone']].isna().all(axis = 1)) & (test['Equal_N_of_Conc_and_Flow']))
a.index[a].to_list()

In [ ]:
test['Equal_N_of_Conc_and_Débit'].value_counts()

In [ ]:
(test[['PSE_labeled','medication_labeled']].sum(axis = 1) == test['total_lines']).sum()

In [ ]:
len(icca_subs)

In [ ]:
test2 = pd.DataFrame(rows_rel).T[['PSE_labeled','PSE_alone','medication_labeled','medication_alone','other']]
# test2.to_excel(base_folder / 'icca_review' / 'short_label_col_exploration_rel.xlsx')
test2

In [ ]:
test2.mean()

In [ ]:
pd.concat([test.notna().sum(), test.notna().sum() / len(icca_subs)], axis = 1).rename(columns = {0:'N',1 : 'Proportion'})

## Crack ID

In [ ]:
def get_admin_type(x):
    if 'Adm.' in x:
        if 'PSE' in x:
            return 'PSE'
        else:
            return 'medication'
    else:
        return 'other'

def load_tt(sub):
    df = load_icca_raw(sub, 'treatment')
    df['short_label'] = df['short Label'].apply(lambda x:x.split(':')[-1][1:])
    df['administration_type'] = df['short Label'].apply(get_admin_type)
    return df

In [ ]:
icca_subs = get_icca_subs()
all_tt = []
for sub in icca_subs:
    try:
        all_tt.append(load_tt(sub))
    except:
        print(sub)
        continue
all_tt = pd.concat(all_tt).reset_index(drop = True)
all_tt = all_tt[all_tt['administration_type'] != 'other']
all_tt.head(5)

In [ ]:
# all_tt_save = all_tt.reset_index(drop = True)
# all_tt_save = all_tt_save[all_tt_save.columns[(all_tt_save.isna().sum() != all_tt_save.isna().shape[0])]]
# remove_cols = ['DDN','long Label','Valeur','Valeur num','IPP']
# keep_cols = [col for col in all_tt_save.columns if not col in remove_cols]
# all_tt_save = all_tt_save[keep_cols]
# all_tt_save.to_csv(base_folder / 'icca_review' / 'all_treatments.csv')

In [ ]:
test.set_index('intervention Id')

In [ ]:
# keep = all_tt['short Label'].apply(lambda x:True if 'Adm.' in x else False)
keep = (all_tt['short Label'].apply(lambda x:True if 'Adm.' in x else False)) & (all_tt['short Label'].apply(lambda x:False if ':' in x else True))
# keep = (all_tt['short Label'].apply(lambda x:True if 'Adm.' in x else False)) & (all_tt['short Label'].apply(lambda x:True if ':' in x else False))
# all_tt2 = all_tt[keep].reset_index(drop = True)
# all_tt2['short_label'] = all_tt2['short Label'].apply(lambda x:x.split(':')[1][1:])
# all_tt2.drop_duplicates(subset = ['intervention Id','short_label'])[['intervention Id','short Label','short_label','administration_type']].reset_index(drop = True)#.to_excel(base_folder / 'icca_review' / 'intervention_id_to_short_label_only_labeled.xlsx')
all_tt[keep].drop_duplicates(subset = ['intervention Id','short Label'])[['intervention Id','short Label']]#.to_excel(base_folder / 'icca_review' / 'intervention_id_to_short_label_raw_tt_no_label_only.xlsx')
# all_tt[keep].drop_duplicates(subset = ['intervention Id','short Label'])[['intervention Id','short Label']]#.to_excel(base_folder / 'icca_review' / 'intervention_id_to_short_label_raw_tt_no_label_only.xlsx')

In [ ]:
keep = (all_tt['short Label'].apply(lambda x:False if ':' in x else True)) & (all_tt['administration_type'] == 'PSE')
interv_id_nude = all_tt[keep].drop_duplicates(subset = ['intervention Id','short Label'])[['intervention Id','short Label']]

res = {}
res_pooled = {}
smart_res = {}

for interv_id in interv_id_nude['intervention Id'].unique():
    res_id = {}
    df_interv_id = all_tt[all_tt['intervention Id'] == interv_id]
    admin_type = df_interv_id['administration_type'].unique()[0]
    print(f'Intervention ID n°{interv_id} is administered by this way : {admin_type}')
    if admin_type == 'medication':
        continue
    param_court = 'Conc' if admin_type == 'PSE' else 'Quantité'
    if param_court == 'Conc':
        df_interv_id = df_interv_id[df_interv_id['Parametre court'] == param_court]
    possible_units = df_interv_id[df_interv_id['Parametre court'] == param_court]['Unite'].unique()
    print(f'It could be administered with such units : {possible_units} ...')
    assert possible_units.size == 1, print(interv_id, df_interv_id[df_interv_id['Parametre court'] == param_court]['Unite'].unique())
    unit = possible_units[0]
    possible_names_pooled = []
    possible_values = df_interv_id['value Number'].unique()
    for possible_conc in possible_values:
        print(f'... at such concentrations {possible_conc} {unit}')
        possible_names = all_tt[(all_tt['value Number'] == possible_conc) & (all_tt['Unite'] == unit)]['short Label'].unique().tolist()
        res_id[f'{possible_conc} {unit}'] = possible_names
        possible_names_pooled.extend(possible_names)
        print(f'... such concentrations / units could correspond to these treatments : {possible_names}')
    res[interv_id] = res_id
    res_pooled[interv_id] = list(possible_names_pooled)
    smart_list = []
    for possible_name in list(set(possible_names_pooled)):
        if possible_names_pooled.count(possible_name) == possible_values.size:
            smart_res[interv_id] = smart_list + [possible_name]

    print()

In [ ]:
smart_res

In [ ]:
res

In [ ]:
all_tt[all_tt['intervention Id']==282295]['Patient'].value_counts()

In [ ]:
res[14845]

In [ ]:
keep = (all_tt['short Label'].apply(lambda x:False if ':' in x else True)) & (all_tt['administration_type'] == 'medication')
interv_id_nude = all_tt[keep].drop_duplicates(subset = ['intervention Id','short Label'])[['intervention Id','short Label']]

res = {}
res_pooled = {}

for interv_id in interv_id_nude['intervention Id'].unique():
    res_id = {}
    df_interv_id = all_tt[all_tt['intervention Id'] == interv_id]
    admin_type = df_interv_id['administration_type'].unique()[0]
    print(f'Intervention ID n°{interv_id} is administered by this way : {admin_type}')
    if admin_type == 'PSE':
        continue
    param_court = 'Conc' if admin_type == 'PSE' else 'Quantité'
    possible_units = df_interv_id[df_interv_id['Parametre court'] == param_court]['Unite'].unique()
    print(f'It could be administered with such units : {possible_units} ...')
    if possible_units .size > 1:
        print(f'{interv_id} is skipped because has several unit possiblities')
    # assert possible_units.size == 1, print(interv_id, df_interv_id[df_interv_id['Parametre court'] == param_court]['Unite'].unique())
    unit = possible_units[0]
    possible_names_pooled = []
    for possible_conc in df_interv_id['value Number'].unique():
        print(f'... at such concentrations {possible_conc} {unit}')
        possible_names = all_tt[(all_tt['value Number'] == possible_conc) & (all_tt['Unite'] == unit)]['short Label'].unique().tolist()
        res_id[f'{possible_conc} {unit}'] = possible_names
        possible_names_pooled.extend(possible_names)
        print(f'... such concentrations / units could correspond to these treatments : {possible_names}')
    res[interv_id] = res_id
    res_pooled[interv_id] = list(set(possible_names_pooled))
    print()

In [ ]:
res

# CPP opt

In [ ]:
def clean_abp_icp(sig, threshs = (0,100)):
    med = np.nanmedian(sig)
    mask = (sig < threshs[0]) | (sig > threshs[1])
    inds_cleaning = np.where(mask)[0]
    clean_sig = sig.copy()
    clean_sig[inds_cleaning] = med
    return clean_sig

In [ ]:
np.array(get_patient_list(['ICP','ABP'], threshold_duration_mins=60*24))

In [ ]:
# sub = 'MF12'
# sub = 'GA9'
sub = 'BM3'


cns_reader = CnsReader(data_path / sub)
get_patient_dates(sub)

In [ ]:
# t0 = '2022-09-09T18:25' # MF12
# t1 = '2022-09-15T06:25'

# t0 = '2021-08-25T23:25' # GA9
# t1 = '2021-08-31T10:25'

# t0 = '2021-02-28T00' # BM3
# t1 = '2021-02-28T15'

savefolder = base_folder / 'figures' / 'CPPopt'
subs_saved = [str(i).split('/')[-1].split('.')[0] for i in savefolder.iterdir()]
continue_subs = subs_saved + ['P39', 'P20','P14','P1','P55']

for sub in get_patient_list(['ICP','ABP'], threshold_duration_mins=60*24):
    
    if sub in continue_subs:
        continue
    cns_reader = CnsReader(data_path / sub)
    t0, t1 = get_patient_dates(sub)

    stream_names = ['ICP_Mean','ABP_Mean']
    srate = np.max([cns_reader.streams[stream_name].sample_rate for stream_name in stream_names])
    srate = np.round(srate)

    ds = cns_reader.export_to_xarray(stream_names=stream_names, start = t0, stop = t1, resample = True, sample_rate=srate)
    raw_abp = ds['ABP_Mean'].values
    raw_icp = ds['ICP_Mean'].values
    n_secs_roll_win = 60
    # raw_abp = pd.Series(raw_abp).rolling(int(srate * n_secs_roll_win)).median().values
    # raw_icp = pd.Series(raw_icp).rolling(int(srate * n_secs_roll_win)).median().values
    raw_abp = clean_abp_icp(raw_abp, threshs=(0, 100))
    raw_icp = clean_abp_icp(raw_icp, threshs=(0, 60))
    cpp = raw_abp - raw_icp
    times_icp_abp = ds['times'].values

    da_prx = prx_job.get(sub)['prx']
    prx = da_prx.loc[t0:t1].values
    times_prx = da_prx.loc[t0:t1]['date'].values

    inds_sel_abp_icp = np.searchsorted(times_icp_abp, times_prx) - 1

    fig, axs = plt.subplots(nrows = 2, figsize = (8,6), constrained_layout = True)
    ax = axs[0]
    ax.plot(times_prx, raw_abp[inds_sel_abp_icp], label = 'abp')
    ax.plot(times_prx, raw_icp[inds_sel_abp_icp], label = 'icp')
    ax.plot(times_prx, cpp[inds_sel_abp_icp], label = 'cpp')
    ax.set_title(sub)
    ax.set_ylim(-5, 110)
    ax.legend()
    ax.set_ylabel('mmHg')
    ax2 = ax.twinx()
    ax2.plot(times_prx, prx, color = 'k', lw = 2, ls = '--', alpha = 0.5)
    ax2.set_ylim(-1,1)
    ax2.set_ylabel('PRx')

    ax = axs[1]
    ax.scatter(cpp[inds_sel_abp_icp], prx, color = 'k', alpha = 0.1)
    ax.set_ylim(-1, 1)
    ax.set_ylabel('PRx')
    ax.set_xlabel('CPP')
    ax.set_xlim(0, 150)
    ax.axhline(0.3, color = 'r')
    ax.set_title('U-shape curve ?')

    fig.savefig(savefolder / f'{sub}.png', dpi = 200, bbox_inches = 'tight')
    plt.close(fig)

# Cardio Resp Synchro

In [ ]:
get_patient_dates(sub)

In [ ]:
np.array(get_patient_list(['CO2','ECG_II']))

In [ ]:
sub = 'MF12'

In [ ]:
cns_reader = CnsReader(data_path / sub)

In [ ]:
meta = load_overview_data().set_index('Patient').loc[sub,['Start_Date','Stop_Date']]
meta

In [ ]:
t0 = np.datetime64('2022-09-10T14')
t1 = t0 + np.timedelta64(120, 'm')

In [ ]:
co2_stream = cns_reader.streams['CO2']
raw_co2, dates_co2 = co2_stream.get_data(sel = slice(t0,t1), with_times = True, apply_gain = True)
assert raw_co2.size != 0
co2, resp_cycles = physio.compute_respiration(raw_co2, co2_stream.sample_rate, parameter_preset='human_co2')
resp_cycles['inspi_date'] = dates_co2[resp_cycles['inspi_index']]
co2_times = np.arange(raw_co2.size) / co2_stream.sample_rate

In [ ]:
ecg_stream = cns_reader.streams['ECG_II']
raw_ecg, dates_ecg = ecg_stream.get_data(sel = slice(t0,t1), with_times = True, apply_gain = True)
assert raw_ecg.size != 0
ecg, ecg_peaks = physio.compute_ecg(raw_ecg, ecg_stream.sample_rate, parameter_preset='human_ecg')
ecg_times = np.arange(raw_ecg.size) / ecg_stream.sample_rate

In [ ]:
inspi_ratio = resp_cycles['cycle_ratio'].median()

cycle_times = resp_cycles[['inspi_time', 'expi_time', 'next_inspi_time']].values

rpeak_phase = physio.time_to_cycle(ecg_peaks['peak_time'].values, cycle_times, segment_ratios=[inspi_ratio])


count, bins = np.histogram(rpeak_phase % 1, bins=np.linspace(0, 1, 101))

fig, axs = plt.subplots(nrows=2, sharex=True, figsize=(8, 6))
ax = axs[0]
ax.scatter(rpeak_phase % 1, np.floor(rpeak_phase), s=0.1)
ax.set_ylabel('resp cycle')

ax = axs[1]
ax.bar(bins[:-1], count, width=bins[1] - bins[0], align='edge')
ax.set_xlabel('resp phase')
ax.set_ylabel('count R peaks')

for ax in axs:
    ax.axvline(0, color='g', label='inspiration', alpha=.6)
    ax.axvline(inspi_ratio, color='r', label='expiration', alpha=.6)
    ax.axvline(1, color='g', label='next_inspiration', alpha=.6)
ax.legend()
ax.set_xlim(-0.01, 1.01)

In [ ]:
ecg_peaks['resp_phase'] = rpeak_phase % 1
ecg_peaks['resp_cycle'] = np.floor(rpeak_phase)
ecg_peaks['peak_date'] = dates_ecg[ecg_peaks['peak_index']]

In [ ]:
%matplotlib inline

In [ ]:
import ghibtools as gh

In [ ]:
bins = np.arange(0,1,0.05)
fig, ax = plt.subplots()

start = '2022-09-10T14:00'
stop = '2022-09-10T14:05'
mask = (ecg_peaks['peak_date'] > np.datetime64(start)) & (ecg_peaks['peak_date'] <= np.datetime64(stop))
count, bins = np.histogram(ecg_peaks[mask]['resp_phase'], bins)
count = count / np.sum(count)
ax.bar(bins[:-1], count, width = np.median(np.diff(bins)), edgecolor = 'k')
plt.show()

In [ ]:
step = 30 # sesc
win_duration_mins = 3
start_compute = np.datetime64("2022-09-10T14:00")
stop_compute = np.datetime64("2022-09-10T16:00")
starts = np.arange(start_compute, stop_compute, np.timedelta64(step, "s"))
stops = starts + np.timedelta64(win_duration_mins, "m")

In [ ]:
bins = np.arange(0,1,0.05)
crpss = np.zeros((starts.size))
Ns = []
count_matrix = np.zeros((bins.size - 1, starts.size))
for i, start, stop in zip(range(starts.size), starts, stops):
    if stop < stop_compute:
        mask = (ecg_peaks['peak_date'] > start) & (ecg_peaks['peak_date'] <= stop)
        local_peaks = ecg_peaks[mask]
        Ns.append(local_peaks.shape[0])
        count, bins = np.histogram(local_peaks['resp_phase'], bins)
        count_matrix[:,i] = count
        count = count / np.sum(count)
        crpss[i] = Modulation_Index(count)
    else:
        crpss[i] = np.nan
        count_matrix[:,i] = np.nan

In [ ]:
fig, ax = plt.subplots()
ax.pcolormesh(starts, bins[:-1], count_matrix)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.plot(starts, crpss)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))
ax.scatter(ecg_peaks['peak_date'], ecg_peaks['resp_phase'], s=0.5)
ax.set_xlabel('date')
ax.set_ylabel('resp phase')
ax.set_xlim(np.nanmin(ecg_peaks['peak_date']), np.nanmax(ecg_peaks['peak_date']))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(25, 8))
# ax.scatter(rpeak_phase % 1, np.floor(rpeak_phase), s=0.1)
x = np.floor(rpeak_phase)
y = rpeak_phase % 1
ax.scatter(x, y, s=0.5)
ax.set_xlabel('resp cycle')
ax.set_ylabel('resp phase')
ax.set_xlim(np.nanmin(x), np.nanmax(x))
plt.show()

In [ ]:
from pycns import get_viewer
from custom_view import *

In [ ]:
%matplotlib widget

In [ ]:
resp_cycles_all = detect_resp_job.get(sub).to_dataframe()
r_peaks_all = detect_ecg_job.get(sub).to_dataframe()
rsa_cycles_all = rsa_job.get(sub).to_dataframe()

rate_bins_resp = np.arange(5, 30, 0.5)
step_bins_ecg = 1
eeg_chan = 'F3'

ext_plots = {
            # ABP_Detections(cns_reader.streams['ABP'], abp_features),
            # ICP_Detections(cns_reader.streams['ICP'], icp_features),
            
            # Wavelet_Power(cns_reader.streams['EEG'], eeg_chan, f_start = 4, f_stop = 8, n_steps = 30, n_cycles = 15, scaling = None),
            # Bipolar(eeg_stream, 'ECoGA4', 'ECoGA3', lowcut = 0.0008, highcut = 0.5, down_sampling_factor = 100),
            # AC_Envelope(eeg_stream, eeg_chan, highcut_amp=0.01),
            # Spreading_depol_bipol(cns_reader.streams['EEG'], down_sampling_factor = 10),
            # Depol(eeg_stream, eeg_chan, min_amp_mv = 15, max_amp_mv = None, min_duration_sec = 60, reverse_sig = False),
            # Spectrogram_bio(co2_stream, wsize = 300, lf = 0.1, hf = 0.4, overlap_prct = 0.75),
            # Spectrogram_eeg(cns_reader.streams['EEG'], eeg_chan, wsize = 10, lf = 1, hf = 20),
            'resp':Respi_Rate(resp_cycles_all, resp_wsize_in_mins = 3, ratio_sat = 4, rate_bins_resp=rate_bins_resp),
            'heat':Heart_Rate(r_peaks_all, step_bins_ecg = step_bins_ecg, hrv_wsize_in_mins = 3, ratio_sat = 3, plot_type = '2d'),
            'RSA':RSA(rsa_cycles_all, win_cycles_smooth = 10, n_mads_cleaning = 5), 
            # Pulse_Pressure(abp_features, n_mads_cleaning = 2),
            # Traube_Herring(ds['ABP'].values, ds['CO2'].values, srate_traube, ds['times'].values, wsize = 60), 
}

# w = get_viewer(cns_reader, stream_names=['ECG_II','CO2'], ext_plots=ext_plots)
w = get_viewer(cns_reader, stream_names=['CO2'], ext_plots=ext_plots)
w

In [ ]:
bins = np.linspace(-3, 3, 100)


count, bins = physio.crosscorrelogram(ecg_peaks['peak_time'].values,
                               resp_cycles['expi_time'].values,
                               bins=bins)

fig, axs = plt.subplots(nrows=2, sharex=True)
ax = axs[0]
ax.bar(bins[:-1], count, align='edge', width=bins[1] - bins[0])
ax.set_xlabel('time lag')
ax.set_ylabel('count')
ax.axvline(0, color='r', label='expiration', alpha=.6)
ax.legend()


ax = axs[1]
count, bins = physio.crosscorrelogram(ecg_peaks['peak_time'].values,
                               resp_cycles['inspi_time'].values,
                               bins=bins)
ax.bar(bins[:-1], count, align='edge', width=bins[1] - bins[0])
ax.set_xlabel('time lag')
ax.set_ylabel('count')
ax.axvline(0, color='g', label='inspiration', alpha=.6)
ax.legend()


plt.show()

# STATS PHYSIO VS CLINIQUE

In [ ]:
overview = load_overview_data().set_index('Patient')

subs = get_patient_list(['CO2','ECG_II','ICP'])
error_subs = ['P61','PatientData_1647020686738207']
subs = [sub for sub in subs if not sub in error_subs]

rows = []
for sub in subs:
    all_rsa = rsa_job.get(sub).to_dataframe()
    all_psi = psi_job.get(sub)['psi']
    all_resp = detect_resp_job.get(sub).to_dataframe()
    all_icp = detect_icp_job.get(sub).to_dataframe()
    all_abp = detect_abp_job.get(sub).to_dataframe()

    gcs_sortie = overview.loc[sub, 'GCS_sortie']
    mrs_sortie = overview.loc[sub, 'mRS_sortie']
    mrs_1_mois = overview.loc[sub, 'mRS_1mois']
    mrs_3_mois = overview.loc[sub, 'mRS_3mois']
    mrs_6_mois = overview.loc[sub, 'mRS_6mois']

    # rsa = np.mean(all_rsa['decay_amplitude'])
    # psi = np.mean(all_psi.values)

    # rsa = np.median(all_rsa['decay_amplitude'])
    # psi = np.median(all_psi.values)
    
    med_resp, mad_resp = physio.compute_median_mad(all_resp['cycle_freq'])
    icp = np.median(all_icp['amplitude_at_peak'].values)
    icp = np.nan if icp > 100 else icp
    abp = np.median(all_abp['amplitude_at_peak'].values)
    abp = np.nan if abp > 100 else abp

    rsa = np.median(all_rsa['rsa_amplitude_clean'][-1000:])
    psi = np.median(all_psi.values[-1000:])

    # rsa = np.median(all_rsa['decay_amplitude'][-10:]) - np.median(all_rsa['decay_amplitude'][:10])
    # psi = np.median(all_psi.values[-10:]) - np.median(all_psi.values[:10])

    # rsa = np.median(all_rsa['decay_amplitude'][-10:]) - np.median(all_rsa['decay_amplitude'])
    # psi = np.median(all_psi.values[-10:]) - np.median(all_psi.values)

    row = [sub, gcs_sortie, mrs_sortie, mrs_1_mois,  mrs_3_mois, mrs_6_mois, rsa, psi, mad_resp, icp, abp]
    rows.append(row)
df = pd.DataFrame(rows, columns = ['patient','gcs_sortie','mrs_sortie','mrs_1_mois', 'mrs_3_mois', 'mrs_6_mois', 'rsa','psi','mad_resp','icp','abp'])

In [ ]:
df_clean = df.dropna()

In [ ]:
%matplotlib inline

In [ ]:
# col_sel = ['gcs_sortie','mrs_sortie','rsa','psi','mad_resp','icp','abp']
col_sel = ['gcs_sortie','mrs_sortie','rsa','psi','mad_resp']
# col_sel = ['psi','mad_resp']
# sns.pairplot(data = df[col_sel])
sns.pairplot(data = df[df.columns[df.dtypes != 'object']])

In [ ]:
stats_quantitative(df = df[['psi','icp']].dropna(), xlabel = 'psi', ylabel = 'icp')

In [ ]:
vars = ['gcs_sortie','mrs_sortie','mrs_1_mois', 'mrs_3_mois', 'mrs_6_mois', 'rsa','psi']

nrows = len(vars)
ncols = len(vars)

fig, axs = plt.subplots(nrows,ncols, figsize = (25,20), constrained_layout = True)
for r, x in enumerate(vars):
    for c, y in enumerate(vars):
        ax = axs[r,c]
        df_compute = df[[x,y]].dropna()
        stats_quantitative(df = df_compute, xlabel = x, ylabel = y, ax=ax)

# FIGS Poster Neurotrauma

In [ ]:
sub = 'NN7'
cns_reader = CnsReader(data_path / sub)
icp_stream = cns_reader.streams['ICP']
srate_icp = icp_stream.sample_rate

In [ ]:
save_folder = base_folder / 'figures' / 'icp_co2' / 'figs_poster'
fontsizes_ticks = 20
fontsizes_titles = 25
figsize_methods = (8,5)

In [ ]:
get_patient_dates(sub)

## Introduction

## Methods

## Population data

In [ ]:
analyzed_subs = get_nory_sub_keys()
N_analyzed_subs = len(analyzed_subs)
meta = pd.read_excel(base_folder / 'overview_data_pycns.xlsx').set_index('Patient').loc[analyzed_subs,'motif'].value_counts()
meta

## Params for next figures

In [ ]:
ylim_icp_time = (0,50)
ylim_icp_freq = (0,25)

### Time domain high compliance ICP

In [ ]:
t0 = '2021-06-30T14:00:00'
t1 = '2021-06-30T14:05:00'

In [ ]:
srate = 200
ds = cns_reader.export_to_xarray(['ICP','CO2'], start=t0, stop=t1, resample=True, sample_rate=srate)
times = np.arange(ds['times'].size) / srate
raw_co2 = ds['CO2'].values
raw_icp = ds['ICP'].values
raw_co2 = (raw_co2 - np.mean(raw_co2)) / np.std(raw_co2)
raw_co2 = raw_co2 * np.std(raw_icp) + np.mean(raw_icp)

In [ ]:
mask_t = times < 10
fig, ax = plt.subplots(figsize = figsize_methods)
ax.plot(times[mask_t], raw_icp[mask_t], color = 'k', label = 'Intracranial Pressure')
ax.set_xlabel('Time (sec)', fontsize = fontsizes_ticks)
ax.set_ylabel('Intracranial Pressure (mmHg)', fontsize = fontsizes_ticks)
ax.set_ylim(ylim_icp_time[0],ylim_icp_time[1])
ax.set_title('B)  ICP : High Compliance', fontsize = fontsizes_titles, fontstyle='italic', fontweight='bold')
fig.savefig(save_folder / 'icp_time_high_compliance.png', dpi = 500, bbox_inches = 'tight')

### Frequency domain high compliance ICP

In [ ]:
nperseg = int(srate * 60)
f, Pxx = scipy.signal.welch(raw_icp, srate, nperseg = nperseg, nfft = nperseg)
mask_f = (f < 2) & (f > 0.15)

x = f[mask_f]
y = np.sqrt(Pxx[mask_f])
fig, ax = plt.subplots(figsize = figsize_methods)
ax.plot(x, y, color = 'k')
ax.set_xlabel('Frequency (Hz)', fontsize = fontsizes_ticks)
ax.set_ylabel('Intracranial Pressure (mmHg)', fontsize = fontsizes_ticks)
ax.set_ylim(ylim_icp_freq[0],ylim_icp_freq[1])
ax.axvspan(0.3,0.36, color = 'b', alpha = 0.2, label = 'Respi')
ax.axvspan(1.15,1.35, color = 'r', alpha = 0.2, label = 'Heart')
ax.legend(loc = 2)
ax.set_title('B)  ICP : High Compliance', fontsize = fontsizes_titles, fontstyle='italic', fontweight='bold')
fig.savefig(save_folder / 'icp_freq_high_compliance.png', dpi = 500, bbox_inches = 'tight')
plt.show()

### Time domain low compliance ICP

In [ ]:
t0 = '2021-06-30T15:40:00'
t1 = '2021-06-30T15:45:00'

In [ ]:
srate = 200
ds = cns_reader.export_to_xarray(['ICP','CO2'], start=t0, stop=t1, resample=True, sample_rate=srate)
times = np.arange(ds['times'].size) / srate
raw_co2 = ds['CO2'].values
raw_icp = ds['ICP'].values
raw_co2 = (raw_co2 - np.mean(raw_co2)) / np.std(raw_co2)
raw_co2 = raw_co2 * np.std(raw_icp) + np.mean(raw_icp)

In [ ]:
mask_t = times < 10
fig, ax = plt.subplots(figsize = figsize_methods)
ax.plot(times[mask_t], raw_icp[mask_t], color = 'k', label = 'Intracranial Pressure')
ax.set_xlabel('Time (sec)', fontsize = fontsizes_ticks)
ax.set_ylabel('Intracranial Pressure (mmHg)', fontsize = fontsizes_ticks)
ax.set_ylim(ylim_icp_time[0],ylim_icp_time[1])
ax.set_title('C)  ICP : Low Compliance', fontsize = fontsizes_titles, fontstyle='italic', fontweight='bold')
fig.savefig(save_folder / 'icp_time_low_compliance.png', dpi = 500, bbox_inches = 'tight')

### Frequency domain low compliance ICP

In [ ]:
nperseg = int(srate * 60)
f, Pxx = scipy.signal.welch(raw_icp, srate, nperseg = nperseg, nfft = nperseg)
mask_f = (f < 2) & (f > 0.15)

fig, ax = plt.subplots(figsize = figsize_methods)
ax.plot(f[mask_f], np.sqrt(Pxx[mask_f]), color = 'k')
ax.set_xlabel('Frequency (Hz)', fontsize = fontsizes_ticks)
ax.set_ylabel('Intracrnial Pressure (mmHg)', fontsize = fontsizes_ticks)
ax.set_ylim(ylim_icp_freq[0],ylim_icp_freq[1])
ax.axvspan(0.3,0.36, color = 'b', alpha = 0.2, label = 'Respi')
ax.axvspan(1.1,1.35, color = 'r', alpha = 0.2, label = 'Heart')
ax.set_title('C)  ICP : Low Compliance', fontsize = fontsizes_titles, fontstyle='italic', fontweight='bold')
ax.legend(loc = 2)
fig.savefig(save_folder / 'icp_freq_low_compliance.png', dpi = 500, bbox_inches = 'tight')
plt.show()

### ICP whole signal

In [ ]:
t0 = '2021-06-30T14:00:00'
t1 = '2021-06-30T15:45:00'

In [ ]:
srate = 200
ds = cns_reader.export_to_xarray(['ICP','CO2'], start=t0, stop=t1, resample=True, sample_rate=srate)
times = np.arange(ds['times'].size) / srate
raw_co2 = ds['CO2'].values
raw_icp = ds['ICP'].values
raw_co2 = (raw_co2 - np.mean(raw_co2)) / np.std(raw_co2)
raw_co2 = raw_co2 * np.std(raw_icp) + np.mean(raw_icp)

In [ ]:
icp_plot = iirfilt(raw_icp, srate, highcut = 0.1)
fig, ax = plt.subplots(figsize = figsize_methods)
ax.plot(times/60, icp_plot, color = 'k')
ax.set_ylim(ylim_icp_time[0],ylim_icp_time[1])
ax.set_xlabel('Time (mins)', fontsize = fontsizes_ticks)
ax.set_ylabel('Intracranial Pressure (mmHg)', fontsize = fontsizes_ticks)
ax.set_title('A)  Acute ICP rise', fontsize = fontsizes_titles, fontstyle='italic', fontweight='bold')
fig.savefig(save_folder / 'icp_whole_signal.png', dpi = 500, bbox_inches = 'tight')

### Spectrogram whole signal

In [ ]:
f, t, Sxx = scipy.signal.spectrogram(raw_icp, srate, nperseg = int(srate * 60))
mask_f = (f > 0.1) & (f < 1.7)
q = 0.05
Sxx_masked = np.sqrt(Sxx[mask_f,:])
da = xr.DataArray(data = Sxx_masked, dims = ['freq','time'], coords = {'freq':f[mask_f], 'time':t})
freq_resp = da.loc[0.15:0.6,:].idxmax('freq').values
freq_heart = da.loc[0.8:2.2,:].idxmax('freq').values
max_resp = da.loc[0.15:0.6,:].max('freq').values
max_heart = da.loc[0.8:2.2,:].max('freq').values
vmin = np.quantile(Sxx_masked, q)
vmax = np.quantile(Sxx_masked, 1-q)
fig, ax = plt.subplots(figsize = figsize_methods)
ax.pcolormesh(t/60, f[mask_f], Sxx_masked, vmin=vmin, vmax=vmax)
ax.plot(t/60, freq_resp, color = 'b', lw = 2, label = 'resp frequency')
ax.plot(t/60, freq_heart, color = 'r', lw = 2, label = 'heart frequency')
ax.set_ylabel('Frequency (Hz)', fontsize = fontsizes_ticks)
ax.set_xlabel('Time (mins)', fontsize = fontsizes_ticks)
ax.legend(loc = 'upper center', ncols = 2, fontsize = 17)
ax.set_title('A)  ICP : Spectrogram', fontsize = fontsizes_titles, fontstyle='italic', fontweight='bold')
fig.savefig(save_folder / 'icp_spectrogram.png', dpi = 500, bbox_inches = 'tight')
plt.show()

### Spectral peaks amplitude

In [ ]:
fig, ax = plt.subplots()
ax.plot(t/60, max_heart, color = 'r', lw = 2, label = 'heart peak size')
ax2 = ax.twinx()
ax2.plot(t/60, max_resp, color = 'b', lw = 2, label = 'resp peak size')
ax2.set_ylabel('Resp in ICP (mmHg)', fontsize = fontsizes_ticks)
ax.set_xlabel('Time (min)', fontsize = fontsizes_ticks)
ax.set_ylabel('Heart in ICP (mmHg)', fontsize = fontsizes_ticks)
ax.set_title('Heart & Resp spectral amplitude in ICP', fontsize = fontsizes_titles)
ax.legend(loc = 'upper left')
ax2.legend(loc = 'lower right')
fig.savefig(save_folder / 'icp_heart_resp_peak_amplitude.png', dpi = 500, bbox_inches = 'tight')
plt.show()

### Correlation example

In [ ]:
x = raw_icp[np.searchsorted(times, t)]
y = max_heart
res_corr = scipy.stats.spearmanr(x,y)
res_reg = scipy.stats.linregress(x,y)
ps = '*' if res_corr.pvalue < 0.05 else 'ns'

fig, ax = plt.subplots()

ax.scatter(x , y, color = 'k')
ax.plot(x, res_reg.intercept + res_reg.slope*x, color = 'r', lw = 2)
ax.set_title(f'Correlation - R : {round(res_corr.statistic, 3)} (p-value : < 0.001)', fontsize = fontsizes_titles)
ax.set_xlabel('ICP (mmHg)', fontsize = fontsizes_ticks)
ax.set_ylabel('Heart in ICP (mmHg)', fontsize = fontsizes_ticks)
fig.savefig(save_folder / 'icp_heart_correlation_example.png', dpi = 500, bbox_inches = 'tight')


## Results

### Example traces

In [ ]:
fig, axs = plt.subplots(nrows = 3, figsize = (9,11), constrained_layout = True)

icp_plot = iirfilt(raw_icp, srate, highcut = 0.1)

ax = axs[0]
ax.plot(times/60, icp_plot, color = 'k')
ax.set_ylabel('ICP (mmHg)', fontsize = fontsizes_ticks)
ax.set_title('A)  Intracranial pressure (acute rise)', fontsize = fontsizes_titles, fontstyle='italic', fontweight='bold')
ax.set_xlabel('Time (mins)', fontsize = fontsizes_ticks)

ax = axs[1]
ax.plot(t/60, max_heart, color = 'r', lw = 2, label = 'Heart spectral peak size')
ax2 = ax.twinx()
ax2.plot(t/60, max_resp, color = 'b', lw = 2, label = 'Resp spectral peak size')
ax2.set_ylabel('Resp in ICP (mmHg)', fontsize = fontsizes_ticks)
ax.set_xlabel('Time (mins)', fontsize = fontsizes_ticks)
ax.set_ylabel('Heart in ICP (mmHg)', fontsize = fontsizes_ticks)
ax.set_title('B)  Heart and Resp spectral amplitude in ICP', fontsize = fontsizes_titles, fontstyle='italic', fontweight='bold')
ax.legend(loc = 'upper left', fontsize = 17)
ax2.legend(loc = 'lower right', fontsize = 17)

ax = axs[2]
ax2 = ax.twinx()
for y, c, label in zip([max_heart, max_resp],['r','b'],['Heart','Resp']):
    x = raw_icp[np.searchsorted(times, t)]
    res_corr = scipy.stats.spearmanr(x,y)
    res_reg = scipy.stats.linregress(x,y)
    ps = '*' if res_corr.pvalue < 0.05 else 'ns'

    if label == 'Heart':
        ax.scatter(x , y, color = c, alpha = 0.8)
        ax.plot(x, res_reg.intercept + res_reg.slope*x, color = c, lw = 2, ls = '-', label = f'{label} - R : {round(res_corr.statistic, 3)} ({ps})')
        ax.legend(loc = 'upper left', fontsize = 17)
    else:
        
        ax2.scatter(x , y, color = c, alpha = 0.8)
        ax2.plot(x, res_reg.intercept + res_reg.slope*x, color = c, lw = 2, ls = '-', label = f'{label} - R : {round(res_corr.statistic, 3)} ({ps})')
        ax2.legend(loc = 'lower right', fontsize = 17)
        
ax.set_ylabel('Heart in ICP (mmHg)', fontsize = fontsizes_ticks)
ax2.set_ylabel('Resp in ICP (mmHg)', fontsize = fontsizes_ticks)
ax.set_xlabel('ICP (mmHg)', fontsize = fontsizes_ticks)
ax.set_title('C)  Correlations with ICP', fontsize = fontsizes_titles, fontstyle='italic', fontweight='bold')


fig.savefig(save_folder / 'res_example_traces.png', dpi = 500, bbox_inches = 'tight')
plt.show()

### Correlations

In [ ]:
from ICP_to_resp_analysis import get_nory_sub_keys, erp_like_fig_sub_event_job

subs = get_nory_sub_keys()
concat = []
bug_subs = []
for sub in subs:
    try:
        df_sub = erp_like_fig_sub_event_job.get(sub).to_dataframe()
    except:
        bug_subs.append(sub)
    else:
        concat.append(df_sub)
# print(bug_subs)
data = pd.concat(concat)
# print(data)
data_without_nan = data.dropna()

In [ ]:
metrics_selection = ['heart_in_icp_spectrum','resp_in_icp_spectrum']
metrics_name = ['Heart Spectral Peak Size','Resp Spectral Peak Size']
N = data_without_nan['Patient'].unique().size
for target_corr_label, clean_label, ylabel in zip(['icp','psi'], ['ICP','Pulse Shape Index'], ['ICP','PSI']):
        data_clean = data_without_nan[data_without_nan['Target_Metric'] == target_corr_label]
        data_clean = data_clean[(data_clean['Metric_to_corr'] != target_corr_label) & (data_clean['Metric_to_corr'].isin(metrics_selection))].reset_index(drop = True)

        fig, ax = plt.subplots(figsize = (8,4), constrained_layout = True)
        order = data_clean.groupby('Metric_to_corr')['R'].mean().sort_values(ascending = False).index
        sns.barplot(data = data_clean,
                    x = 'Metric_to_corr', 
                    y = 'R',
                    ax=ax,
                    order = order,
                    palette = ['r','b']
                    )
        ax.set_title(f'A)  Correlations with {clean_label}', fontsize= fontsizes_titles, fontstyle='italic', fontweight='bold')
        ax.set_ylim(-1, 1)
        ax.set_ylabel(f'Correlation coefficients', fontsize = fontsizes_ticks)
        ax.set_xticks(ax.get_xticks(), labels = metrics_name, rotation = 0, fontsize = fontsizes_ticks)
        ax.set_xlabel('')
        fig.savefig(save_folder  / f'correlation_coef_with_{ylabel}.png', dpi = 500, bbox_inches = 'tight')
        plt.show()

### Significance

In [ ]:
for target_corr_label, clean_label, ylabel in zip(['icp','psi'], ['ICP','Pulse Shape Index'], ['ICP','PSI']):
    data_clean = data_without_nan[data_without_nan['Target_Metric'] == target_corr_label]
    data_clean = data_clean[(data_clean['Metric_to_corr'] != target_corr_label) & (data_clean['Metric_to_corr'].isin(metrics_selection))].reset_index(drop = True)

    fig, ax = plt.subplots(figsize = (8,5), constrained_layout = True)
    proportion_significant_metric = data_clean.groupby('Metric_to_corr')['Is_Significant'].mean().sort_values(ascending = False)
    proportion_significant_metric.round(2).plot.bar(ax=ax, color = ['r','b'])
    ax.set_title(f'B)  Proportion of significant correlations (p < 0.05)', fontsize = fontsizes_titles, fontstyle='italic', fontweight='bold')
    for bar in ax.containers:
        ax.bar_label(bar, fontsize = 17)
    ax.set_xticks(ax.get_xticks(), labels = metrics_name, rotation = 0, fontsize = fontsizes_ticks)
    ax.set_xlabel('')
    ax.set_ylim(0,1)
    ax.set_ylabel('Proportion with p-value < 0.05', fontsize = fontsizes_ticks)
    fig.savefig(save_folder  / f'stats_proportion_signif_{ylabel}.png', dpi = 500, bbox_inches = 'tight')
    plt.show()

# ICP WAVEFORM HOMEMADE

In [ ]:
sub = 'NY15'

In [ ]:
cns_reader = pycns.CnsReader(data_path / sub)
icp_stream = cns_reader.streams['ICP']
srate = icp_stream.sample_rate
raw_icp, dates = icp_stream.get_data(with_times = True)
detections = detect_icp_job.get(sub).to_dataframe()

In [ ]:
detections['total_duration'].plot.hist(bins = np.arange(0,1.5,0.01))

In [ ]:
half_win_size_secs = 0.5
half_win_size = int(srate * half_win_size_secs)
win_size = int(half_win_size * 2)

starts = detections['peak_ind'].values - half_win_size
starts = starts[starts > 0]
stops = starts + win_size
stops = stops[stops < raw_icp.size]

assert starts.size == stops.size

epochs = np.zeros((starts.size, win_size))

for i, start, stop in zip(np.arange(starts.size),starts,stops):
    epoch = raw_icp[start:stop]
    epochs[i,:] = epoch - np.mean(epoch)

In [ ]:
t_epoch = np.arange(win_size) / srate
m = np.mean(epochs, axis = 0)
s = np.std(epochs, axis = 0)
fig, ax = plt.subplots()
t_epoch = np.arange(win_size) / srate
ax.plot(t_epoch, m, color = 'k')
# ax.plot(t_epoch, s, color = 'r')
ax.fill_between(t_epoch, m-s, m+s, color = 'k', alpha = 0.1)

In [ ]:
epochs.shape

In [ ]:
epochs[::100,:].shape

In [ ]:
to_plot = epochs[::5000,:]
q = 0.05
vmin = np.quantile(to_plot, q)
vmax = np.quantile(to_plot, 1-q)
fig, ax = plt.subplots()
ax.pcolormesh(t_epoch, np.arange(to_plot.shape[0]), to_plot, vmin=vmin, vmax=vmax)
plt.show()

fig, ax = plt.subplots()
ax.plot(t_epoch, to_plot.T, color = 'k', alpha = 0.1)
plt.show()

# COHERENCE SPECTROGRAM

In [ ]:
sub = 'MF12'
cns_reader = CnsReader(data_path / sub)
stream_names = ['ICP', 'ABP']
srate = 200
start = "2022-09-10T17:00"
stop = "2022-09-10T17:30"
ds = cns_reader.export_to_xarray(stream_names, start=start, stop=stop, resample=True, sample_rate=srate)
raw_abp = ds['ABP'].values
raw_icp = ds['ICP'].values

In [ ]:
def coherence_spectrogram2(x, y, fs=1.0, window='hann', nperseg=256, noverlap=None):

    if noverlap is None:
        noverlap = nperseg // 2

    step = nperseg - noverlap
    starts = (np.arange(0, x.size, step)).astype(int)
    stops = starts + nperseg
    stops[-1] = x.size
    nsegments = stops.size - 1

    Cxys = None

    for i in range(nsegments):
        start = starts[i]
        stop = stops[i]
        # f, Cxy = scipy.signal.coherence(x[start:stop],y[start:stop],fs=fs, nperseg=nperseg, noverlap=0)
        x_local = x[start:stop]
        y_local = y[start:stop]
        f, Pxy = scipy.signal.csd(x_local, y_local, fs=srate, nperseg=raw_abp.size, noverlap=0)
        f, Pxx = scipy.signal.welch(x_local, fs=srate, nperseg=raw_abp.size, noverlap=0)
        f, Pyy = scipy.signal.welch(y_local, fs=srate, nperseg=raw_icp.size, noverlap=0)
        Cxy = np.abs(Pxy) / (Pxx * Pyy)
        Cxy = 1-Cxy
        if Cxys is None:
            Cxys = np.zeros((f.size, nsegments))
        Cxys[:,i] = Cxy

    # Temps correspondant à chaque segment
    t = stops[:-1] / srate

    return f, t, Cxys

In [ ]:
window_size_secs = 60
nperseg = int(window_size_secs * srate)
f, t, Cxys = coherence_spectrogram2(raw_abp, raw_icp, fs=srate, nperseg=nperseg)
f_mask = (f < 1)
data_plot = Cxys[f_mask,:]
# Afficher le résultat
fig, ax = plt.subplots()
ax.pcolormesh(t, f[f_mask], data_plot, vmin = 0, vmax = 1)
ax.set_ylabel('Fréquence [Hz]')
ax.set_xlabel('Temps [sec]')
ax.set_title('Spectrogramme de cohérence')
plt.show()


# OVERVIEW DATA STATS

In [ ]:
import openpyxl
overview = pd.read_excel(base_folder / 'overview_data_pycns_25_11_2025.xlsx')

In [ ]:
overview['Duration_Days'] = overview['Duration_Mins'] / (60 * 24)

In [ ]:
overview.head(2)

## - Stats descriptives

In [ ]:
cols = ['motif','GCS_sortie','mRS_sortie']
nrows = 2
ncols = 2
poss = attribute_subplots(cols, nrows,ncols)

N = overview.shape[0]

fig, axs = plt.subplots(nrows, ncols, figsize = (8,6), constrained_layout = True)
fig.suptitle(f'Descriptive statistics (N : {N})', fontsize = 15)
for colname, pos in poss.items():
    ax = axs[pos[0], pos[1]]    
    series = overview[colname]
    if series.dtypes != 'object':
        vcount = series.value_counts().sort_index()
    else:
        vcount = series.value_counts()
    vcount.plot.bar(ax=ax)
    for bar in ax.containers:
        ax.bar_label(bar)
    # ax.set_title(col)
    ax.set_ylabel('N')
ax = axs[1,1]
ser = overview['Duration_Days']
# m, s = ser.mean()
ax.hist(ser, bins = 15, edgecolor = 'k')
ax.set_ylabel('N')
ax.set_xlabel('Duration of recording (days)')
ax.set_title(ser.describe()[['mean','50%','std','min','max']].round(2).to_dict(), fontsize = 8)
fig.savefig(base_folder / 'figures' / 'stats_overview' / 'descriptive_stats.png', dpi = 500, bbox_inches = 'tight')
plt.show()

In [ ]:
overview_detailed = detailed_view_streams_job.get('concat').to_dataframe()
stream_sel = ['CO2','ECG_II','ICP','ABP','Scalp','ECoG']
overview_detailed_clean = overview_detailed[overview_detailed['stream'].isin(stream_sel)].reset_index(drop = True).iloc[:,1:]
overview_detailed_clean['duration_days_'] = overview_detailed_clean['duration_mins'] / (60 * 24)
overview_detailed_clean

In [ ]:
overview_detailed['patient'].unique().size

In [ ]:
def mad_sp(x):
    return scipy.stats.median_abs_deviation(x, scale = 'normal')

In [ ]:
overview_detailed_clean.groupby('stream')['duration_days_'].agg(['mean','std','median',scipy.stats.iqr, mad_sp])

In [ ]:
fontsize_labels = 14
fig, axs = plt.subplots(nrows = 3, figsize = (8,7), constrained_layout = True)

ax = axs[0]
ax.set_title(f'N patients : {overview.shape[0]}', fontsize = 20)
# overview[['CO2','ECG_II','ICP','ABP','Scalp','ECoG']].sum().plot.bar(ax=ax)
sns.barplot(data = overview[['CO2','ECG_II','ICP','ABP','Scalp','ECoG']].sum().reset_index(), x = 'index', y = 0, ax=ax)
ax.set_ylabel('N patients recorded', fontstyle = 'italic', fontsize = fontsize_labels)
ax.set_xlabel('', fontstyle = 'italic', fontsize = fontsize_labels)
for bar in ax.containers:
    ax.bar_label(bar)
ax.set_ylim(0, 110)


ax = axs[1]
sns.boxplot(data = overview_detailed_clean, 
            x = 'stream',
            y = 'duration_days_',
            ax=ax,
            order=stream_sel
            )
sns.stripplot(data = overview_detailed_clean, 
            x = 'stream',
            y = 'duration_days_',
            ax=ax,
            color = 'k',
            order=stream_sel,
            size=2
            )
ax.set_ylim(-1, 20)
ax.set_xlabel('', fontstyle = 'italic', fontsize = fontsize_labels)
ax.set_ylabel('Recording\nduration (days)', fontstyle = 'italic', fontsize = fontsize_labels)

ax = axs[2]
sns.barplot(data = overview_detailed_clean.groupby('stream')['duration_days_'].sum().round(0).reset_index(), x = 'stream', y = 'duration_days_', ax=ax, order=stream_sel)
for bar in ax.containers:
    ax.bar_label(bar)
ax.set_ylim(0, 700)
ax.set_xlabel('Stream', fontstyle = 'italic', fontsize = fontsize_labels)
ax.set_ylabel('Cumulative\nrecording duration (days)', fontstyle = 'italic', fontsize = fontsize_labels)

fig.savefig(base_folder / 'figures' / 'stats_overview' / 'overview_recording_durations.png' , dpi = 300, bbox_inches = 'tight')

## - Stats inférentielles

### Motif --> Duration_Days

In [ ]:
import ghibtools as gh

In [ ]:
gh.auto_stats(df = overview,
              predictor='motif',
              outcome = 'Duration_Days',
              design='between'
              )

In [ ]:
gh.auto_stats(df=overview,
                     predictor='motif',
                     outcome = 'GCS_sortie',
                     design='between'
                     )

In [ ]:
gh.auto_stats(df=overview,
                     predictor='motif',
                     outcome = 'N_SDs',
                     design='between'
                     )

In [ ]:
gh.stats_quantitative(df=overview,
                     xlabel='mRS_sortie',
                     ylabel = 'Duration_Days',
                    #  design='between'
                     )

# FIGURE FOR PAPER ARTIFACTs

In [ ]:
patient_id = 'FC13'
raw_folder = data_path / patient_id
cns_reader = CnsReader(raw_folder)

apply_gain = True

In [ ]:
def center(sig):
    return sig - np.mean(sig)

In [ ]:
def notch50Hz(sig, srate, bandcut = (48,52), order = 4, ftype = 'butter', verbose = False, show = False, axis = -1):

    """
    IIR-Filter to notch/cut 50 Hz of signal
    """

    band = [bandcut[0], bandcut[1]]
    Wn = [e / srate * 2 for e in band]
    sos = signal.iirfilter(order, Wn, analog=False, btype='bandstop', ftype=ftype, output='sos')
    filtered_sig = signal.sosfiltfilt(sos, sig, axis=axis)

    if verbose:
        print(f'{ftype} iirfilter of {order}th-order')
        print(f'btype : {btype}')


    if show:
        w, h = signal.sosfreqz(sos,fs=srate, worN = 2**18)
        fig, ax = plt.subplots()
        ax.plot(w, np.abs(h))
        ax.scatter(w, np.abs(h), color = 'k', alpha = 0.5)
        full_energy = w[np.abs(h) >= 0.99]
        ax.axvspan(xmin = full_energy[0], xmax = full_energy[-1], alpha = 0.1)
        ax.set_title('Frequency response')
        ax.set_xlabel('Frequency [Hz]')
        ax.set_ylabel('Amplitude')
        plt.show()

    return filtered_sig

In [ ]:
# chan = 'C3'
chan = 'ECoG4'
sigs = cns_reader.streams['EEG'].get_data(sel=slice('2022-02-10T15:05:00','2022-02-10T15:35:00'), with_times=False, apply_gain=apply_gain)
chans = cns_reader.streams['EEG'].channel_names
srate =  cns_reader.streams['EEG'].sample_rate
units = cns_reader.streams['EEG'].units
sig = sigs[:,chans.index(chan)]
sig_notch = notch50Hz(sig, srate, show = False)
times = np.arange(sig.size)/srate

In [ ]:
save_dpi = 500
save_extension = '.png'

In [ ]:
duration_example = 5
start_win_good = 100
start_win_bad = 1100

sig_example_good = sig_notch[(times > start_win_good) & (times <=  start_win_good + duration_example)]
sig_example_bad = sig_notch[(times > start_win_bad) & (times <=  start_win_bad + duration_example)]
t_example = np.arange(sig_example_good.size) / srate

sig_plots = [sig_example_good, sig_example_bad]
titles = ['High Quality Signal','Low Quality Signal']
save_names = ['1A','1B']

for sig_plot, title, save_name in zip(sig_plots,titles,save_names):
    fig, ax = plt.subplots(figsize = (4,2), constrained_layout = True)
    ax.plot(t_example, center(sig_plot), color = 'k', lw = 0.5)
    ax.set_xlabel('Time (sec)')
    ax.set_ylabel(f'Amplitude ({units})')
    ax.ticklabel_format(scilimits = [-2,2])
    ax.set_title(title)
    ax.set_xlim(t_example[0], t_example[-1])
    fig.savefig(base_folder / 'figures' / 'data_quality' / 'pour_article' / f'{save_name}{save_extension}', dpi = save_dpi, bbox_inches = 'tight')
    plt.show()

In [ ]:
wsize = 30
f, t , Sxx = scipy.signal.spectrogram(sig, srate, nperseg = int(wsize*srate), scaling='spectrum')
slopes_from_Sxx = np.apply_along_axis(compute_spectrum_log_slope, axis = 0, arr=Sxx, freqs=f)
total_powers = np.sum(Sxx, axis = 0)

In [ ]:
fig, ax = plt.subplots(figsize = (8,2), constrained_layout = True)
ax.plot(times, sig, color = 'k', lw = 0.5)
ax.set_xlabel('Time (sec)')
ax.set_ylabel(f'Amplitude ({units})')
ax.set_title('Raw signal - Time domain')
ax.ticklabel_format(axis = 'y', scilimits = [-2,2])
ax.set_xlim(times[0], times[-1])
fig.savefig(base_folder / 'figures' / 'data_quality' / 'pour_article' / f'1C{save_extension}', dpi = save_dpi, bbox_inches = 'tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (8,2), constrained_layout = True)
im = ax.pcolormesh(t, f, np.log(Sxx))
ax.set_xlabel('Time (sec)')
ax.set_ylabel('Frequency (Hz)')
clb = fig.colorbar(im, ax=ax,orientation="horizontal")
clb.set_label('Log-Power (log[µV²])')
ax.set_title('Raw signal - Frequency domain : Spectrogram of log-transformed power')
ax.set_xlim(t[0], t[-1])
fig.savefig(base_folder / 'figures' / 'data_quality' / 'pour_article' / f'1D{save_extension}', dpi = save_dpi, bbox_inches = 'tight')
plt.show()

In [ ]:
# ylims = (1e-7,1e7)
ylims = (-16,20)

inds = [10, 40]
titles = ['High','Low']
save_names = ['1E','1F']
freq_range = [1,100]


for ind, title, save_name in zip(inds, titles,save_names):
    spectrum = Sxx[:,ind]
    total_power = np.sum(spectrum)
    
    fig, ax = plt.subplots(figsize = (4,2), constrained_layout = True)
    
    mask = (f >= freq_range[0]) & (f <= freq_range[1])
    f_log = np.log(f[mask])
    spectrum_log = np.log(spectrum[mask])

    res = scipy.stats.linregress(f_log, spectrum_log)
    a = res.slope
    b = res.intercept
    
    fit_log = a * f_log + b
    fit = np.exp(a * f_log + b)
    
    ax.plot(f_log, spectrum_log, color = 'k', lw = 0.5)
    ax.plot(f_log,  fit_log, lw = 2, color = 'darkorange')
        
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Log-Power (log[µV²])')
    ax.set_ylim(ylims[0] , ylims[1])
    # ax.set_title(f'Log Power Spectrum - {title} Quality Signal\nSlope : {round(a, 2)} - Total power : {round(total_power, 2)} µV²')
    ax.set_title('Log Power Spectrum - {} Quality Signal\nSlope : {} - Total power : {:.1e} µV²'.format(title, round(a,2), round(total_power,2)))
    ax.set_xticks(range(5), np.round(np.exp(range(5)) , 2))
    ax.set_ylim(ylims[0], ylims[1])
    ax.set_xlim(f_log[0], f_log[-1])
    fig.savefig(base_folder / 'figures' / 'data_quality' / 'pour_article' / f'{save_name}{save_extension}', dpi = save_dpi, bbox_inches = 'tight')
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (8,2), constrained_layout = True)
ax.plot(t, slopes_from_Sxx, color = 'k')
ax.set_ylabel('Log Spectrum Slope')
ax.set_xlabel('Time (sec)')
ax.axhline(-1, color = 'r', lw = 2, label = 'Threshold Quality')
ax.legend(loc = 2)
ax.set_title('Log Spectrum Slope')
ax.set_xlim(t[0], t[-1])
fig.savefig(base_folder / 'figures' / 'data_quality' / 'pour_article' / f'1G{save_extension}', dpi = save_dpi, bbox_inches = 'tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize = (8,2), constrained_layout = True)
ax.plot(t, np.log(total_powers), color = 'k')
ax.set_ylabel('Log-Power (log[µV²])')
ax.set_xlabel('Time (sec)')
ax.axhline(20, color = 'r', lw = 2, label = 'Threshold Quality')
ax.legend(loc = 2)
ax.set_xlim(t[0], t[-1])
ax.set_title('Total Log-Power')
fig.savefig(base_folder / 'figures' / 'data_quality' / 'pour_article' / f'1H{save_extension}', dpi = save_dpi, bbox_inches = 'tight')
plt.show()

In [ ]:
boolean_mask = np.all(np.array([np.log(total_powers) < 20, slopes_from_Sxx < 1]), axis = 0)

fig, ax = plt.subplots(figsize = (8,2), constrained_layout = True)
ax.plot(t, boolean_mask, color = 'k')
ax.set_ylabel('Is High Quality')
ax.set_xlabel('Time (sec)')
ax.set_title('Boolean mask of High Quality Signal')
ax.set_yticks([0,1],['No','Yes'])
ax.set_xlim(t[0], t[-1])
fig.savefig(base_folder / 'figures' / 'data_quality' / 'pour_article' / f'1I{save_extension}', dpi = save_dpi, bbox_inches = 'tight')
plt.show()

# RESPI WAVEFORM IN or OUT of RSA OSCILLATIONS WINDOWS

In [ ]:
# windows for patients in VAC but in or out of RSA oscillations periods to compare respi waveform
windows_rsa_oscillations = {
        'MF12':{'in':('2022-09-11T11:00:00','2022-09-11T12:00:00'), 'out':('2022-09-10T13:00:00','2022-09-10T14:00:00')},
        'SP2':{'in':('2021-02-08T12:45:00','2021-02-08T13:45:00'), 'out':('2021-02-08T11:53:00','2021-02-08T12:53:00')},
        'WJ14':{'in':('2022-03-02T02:23:00','2022-03-02T03:23:00'), 'out':('2022-02-28T20:00:00','2022-02-28T21:00:00')},
        'GA9':{'in':('2021-08-25T23:16:00','2021-08-26T00:16:00'), 'out':('2021-08-27T08:00:00','2021-08-27T09:00:00')},
       }

In [ ]:
n_phase_bins = 40
phase_resp = np.linspace(0,1,n_phase_bins)

subs = windows_rsa_oscillations.keys()

fig, axs = plt.subplots(nrows = len(subs), ncols = 2, figsize = (9, 15), constrained_layout = True)
fig.suptitle('CO2 average aveform in or out RSA oscillation window', fontsize = 20)

for r, sub in enumerate(subs):
    print(sub)
    reader = CnsReader(data_path / sub)
    co2_stream = reader.streams['CO2']
    raw_co2, dates = co2_stream.get_data(with_times=True)
    srate = co2_stream.sample_rate
    co2, resp_cycles = physio.compute_respiration(raw_co2, srate, parameter_preset='human_co2')
    resp_cycles['inspi_date'] = dates[resp_cycles['inspi_index']]
    # cycle_times = resp_cycles[['inspi_time','next_inspi_time']].values
    expi_times = resp_cycles['expi_time'].values[:-1]
    next_expi_times = resp_cycles['expi_time'].values[1:]
    cycle_times = np.concatenate([expi_times[:,None], next_expi_times[:,None]], axis = 1)
    co2_cycles = physio.deform_traces_to_cycle_template(data = co2, 
                                                        times = np.arange(0,co2.size/srate, 1 / srate), 
                                                        cycle_times = cycle_times,
                                                        points_per_cycle = n_phase_bins,
                                                        progress_bar = False
                                                       )
    for c, win in enumerate(windows_rsa_oscillations[sub].keys()):
        ax = axs[r,c]
        start = np.datetime64(windows_rsa_oscillations[sub][win][0])
        stop = np.datetime64(windows_rsa_oscillations[sub][win][1])
        
        local_resp_cycles = resp_cycles[(resp_cycles['inspi_date'] >= start) & (resp_cycles['inspi_date'] < stop)]
        n_cycles = local_resp_cycles.shape[0]
        local_resp_cycles_inds = local_resp_cycles.index.values
        local_co2_cycles = co2_cycles[local_resp_cycles_inds,:]
        local_co2_cycles = local_co2_cycles.T - np.mean(local_co2_cycles, axis = 1) # centering. cycle axis come from 0 and go to 1
        
        # med_ratio = local_resp_cycles['cycle_ratio'].median()
        med_ratio = 1 - local_resp_cycles['cycle_ratio'].median()
        mean_waveform = local_co2_cycles.mean(axis = 1)
        ax.plot(phase_resp, local_co2_cycles, color = 'k', alpha = 0.1, lw = 0.8)
        ax.plot(phase_resp, mean_waveform, color = 'darkorange', lw = 2)
        ax.axvline(med_ratio, color = 'r', ls = '--', alpha = 0.8)
        ax.set_xlabel('CO2 phase')
        ax.set_ylabel('CO2 amplitude [AU]')
        ax.set_title(f'{sub} - {win} (N cycles = {n_cycles})', fontsize = 15)
        
# fig.savefig(base_folder / 'figures' / 'autres' / 'co2_waveform_in_out_rsa_oscillations.png', dpi = 500, bbox_inches = 'tight')
plt.show()

# FIGURE LIKE HARTINGS 2020 : QUALITY + SD , FOR GWENDAN

In [ ]:
from compute_artifacts import mask_artifacting_job

In [ ]:
# sd_df = pd.read_excel(base_folder / 'scripts_analyse_physio_neuro_rea' / 'metadata' / 'csd_dates_gwendan.xlsx')
sd_df = pd.read_excel(base_folder / 'scripts_analyse_physio_neuro_rea' / 'metadata' / 'new_csd_dates_gwendan_vg.xlsx')
overview_df = pd.read_excel(base_folder / 'overview_data_pycns_05_02_25.xlsx').set_index('Patient')

sd_df = sd_df[sd_df['window'] == 'per_csd'].reset_index(drop = True)

days_after_start_hospit_list = []

for i, row in sd_df.iterrows():
    start_hospit = overview_df.loc[row['patient'],'Start_Date']
    start_csd = row['start_csd']
    days_after_start_hospit_mins = (np.timedelta64(np.datetime64(start_csd) - np.datetime64(start_hospit))).astype('timedelta64[m]').astype(int)
    days_after_start_hospit_days = days_after_start_hospit_mins / (60*24)
    days_after_start_hospit_list.append(days_after_start_hospit_days)

sd_df['n_days_after_start_hospit'] = days_after_start_hospit_list
sd_df['num_patient'] = sd_df['patient'].map({patient:i+1 for i,patient in enumerate(sd_df['patient'].unique())})
sd_df['type_of_event2'] = ['iSD' if 'iSD' in _type else 'CSD' for _type in list(sd_df['type_of_event'])]

In [ ]:
fig, ax = plt.subplots(figsize = (7,5), constrained_layout = True)

for h in np.arange(0,11):
    ax.axhline(h, color='k', alpha = 0.2)
    
subs = list(sd_df['patient'].unique())

ax.set_xlim(0, len(subs) + 1)
ax.set_ylim(0, sd_df['n_days_after_start_hospit'].max() + 1)
ax.set_ylabel('Days after start of recording')
ax.set_yticks(np.arange(0,11))
n_sd_by_sub = sd_df['patient'].value_counts().to_dict()
labels = [f'{sub}\n({n_sd_by_sub[sub]})' for sub in subs]
ax.set_xticks(np.arange(1,len(subs)+1), labels = labels)
ax.set_xlabel('Patient\n(n of events)')

color_events = {'CSD':'k','SD_Cluster':'dodgerblue','iSD':'r','IEE':'limegreen'}
for _type in sd_df['type_of_event'].unique():
    sd_df_type = sd_df[sd_df['type_of_event'] == _type]
    ax.scatter(sd_df_type['num_patient'], sd_df_type['n_days_after_start_hospit'], color = color_events[_type], marker = '_', alpha = 0.8, label = _type)

n_sds = sd_df.shape[0]
ax.set_title(f'Overview of pathological brain events - Total number = {n_sds}', fontsize = 12)

for i, sub in enumerate(subs):
    quality_mask = mask_artifacting_job.get(sub)['mask_artifacting'].loc['ECoG']
    dates_mask = quality_mask['date'].values
    dates_mask_delta = (dates_mask - np.datetime64(dates_mask[0])).astype('timedelta64[s]').astype(int) / (60 * 60 * 24)
    quality_mask = quality_mask.values
    dates_mask_delta[np.where(~quality_mask)[0]] = np.nan
    if i == len(subs) - 1:
        legend_label = 'High-quality recording'
    else:
        legend_label = None
    ax.plot([i+1] * quality_mask.size, dates_mask_delta, color = 'k', alpha = 0.2, lw = 3, label = legend_label)
ax.legend(ncols = 3)

# fig.savefig(base_folder / 'figures' / 'autres' / 'sd_and_quality_overview.png', dpi = 500, bbox_inches = 'tight')

plt.show()

# FIGURE ARTIFACTING

In [ ]:
from compute_artifacts import mask_artifacting_job, mask_muscle_job, mask_6Hz_job

In [ ]:
subs = get_patient_list(['Scalp','ECoG'], patient_type='SD_ICU')
subs

In [ ]:
days_max = 15
mask_types = ['ECoG','Scalp']
job = mask_artifacting_job
da_name = 'mask_artifacting'
title = f'Overview of Signal Quality'
figsize = (9,5)
x_delta_range = 0.2
lw = 0.5

fig, ax = plt.subplots(figsize = figsize, constrained_layout = True)

ax.set_title(title, fontsize = 12)

start_dates = {}
stop_dates = {}
stop_days = {}
labels = []

x_deltas = np.linspace(-x_delta_range,x_delta_range,len(mask_types))

stats_rows = []

for i, sub in enumerate(subs):
    masks = job.get(sub)[da_name]
    stats = {_type:masks.attrs['proportion_good_quality'][k] for k, _type in enumerate(masks['chan_type'].values)} 
    for j, mask_type in enumerate(mask_types):
        stats_rows.append([sub, mask_type, stats[mask_type] * 100])
        mask = masks.loc[mask_type]
        dates_mask = mask['date'].values
        start_date = dates_mask[0]
        stop_date = dates_mask[-1]
        dates_mask_delta = (dates_mask - np.datetime64(start_date)).astype('timedelta64[s]').astype(int) / (60 * 60 * 24)
        stop_day = dates_mask_delta[-1]
        dates_not_mask_delta = dates_mask_delta.copy()
        mask = mask.values
        dates_mask_delta[np.where(~mask)[0]] = np.nan
        dates_not_mask_delta[np.where(mask)[0]] = np.nan
        x_pos = i+1 + x_deltas[j]
        if sub == subs[-1] and mask_type == mask_types[-1]:
            label_good = 'High-Quality Recording'
            label_bad = 'Low-Quality Recording'
        else:
            label_good, label_bad = None, None
        ax.plot([x_pos] * mask.size, dates_mask_delta, color = 'g', alpha = 1, lw = lw, label = label_good)
        ax.plot([x_pos] * mask.size, dates_not_mask_delta, color = 'r', alpha = 1, lw = lw, label = label_bad)
        ax.text(x_pos, -1.5, mask_type, rotation = 90, fontsize = 8, ha = 'center')
    stop_days[sub] = stop_day

ax.legend(ncols = 2, loc = 'upper center', framealpha = 1)

# days_max = int(max(stop_days.values()))
days_max = 14
for h in np.arange(0,days_max+2):
    ax.axhline(h, color='k', alpha = 0.2)
    
ax.set_xlim(0, len(subs) + 1)
ax.set_ylim(-2, days_max+2)
ax.set_ylabel('Days after start of recording')
ax.set_yticks(np.arange(0,days_max+2))

labels = subs
ax.set_xticks(np.arange(1,len(subs)+1), labels = labels, rotation = 90)
ax.set_xlabel('Patient')


# fig.savefig(base_folder / 'figures' / 'data_quality' / 'artifacting_overview_v2.png', dpi = 500, bbox_inches = 'tight')

plt.show()

stats_artifacting_df = pd.DataFrame(stats_rows, columns = ['Patient','Channel Type','Percentage of Good Quality'])

In [ ]:
summary_artifacting = stats_artifacting_df.groupby('Channel Type')['Percentage of Good Quality'].agg(['mean','std','max','min']).round(2)
# summary_artifacting.to_excel(base_folder / 'figures' / 'data_quality' / 'artifacting_overview_stats.xlsx')
summary_artifacting


In [ ]:
stats_artifacting_df.columns

In [ ]:
import ghibtools as gh

In [ ]:
stats_artifacting_df.groupby('Channel Type')['Percentage of Good Quality'].describe()

In [ ]:
days_max = 15
mask_types = ['ECoG','Scalp','Both']
job = mask_muscle_job
da_name = 'mask_muscle'
title = f'Overview of Muscle Artifacting'
figsize = (9,5)
x_delta_range = 0.3
lw = 0.2
decimate = True
decim_factor = 10

fig, ax = plt.subplots(figsize = figsize, constrained_layout = True)

ax.set_title(title, fontsize = 12)

start_dates = {}
stop_dates = {}
stop_days = {}
labels = []

x_deltas = np.linspace(-x_delta_range,x_delta_range,len(mask_types))

stats_rows = []

for i, sub in enumerate(subs):
    masks = job.get(sub)[da_name]
    for j, mask_type in enumerate(mask_types):
        mask = masks.loc[mask_type]
        dates_mask = mask['date'].values
        start_date = dates_mask[0]
        stop_date = dates_mask[-1]
        dates_mask_delta = (dates_mask - np.datetime64(start_date)).astype('timedelta64[s]').astype(int) / (60 * 60 * 24)
        if decimate:
            dates_mask_delta = dates_mask_delta[::decim_factor]
        stop_day = dates_mask_delta[-1]
        mask = mask.values
        stats_rows.append([sub, mask_type, (mask.sum() / mask.size)*100])
        if decimate:
            mask = mask[::decim_factor]
        dates_mask_delta[np.where(~mask)[0]] = np.nan
        x_pos = i+1 + x_deltas[j]
        if sub == subs[-1] and mask_type == mask_types[-1]:
            label = 'Muscle Activity'
        else:
            label = None
        ax.plot([x_pos] * mask.size, dates_mask_delta, color = 'tab:blue', alpha = 1, lw = lw, label = label)
        ax.text(x_pos, -1.5, mask_type, rotation = 90, fontsize = 8, ha = 'center')
    stop_days[sub] = stop_day

ax.legend(ncols = 2, loc = 'upper center', framealpha = 1)

# days_max = int(max(stop_days.values()))
days_max = 14
for h in np.arange(0,days_max+2):
    ax.axhline(h, color='k', alpha = 0.2)
    
ax.set_xlim(0, len(subs) + 1)
ax.set_ylim(-2, days_max+2)
ax.set_ylabel('Days after start of recording')
ax.set_yticks(np.arange(0,days_max+2))

labels = subs
ax.set_xticks(np.arange(1,len(subs)+1), labels = labels, rotation = 90)
ax.set_xlabel('Patient')


fig.savefig(base_folder / 'figures' / 'data_quality' / 'muscle_overview.png', dpi = 500, bbox_inches = 'tight')

plt.show()

stats_muscle_df = pd.DataFrame(stats_rows, columns = ['Patient','Channel Type','Percentage of Muscle Activity'])

In [ ]:
summary_muscle = stats_muscle_df.groupby('Channel Type')['Percentage of Muscle Activity'].agg(['mean','std','max','min']).round(2)
summary_muscle.to_excel(base_folder / 'figures' / 'data_quality' / 'muscle_overview_stats.xlsx')
summary_muscle

In [ ]:
days_max = 15
mask_types = ['ECoG','Scalp','Both']
job = mask_6Hz_job
da_name = 'mask_6Hz'
title = f'Overview of 6Hz Artifacting'
figsize = (9,5)
x_delta_range = 0.3
lw = 1
decimate = True
decim_factor = 1000

fig, ax = plt.subplots(figsize = figsize, constrained_layout = True)

ax.set_title(title, fontsize = 12)

start_dates = {}
stop_dates = {}
stop_days = {}
labels = []

x_deltas = np.linspace(-x_delta_range,x_delta_range,len(mask_types))

stats_rows = []

for i, sub in enumerate(subs):
    masks = job.get(sub)[da_name]
    for j, mask_type in enumerate(mask_types):
        mask = masks.loc[mask_type]
        dates_mask = mask['date'].values
        start_date = dates_mask[0]
        stop_date = dates_mask[-1]
        dates_mask_delta = (dates_mask - np.datetime64(start_date)).astype('timedelta64[s]').astype(int) / (60 * 60 * 24)
        if decimate:
            dates_mask_delta = dates_mask_delta[::decim_factor]
        stop_day = dates_mask_delta[-1]
        mask = mask.values
        stats_rows.append([sub, mask_type, (mask.sum() / mask.size)*100])
        if decimate:
            mask = mask[::decim_factor]
        dates_mask_delta[np.where(~mask)[0]] = np.nan
        x_pos = i+1 + x_deltas[j]
        if sub == subs[-1] and mask_type == mask_types[-1]:
            label = '6Hz Artifacting'
        else:
            label = None
        ax.plot([x_pos] * mask.size, dates_mask_delta, color = 'k', alpha = 1, lw = lw, label = label)
        ax.text(x_pos, -1.5, mask_type, rotation = 90, fontsize = 8, ha = 'center')
    stop_days[sub] = stop_day

ax.legend(ncols = 2, loc = 'upper center', framealpha = 1)

# days_max = int(max(stop_days.values()))
days_max = 15
for h in np.arange(0,days_max+2):
    ax.axhline(h, color='k', alpha = 0.2)
    
ax.set_xlim(0, len(subs) + 1)
ax.set_ylim(-2, days_max+2)
ax.set_ylabel('Days after start of recording')
ax.set_yticks(np.arange(0,days_max+2))

labels = subs
ax.set_xticks(np.arange(1,len(subs)+1), labels = labels, rotation = 90)
ax.set_xlabel('Patient')


fig.savefig(base_folder / 'figures' / 'data_quality' / '6Hz_overview.png', dpi = 500, bbox_inches = 'tight')

plt.show()

stats_6Hz_df = pd.DataFrame(stats_rows, columns = ['Patient','Channel Type','Percentage of 6Hz Artifacting'])

In [ ]:
summary_6Hz = stats_6Hz_df.groupby('Channel Type')['Percentage of 6Hz Artifacting'].agg(['mean','std','max','min']).round(2)
summary_6Hz.to_excel(base_folder / 'figures' / 'data_quality' / '6Hz_overview_stats.xlsx')
summary_6Hz

# FIGURE LIKE HARTINGS 2020 : QUALITY + SD from ALL PATIENTS FROM BAPTISTE'S SD TABLE

## functions

In [ ]:
def process_events(events):
    events_processed = []
    for event in events:
        # if 'SD' in event:
        if 'SD' in event and not '?' in event:
            # if 'iSD' in event:
            #     event_processed = 'iSD'
            # else:
            #     event_processed = 'CSD'
            event_processed = 'SD'
        else:
            event_processed = np.nan
        events_processed.append(event_processed)
    return events_processed

In [ ]:
def compute_days_after_start_hospit(dates, start_hospit_date):
    days_after_start_hospit_list = []
    if type(dates) is str:
        dates = [dates]
    for date in dates:
        days_after_start_hospit_mins = (np.timedelta64(np.datetime64(date) - np.datetime64(start_hospit_date))).astype('timedelta64[m]').astype(int)
        days_after_start_hospit_days = days_after_start_hospit_mins / (60*24)
        days_after_start_hospit_list.append(days_after_start_hospit_days)
    return days_after_start_hospit_list

In [ ]:
overview_df = pd.read_excel(base_folder / 'overview_data_pycns.xlsx').set_index('Patient')

In [ ]:
subs = get_patient_list(['ECoG','Scalp'], patient_type = 'SD_ICU', threshold_duration_mins = 0, verbose = True)

In [ ]:
subs_sorted = overview_df.loc[subs].sort_values(by = 'Start_Date').index.tolist()
mapper_sub_num = {s:int(s[2:]) for s in subs_sorted}
subs_sorted = pd.Series(mapper_sub_num).sort_values().index.tolist()
subs_sorted

In [ ]:
overview_df.loc[subs,'Duration_Mins'].sum() / (60 * 24)

In [ ]:
# sd_df_all = pd.read_excel(base_folder / 'scripts_analyse_physio_neuro_rea' / 'metadata' / 'csd_dates_gwendan.xlsx')
sd_df_all = pd.read_excel(base_folder / 'scripts_analyse_physio_neuro_rea' / 'metadata' / 'new_csd_dates_gwendan.xlsx')
sd_df_all = sd_df_all[sd_df_all['window'] == 'per_csd']
sd_df_all = sd_df_all.drop_duplicates()
# sd_df_all['type_of_event'] = ['iSD' if 'iSD' in _type else 'CSD' for _type in sd_df_all['type_of_event']]

gwendan_subs = list(sd_df_all['patient'].unique())

In [ ]:
other_subs = [sub for sub in subs if not sub in gwendan_subs]
other_subs

In [ ]:
def get_pycns_events(sub):
    cns_reader = pycns.CnsReader(data_path / sub, with_events = True, event_time_zone='Europe/Paris')
    evs = pd.DataFrame(cns_reader.events)
    return evs


## fig

In [ ]:
# subs_for_fig = subs
subs_for_fig = subs_sorted

ymax = 14

fig, ax = plt.subplots(figsize = (9,5), constrained_layout = True)

for h in np.arange(0,ymax+1):
    ax.axhline(h, color='k', alpha = 0.2)
    
ax.set_xlim(0, len(subs_for_fig) + 1)
ax.set_ylim(0, ymax + 1)
ax.set_ylabel('Days after start of recording')
ax.set_yticks(np.arange(0,ymax+1))


color_events = {'SD':'k','SD_Cluster':'dodgerblue','iSD':'r','IEE':'limegreen'}

n_evs_by_sub = {}

sds_days_after_start_hospit = []

for i, sub in enumerate(subs_for_fig):
    # print(sub)
    # if sub in gwendan_subs:
    if sub in []:
        sd_df = sd_df_all.set_index('patient').loc[sub,:]
        sd_df['n_days_after_start_hospit'] = compute_days_after_start_hospit(sd_df['start_csd'], overview_df.loc[sub,'Start_Date'])
        if isinstance(sd_df, pd.DataFrame):
            n_ev_sub = sd_df.shape[0]
        elif isinstance(sd_df, pd.Series):
            n_ev_sub = 1
    else:
        # sd_df = read_events(sub)
        # sd_df = read_events(sub)
        sd_df = get_pycns_events(sub)
        # print(sub, sd_df['Name'])
        # sd_df['StartTimeGMT'] = pd.to_datetime(sd_df['StartTime']).dt.tz_localize('Europe/Paris').dt.tz_convert('GMT')
        sd_df['type_of_event'] = process_events(sd_df['name'])
        sd_df['n_days_after_start_hospit'] = compute_days_after_start_hospit(sd_df['start_time'], overview_df.loc[sub,'Start_Date'])
        sd_df = sd_df.dropna(subset = 'type_of_event')
        n_ev_sub = sd_df.shape[0]
        print(sub, n_ev_sub)
    
    sds_days_after_start_hospit.extend(sd_df['n_days_after_start_hospit'])
    n_evs_by_sub[sub] = n_ev_sub
    
    if isinstance(sd_df, pd.Series):
        sd_df = sd_df.to_frame().T
    for _type in sd_df['type_of_event'].unique():
        if _type in color_events.keys():
            sd_df_type = sd_df[sd_df['type_of_event'] == _type]
            # print(sd_df_type.shape)
            if sd_df_type.shape[0] > 1:
                y = sd_df_type['n_days_after_start_hospit'].values
            else:
                y = np.array(sd_df_type['n_days_after_start_hospit'].values[0])
            x = [i+1] * y.size
            # print(x,y)
            if sub == 'MF12':
                legend_label = _type
            else:
                legend_label = None
            ax.scatter(x, y, color = color_events[_type], marker = '_', alpha = 0.8, label = legend_label)

    quality_mask = eeg_quality_averaged_job.get(sub)['eeg_quality_averaged'].loc['ecog']
    dates_mask = quality_mask['datetime'].values
    dates_mask_delta = (dates_mask - np.datetime64(dates_mask[0])).astype('timedelta64[s]').astype(int) / (60 * 60 * 24)
    quality_mask = quality_mask.values
    dates_mask_delta[np.where(~quality_mask.astype(bool))[0]] = np.nan
    if i == len(subs_for_fig) - 1:
        legend_label = 'High-quality ECoG recording'
    else:
        legend_label = None
    ax.plot([i+1] * quality_mask.size, dates_mask_delta, color = 'k', alpha = 0.2, lw = 3, label = legend_label)
ax.legend(ncols = 4, loc = 'upper center')

n_sds = sum(n_evs_by_sub.values())
ax.set_title(f'Overview of pathological brain events - Total number = {n_sds}', fontsize = 12)

# labels = [f'{sub}\n({n_evs_by_sub[sub]})' for sub in subs_for_fig]
# labels = [f'{sub[2:]}\n({n_evs_by_sub[sub]})' for sub in subs_for_fig]
labels = [f'{sub[2:]}' for sub in subs_for_fig]
ax.set_xticks(np.arange(1,len(subs)+1), labels = labels)
# ax.set_xlabel('Patient\n(n of events)')
ax.set_xlabel('Patient Number')

fig.savefig(base_folder / 'figures' / 'autres' / 'sd_and_quality_overview_v5_sorted_by_date.png', dpi = 500, bbox_inches = 'tight')

plt.show()

In [ ]:
bins = np.arange(0, 15, 1)
fig, ax = plt.subplots(figsize = (8,3))
count, bins = np.histogram(sds_days_after_start_hospit, bins = bins)
ax.bar(bins[:-1], count, width = np.median(np.diff(bins)), color = 'k')
# ax.plot(bins[:-1], count, color = 'k')
# ax.fill_between(bins[:-1], count, color = 'k')
ax.set_xticks(bins)
fontsize = 20
ax.set_ylabel('Count of SDs', fontsize = fontsize)
ax.set_xlabel('Days after start of recording', fontsize = fontsize)
ax.set_ylim(0, 50)
ax.set_xticks(ax.get_xticks(), labels = ax.get_xticks(), fontsize = 15)
ax.set_yticks(ax.get_yticks(), labels = list(map(int, ax.get_yticks())), fontsize = 15)
ax.invert_xaxis()
fig.savefig(base_folder / 'figures' / 'autres' / 'sd_distribution_by_day_v5.png', dpi = 500, bbox_inches = 'tight')
plt.show()

In [ ]:
days_max = 15
mask_types = ['ECoG','Scalp']
job = eeg_quality_averaged_job
da_name = 'mask_artifacting'
title = f'Overview of Signal Quality'
figsize = (9,5)
x_delta_range = 0.2
lw = 0.5

fig, ax = plt.subplots(figsize = figsize, constrained_layout = True)

ax.set_title(title, fontsize = 12)

start_dates = {}
stop_dates = {}
stop_days = {}
labels = []

x_deltas = np.linspace(-x_delta_range,x_delta_range,len(mask_types))

stats_rows = []

subs = subs_sorted
for i, sub in enumerate(subs):
    masks = job.get(sub)['eeg_quality_averaged'].astype(bool)
    stats = {_type:masks.attrs['proportion_good_quality'][k] for k, _type in enumerate(masks['mode'].values)} 
    for j, mask_type in enumerate(mask_types):
        stats_rows.append([sub, mask_type, stats[mask_type] * 100])
        mask = masks.loc[mask_type]
        dates_mask = mask['date'].values
        start_date = dates_mask[0]
        stop_date = dates_mask[-1]
        dates_mask_delta = (dates_mask - np.datetime64(start_date)).astype('timedelta64[s]').astype(int) / (60 * 60 * 24)
        stop_day = dates_mask_delta[-1]
        dates_not_mask_delta = dates_mask_delta.copy()
        mask = mask.values
        dates_mask_delta[np.where(~mask)[0]] = np.nan
        dates_not_mask_delta[np.where(mask)[0]] = np.nan
        x_pos = i+1 + x_deltas[j]
        if sub == subs[-1] and mask_type == mask_types[-1]:
            label_good = 'High-Quality Recording'
            label_bad = 'Low-Quality Recording'
        else:
            label_good, label_bad = None, None
        ax.plot([x_pos] * mask.size, dates_mask_delta, color = 'g', alpha = 1, lw = lw, label = label_good)
        ax.plot([x_pos] * mask.size, dates_not_mask_delta, color = 'r', alpha = 1, lw = lw, label = label_bad)
        ax.text(x_pos, -1.5, mask_type, rotation = 90, fontsize = 8, ha = 'center')
    stop_days[sub] = stop_day

ax.legend(ncols = 2, loc = 'upper center', framealpha = 1)

# days_max = int(max(stop_days.values()))
days_max = 14
for h in np.arange(0,days_max+2):
    ax.axhline(h, color='k', alpha = 0.2)
    
ax.set_xlim(0, len(subs) + 1)
ax.set_ylim(-2, days_max+2)
ax.set_ylabel('Days after start of recording')
ax.set_yticks(np.arange(0,days_max+2))

labels = subs
ax.set_xticks(np.arange(1,len(subs)+1), labels = [s[2:] for s in labels], rotation = 90)
ax.set_xlabel('Patient Number')


fig.savefig(base_folder / 'figures' / 'autres' / 'artifacting_overview_eeg_ecog_v4.png', dpi = 500, bbox_inches = 'tight')

plt.show()

stats_artifacting_df = pd.DataFrame(stats_rows, columns = ['Patient','Channel Type','Percentage of Good Quality'])

In [ ]:
days_max = 15
mask_types = ['ECoG','Scalp']
job = mask_artifacting_job
da_name = 'mask_artifacting'
title = f'Overview of Signal Quality'
figsize = (9,5)
x_delta_range = 0.2
lw = 4

fig, ax = plt.subplots(figsize = figsize, constrained_layout = True)

ax.set_title(title, fontsize = 12)

start_dates = {}
stop_dates = {}
stop_days = {}
labels = []

x_deltas = np.linspace(-x_delta_range,x_delta_range,len(mask_types))

stats_rows = []

subs = subs_sorted
for i, sub in enumerate(subs):
    masks = job.get(sub)[da_name]
    stats = {_type:masks.attrs['proportion_good_quality'][k] for k, _type in enumerate(masks['chan_type'].values)} 
    for j, mask_type in enumerate(mask_types):
        stats_rows.append([sub, mask_type, stats[mask_type] * 100])
        mask = masks.loc[mask_type]
        dates_mask = mask['date'].values
        start_date = dates_mask[0]
        stop_date = dates_mask[-1]
        dates_mask_delta = (dates_mask - np.datetime64(start_date)).astype('timedelta64[s]').astype(int) / (60 * 60 * 24)
        stop_day = dates_mask_delta[-1]
        dates_not_mask_delta = dates_mask_delta.copy()
        mask = mask.values
        dates_mask_delta[np.where(~mask)[0]] = np.nan
        dates_not_mask_delta[np.where(mask)[0]] = np.nan
        x_pos = i+1 + x_deltas[j]
        if sub == subs[-1] and mask_type == mask_types[-1]:
            label_good = 'High-Quality Recording'
            label_bad = 'Low-Quality Recording'
        else:
            label_good, label_bad = None, None
        ax.plot([x_pos] * mask.size, dates_mask_delta, color = 'k', alpha = 1, lw = lw, label = label_good)
        ax.plot([x_pos] * mask.size, dates_not_mask_delta, color = 'gray', alpha = 1, lw = lw, label = label_bad)
        ax.text(x_pos, -1.5, mask_type, rotation = 90, fontsize = 8, ha = 'center')
    stop_days[sub] = stop_day

ax.legend(ncols = 2, loc = 'upper center', framealpha = 1)

# days_max = int(max(stop_days.values()))
days_max = 14
for h in np.arange(0,days_max+2):
    ax.axhline(h, color='k', alpha = 0.2)
    
ax.set_xlim(0, len(subs) + 1)
ax.set_ylim(-2, days_max+2)
ax.set_ylabel('Days after start of recording')
ax.set_yticks(np.arange(0,days_max+2))

labels = subs
ax.set_xticks(np.arange(1,len(subs)+1), labels = [s[2:] for s in labels], rotation = 90)
ax.set_xlabel('Patient Number')


fig.savefig(base_folder / 'figures' / 'autres' / 'artifacting_overview_eeg_ecog_v4.png', dpi = 500, bbox_inches = 'tight')

plt.show()

stats_artifacting_df = pd.DataFrame(stats_rows, columns = ['Patient','Channel Type','Percentage of Good Quality'])

In [ ]:
days_max = 15
mask_types = ['ECoG','Scalp']
job = mask_artifacting_job
da_name = 'mask_artifacting'
title = f'Overview of Signal Quality'
figsize = (9,5)
x_delta_range = 0.2
lw = 3

fig, ax = plt.subplots(figsize = figsize, constrained_layout = True)

ax.set_title(title, fontsize = 12)

start_dates = {}
stop_dates = {}
stop_days = {}
labels = []

x_deltas = np.linspace(-x_delta_range,x_delta_range,len(mask_types))

stats_rows = []

subs = subs_sorted
for i, sub in enumerate(subs):
    masks = job.get(sub)[da_name]
    stats = {_type:masks.attrs['proportion_good_quality'][k] for k, _type in enumerate(masks['chan_type'].values)} 
    for j, mask_type in enumerate(mask_types):
        stats_rows.append([sub, mask_type, stats[mask_type] * 100])
        mask = masks.loc[mask_type]
        dates_mask = mask['date'].values
        start_date = dates_mask[0]
        stop_date = dates_mask[-1]

        dates_mask_delta = (dates_mask - np.datetime64(start_date)).astype('timedelta64[s]').astype(int) / (60 * 60 * 24)
        stop_day = dates_mask_delta[-1]
        dates_not_mask_delta = dates_mask_delta.copy()
        mask = mask.values
        dates_mask_delta[np.where(~mask)[0]] = np.nan
        dates_not_mask_delta[np.where(mask)[0]] = np.nan
        
        x_pos = i+1 + x_deltas[j]

        if sub == subs[-1] and mask_type == mask_types[-1]:
            label_good = 'High-Quality Recording'
            label_bad = 'Low-Quality Recording'
        else:
            label_good, label_bad = None, None
        x = [x_pos] * mask.size
        ax.plot(x[::10], dates_mask_delta[::10], color = 'k', alpha = 1, lw = lw, label = label_good)
        ax.plot(x[::10], dates_not_mask_delta[::10], color = 'white', alpha = 1, lw = lw * 1.5, label = label_bad)
        # ax.plot(x, dates_not_mask_delta, color = 'r', alpha = 1, lw = lw, label = label_bad)
        # Remplacer les tracés par des scatter pour éviter les problèmes de linewidth
        # ax.scatter([x_pos] * mask.size, dates_mask_delta, color='k', alpha=1, s=10, label=label_good, marker='_')  # Points noirs pour bonne qualité
        # ax.scatter([x_pos] * mask.size, dates_not_mask_delta, color='r', alpha=1, s=10, label=label_bad)  # Points rouges pour mauvaise qualité

        ax.text(x_pos, -1.5, mask_type, rotation = 90, fontsize = 8, ha = 'center')
    stop_days[sub] = stop_day

handles, labels = ax.get_legend_handles_labels()
del handles[1]
del labels[1]
ax.legend(handles, labels, ncols = 2, loc = 'upper center', framealpha = 1)

# days_max = int(max(stop_days.values()))
days_max = 14
for h in np.arange(0,days_max+2):
    ax.axhline(h, color='k', alpha = 0.2)
    
ax.set_xlim(0, len(subs) + 1)
ax.set_ylim(-2, days_max+2)
ax.set_ylabel('Days after start of recording')
ax.set_yticks(np.arange(0,days_max+2))

labels = subs
ax.set_xticks(np.arange(1,len(subs)+1), labels = [s[2:] for s in labels], rotation = 90)
ax.set_xlabel('Patient Number')


fig.savefig(base_folder / 'figures' / 'autres' / 'artifacting_overview_eeg_ecog_v5.png', dpi = 500, bbox_inches = 'tight')

plt.show()

stats_artifacting_df = pd.DataFrame(stats_rows, columns = ['Patient','Channel Type','Percentage of Good Quality'])

## Supplementary post reviewing 1

### class view

In [ ]:
class CSD_view:
    name = 'CSD view'

    def __init__(self, cns_reader, chan_type = 'ECoG', with_detections = True, mode = 'mono', dc_band = (0.001, 0.01), ac_band = (0.5, 20), down_sampling_factor = 5, reref_median_ac = True, notch = True, offset_split_factor_uV = 100, dc_gain_factor = 0.1, ac_gain_factor = 1):
        self.stream = cns_reader.streams['EEG']
        self.detections = pd.DataFrame(cns_reader.events)
        self.with_detections = with_detections
        self.down_sampling_factor = down_sampling_factor
        self.mode = mode
        self.dc_band = dc_band
        self.ac_band = ac_band
        self.reref_median_ac = reref_median_ac
        self.chan_type = chan_type
        self.notch = notch
        self.offset_split_factor_uV = offset_split_factor_uV
        self.dc_gain_factor = dc_gain_factor
        self.ac_gain_factor = ac_gain_factor

    def plot(self, ax, t0, t1):
        chans = self.stream.channel_names
        srate = self.stream.sample_rate
        if self.chan_type == 'ECoG':
            chan_names = [chan for chan in chans if 'ECoG' in chan]
        else:
            chan_names = [chan for chan in chans if not 'ECoG' in chan]
        sigs, datetimes = self.stream.get_data(sel=slice(t0, t1), with_times=True, apply_gain=True)
        
        da_init = xr.DataArray(data = sigs.T, dims = ['chan','datetime'], coords = {'chan':chans, 'datetime':datetimes.astype('datetime64[ns]')})
        # da_init[:] = notch_filter(da_init.values, srate=srate, axis = 1)
        new_datetimes = datetimes[::self.down_sampling_factor]
        new_srate = srate / self.down_sampling_factor

        if self.mode == 'bipol':
            bipol_chan_names = [f'{chan_names[i]}-{chan_names[i-1]}' for i in np.arange(len(chan_names)-1,0,-1)]
            coords = {'chan':bipol_chan_names, 'datetime':da_init['datetime'].values}
            da = xr.DataArray(data = np.nan , dims = coords.keys(), coords = coords)
            for bipol_chan in bipol_chan_names:
                chan1, chan2 = bipol_chan.split('-')
                da.loc[bipol_chan,: ] = da_init.loc[chan1] - da_init.loc[chan2]
            
        elif self.mode == 'mono':
            da = da_init.loc[chan_names]

        if self.notch:
            da[:] = notch_filter(da.values, srate, axis = 1)
            
        da_dcs = da.copy()
        da_dcs[:] = iirfilt(da.values, srate, lowcut = self.dc_band[0], highcut = self.dc_band[1], ftype = 'bessel', axis = 1)
        da_acs = da.copy()
        da_acs[:] = iirfilt(da.values, srate, lowcut = self.ac_band[0], highcut = self.ac_band[1], ftype = 'butter', axis = 1)
        if self.reref_median_ac:
            median_ac = da_acs.median('chan')
            da_acs = da_acs - median_ac

        da_dcs = da_dcs[:,::self.down_sampling_factor]
        da_acs = da_acs[:,::self.down_sampling_factor]

        offsets_dc = da_dcs.median('datetime')
        offsets_ac = da_acs.median('datetime')

        offsets_plot = np.arange(offsets_dc.size) * self.offset_split_factor_uV
        offsets_plot = xr.DataArray(data = offsets_plot, dims = ['chan'], coords = {'chan':da_dcs['chan'].values})

        dcs_centered = (da_dcs - offsets_dc)
        acs_centered = (da_acs - offsets_ac)

        dcs_centered_rescaled = dcs_centered * self.dc_gain_factor
        das_centered_rescaled = acs_centered * self.ac_gain_factor

        dcs_plot = dcs_centered_rescaled + offsets_plot
        acs_plot = das_centered_rescaled + offsets_plot    
        
        new_datetimes = np.arange(0, new_datetimes.size / new_srate, 1 / new_srate) / 60
        ax.plot(new_datetimes, acs_plot.T, color = 'k', lw = 0.5, alpha = 0.9)
        ax.plot(new_datetimes, dcs_plot.T, color = 'r', lw = 1)
        
        vmin = np.min(offsets_plot.values) - self.offset_split_factor_uV
        vmax = np.max(offsets_plot.values) + self.offset_split_factor_uV

        ax.set_ylim(vmin, vmax)

        ax.set_yticks(offsets_plot.values, labels=[ch.replace("A", "") for ch in da_dcs['chan'].values])
        ax.set_ylabel(f'{self.chan_type}\n[µV]')
        
        if self.with_detections:
            detections = self.detections
            detections = detections[detections['name'].apply(lambda x:'SD' in x)]
            detections = detections.sort_values(by = 'start_time')
            local_detections = detections[(detections['start_time'].values > np.datetime64(t0)) & (detections['start_time'].values < np.datetime64(t1))]
            if local_detections.shape[0] > 0:
                for i, row in local_detections.iterrows():
                    start = row['start_time']
                    duration = float(row['duration'])
                    stop = start + pd.Timedelta(duration, 's')
                ax.axvspan(start, stop, color = 'k', alpha = 0.1)

In [ ]:
# figsize = (6,3)

# offset_split_factor_uV = 1000
# dc_gain_factor = 0.25
# ac_gain_factor = 2

# view = CSD_view(reader, with_detections=False, mode = mode, dc_band = (0.001,0.01), ac_band = ac_band, reref_median_ac = reref_median_ac, notch=notch, offset_split_factor_uV=offset_split_factor_uV, dc_gain_factor=dc_gain_factor, ac_gain_factor=ac_gain_factor)

# local_evs = evs[(evs['start_time'] > d0_load) & (evs['start_time'] < d1_load)].reset_index(drop = True)
# fig, ax = plt.subplots(figsize = figsize)
# view.plot(ax, d0_load, d1_load)
# for i, row in local_evs.iterrows():
#     d0_ev = row['start_time']
#     d0_ev = ((d0_ev - d0_load).total_seconds()) / 60
#     d1_ev = d0_ev + row['duration'] / 60
#     ax.axvspan(d0_ev,d1_ev,  color = 'k', alpha = 0.2, label = 'SD' if i == 0 else None)
# ax.set_ylabel('')
# ax.set_xlabel('Time (min)')
# ax.legend(loc = 'upper right', framealpha = 1)
# ax.set_xlim(ax.get_xticks()[1], ax.get_xticks()[-2])
# ax.set_title('Isolated SD in patient n°12 (Strip)')

In [ ]:
subs = get_patient_list(['ECoG','Scalp'], patient_type = 'SD_ICU', threshold_duration_mins = 0, verbose = True)

In [ ]:
overview_df = pd.read_excel(base_folder / 'overview_data_pycns.xlsx').set_index('Patient')
subs_sorted = overview_df.loc[subs].sort_values(by = 'Start_Date').index.tolist()
mapper_sub_num = {s:int(s[2:]) for s in subs_sorted}
subs_sorted = pd.Series(mapper_sub_num).sort_values().index.tolist()
subs_sorted

### loop modes

In [ ]:
p_mono = dict(
offset_split_factor_uV = 1000,
dc_gain_factor = 0.25,
ac_gain_factor = 2,
dc_band = (0.001,0.025),
nup_band = (0.0001,0.025),
nup_dc_gain_factor = 0.1,
ac_band = (0.5,70),
notch = True,
reref_median_ac = True,
mode = 'mono',
figsize = (6,3),
x0_letter = -0.1,
)

params_mono = {
    'B)':{'sub':'MF12', 'd0':'2022-09-09T21:15', 'd1':'2022-09-09T23:45', 'title':'Isolated SD in patient n°12 (Strip)'} | p_mono,
    'C)':{'sub':'MF12', 'd0':'2022-09-11T13:30', 'd1':'2022-09-11T18:30', 'title':'Cluster of SD in patient n°12 (Strip)'} | p_mono,
    'D)':{'sub':'MF12', 'd0':'2022-09-14T15:15', 'd1':'2022-09-14T23:15', 'title':'NUP in patient n°12 (Strip)'} | p_mono,
    'E)':{'sub':'FC13', 'd0':'2022-02-13T06:00', 'd1':'2022-02-13T08:00', 'title':'Cluster of SD in patient n°13 (Depth)'} | p_mono,
    'F)':{'sub':'JR10', 'd0':'2021-11-16T18:55', 'd1':'2021-11-16T19:30', 'title':'Isolated SD in patient n°10 (Depth)'} | p_mono,
}

p_bipol = dict(
offset_split_factor_uV = 1000,
dc_gain_factor = 0.1,
ac_gain_factor = 2,
dc_band = (0.001,0.025),
nup_band = (0.0001,0.025),
nup_dc_gain_factor = 0.05,
ac_band = (0.5,70),
notch = True,
reref_median_ac = False,
mode = 'bipol',
figsize = (6,3),
x0_letter = -0.25,
)

params_bipol = {
    'B)':{'sub':'MF12', 'd0':'2022-09-09T21:15', 'd1':'2022-09-09T23:45', 'title':'Isolated SD in patient n°12 (Strip)'} | p_bipol,
    'C)':{'sub':'MF12', 'd0':'2022-09-11T13:30', 'd1':'2022-09-11T18:30', 'title':'Cluster of SD in patient n°12 (Strip)'} | p_bipol,
    'D)':{'sub':'MF12', 'd0':'2022-09-14T15:15', 'd1':'2022-09-14T23:15', 'title':'NUP in patient n°12 (Strip)'} | p_bipol,
    'E)':{'sub':'FC13', 'd0':'2022-02-13T06:00', 'd1':'2022-02-13T08:00', 'title':'Cluster of SD in patient n°13 (Depth)'} | p_bipol,
    'F)':{'sub':'JR10', 'd0':'2021-11-16T18:55', 'd1':'2021-11-16T19:30', 'title':'Isolated SD in patient n°10 (Depth)'} | p_bipol,
}

In [ ]:
for mode, params in zip(['bipol','mono'],[params_bipol, params_mono]): 
    for letter, p in  params.items():
        sub = p['sub']
        reader = pycns.CnsReader(data_path / sub, with_events = True, event_time_zone='Europe/Paris')
        evs = pd.DataFrame(reader.events)
        search = 'NUP' if 'NUP' in p['title'] else 'SD'
        evs = filter_events(evs, search,  mode = 'with')
        
        label_vspan = 'NUP' if 'NUP' in p['title'] else 'SD'
        dc_band = p['nup_band'] if 'NUP' in p['title'] else p['dc_band']
        dc_gain_factor = p['nup_dc_gain_factor'] if 'NUP' in p['title'] else p['dc_gain_factor']
        
        view = CSD_view(reader, with_detections=False, mode = p['mode'], dc_band = dc_band, ac_band = p['ac_band'], reref_median_ac = p['reref_median_ac'], notch=p['notch'], offset_split_factor_uV=p['offset_split_factor_uV'], dc_gain_factor=dc_gain_factor, ac_gain_factor=p['ac_gain_factor'],)

        local_evs = evs[(evs['start_time'] > p['d0']) & (evs['start_time'] < p['d1'])].reset_index(drop = True)

        d0_load = np.datetime64(p['d0'])
        d1_load = np.datetime64(p['d1'])
        duration_mins = (d1_load - d0_load).astype(float)

        fig, ax = plt.subplots(figsize = figsize)
        view.plot(ax, d0_load , d1_load)
        for i, row in local_evs.iterrows():
            d0_ev = row['start_time']
            d0_ev = ((d0_ev - d0_load).total_seconds()) / 60
            d1_ev = d0_ev + row['duration'] / 60
            if d1_ev > duration_mins:
                d1_ev = duration_mins
            ax.axvspan(d0_ev,d1_ev,  color = 'k', alpha = 0.2, label = label_vspan if i == 0 else None)
        ax.set_ylabel('')
        ax.set_xlabel('Time (min)')
        ax.legend(loc = 'upper right', framealpha = 1)
        # ax.set_xlim(ax.get_xticks()[1], ax.get_xticks()[-2])
        ax.set_xlim(0, duration_mins)
        ax.set_title(p['title'])
        ax.text(p['x0_letter'],1.02, letter, transform=ax.transAxes, fontsize = 15, weight = 'bold')
        fig.savefig(base_folder / 'figures' / 'autres' / 'figs_example_sd_cluster_isolated' / f'{letter[0]}_{mode}.png', dpi = 300, bbox_inches = 'tight')
        plt.show()